# Tribal Preference Study V15 — Primary-4 Submission-Release Pipeline

**Submission-release copy, June 2026.** This notebook makes the conservative **Primary-4 competitive-family analysis** the official main pipeline.

Core policy:

- **Falcon remains preserved in the raw responses, raw judgments, and full 5-family appendix sensitivity.**
- **Falcon is excluded as a judge.** This was already implemented in the prior notebook.
- **Falcon is also excluded from the main candidate pool** because it creates a quality-saturation/anchor-distortion problem and has documented pipeline-level generation artifacts.
- The main paper should report the Primary-4 estimate. The 5-family estimate is retained only as an appendix/sensitivity result.

Primary analysis families:

```text
Judges:     llama, qwen, gemma, yi
Candidates: llama, qwen, gemma, yi
```

Appendix sensitivity:

```text
Judges:     llama, qwen, gemma, yi
Candidates: llama, qwen, falcon, gemma, yi
```

The major changes from the previous notebook are:

1. Separate `*_primary` and `*_full` artifacts.
2. Forced rerun of Phase 6 to Phase 9 on the Primary-4 dataset.
3. Falcon diagnostics retained as formal appendix evidence.
4. Primary-4 BT, bootstrap, permutation, GEE, nested-logit, multiverse, and figures.
5. Human annotation export fixed by joining `response_a` and `response_b` from `trial_master` when absent from `master_judgments`.
6. Additional diagnostics: candidate-subset audit, BT/separation diagnostics, judge-panel validation, position-bias audit, placebo family-label shuffle, leave-one-judge-out, leave-one-candidate-out, and consistent-wins-only robustness.


# V15 submission-clean notebook notes

This copy preserves the canonical analyses and removes cells that could create reviewer-facing contradictions unless they are regenerated as documented appendix artifacts.

Canonical paper values are listed in `tribal_pref_v15_FINAL_NUMBERS.md`.

Important release decisions:

1. The notebook uses a seeded **67/33 exploratory/confirmatory split**. The paper and artifact documentation should use only this split wording.
2. The canonical human-calibration result is Cells 10.5–10.7: two retained annotators, 800 final human judgments, 266 consensus rows, and 57.1% LLM-panel exact match on consensus rows.
3. The secondary human-baseline diagnostic is not the paper's main human-calibration table unless it is regenerated and explicitly documented as appendix-only.
4. Exploratory float-precision checks are not part of the matched headline GGUF Primary-4 design and should not be used as paper support.
5. The style-similarity method should be confirmed from `style_similarity_summary.json` before final submission.


# Release-clean artifact patch log

This notebook is the paper-facing V15 release-clean copy. The canonical executed analyses and outputs are preserved, but non-canonical reviewer-risk material has been removed or clarified.

Changes applied in this release copy:

1. Removed a stale unexecuted reviewer-response scratch cell that contained partial output and AI-workflow residue.
2. Removed the disabled exploratory float-precision scratch check because it is not part of the headline GGUF Primary-4 design.
3. Removed an unexecuted off-diagonal BT scratch cell from the paper-facing notebook; keep that analysis only if regenerated as a documented appendix artifact.
4. Standardized the paper-facing split language to the seeded 67/33 exploratory/confirmatory split.
5. Clarified that the human-calibration paper result uses two retained annotators and 800 final judgments.
6. Replaced the stale human-agreement checklist threshold with an honest-reporting requirement.

The Drive storage directory may still be named `tribal_pref_v11` because it is the stable project directory used by the cached run artifacts; this notebook release version is V15.

Final GPU cleanup applied:

7. Removed the slow source-build `llama-cpp-python` setup from the early cells.
8. Replaced it with a GPU-only CUDA-wheel setup that fails fast instead of compiling a CPU/source build.
9. Updated release documentation so model-generation instructions are GPU-first.


## PHASE 1 — Bootstrap, Configuration, Engineering Primitives

Everything downstream depends on this phase. Paths, registries, seeds, logging,
atomic writes, run_step, hashing, validators.



In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu124

In [ ]:
# ============================================================================
# Cell 1.0B — GPU llama-cpp-python setup (fast CUDA wheel, no CPU source build)
# ============================================================================
"""
Run this only in a GPU-backed Colab/runtime.

This replaces the previous slow source-build setup. It intentionally avoids cleanup loops, forced reinstalls, cache-bypass installs, and CPU-only compilation.

If the CUDA wheel is unavailable, this cell fails fast instead of compiling a
slow source build. For cached analysis-only phases, you may skip this cell.
"""

import sys
import subprocess

print("=" * 100)
print("GPU setup: installing CUDA-enabled llama-cpp-python wheel")
print("=" * 100)

# Require a GPU runtime for model-generation / judging phases.
gpu_check = subprocess.run(
    ["nvidia-smi"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    check=False,
)
if gpu_check.returncode != 0:
    raise RuntimeError(
        "No GPU detected by nvidia-smi. Switch Colab Runtime → Change runtime type → GPU, "
        "then rerun this setup cell. The previous source-build setup has been removed."
    )

# Install project dependencies. These are normal wheels and should not trigger
# llama.cpp CPU compilation.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "statsmodels",
        "sentence-transformers",
        "datasets",
        "huggingface_hub",
        "tqdm",
        "matplotlib",
        "seaborn",
        "krippendorff",
        "irrCAC",
        "pingouin",
        "jupytext",
        "diskcache",
        "jinja2",
        "typing-extensions",
    ],
    check=True,
)

# CUDA 12.4 llama-cpp-python wheel. --only-binary prevents accidental source
# builds; if no compatible wheel exists, the cell stops instead of taking hours.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "--prefer-binary",
        "--only-binary=:all:",
        "llama-cpp-python",
        "--extra-index-url",
        "https://abetlen.github.io/llama-cpp-python/whl/cu124",
    ],
    check=True,
)

print("=" * 100)
print("GPU llama-cpp-python setup complete. Restart runtime once, then continue from Cell 1.1.")
print("=" * 100)


In [ ]:
# ============================================================================
# Cell 1.1 — Imports, Drive mount, environment checks
# ============================================================================
"""
Cell 1.1 — Imports & environment.

GPU-first version.
- No CPU/source-build install.
- Fixes NumPy/Pandas binary incompatibility if detected.
- If it repairs packages, it restarts the runtime once. After restart, rerun this cell.
"""

import os
import re
import gc
import io
import sys
import json
import math
import time
import shutil
import hashlib
import logging
import tempfile
import warnings
import platform
import subprocess
import contextlib
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations, product
from collections import defaultdict, Counter
from typing import Any, Callable, Dict, List, Optional, Tuple, Iterable, Sequence


# ---------------------------------------------------------------------------
# 1. NumPy/Pandas ABI guard
# ---------------------------------------------------------------------------

def _pip_install_compatible_numpy_pandas() -> None:
    """
    Repair common Colab ABI issue:
    ValueError: numpy.dtype size changed, may indicate binary incompatibility.
    """
    print("⚠ NumPy/Pandas binary incompatibility detected.")
    print("→ Reinstalling compatible NumPy/Pandas wheels...")

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "numpy==1.26.4",
        "pandas==2.2.2",
        "scipy==1.13.1",
        "scikit-learn==1.5.2",
        "statsmodels==0.14.2",
    ]
    subprocess.check_call(cmd)

    print("✓ Compatible scientific stack installed.")
    print("⚠ Runtime must restart once so Python unloads old binary modules.")

    # Colab-safe hard restart
    os.kill(os.getpid(), 9)


try:
    import numpy as np
    import pandas as pd
except Exception as e:
    msg = str(e)
    if (
        "numpy.dtype size changed" in msg
        or "binary incompatibility" in msg
        or "numpy.core.multiarray failed to import" in msg
        or "numpy.random.mtrand" in msg
    ):
        _pip_install_compatible_numpy_pandas()
    else:
        raise


warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


# ---------------------------------------------------------------------------
# 2. Environment detection
# ---------------------------------------------------------------------------

IS_COLAB = "google.colab" in sys.modules

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"In Colab: {IS_COLAB}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")


# ---------------------------------------------------------------------------
# 3. Drive mount
# ---------------------------------------------------------------------------

if IS_COLAB:
    from google.colab import drive  # type: ignore

    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

    print("✓ Drive mounted")


# ---------------------------------------------------------------------------
# 4. GPU check
# ---------------------------------------------------------------------------

def _check_gpu_or_warn() -> None:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            check=False,
        )

        if out.returncode == 0 and out.stdout.strip():
            print(f"GPU: {out.stdout.strip()}")
        else:
            print("⚠ nvidia-smi did not return a GPU.")
            print("⚠ Set Runtime → Change runtime type → GPU before running generation cells.")

    except FileNotFoundError:
        print("⚠ nvidia-smi not found.")
        print("⚠ Set Runtime → Change runtime type → GPU before running generation cells.")


_check_gpu_or_warn()


# ---------------------------------------------------------------------------
# 5. Determinism
# ---------------------------------------------------------------------------

RANDOM_SEED = 20260502
np.random.seed(RANDOM_SEED)

print(f"Seed: {RANDOM_SEED}")
print("✓ Cell 1.1 complete")

## Superseded duplicate configuration cell

The previous notebook contained two `Cell 1.2` configuration cells. This duplicate is intentionally disabled so that only the updated configuration cell below controls file paths, family subsets, and analysis policy.


In [ ]:
# ============================================================================
# Cell 1.2 — Centralised configuration (paths, models, hyperparameters)
# ============================================================================
"""
Cell 1.2 — All configuration in one place.

Stable storage root: /content/drive/MyDrive/tribal_pref_v11. The directory
name is retained for reproducibility/cache compatibility; this notebook release
version is V15. Local staging remains on /content for hot model files.

IMPORTANT:
- Original variable names are preserved.
- Mixtral/Mistral has been replaced with Falcon.
- Quant ablation is restricted to LLaMA Q3 and Qwen Q3.
- Falcon uses chat_format="alpaca" because llama-cpp-python has no native
  "falcon" handler and Falcon-Instruct's prompt template (User:/Assistant:)
  is identical to the alpaca format.
"""

# ---- Paths ----------------------------------------------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive/tribal_pref_v11") if IS_COLAB \
             else Path.home() / "tribal_pref_v11"  # stable storage directory
LOCAL_STAGE = Path("/content/local_stage_v11") if IS_COLAB \
              else Path.home() / "tribal_pref_v11_local_stage"

PATHS: Dict[str, Path] = {
    "root":                DRIVE_ROOT,
    "data":                DRIVE_ROOT / "data",
    "datasets_raw":        DRIVE_ROOT / "datasets_raw",

    # Model folders
    "models_drive":        DRIVE_ROOT / "models",
    "models_large":        DRIVE_ROOT / "models" / "large",
    "models_small":        DRIVE_ROOT / "models" / "small",
    "models_quant":        DRIVE_ROOT / "models" / "quant",

    # Local model staging folders
    "models_local":        LOCAL_STAGE / "models",
    "models_local_large":  LOCAL_STAGE / "models" / "large",
    "models_local_small":  LOCAL_STAGE / "models" / "small",
    "models_local_quant":  LOCAL_STAGE / "models" / "quant",

    "responses_large":     DRIVE_ROOT / "responses" / "large",
    "responses_small":     DRIVE_ROOT / "responses" / "small",
    "responses_quant":     DRIVE_ROOT / "responses" / "quant_ablation",
    "responses_combined":  DRIVE_ROOT / "responses" / "combined",

    "trials":              DRIVE_ROOT / "trials",
    "judgments_large":     DRIVE_ROOT / "judgments" / "large_rubric",
    "judgments_large_neu": DRIVE_ROOT / "judgments" / "large_neutral",
    "judgments_small":     DRIVE_ROOT / "judgments" / "small_rubric",
    "judgments_master":    DRIVE_ROOT / "judgments" / "master",

    "analysis_pref":       DRIVE_ROOT / "analysis" / "preference",
    "analysis_stats":      DRIVE_ROOT / "analysis" / "stats",
    "analysis_bt":         DRIVE_ROOT / "analysis" / "bradley_terry",
    "analysis_decomp":     DRIVE_ROOT / "analysis" / "mechanism_decomp",
    "analysis_robust":     DRIVE_ROOT / "analysis" / "robustness",
    "analysis_human":      DRIVE_ROOT / "analysis" / "human_calibration",
    "analysis_leakage":    DRIVE_ROOT / "analysis" / "preference_leakage",
    "analysis_diagnostics":DRIVE_ROOT / "analysis" / "diagnostics",

    "embeddings":          DRIVE_ROOT / "embeddings",
    "cache":               DRIVE_ROOT / "cache",
    "logs":                DRIVE_ROOT / "logs",
    "figures":             DRIVE_ROOT / "figures",
    "tables":              DRIVE_ROOT / "tables",
    "release":             DRIVE_ROOT / "release",
    "config_snapshots":    DRIVE_ROOT / "config_snapshots",
    "local_stage":         LOCAL_STAGE,
}

for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

# ---- Canonical filenames ---------------------------------------------------
FILES: Dict[str, Path] = {
    # data
    "master_prompts":       PATHS["data"] / "master_prompts.csv",
    "prompt_split":         PATHS["data"] / "prompt_split.csv",
    "calibration_subset":   PATHS["data"] / "calibration_subset.csv",
    "alpaca_full":          PATHS["datasets_raw"] / "alpaca_eval_full.csv",
    "mt_bench_jsonl":       PATHS["datasets_raw"] / "mt_bench.jsonl",
    "wildbench_jsonl":      PATHS["datasets_raw"] / "wildbench_subset.jsonl",

    # responses
    "all_responses":        PATHS["responses_combined"] / "all_responses.csv",
    "response_health":      PATHS["analysis_diagnostics"] / "response_health.csv",

    # trials
    "trial_master":         PATHS["trials"] / "trial_master.csv",
    "trial_audit":          PATHS["trials"] / "trial_audit.json",

    # judgments
    "master_judgments":     PATHS["judgments_master"] / "master_judgments.csv",
    "master_judgments_neu": PATHS["judgments_master"] / "master_judgments_neutral.csv",
    "master_judgments_sm":  PATHS["judgments_master"] / "master_judgments_small.csv",

    # caches
    "logprob_cache":        PATHS["cache"] / "logprob_cache.json",
    "embedding_cache":      PATHS["cache"] / "embedding_cache.npz",

    # core analysis
    "effective_winners":    PATHS["analysis_pref"] / "effective_winners_primary.csv",
    "effective_winners_primary": PATHS["analysis_pref"] / "effective_winners_primary.csv",
    "effective_winners_full":    PATHS["analysis_pref"] / "effective_winners_full.csv",
    "preference_matrix":    PATHS["analysis_pref"] / "preference_matrix_primary.csv",
    "preference_matrix_primary": PATHS["analysis_pref"] / "preference_matrix_primary.csv",
    "preference_matrix_full":    PATHS["analysis_pref"] / "preference_matrix_full.csv",
    "preference_support":   PATHS["analysis_pref"] / "preference_support_primary.csv",
    "preference_support_primary": PATHS["analysis_pref"] / "preference_support_primary.csv",
    "preference_support_full":    PATHS["analysis_pref"] / "preference_support_full.csv",
    "tps_summary":          PATHS["analysis_pref"] / "tps_summary_primary.json",
    "tps_summary_primary":  PATHS["analysis_pref"] / "tps_summary_primary.json",
    "tps_summary_full":     PATHS["analysis_pref"] / "tps_summary_full.json",
    "candidate_subset_audit": PATHS["analysis_pref"] / "candidate_subset_audit.json",

    # inference
    "cluster_bootstrap":    PATHS["analysis_stats"] / "cluster_bootstrap_tps_primary.json",
    "permutation_result":   PATHS["analysis_stats"] / "permutation_result_primary.json",
    "per_family_bh":        PATHS["analysis_stats"] / "per_family_tps_bh_primary.json",
    "bt_results":           PATHS["analysis_bt"] / "bt_results_primary.json",
    "bt_results_primary":   PATHS["analysis_bt"] / "bt_results_primary.json",
    "bt_results_full":      PATHS["analysis_bt"] / "bt_results_full.json",
    "bt_diagnostics":       PATHS["analysis_bt"] / "bt_diagnostics_primary_vs_full.json",
    "bt_residual_matrix":   PATHS["analysis_bt"] / "bt_residual_matrix_primary.csv",
    "bt_residual_matrix_primary": PATHS["analysis_bt"] / "bt_residual_matrix_primary.csv",
    "bt_residual_matrix_full":    PATHS["analysis_bt"] / "bt_residual_matrix_full.csv",

    # mechanism
    "decomp_table":         PATHS["analysis_decomp"] / "nested_logit_decomposition_primary.csv",
    "regression_features":  PATHS["analysis_decomp"] / "regression_features_primary.csv",
    "gee_full":             PATHS["analysis_decomp"] / "gee_full_summary_primary.txt",
    "mixed_glmm":           PATHS["analysis_decomp"] / "mixed_effects_glmm_primary.json",
    "separation_diagnostics": PATHS["analysis_decomp"] / "separation_diagnostics_primary.json",
    "style_sim_matrix":     PATHS["embeddings"] / "style_similarity_matrix.csv",

    # robustness
    "multiverse_results":   PATHS["analysis_robust"] / "multiverse_grid_with_candidate_subset.csv",
    "neutral_compare":      PATHS["analysis_robust"] / "rubric_vs_neutral_primary.json",
    "judge_panel_validation": PATHS["analysis_robust"] / "judge_panel_validation.json",
    "position_bias_audit": PATHS["analysis_robust"] / "position_bias_audit.json",
    "placebo_family_shuffle": PATHS["analysis_robust"] / "placebo_family_shuffle.json",
    "consistent_wins_only": PATHS["analysis_robust"] / "consistent_wins_only_primary.json",
    "leave_one_judge_tps": PATHS["analysis_robust"] / "leave_one_judge_tps_primary.csv",
    "leave_one_candidate_tps": PATHS["analysis_robust"] / "leave_one_candidate_tps_primary.csv",
    "contamination_split":  PATHS["analysis_robust"] / "contamination_free_vs_classic_primary.json",
    "confirmatory_result":  PATHS["analysis_robust"] / "confirmatory_holdout_primary.json",
    "quant_ablation":       PATHS["analysis_robust"] / "quant_ablation.json",
    "scale_compare":        PATHS["analysis_robust"] / "scale_comparison_primary.json",

    # human calibration
    "human_export":         PATHS["analysis_human"] / "human_annotation_export_primary.csv",
    "human_filled":         PATHS["analysis_human"] / "human_annotations_filled.csv",
    "human_metrics":        PATHS["analysis_human"] / "agreement_metrics.json",
    "human_gold_tps":       PATHS["analysis_human"] / "human_gold_tps.json",

    # leakage
    "leakage_audit":        PATHS["analysis_leakage"] / "leakage_audit.json",

    # release
    "hf_dataset_dir":       PATHS["release"] / "hf_dataset",
    "model_card":           PATHS["release"] / "MODEL_CARD.md",
    "data_statement":       PATHS["release"] / "DATA_STATEMENT.md",
    "reproduce_readme":     PATHS["release"] / "REPRODUCE.md",
    "final_audit":          PATHS["release"] / "final_audit.json",
}

# ---- Experimental constants -----------------------------------------------
FAMILIES: List[str] = ["llama", "qwen", "falcon", "gemma", "yi"]

# Main analysis policy -------------------------------------------------------
# Full raw material is retained, but Falcon is not used in the main inference.
# Falcon was already excluded as judge; the Primary-4 update also excludes it
# as a main candidate and retains it only for appendix/sensitivity diagnostics.
EXCLUDED_JUDGE_FAMILIES: List[str] = ["falcon"]
EXCLUDED_PRIMARY_CANDIDATE_FAMILIES: List[str] = ["falcon"]
PRIMARY_JUDGE_FAMILIES: List[str] = [f for f in FAMILIES if f not in EXCLUDED_JUDGE_FAMILIES]
ACTIVE_JUDGE_FAMILIES: List[str] = list(PRIMARY_JUDGE_FAMILIES)
PRIMARY_CANDIDATE_FAMILIES: List[str] = [f for f in FAMILIES if f not in EXCLUDED_PRIMARY_CANDIDATE_FAMILIES]
FULL_CANDIDATE_FAMILIES: List[str] = list(FAMILIES)
CANDIDATE_FAMILIES: List[str] = list(PRIMARY_CANDIDATE_FAMILIES)
PRIMARY_ANALYSIS_LABEL = "primary_4"
FULL_SENSITIVITY_LABEL = "all_5"

# Categories used for moderation analyses
OBJECTIVE_CATEGORIES = {"math", "coding", "extraction", "reasoning_objective"}
CREATIVE_CATEGORIES  = {"writing", "roleplay", "brainstorm", "creative"}

# Contamination-free framing: which sources count as "fresh"
FRESH_SOURCES = {"wildbench"}
CLASSIC_SOURCES = {"mt_bench", "alpaca_eval"}

# Pre-registration: confirmatory holdout fraction
CONFIRMATORY_FRAC = 0.33
PRESPLIT_SEED = RANDOM_SEED + 1

# Generation hyperparameters
GEN_MAX_TOKENS   = 768
GEN_TEMPERATURE  = 0.7
GEN_TOP_P        = 0.95
GEN_REPEAT_PEN   = 1.05

# Judge hyperparameters
JUDGE_MAX_TOKENS  = 512
JUDGE_TEMPERATURE = 0.0
JUDGE_TOP_P       = 1.0
JUDGE_CONTEXT     = 8192

# Logprob proxy
LOGPROB_MAX_CHARS = 400
LOGPROB_SAVE_EVERY = 50

# Bootstrap / permutation
N_BOOTSTRAP = 2000
N_PERMUTATIONS = 5000
BOOTSTRAP_CLUSTER = "prompt_id"
ALPHA = 0.05

# Multiverse grid axes
MULTIVERSE_AXES = {
    "tie_weight":        [0.0, 0.5, 1.0],
    "drop_inconsistent": [True, False],
    "judge_subset":      ["all_primary", "leave_one_out"],
    "candidate_subset":  ["primary_4", "all_5"],
    "prompt_subset":     ["all", "fresh_only", "classic_only"],
    "rubric":            ["rubric", "neutral"],
}

# Model registry: large + small + quant-ablation
# `candidates` lists possible filenames; first match wins.
MODEL_REGISTRY: Dict[str, Dict[str, Dict[str, Any]]] = {
    "large": {
        "llama": {
            "display": "Meta-Llama-3.1-70B-Instruct",
            "candidates": [
                "Meta-Llama-3.1-70B-Instruct-Q4_K_M.gguf",
                "Meta-Llama-3.1-70B-Instruct-Q3_K_M.gguf",
            ],
            "chat_format": "llama-3",
        },
        "qwen": {
            "display": "Qwen2.5-72B-Instruct",
            "candidates": [
                "Qwen2.5-72B-Instruct-Q4_K_M.gguf",
                "Qwen2.5-72B-Instruct-Q3_K_M.gguf",
            ],
            "chat_format": "chatml",
        },
        "falcon": {
            "display": "Falcon-40B-Instruct",
            "candidates": [
                "falcon-40b-instruct.i1-Q4_K_M.gguf",        # mradermacher imatrix quant
                "falcon-40b-instruct.Q4_K_M.gguf",
                "Falcon-40B-Instruct-Q4_K_M.gguf",
                "falcon-40b-instruct-q4_k_m.gguf",
            ],
            "chat_format": "alpaca",     # was "falcon" — llama-cpp has no falcon handler
        },
        "gemma": {
            "display": "Gemma-2-27B-it",
            "candidates": [
                "gemma-2-27b-it-Q4_K_M.gguf",
                "Gemma-2-27B-it-Q4_K_M.gguf",
            ],
            "chat_format": "gemma",
        },
        "yi": {
            "display": "Yi-1.5-34B-Chat",
            "candidates": [
                "Yi-1.5-34B-Chat-Q4_K_M.gguf",
            ],
            "chat_format": "chatml",
        },
    },

    "small": {
        "llama": {
            "display": "Meta-Llama-3.1-8B-Instruct",
            "candidates": [
                "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
            ],
            "chat_format": "llama-3",
        },
        "qwen": {
            "display": "Qwen2.5-7B-Instruct",
            "candidates": [
                "Qwen2.5-7B-Instruct-Q4_K_M.gguf",
            ],
            "chat_format": "chatml",
        },
        "falcon": {
            "display": "Falcon-7B-Instruct",
            "candidates": [
                "falcon-7b-instruct.Q4_K_M.gguf",
                "Falcon-7B-Instruct-Q4_K_M.gguf",
                "falcon-7b-instruct-q4_k_m.gguf",
            ],
            "chat_format": "alpaca",     # was "falcon" — llama-cpp has no falcon handler
        },
        "gemma": {
            "display": "Gemma-2-9B-it",
            "candidates": [
                "gemma-2-9b-it-Q4_K_M.gguf",
            ],
            "chat_format": "gemma",
        },
        "yi": {
            "display": "Yi-1.5-9B-Chat",
            "candidates": [
                "Yi-1.5-9B-Chat-Q4_K_M.gguf",
            ],
            "chat_format": "chatml",
        },
    },

    # Quantization ablation:
    # Only LLaMA Q3 and Qwen Q3 are active.
    "quant_ablation": {
        "llama_q3": {
            "family": "llama",
            "display": "Meta-Llama-3.1-70B-Instruct (Q3_K_M)",
            "candidates": [
                "Meta-Llama-3.1-70B-Instruct-Q3_K_M.gguf",
            ],
            "chat_format": "llama-3",
            "quant": "Q3_K_M",
        },
        "qwen_q3": {
            "family": "qwen",
            "display": "Qwen2.5-72B-Instruct (Q3_K_M)",
            "candidates": [
                "Qwen2.5-72B-Instruct-Q3_K_M.gguf",
            ],
            "chat_format": "chatml",
            "quant": "Q3_K_M",
        },
    },
}

# Llama.cpp runtime
LLAMA_CPP_KWARGS: Dict[str, Any] = {
    "n_gpu_layers": -1,
    "n_ctx":         JUDGE_CONTEXT,
    "n_batch":       512,
    "n_threads":     8,
    "verbose":       False,
    "logits_all":    False,
    "use_mmap":      True,
    "use_mlock":     False,
    "flash_attn":    True,
}

print(f"Drive root: {PATHS['root']}")
print(f"Local stage: {PATHS['local_stage']}")
print(f"Families: {FAMILIES}")
print(f"Primary judge families: {PRIMARY_JUDGE_FAMILIES}")
print(f"Primary candidate families: {PRIMARY_CANDIDATE_FAMILIES}")
print(f"Full appendix candidate families: {FULL_CANDIDATE_FAMILIES}")
print(f"Large models: {list(MODEL_REGISTRY['large'].keys())}")
print(f"Small models: {list(MODEL_REGISTRY['small'].keys())}")
print(f"Quant ablation models: {list(MODEL_REGISTRY['quant_ablation'].keys())}")
print(f"Confirmatory holdout: {int(100*CONFIRMATORY_FRAC)}%")
print(f"Bootstrap iters: {N_BOOTSTRAP}, Permutations: {N_PERMUTATIONS}")
print("✓ Configuration loaded")

In [ ]:
# ============================================================================
# Cell 1.3 — Engineering primitives (logging, atomic IO, hashing, run_step)
# ============================================================================
"""
Cell 1.3 — The unsexy plumbing that makes the rest survivable.

Includes: structured logging, atomic JSON/CSV writes, content-hash-based cache
keys, a universal run_step() that skips work when validated outputs already
exist, and a config snapshot for reproducibility.
"""

# ---- Logging --------------------------------------------------------------
LOG_PATH = PATHS["logs"] / f"pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
LOG_LATEST = PATHS["logs"] / "pipeline_latest.log"

logger = logging.getLogger("tribal_pref_v15_release")
logger.setLevel(logging.INFO)
logger.handlers.clear()

_fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
_sh = logging.StreamHandler()
_sh.setFormatter(_fmt)
logger.addHandler(_sh)

_fh = logging.FileHandler(LOG_PATH, mode="a", encoding="utf-8")
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

try:
    if LOG_LATEST.exists() or LOG_LATEST.is_symlink():
        LOG_LATEST.unlink()
    LOG_LATEST.symlink_to(LOG_PATH)
except OSError:
    pass  # Drive doesn't always support symlinks

log = logger
log.info(f"=== Pipeline session started: {LOG_PATH.name} ===")


# ---- Atomic IO ------------------------------------------------------------
def atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    tmp.replace(path)

def atomic_write_json(obj: Any, path: Path) -> None:
    atomic_write_text(path, json.dumps(obj, indent=2, default=str))

def atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def atomic_write_npy(arr: np.ndarray, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    np.save(tmp, arr)
    tmp.replace(path)


# ---- Hashing --------------------------------------------------------------
def sha256_short(text: str, n: int = 16) -> str:
    return hashlib.sha256(("" if text is None else str(text)).encode("utf-8")
                          ).hexdigest()[:n]

def sha1_short(text: str, n: int = 8) -> str:
    return hashlib.sha1(("" if text is None else str(text)).encode("utf-8")
                        ).hexdigest()[:n]

def stable_hash(text: str) -> str:
    """24-char SHA256 prefix, used for cache keys."""
    return sha256_short(text, 24)


# ---- Validators ----------------------------------------------------------
def csv_has(path: Path, required_cols: Optional[Sequence[str]] = None,
            min_rows: int = 1) -> bool:
    """Validate a CSV: exists, parseable, has required columns and min rows."""
    try:
        if not Path(path).exists():
            return False
        df = pd.read_csv(path, nrows=5 if required_cols else min_rows)
        if required_cols:
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                log.warning(f"{path.name}: missing columns {missing}")
                return False
        # cheap row count
        n = sum(1 for _ in open(path, "r", encoding="utf-8")) - 1
        return n >= min_rows
    except Exception as e:
        log.warning(f"csv_has({path}) failed: {type(e).__name__}: {e}")
        return False

def json_has(path: Path, required_keys: Optional[Sequence[str]] = None) -> bool:
    try:
        if not Path(path).exists():
            return False
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        if required_keys:
            missing = [k for k in required_keys if k not in obj]
            if missing:
                log.warning(f"{path.name}: missing keys {missing}")
                return False
        return True
    except Exception as e:
        log.warning(f"json_has({path}) failed: {type(e).__name__}: {e}")
        return False

def file_nonempty(path: Path, min_bytes: int = 1) -> bool:
    p = Path(path)
    return p.exists() and p.stat().st_size >= min_bytes


# ---- run_step: the universal cache wrapper -------------------------------
def run_step(outputs: Sequence[Path],
             func: Callable[[], Any],
             name: str,
             *,
             force: bool = False,
             validators: Optional[Dict[Path, Callable[[Path], bool]]] = None,
             require_all: bool = True) -> Any:
    """
    Run `func()` only if outputs are missing or fail validation. Otherwise skip.

    - outputs: list of paths the step produces.
    - validators: optional {path: callable(path) -> bool}; all must pass.
    - force: skip cache check, always rerun.
    - require_all: if True, ALL outputs must be valid to skip; else ANY is enough.

    Returns whatever `func()` returns (or None if skipped).
    """
    outputs = [Path(p) for p in outputs]
    validators = validators or {}

    def _is_valid(p: Path) -> bool:
        if p in validators:
            return validators[p](p)
        return file_nonempty(p)

    if not force:
        ok_flags = [_is_valid(p) for p in outputs]
        if (require_all and all(ok_flags)) or ((not require_all) and any(ok_flags)):
            log.info(f"[skip] {name}: validated outputs already present")
            return None

    log.info(f"[run]  {name}")
    t0 = time.time()
    try:
        result = func()
        dt = time.time() - t0
        log.info(f"[done] {name} in {dt:.1f}s")
        return result
    except Exception as e:
        log.exception(f"[FAIL] {name}: {e}")
        raise


# ---- Snapshot the config to Drive (for forensic reproducibility) ----------
def snapshot_config() -> Path:
    """Persist the current config so we can audit drift later."""
    snap = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "random_seed": RANDOM_SEED,
        "families": FAMILIES,
        "model_registry": MODEL_REGISTRY,
        "files": {k: str(v) for k, v in FILES.items()},
        "paths": {k: str(v) for k, v in PATHS.items()},
        "gen": {"max_tokens": GEN_MAX_TOKENS, "temperature": GEN_TEMPERATURE,
                "top_p": GEN_TOP_P, "repeat_pen": GEN_REPEAT_PEN},
        "judge": {"max_tokens": JUDGE_MAX_TOKENS, "temperature": JUDGE_TEMPERATURE,
                  "context": JUDGE_CONTEXT},
        "stats": {"n_bootstrap": N_BOOTSTRAP, "n_permutations": N_PERMUTATIONS,
                  "alpha": ALPHA, "cluster": BOOTSTRAP_CLUSTER},
        "presplit": {"frac": CONFIRMATORY_FRAC, "seed": PRESPLIT_SEED},
        "multiverse_axes": MULTIVERSE_AXES,
    }
    out = PATHS["config_snapshots"] / f"config_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    atomic_write_json(snap, out)
    latest = PATHS["config_snapshots"] / "config_latest.json"
    atomic_write_json(snap, latest)
    log.info(f"Config snapshotted → {out.name}")
    return out

snapshot_config()
print("✓ Engineering primitives ready")




## PHASE 2 — Data Pipeline

Three prompt sources:
- **MT-Bench** (80 prompts): classic, established, but training-set contaminated.
- **AlpacaEval** (70 prompts): classic, similar concern.
- **WildBench** (50 prompts): post-2024, treated as our **contamination-free**
  subset for decontamination defense.

Then a pre-registered 67/33 split into exploratory and confirmatory subsets.
Headline tests are reported on the confirmatory subset; everything else is
exploratory and tagged as such.



In [ ]:
# ============================================================================
# Cell 2.1 — Fetch raw prompt sources (MT-Bench, AlpacaEval, WildBench)
# ============================================================================
"""
Cell 2.1 — Pull raw datasets if not already cached.

Sources:
- MT-Bench: official FastChat MT-Bench question.jsonl.
- AlpacaEval: raw alpaca_eval.json from tatsu-lab/alpaca_eval.
- WildBench: allenai/WildBench configs v2, v2-hard, v1-legacy.

This cell is idempotent and safe to rerun.
"""

import urllib.request
import json
import ast
from pathlib import Path
from typing import Any, Dict, List


def _download_url(url: str, out_path: Path, min_bytes: int = 1_000) -> Path:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    tmp = out_path.with_suffix(out_path.suffix + ".tmp")
    tmp.unlink(missing_ok=True)

    log.info(f"Downloading: {url}")
    urllib.request.urlretrieve(url, tmp)

    if not tmp.exists() or tmp.stat().st_size < min_bytes:
        raise RuntimeError(
            f"Downloaded file too small or missing: {tmp} "
            f"({tmp.stat().st_size if tmp.exists() else 0} bytes)"
        )

    tmp.replace(out_path)
    return out_path


def _jsonl_count(path: Path) -> int:
    path = Path(path)
    if not path.exists():
        return 0

    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


def _jsonl_valid(path: Path, min_rows: int = 1) -> bool:
    try:
        n = 0
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    json.loads(line)
                    n += 1
        return n >= min_rows
    except Exception:
        return False


# ============================================================
# MT-BENCH
# ============================================================

def _fetch_mt_bench() -> Path:
    """
    Fetch official MT-Bench from FastChat.

    FastChat is the official source for MT-Bench evaluation data.
    """
    out = FILES["mt_bench_jsonl"]

    if file_nonempty(out, min_bytes=1024) and _jsonl_valid(out, min_rows=80):
        log.info(f"MT-Bench already cached: {out}")
        return out

    log.info("Fetching MT-Bench...")

    url = (
        "https://raw.githubusercontent.com/lm-sys/FastChat/main/"
        "fastchat/llm_judge/data/mt_bench/question.jsonl"
    )

    _download_url(url, out, min_bytes=1024)

    n = 0
    with open(out, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)

                if "turns" not in obj:
                    raise RuntimeError("MT-Bench row missing 'turns'")

                n += 1

    if n < 80:
        raise RuntimeError(f"MT-Bench too small: {n} rows")

    log.info(f"MT-Bench saved: {out} ({n} rows)")
    return out


# ============================================================
# ALPACA EVAL
# ============================================================

def _fetch_alpaca_eval() -> Path:
    """
    Fetch AlpacaEval directly from raw HF JSON.

    This avoids datasets.load_dataset because the old AlpacaEval dataset script
    is no longer supported by recent datasets versions.
    """
    out = FILES["alpaca_full"]

    if file_nonempty(out, min_bytes=10_000) and csv_has(out, ["instruction"], min_rows=700):
        log.info(f"AlpacaEval already cached: {out}")
        return out

    log.info("Fetching AlpacaEval from raw JSON...")

    urls = [
        "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json",
        "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/5d2dbf0eee6823c233f75e721c48214f43af2dda/alpaca_eval.json",
    ]

    last_err = None

    for url in urls:
        try:
            tmp_json = out.with_suffix(".json")
            _download_url(url, tmp_json, min_bytes=100_000)

            with open(tmp_json, "r", encoding="utf-8") as f:
                obj = json.load(f)

            if isinstance(obj, dict):
                if "data" in obj and isinstance(obj["data"], list):
                    data = obj["data"]
                elif "examples" in obj and isinstance(obj["examples"], list):
                    data = obj["examples"]
                else:
                    data = list(obj.values())
            elif isinstance(obj, list):
                data = obj
            else:
                raise RuntimeError(f"Unexpected AlpacaEval JSON type: {type(obj)}")

            df = pd.DataFrame(data)

            if "instruction" not in df.columns:
                for c in ["prompt", "question", "input"]:
                    if c in df.columns:
                        df = df.rename(columns={c: "instruction"})
                        break

            if "instruction" not in df.columns:
                raise RuntimeError(
                    f"AlpacaEval file has no instruction/prompt/question/input column. "
                    f"Columns: {list(df.columns)}"
                )

            df["instruction"] = df["instruction"].fillna("").astype(str).str.strip()
            df = df[df["instruction"].ne("")].copy()

            if len(df) < 700:
                raise RuntimeError(f"AlpacaEval too small after cleaning: {len(df)} rows")

            atomic_write_csv(df, out)
            tmp_json.unlink(missing_ok=True)

            log.info(f"AlpacaEval saved: {out} ({len(df)} rows)")
            return out

        except Exception as e:
            last_err = e
            log.warning(f"AlpacaEval fetch failed from {url}: {e}")

    raise RuntimeError(f"All AlpacaEval download attempts failed: {last_err}")


# ============================================================
# WILDBENCH
# ============================================================

def _safe_isna(x) -> bool:
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


def _to_python_obj(x):
    """
    Convert WildBench cell values into Python objects.

    Handles:
    - list
    - tuple
    - dict
    - numpy array
    - stringified JSON
    - stringified Python literals
    """
    if x is None:
        return None

    try:
        import numpy as np
        if isinstance(x, np.ndarray):
            return x.tolist()
    except Exception:
        pass

    if isinstance(x, (list, tuple, dict)):
        return x

    if isinstance(x, str):
        s = x.strip()

        if not s:
            return None

        try:
            return json.loads(s)
        except Exception:
            pass

        try:
            return ast.literal_eval(s)
        except Exception:
            return s

    if _safe_isna(x):
        return None

    return x


def _extract_text_from_any(obj) -> str:
    """
    Recursively extract the first useful prompt-like text.
    """
    obj = _to_python_obj(obj)

    if obj is None:
        return ""

    if isinstance(obj, str):
        s = obj.strip()

        if not s:
            return ""

        # Avoid returning raw serialized containers as prompts.
        if s.startswith("{") or s.startswith("["):
            return ""

        return s

    if isinstance(obj, dict):
        # Prefer direct content fields.
        for k in [
            "content",
            "text",
            "value",
            "prompt",
            "instruction",
            "question",
            "input",
            "query",
        ]:
            if k in obj:
                txt = _extract_text_from_any(obj.get(k))
                if txt:
                    return txt

        # Then scan all values.
        for v in obj.values():
            txt = _extract_text_from_any(v)
            if txt:
                return txt

    if isinstance(obj, (list, tuple)):
        # Prefer user/human turns if present.
        for item in obj:
            item = _to_python_obj(item)

            if isinstance(item, dict):
                role = str(item.get("role", item.get("from", ""))).lower()

                if role in ["user", "human"]:
                    txt = _extract_text_from_any(item)
                    if txt:
                        return txt

        # Fallback: first useful text anywhere.
        for item in obj:
            txt = _extract_text_from_any(item)
            if txt:
                return txt

    return ""


def _extract_wildbench_prompt(row, columns) -> str:
    """
    Robustly extract prompt text from WildBench rows.
    """
    # Main WildBench column.
    if "conversation_input" in columns:
        txt = _extract_text_from_any(row.get("conversation_input"))
        if txt:
            return txt.strip()

    # Other possible conversation columns.
    for c in ["conversation", "messages"]:
        if c in columns:
            txt = _extract_text_from_any(row.get(c))
            if txt:
                return txt.strip()

    # Direct fallback fields.
    for c in ["prompt", "instruction", "question", "input", "query", "intent"]:
        if c in columns:
            txt = _extract_text_from_any(row.get(c))
            if txt:
                return txt.strip()

    return ""


def _fetch_wildbench(n: int = 50) -> Path:
    """
    Fetch WildBench using valid configs and extract 50 usable prompts.
    """
    out = FILES["wildbench_jsonl"]

    if file_nonempty(out, min_bytes=500) and _jsonl_count(out) >= n:
        log.info(f"WildBench already cached: {out}")
        return out

    log.info("Fetching WildBench...")

    rows: List[Dict[str, Any]] = []

    try:
        from datasets import load_dataset

        candidates = [
            ("allenai/WildBench", "v2", "test"),
            ("allenai/WildBench", "v2-hard", "test"),
            ("allenai/WildBench", "v1-legacy", "test"),
        ]

        for ds_name, config, split in candidates:
            if len(rows) >= n:
                break

            try:
                log.info(f"Trying WildBench: {ds_name}, config={config}, split={split}")

                ds = load_dataset(ds_name, config, split=split)
                df = ds.to_pandas()

                log.info(
                    f"WildBench loaded: config={config}, rows={len(df)}, "
                    f"columns={list(df.columns)}"
                )

                for idx, r in df.iterrows():
                    prompt = _extract_wildbench_prompt(r, df.columns)

                    if not prompt:
                        continue

                    # Remove obvious non-prompts.
                    if len(prompt.split()) < 4:
                        continue

                    rows.append({
                        "id": str(r.get("session_id", r.get("id", f"wb_{len(rows):04d}"))),
                        "prompt": prompt,
                        "category": str(
                            r.get(
                                "primary_tag",
                                r.get("category", r.get("tag", "open"))
                            )
                        ).lower(),
                        "wildbench_config": config,
                    })

                    if len(rows) >= n:
                        break

            except Exception as e:
                log.warning(f"WildBench candidate failed: {ds_name}/{config}/{split}: {e}")

    except Exception as e:
        log.warning(f"WildBench load_dataset failed: {e}")

    # Deduplicate and clean.
    clean_rows = []
    seen = set()

    for r in rows:
        prompt = str(r.get("prompt", "")).strip()

        if not prompt:
            continue

        key = prompt.lower()

        if key in seen:
            continue

        seen.add(key)

        clean_rows.append({
            "id": str(r.get("id", f"wb_{len(clean_rows):04d}")),
            "prompt": prompt,
            "category": str(r.get("category", "open")).lower(),
            "wildbench_config": str(r.get("wildbench_config", "unknown")),
        })

        if len(clean_rows) >= n:
            break

    if len(clean_rows) < n:
        raise RuntimeError(
            f"WildBench extraction failed: only {len(clean_rows)}/{n} usable prompts. "
            "Do not continue because the contamination-free subset would be underpowered."
        )

    with open(out, "w", encoding="utf-8") as f:
        for r in clean_rows[:n]:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    log.info(f"WildBench subset saved: {out} ({len(clean_rows[:n])} rows)")
    return out


# ============================================================
# RUN STEPS
# ============================================================

# Remove broken/undersized WildBench cache if present.
try:
    if FILES["wildbench_jsonl"].exists():
        wb_n = _jsonl_count(FILES["wildbench_jsonl"])

        if wb_n < 50:
            log.info(f"Removing undersized WildBench cache ({wb_n} rows) to refetch.")
            FILES["wildbench_jsonl"].unlink(missing_ok=True)
except Exception as e:
    log.warning(f"Could not inspect existing WildBench cache: {e}")


run_step(
    [FILES["mt_bench_jsonl"]],
    _fetch_mt_bench,
    "fetch_mt_bench",
    validators={
        FILES["mt_bench_jsonl"]: lambda p: (
            file_nonempty(p, min_bytes=1024) and _jsonl_valid(p, min_rows=80)
        )
    },
)

run_step(
    [FILES["alpaca_full"]],
    _fetch_alpaca_eval,
    "fetch_alpaca_eval",
    validators={
        FILES["alpaca_full"]: lambda p: csv_has(p, ["instruction"], min_rows=700)
    },
)

run_step(
    [FILES["wildbench_jsonl"]],
    _fetch_wildbench,
    "fetch_wildbench",
    validators={
        FILES["wildbench_jsonl"]: lambda p: (
            file_nonempty(p, min_bytes=500) and _jsonl_count(p) >= 50
        )
    },
)

print("✓ Raw prompt sources cached")

In [ ]:
# ============================================================================
# Cell 2.2 — Build canonical master prompt set with stable IDs
# ============================================================================
"""
Cell 2.2 — Combine the three sources into a master prompt set with stable IDs.

Stable IDs are content-hashed; if a prompt's text changes upstream we'll see it.
"""
TARGET_MTB    = 80   # all of MT-Bench
TARGET_ALPACA = 70   # stratified sample
TARGET_WB     = 50   # contamination-free subset
TARGET_TOTAL  = TARGET_MTB + TARGET_ALPACA + TARGET_WB  # 200

def _normalize_category(c: str) -> str:
    if c is None:
        return "unknown"
    c = str(c).strip().lower()
    aliases = {
        "math": "math", "mathematics": "math",
        "code": "coding", "coding": "coding", "programming": "coding",
        "extraction": "extraction", "extract": "extraction",
        "reasoning": "reasoning", "logic": "reasoning",
        "writing": "writing", "creative_writing": "writing", "creative": "writing",
        "roleplay": "roleplay", "role-play": "roleplay",
        "stem": "reasoning", "humanities": "writing", "open": "writing",
        "brainstorm": "writing", "brainstorming": "writing",
        "summarization": "extraction", "summary": "extraction",
        "translation": "writing",
    }
    return aliases.get(c, c)

def _infer_alpaca_category(text: str) -> str:
    t = text.lower()
    if any(k in t for k in ["solve", "compute", "calculate", "equation", "integral"]):
        return "math"
    if any(k in t for k in ["python", "code", "function", "javascript", "sql", "regex"]):
        return "coding"
    if any(k in t for k in ["extract", "list all", "find the", "summarize"]):
        return "extraction"
    if any(k in t for k in ["story", "poem", "write a", "compose"]):
        return "writing"
    if any(k in t for k in ["roleplay", "act as", "pretend"]):
        return "roleplay"
    return "writing"

def _build_master_prompts() -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    # MT-Bench
    with open(FILES["mt_bench_jsonl"], "r", encoding="utf-8") as f:
        mt = [json.loads(line) for line in f if line.strip()]
    if len(mt) < TARGET_MTB:
        raise RuntimeError(f"Need ≥{TARGET_MTB} MT-Bench prompts, got {len(mt)}")
    for i, r in enumerate(mt[:TARGET_MTB]):
        prompt = r.get("turns", [r.get("question", "")])[0]
        rows.append({
            "prompt_id": f"MTB_{r.get('question_id', i):03d}",
            "prompt": prompt,
            "source": "mt_bench",
            "category": _normalize_category(r.get("category", "unknown")),
        })

    # AlpacaEval — stratified by inferred category
    alp = pd.read_csv(FILES["alpaca_full"])
    pcol = next((c for c in ["instruction", "prompt", "question"] if c in alp.columns), None)
    if pcol is None:
        raise RuntimeError(f"AlpacaEval has no prompt column: {list(alp.columns)}")
    alp["category"] = alp[pcol].map(_infer_alpaca_category).map(_normalize_category)
    parts, used = [], set()
    cats = sorted(alp["category"].dropna().unique())
    base = max(1, TARGET_ALPACA // max(1, len(cats)))
    for cat in cats:
        sub = alp[alp["category"] == cat]
        take = min(base, len(sub))
        s = sub.sample(take, random_state=RANDOM_SEED)
        parts.append(s); used.update(s.index)
    sample = pd.concat(parts) if parts else pd.DataFrame()
    if len(sample) < TARGET_ALPACA:
        rest = alp.drop(index=list(used), errors="ignore")
        sample = pd.concat([sample, rest.sample(TARGET_ALPACA - len(sample),
                                                 random_state=RANDOM_SEED)])
    sample = sample.head(TARGET_ALPACA).reset_index(drop=True)
    for i, r in sample.iterrows():
        prompt = str(r[pcol])
        rows.append({
            "prompt_id": f"AE_{i:03d}_{sha1_short(prompt, 6)}",
            "prompt": prompt,
            "source": "alpaca_eval",
            "category": _normalize_category(r["category"]),
        })

    # WildBench — contamination-free
    with open(FILES["wildbench_jsonl"], "r", encoding="utf-8") as f:
        wb = [json.loads(line) for line in f if line.strip()]
    wb = wb[:TARGET_WB]
    for r in wb:
        prompt = str(r["prompt"])
        rows.append({
            "prompt_id": f"WB_{sha1_short(prompt, 8)}",
            "prompt": prompt,
            "source": "wildbench",
            "category": _normalize_category(r.get("category", "open")),
        })

    df = pd.DataFrame(rows)
    df["is_calibration"] = False
    if df["prompt_id"].duplicated().any():
        raise RuntimeError("Duplicate prompt_ids — fix collision in stable id scheme")
    if len(df) < (TARGET_MTB + TARGET_ALPACA + 5):  # tolerate WB fallback being smaller
        raise RuntimeError(f"Master prompt set too small: {len(df)}")
    log.info(f"Master prompts: {len(df)} ({df['source'].value_counts().to_dict()})")
    return df

def _step_master_prompts():
    df = _build_master_prompts()
    atomic_write_csv(df, FILES["master_prompts"])

run_step([FILES["master_prompts"]], _step_master_prompts, "build_master_prompts",
         validators={FILES["master_prompts"]:
                     lambda p: csv_has(p, ["prompt_id", "prompt", "source", "category"], 100)})

mp_df = pd.read_csv(FILES["master_prompts"])
print(f"Master prompts: {len(mp_df)}")
print(mp_df.groupby(["source", "category"]).size().unstack(fill_value=0))




In [ ]:
# ============================================================================
# Cell 2.3 — Pre-registered exploratory / confirmatory split
# ============================================================================
"""
Cell 2.3 — Lock a 67/33 stratified split. Headline tests run on confirmatory.

Pre-registration discipline: the split is seeded so this cell is deterministic
and the same prompts will always land in the same arm. Once written, do NOT
modify split logic.
"""
def _step_split():
    df = pd.read_csv(FILES["master_prompts"])
    # Stratify by (source × category) to balance both arms
    rng = np.random.default_rng(PRESPLIT_SEED)
    rows: List[Dict[str, Any]] = []
    for (src, cat), sub in df.groupby(["source", "category"]):
        idx = sub.index.to_numpy()
        rng.shuffle(idx)
        n_conf = max(1, int(round(len(idx) * CONFIRMATORY_FRAC)))
        for i in idx[:n_conf]:
            rows.append({"prompt_id": df.loc[i, "prompt_id"], "split": "confirmatory"})
        for i in idx[n_conf:]:
            rows.append({"prompt_id": df.loc[i, "prompt_id"], "split": "exploratory"})
    out = pd.DataFrame(rows)
    out = out.merge(df[["prompt_id", "source", "category"]], on="prompt_id", how="left")
    counts = out.groupby(["split", "source"]).size().unstack(fill_value=0)
    log.info(f"Split counts:\n{counts}")
    atomic_write_csv(out, FILES["prompt_split"])

run_step([FILES["prompt_split"]], _step_split, "build_prompt_split",
         validators={FILES["prompt_split"]:
                     lambda p: csv_has(p, ["prompt_id", "split"], 100)})

split_df = pd.read_csv(FILES["prompt_split"])
print(split_df.groupby(["split", "source"]).size().unstack(fill_value=0))
print("✓ Pre-registered split locked")




## PHASE 3 — Models & Response Generation

We use llama-cpp-python with full GPU offload. With ~80–90 GB of GPU memory,
any single GGUF in our registry fits comfortably (largest is 70B Q4 ≈ 42 GB).
We stage models from Drive → local SSD before loading (faster mmap, fewer
Drive throttles), then unload + GC between models.

Resume safety: each (model × prompt) result is keyed by content hash.
A disconnect mid-run costs at most one prompt of work.



In [ ]:
# ============================================================================
# CELL 3.0 — Permanent GGUF model download / verification cell
# Updated for the V15 release-clean Falcon pipeline; uses the stable Drive storage directory
# Stores models permanently in Google Drive.
# Skips completed Drive files.
# Does NOT delete partial files.
# ============================================================================

import os
import sys
import json
import shutil
import subprocess
import gc
from pathlib import Path
from datetime import datetime

# ----------------------------------------------------------------------------
# 1. Install downloader dependencies
# ----------------------------------------------------------------------------

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U",
    "huggingface_hub",
    "hf_xet",
    "tqdm",
])

from huggingface_hub import hf_hub_download


# ----------------------------------------------------------------------------
# 2. Permanent Google Drive filesystem
# ----------------------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/tribal_pref_v11")
MODEL_ROOT = ROOT / "models"

MODEL_DIRS = {
    "large": MODEL_ROOT / "large",
    "small": MODEL_ROOT / "small",
    "quant": MODEL_ROOT / "quant",
}

for d in MODEL_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = MODEL_ROOT / "model_download_manifest.json"


# ----------------------------------------------------------------------------
# 3. Runtime download/cache folders
# ----------------------------------------------------------------------------
# Local runtime cache is temporary. Final models are saved to Google Drive.

LOCAL_MODEL_TMP = Path("/content/model_download_tmp")
LOCAL_HF_CACHE = Path("/content/hf_download_cache")

LOCAL_MODEL_TMP.mkdir(parents=True, exist_ok=True)
LOCAL_HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(LOCAL_HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE / "hub")


# ----------------------------------------------------------------------------
# 4. Download switches
# ----------------------------------------------------------------------------
# Turn any of these False if you do not want to check/download that group.

DOWNLOAD_LARGE = True
DOWNLOAD_SMALL = True
DOWNLOAD_QUANT = True

# Keep False unless you intentionally want to redownload.
FORCE_REDOWNLOAD = False

HF_TOKEN = os.environ.get("HF_TOKEN", None)


# ----------------------------------------------------------------------------
# 5. Model registries
# ----------------------------------------------------------------------------
# Active v11 families:
#   llama, qwen, falcon, gemma, yi
#
# Falcon replaces Mixtral/Mistral.
#
# Note:
# - If a valid GGUF already exists in Drive under ANY candidate filename,
#   the cell skips it and does not download.
# - repo_id/source_filename are used only when no valid Drive file is found.

LARGE_MODEL_REGISTRY = {
    "llama": {
        "repo_id": "bullerwins/Meta-Llama-3.1-70B-Instruct-GGUF",
        "source_filename": "Meta-Llama-3.1-70B-Instruct-Q4_K_M.gguf",
        "gguf_filename": "Meta-Llama-3.1-70B-Instruct-Q4_K_M.gguf",
        "candidates": [
            "Meta-Llama-3.1-70B-Instruct-Q4_K_M.gguf",
            "Meta-Llama-3.1-70B-Instruct.Q4_K_M.gguf",
        ],
        "min_gb": 35.0,
    },

    "qwen": {
        "repo_id": "bartowski/Qwen2.5-72B-Instruct-GGUF",
        "source_filename": "Qwen2.5-72B-Instruct-Q4_K_M.gguf",
        "gguf_filename": "Qwen2.5-72B-Instruct-Q4_K_M.gguf",
        "candidates": [
            "Qwen2.5-72B-Instruct-Q4_K_M.gguf",
            "Qwen2.5-72B-Instruct.Q4_K_M.gguf",
        ],
        "min_gb": 35.0,
    },

    "falcon": {
        # The earlier TheBloke/falcon-40b-instruct-GGUF path returned 404.
        # This repo currently exposes Falcon-40B-Instruct GGUF.
        "repo_id": "mradermacher/falcon-40b-instruct-GGUF",
        "source_filename": "falcon-40b-instruct.Q4_K_M.gguf",
        "gguf_filename": "falcon-40b-instruct.Q4_K_M.gguf",
        "candidates": [
            "falcon-40b-instruct.Q4_K_M.gguf",
            "Falcon-40B-Instruct-Q4_K_M.gguf",
            "falcon-40b-instruct-q4_k_m.gguf",
            "tiiuae-falcon-40b-instruct-Q4_K_M.gguf",
            "tiiuae-falcon-40b-instruct.Q4_K_M.gguf",
        ],
        "min_gb": 20.0,
    },

    "gemma": {
        "repo_id": "bartowski/gemma-2-27b-it-GGUF",
        "source_filename": "gemma-2-27b-it-Q4_K_M.gguf",
        "gguf_filename": "gemma-2-27b-it-Q4_K_M.gguf",
        "candidates": [
            "gemma-2-27b-it-Q4_K_M.gguf",
            "Gemma-2-27B-it-Q4_K_M.gguf",
            "gemma-2-27b-it.Q4_K_M.gguf",
        ],
        "min_gb": 12.0,
    },

    "yi": {
        "repo_id": "bartowski/Yi-1.5-34B-Chat-GGUF",
        "source_filename": "Yi-1.5-34B-Chat-Q4_K_M.gguf",
        "gguf_filename": "Yi-1.5-34B-Chat-Q4_K_M.gguf",
        "candidates": [
            "Yi-1.5-34B-Chat-Q4_K_M.gguf",
            "Yi-1.5-34B-Chat.Q4_K_M.gguf",
            "yi-1.5-34b-chat.Q4_K_M.gguf",
        ],
        "min_gb": 15.0,
    },
}


SMALL_MODEL_REGISTRY = {
    "llama": {
        "repo_id": "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
        "source_filename": "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
        "gguf_filename": "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
        "candidates": [
            "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
            "Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf",
        ],
        "min_gb": 3.0,
    },

    "qwen": {
        "repo_id": "bartowski/Qwen2.5-7B-Instruct-GGUF",
        "source_filename": "Qwen2.5-7B-Instruct-Q4_K_M.gguf",
        "gguf_filename": "Qwen2.5-7B-Instruct-Q4_K_M.gguf",
        "candidates": [
            "Qwen2.5-7B-Instruct-Q4_K_M.gguf",
            "Qwen2.5-7B-Instruct.Q4_K_M.gguf",
        ],
        "min_gb": 3.0,
    },

    "falcon": {
        "repo_id": "QuantFactory/falcon-7b-instruct-GGUF",
        "source_filename": "falcon-7b-instruct.Q4_K_M.gguf",
        "gguf_filename": "falcon-7b-instruct.Q4_K_M.gguf",
        "candidates": [
            "falcon-7b-instruct.Q4_K_M.gguf",
            "Falcon-7B-Instruct-Q4_K_M.gguf",
            "falcon-7b-instruct-q4_k_m.gguf",
            "tiiuae-falcon-7b-instruct-Q4_K_M.gguf",
            "tiiuae-falcon-7b-instruct.Q4_K_M.gguf",
        ],
        "min_gb": 3.0,
    },

    "gemma": {
        "repo_id": "bartowski/gemma-2-9b-it-GGUF",
        "source_filename": "gemma-2-9b-it-Q4_K_M.gguf",
        "gguf_filename": "gemma-2-9b-it-Q4_K_M.gguf",
        "candidates": [
            "gemma-2-9b-it-Q4_K_M.gguf",
            "Gemma-2-9B-it-Q4_K_M.gguf",
            "gemma-2-9b-it.Q4_K_M.gguf",
        ],
        "min_gb": 4.0,
    },

    "yi": {
        "repo_id": "bartowski/Yi-1.5-9B-Chat-GGUF",
        "source_filename": "Yi-1.5-9B-Chat-Q4_K_M.gguf",
        "gguf_filename": "Yi-1.5-9B-Chat-Q4_K_M.gguf",
        "candidates": [
            "Yi-1.5-9B-Chat-Q4_K_M.gguf",
            "Yi-1.5-9B-Chat.Q4_K_M.gguf",
            "yi-1.5-9b-chat.Q4_K_M.gguf",
        ],
        "min_gb": 4.0,
    },
}


QUANT_MODEL_REGISTRY = {
    "llama_q3": {
        "repo_id": "bartowski/Meta-Llama-3.1-70B-Instruct-GGUF",
        "source_filename": "Meta-Llama-3.1-70B-Instruct-Q3_K_M.gguf",
        "gguf_filename": "Meta-Llama-3.1-70B-Instruct-Q3_K_M.gguf",
        "candidates": [
            "Meta-Llama-3.1-70B-Instruct-Q3_K_M.gguf",
            "Meta-Llama-3.1-70B-Instruct.Q3_K_M.gguf",
        ],
        "min_gb": 25.0,
    },

    "qwen_q3": {
        "repo_id": "bartowski/Qwen2.5-72B-Instruct-GGUF",
        "source_filename": "Qwen2.5-72B-Instruct-Q3_K_M.gguf",
        "gguf_filename": "Qwen2.5-72B-Instruct-Q3_K_M.gguf",
        "candidates": [
            "Qwen2.5-72B-Instruct-Q3_K_M.gguf",
            "Qwen2.5-72B-Instruct.Q3_K_M.gguf",
        ],
        "min_gb": 25.0,
    },
}


REGISTRIES = {
    "large": LARGE_MODEL_REGISTRY,
    "small": SMALL_MODEL_REGISTRY,
    "quant": QUANT_MODEL_REGISTRY,
}


# ----------------------------------------------------------------------------
# 6. Helpers
# ----------------------------------------------------------------------------

def gb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 3)


def file_ok(path: Path, min_gb: float) -> bool:
    return path.exists() and path.is_file() and gb(path) >= float(min_gb)


def clear_runtime_temp():
    """
    Clears only temporary runtime download/cache folders.
    Does NOT touch Google Drive.
    Does NOT delete partial files in Drive.
    """
    for p in [LOCAL_MODEL_TMP, LOCAL_HF_CACHE]:
        if p.exists():
            shutil.rmtree(p, ignore_errors=True)
        p.mkdir(parents=True, exist_ok=True)

    gc.collect()
    print("[CLEARED] runtime temp/cache")


def load_manifest():
    if MANIFEST_PATH.exists():
        try:
            with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}


def save_manifest(manifest):
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = MANIFEST_PATH.with_suffix(".json.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    tmp.replace(MANIFEST_PATH)


manifest = load_manifest()


def find_existing_candidate(drive_dir: Path, candidates: list, min_gb: float):
    """
    Search Drive for already-downloaded model files.

    Matching order:
    1. exact candidate filenames
    2. case-insensitive exact filename match
    3. loose fallback for Falcon naming differences
    """
    drive_dir = Path(drive_dir)

    # 1. Exact match
    for name in candidates:
        p = drive_dir / name
        if file_ok(p, min_gb):
            return p

    # 2. Case-insensitive exact match
    ggufs = [p for p in drive_dir.glob("*.gguf") if p.is_file()]
    lower_map = {p.name.lower(): p for p in ggufs}

    for name in candidates:
        p = lower_map.get(name.lower())
        if p and file_ok(p, min_gb):
            return p

    # 3. Loose Falcon fallback
    joined = " ".join(candidates).lower()

    if "falcon" in joined and "40b" in joined:
        for p in ggufs:
            n = p.name.lower()
            if "falcon" in n and "40b" in n and file_ok(p, min_gb):
                return p

    if "falcon" in joined and "7b" in joined:
        for p in ggufs:
            n = p.name.lower()
            if "falcon" in n and "7b" in n and file_ok(p, min_gb):
                return p

    return None


# ----------------------------------------------------------------------------
# 7. Download one model with strict Drive caching
# ----------------------------------------------------------------------------

def download_one(scale: str, family: str, spec: dict):
    drive_dir = MODEL_DIRS[scale]
    drive_dir.mkdir(parents=True, exist_ok=True)

    candidates = spec.get("candidates", [spec["gguf_filename"]])
    preferred_name = spec["gguf_filename"]

    final_path = drive_dir / preferred_name
    partial_path = drive_dir / f"{preferred_name}.partial"

    min_gb = float(spec["min_gb"])

    print("\n" + "=" * 110)
    print(f"MODEL:      {scale}/{family}")
    print(f"REPO:       {spec['repo_id']}")
    print(f"HF FILE:    {spec['source_filename']}")
    print(f"DRIVE DIR:  {drive_dir}")
    print(f"CANDIDATES: {candidates}")
    print("=" * 110)

    if FORCE_REDOWNLOAD and final_path.exists():
        print(f"[FORCE DELETE FINAL] {final_path}")
        final_path.unlink()

    # First search Drive for any valid existing candidate.
    existing = None if FORCE_REDOWNLOAD else find_existing_candidate(drive_dir, candidates, min_gb)

    if existing is not None:
        print("[SKIP] already permanently stored in Drive:")
        print(f"       {existing}")
        print(f"       size = {gb(existing):.2f} GB")

        manifest[f"{scale}/{family}"] = {
            "status": "skipped_existing_drive_file",
            "repo_id": spec["repo_id"],
            "source_filename": spec["source_filename"],
            "drive_path": str(existing),
            "size_gb": round(gb(existing), 3),
            "updated_at": datetime.now().isoformat(),
        }
        save_manifest(manifest)
        return existing

    # If preferred final exists but is too small, rename it instead of deleting.
    if final_path.exists() and not file_ok(final_path, min_gb):
        bad_path = final_path.with_suffix(
            final_path.suffix + f".bad_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        )
        print(f"[BAD FINAL] {final_path.name} is only {gb(final_path):.2f} GB")
        print(f"[RENAME BAD FINAL] {final_path.name} -> {bad_path.name}")
        final_path.rename(bad_path)

    # Preserve Drive partial file.
    if partial_path.exists():
        print(f"[NOTICE] existing partial preserved:")
        print(f"         {partial_path}")
        print(f"         size = {gb(partial_path):.2f} GB")

    local_dir = LOCAL_MODEL_TMP / f"{scale}_{family}"
    if local_dir.exists():
        shutil.rmtree(local_dir, ignore_errors=True)
    local_dir.mkdir(parents=True, exist_ok=True)

    clear_runtime_temp()
    local_dir.mkdir(parents=True, exist_ok=True)

    print("[DOWNLOAD] no valid Drive file found, downloading to local runtime first...")

    downloaded_path = hf_hub_download(
        repo_id=spec["repo_id"],
        filename=spec["source_filename"],
        local_dir=str(local_dir),
        cache_dir=str(LOCAL_HF_CACHE),
        token=HF_TOKEN,
    )

    downloaded_path = Path(downloaded_path)

    if not downloaded_path.exists():
        raise FileNotFoundError(f"Download returned missing file: {downloaded_path}")

    if not file_ok(downloaded_path, min_gb):
        raise RuntimeError(
            f"Downloaded local file failed validation: {downloaded_path} "
            f"size={gb(downloaded_path) if downloaded_path.exists() else 0:.2f} GB, "
            f"required>={min_gb:.2f} GB"
        )

    print(f"[LOCAL READY] {downloaded_path.name} ({gb(downloaded_path):.2f} GB)")

    print("[SAVE] copying complete file to Google Drive .partial...")
    shutil.copy2(downloaded_path, partial_path)

    if not file_ok(partial_path, min_gb):
        raise RuntimeError(
            f"Drive partial failed validation: {partial_path} "
            f"size={gb(partial_path) if partial_path.exists() else 0:.2f} GB, "
            f"required>={min_gb:.2f} GB"
        )

    partial_path.replace(final_path)

    if not file_ok(final_path, min_gb):
        raise RuntimeError(
            f"Drive final failed validation: {final_path} "
            f"size={gb(final_path) if final_path.exists() else 0:.2f} GB, "
            f"required>={min_gb:.2f} GB"
        )

    print(f"[DONE] saved permanently in Drive: {final_path.name} ({gb(final_path):.2f} GB)")

    manifest[f"{scale}/{family}"] = {
        "status": "downloaded_and_saved_to_drive",
        "repo_id": spec["repo_id"],
        "source_filename": spec["source_filename"],
        "drive_path": str(final_path),
        "size_gb": round(gb(final_path), 3),
        "updated_at": datetime.now().isoformat(),
    }
    save_manifest(manifest)

    clear_runtime_temp()

    return final_path


# ----------------------------------------------------------------------------
# 8. Download selected groups
# ----------------------------------------------------------------------------

selected_scales = []

if DOWNLOAD_LARGE:
    selected_scales.append("large")

if DOWNLOAD_SMALL:
    selected_scales.append("small")

if DOWNLOAD_QUANT:
    selected_scales.append("quant")


for scale in selected_scales:
    print("\n" + "#" * 110)
    print(f"DOWNLOADING / VERIFYING SCALE: {scale.upper()}")
    print(f"DESTINATION: {MODEL_DIRS[scale]}")
    print("#" * 110)

    for family, spec in REGISTRIES[scale].items():
        download_one(scale, family, spec)


# ----------------------------------------------------------------------------
# 9. Final verification
# ----------------------------------------------------------------------------

print("\n" + "=" * 110)
print("FINAL GOOGLE DRIVE MODEL VERIFICATION")
print("=" * 110)

all_ok = True

for scale in selected_scales:
    print(f"\n[{scale.upper()}] {MODEL_DIRS[scale]}")

    for family, spec in REGISTRIES[scale].items():
        candidates = spec.get("candidates", [spec["gguf_filename"]])
        p = find_existing_candidate(MODEL_DIRS[scale], candidates, float(spec["min_gb"]))

        if p is not None:
            print(f"✅ {family:10s} {p.name:75s} {gb(p):8.2f} GB")
        else:
            all_ok = False
            preferred = MODEL_DIRS[scale] / spec["gguf_filename"]
            partial = MODEL_DIRS[scale] / f"{spec['gguf_filename']}.partial"

            if partial.exists():
                print(f"⚠️ {family:10s} final missing/bad, partial exists: {partial.name} ({gb(partial):.2f} GB)")
            else:
                print(f"❌ {family:10s} missing/bad: {preferred.name}")

print(f"\nManifest saved at: {MANIFEST_PATH}")

if all_ok:
    print("\n✅ All selected models are permanently stored in Google Drive.")
    print("Next run will SKIP them, not redownload.")
else:
    print("\n⚠️ Some models are missing or incomplete. Rerun this cell after checking the missing files.")

In [ ]:
# ============================================================================
# Cell 3.1 — Model loading + STRONG cleanup (GPU + disk + in-memory cache)
#
# Three-tier cache:
#   1. In-memory  — same model object reused across calls (instant).
#   2. Local SSD  — staged GGUF reused across loads (skip ~10 min Drive copy).
#   3. Drive      — authoritative copy, never deleted.
#
# All three layers are checked before any expensive work. Releases are
# explicit so GPU memory is actually returned (llama.cpp uses ggml, not torch).
# ============================================================================

from pathlib import Path
from contextlib import contextmanager
from typing import Any, Dict, Optional
import shutil, time, gc, os, json, hashlib

import numpy as np

# ----------------------------------------------------------------------------
# Logger fallback (in case Cell 1.3 didn't run yet)
# ----------------------------------------------------------------------------
if "logger" not in globals():
    import logging
    logger = logging.getLogger("tribal_pref_v15_release")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        h = logging.StreamHandler()
        h.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
        logger.addHandler(h)


# ============================================================================
# Disk + GPU diagnostics
# ============================================================================
def gb(path) -> float:
    p = Path(path)
    if not p.exists():
        return 0.0
    return p.stat().st_size / (1024 ** 3)

def disk_free_gb(path: str = "/content") -> float:
    try:
        return shutil.disk_usage(path).free / (1024 ** 3)
    except Exception:
        return -1.0

def gpu_free_gb() -> float:
    try:
        import torch
        if torch.cuda.is_available():
            free, _ = torch.cuda.mem_get_info()
            return free / (1024 ** 3)
    except Exception:
        pass
    return -1.0

def cleanup_gpu_memory(verbose: bool = True) -> None:
    """Aggressively reclaim GPU memory after a llama.cpp model release."""
    gc.collect(); gc.collect()                         # break ref cycles
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            torch.cuda.synchronize()
    except Exception:
        pass
    if verbose:
        f = gpu_free_gb()
        if f >= 0:
            logger.info(f"  GPU memory: {f:.1f} GB free")


# ============================================================================
# Native llama.cpp handle release
# ============================================================================
def _safe_close_llm(llm) -> None:
    """Properly release a llama.cpp model — Python wrapper AND native handle.

    `del llm` alone isn't enough because llama_cpp.Llama holds C-level resources
    (model, ctx, batch, sampler) inside a contextlib.ExitStack. We close the
    stack explicitly so GPU memory is actually returned.
    """
    if llm is None:
        return
    try:
        stack = getattr(llm, "_stack", None)
        if stack is not None and hasattr(stack, "close"):
            stack.close()
    except Exception as e:
        logger.debug(f"_stack.close failed: {e}")
    try:
        if hasattr(llm, "close"):
            llm.close()
    except Exception:
        pass
    for attr in ("_model", "_ctx", "_batch", "_sampler", "_lora_adapters", "_scores"):
        try:
            if hasattr(llm, attr):
                delattr(llm, attr)
        except Exception:
            pass


# ============================================================================
# Drive ↔ Local-SSD staging directories
# ============================================================================
def model_drive_and_local_dirs(scale_or_quant: str):
    if scale_or_quant == "large":
        return PATHS["models_large"], PATHS["local_models_large"]
    if scale_or_quant == "small":
        return PATHS["models_small"], PATHS["local_models_small"]
    if scale_or_quant == "quant_ablation":
        return PATHS["models_quant"], PATHS["local_models_quant"]
    raise KeyError(f"Unknown model group: {scale_or_quant}")


# ============================================================================
# LRU disk eviction so we don't accumulate staged GGUFs forever
# ============================================================================
def _list_staged_gguf(scale_or_quant: str):
    _, local_dir = model_drive_and_local_dirs(scale_or_quant)
    local_dir = Path(local_dir)
    if not local_dir.exists():
        return []
    out = []
    for f in local_dir.iterdir():
        if f.is_file() and f.suffix == ".gguf":
            try:
                out.append((f, f.stat().st_mtime, f.stat().st_size))
            except Exception:
                pass
    out.sort(key=lambda x: x[1])         # oldest first
    return out

def cleanup_local_staged_model(scale_or_quant: str, key: str,
                                verbose: bool = True) -> None:
    """Delete the locally staged GGUF for one model. Drive copy is preserved."""
    if scale_or_quant not in MODEL_REGISTRY or key not in MODEL_REGISTRY[scale_or_quant]:
        return
    spec = MODEL_REGISTRY[scale_or_quant][key]
    _, local_dir = model_drive_and_local_dirs(scale_or_quant)
    local_dir = Path(local_dir)
    deleted_gb = 0.0
    for filename in spec.get("candidates", []):
        for p in [local_dir / filename, local_dir / (filename + ".tmp")]:
            if p.exists():
                sz = p.stat().st_size / (1024 ** 3)
                try:
                    p.unlink()
                    deleted_gb += sz
                    if verbose:
                        logger.info(f"  🗑  Removed staged file: {p.name} ({sz:.2f} GB)")
                except Exception as e:
                    logger.warning(f"  Could not delete {p.name}: {e}")
    if verbose and deleted_gb > 0:
        logger.info(f"  💾 Local disk: {disk_free_gb():.1f} GB free")

def ensure_disk_for(needed_gb: float, headroom_gb: float = 15.0,
                     candidate_groups=("large", "small", "quant_ablation"),
                     protect_keys=()) -> None:
    """Before staging, make sure free disk ≥ needed + headroom.
    If not, evict the OLDEST staged GGUFs (across all groups) until it is.

    `protect_keys` — set of (scale_or_quant, key) tuples that must NOT be evicted
    (used to keep a currently-loaded-in-memory model on disk too).
    """
    target = needed_gb + headroom_gb
    free = disk_free_gb()
    if free >= target:
        return
    logger.warning(f"⚠ Low disk: {free:.1f} GB free, need {target:.1f} GB. Evicting LRU…")

    # Build full eviction candidate list with metadata
    candidates = []
    protected_paths = set()
    for so_q in candidate_groups:
        if so_q not in MODEL_REGISTRY:
            continue
        for key, spec in MODEL_REGISTRY[so_q].items():
            if (so_q, key) in protect_keys:
                _, local_dir = model_drive_and_local_dirs(so_q)
                for fname in spec.get("candidates", []):
                    protected_paths.add(str(Path(local_dir) / fname))

        for f, mtime, sz in _list_staged_gguf(so_q):
            if str(f) in protected_paths:
                continue
            candidates.append((mtime, f, sz))

    candidates.sort(key=lambda x: x[0])         # oldest first
    for _, fpath, sz in candidates:
        sz_gb = sz / (1024 ** 3)
        try:
            fpath.unlink()
            logger.info(f"  🗑  Evicted {fpath.name} ({sz_gb:.2f} GB)")
            free = disk_free_gb()
            if free >= target:
                logger.info(f"  ✓ Disk OK: {free:.1f} GB free")
                return
        except Exception as e:
            logger.warning(f"  Could not evict {fpath.name}: {e}")
    logger.warning(f"  ⚠ After eviction {disk_free_gb():.1f} GB free (still < target)")


# ============================================================================
# Drive → local SSD staging with progress bar
# ============================================================================
def copy_with_progress(src, dst, chunk_size=256 * 1024 * 1024):
    src = Path(src); dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    total = src.stat().st_size
    tmp = dst.with_suffix(dst.suffix + ".tmp")

    if dst.exists() and dst.stat().st_size == total:
        print(f"✅ Already staged: {dst} ({gb(dst):.2f} GB)")
        return dst
    if dst.exists() and dst.stat().st_size != total:
        print(f"Removing incomplete staged file: {dst} ({gb(dst):.2f} GB)")
        dst.unlink()
    if tmp.exists():
        print(f"Removing old temp file: {tmp} ({gb(tmp):.2f} GB)")
        tmp.unlink()

    # Make sure there is enough room before we even start.
    # Protect the currently-in-memory model from eviction so we don't tear it
    # down mid-flight to make room for itself.
    protect = ()
    if _LOADED_MODEL.get("scale_or_quant") and _LOADED_MODEL.get("key"):
        protect = ((_LOADED_MODEL["scale_or_quant"], _LOADED_MODEL["key"]),)
    ensure_disk_for(total / (1024 ** 3), protect_keys=protect)

    copied, t0 = 0, time.time()
    print("=" * 90)
    print(f"STAGING {src.name}  ({total/(1024**3):.2f} GB)  →  {dst}")
    print("=" * 90)
    with open(src, "rb") as fsrc, open(tmp, "wb") as fdst:
        while True:
            chunk = fsrc.read(chunk_size)
            if not chunk:
                break
            fdst.write(chunk)
            copied += len(chunk)
            elapsed = max(time.time() - t0, 1e-6)
            speed = copied / elapsed / (1024 ** 2)
            print(f"  copied {copied/(1024**3):.2f}/{total/(1024**3):.2f} GB "
                  f"({100*copied/total:.1f}%) | {speed:.1f} MB/s", flush=True)
    tmp.replace(dst)
    if dst.stat().st_size != total:
        raise RuntimeError(f"Staged size mismatch: expected {total}, got {dst.stat().st_size}")
    print(f"✅ Staging done in {(time.time()-t0)/60:.1f} min")
    return dst


# ============================================================================
# Cache check #1 — is a complete staged GGUF already on the local SSD?
# ============================================================================
def is_model_staged(scale_or_quant: str, key: str) -> bool:
    """Return True if a complete staged copy already exists on local SSD AND
    matches the size of its Drive counterpart."""
    if scale_or_quant not in MODEL_REGISTRY or key not in MODEL_REGISTRY[scale_or_quant]:
        return False
    spec = MODEL_REGISTRY[scale_or_quant][key]
    drive_dir, local_dir = model_drive_and_local_dirs(scale_or_quant)
    drive_dir, local_dir = Path(drive_dir), Path(local_dir)
    for filename in spec.get("candidates", []):
        dp, lp = drive_dir / filename, local_dir / filename
        if dp.exists() and lp.exists() and dp.stat().st_size == lp.stat().st_size:
            return True
    return False


def _resolve_model_path(spec, scale_or_quant, key):
    drive_dir, local_dir = model_drive_and_local_dirs(scale_or_quant)
    drive_dir, local_dir = Path(drive_dir), Path(local_dir)
    drive_dir.mkdir(parents=True, exist_ok=True)
    local_dir.mkdir(parents=True, exist_ok=True)
    candidates = spec.get("candidates", [])
    tried = []
    for filename in candidates:
        drive_path = drive_dir / filename
        local_path = local_dir / filename
        tried.append(str(drive_path))
        if drive_path.exists() and drive_path.stat().st_size > 0:
            drive_size = drive_path.stat().st_size
            # Local SSD cache hit
            if local_path.exists() and local_path.stat().st_size == drive_size:
                logger.info(f"♻  Using already-staged file: {local_path.name} "
                            f"({gb(local_path):.2f} GB)")
                # Touch so LRU eviction prefers other files
                try:
                    local_path.touch()
                except Exception:
                    pass
                return local_path
            # Stage from Drive
            logger.info(f"Staging {filename} to local SSD…")
            return copy_with_progress(drive_path, local_path)
    raise FileNotFoundError(
        f"No GGUF found for {scale_or_quant}/{key}.\n"
        f"Folder: {drive_dir}\nTried:\n  " + "\n  ".join(tried)
    )


# ============================================================================
# Cache check #2 — in-memory cache (the big win for repeat calls)
# ============================================================================
_LOADED_MODEL: Dict[str, Any] = {
    "llm":            None,
    "scale_or_quant": None,
    "key":            None,
    "fingerprint":    None,
    "loaded_at":      None,
}

def _model_fingerprint(scale_or_quant, key, n_ctx, logits_all, path) -> str:
    """Captures everything that matters for safe in-memory reuse."""
    try:
        size = Path(path).stat().st_size
        mtime = int(Path(path).stat().st_mtime)
    except Exception:
        size, mtime = -1, -1
    payload = (f"{scale_or_quant}|{key}|n_ctx={int(n_ctx)}|"
               f"logits_all={int(bool(logits_all))}|size={size}|mtime={mtime}")
    return hashlib.sha1(payload.encode()).hexdigest()[:16]

def is_model_loaded(scale_or_quant: str, key: str,
                    n_ctx=None, logits_all=False) -> bool:
    """True iff the requested model is already in GPU memory with matching
    n_ctx and logits_all flags AND the underlying file hasn't changed."""
    cur = _LOADED_MODEL
    if cur["llm"] is None:
        return False
    if cur["scale_or_quant"] != scale_or_quant or cur["key"] != key:
        return False
    try:
        spec = MODEL_REGISTRY[scale_or_quant][key]
        # Don't trigger a stage from this check; just read paths.
        drive_dir, local_dir = model_drive_and_local_dirs(scale_or_quant)
        drive_dir, local_dir = Path(drive_dir), Path(local_dir)
        path = None
        for fname in spec.get("candidates", []):
            lp = local_dir / fname
            if lp.exists():
                path = lp; break
            dp = drive_dir / fname
            if dp.exists():
                path = dp; break
        if path is None:
            return False
        n_ctx_eff = int(n_ctx or N_CTX)
        fp = _model_fingerprint(scale_or_quant, key, n_ctx_eff, logits_all, path)
        return fp == cur["fingerprint"]
    except Exception:
        return False

def _evict_loaded_model() -> None:
    """Release whatever is currently in the in-memory slot (frees GPU)."""
    cur = _LOADED_MODEL
    if cur["llm"] is None:
        return
    logger.info(f"Evicting in-memory model: {cur['scale_or_quant']}/{cur['key']}")
    try:
        _safe_close_llm(cur["llm"])
    except Exception:
        pass
    cur["llm"] = None
    cur["scale_or_quant"] = None
    cur["key"] = None
    cur["fingerprint"] = None
    cur["loaded_at"] = None
    cleanup_gpu_memory(verbose=True)


# ============================================================================
# Public API: load_model + with_model
# ============================================================================
def load_model(scale_or_quant: str, key: str,
               n_ctx: Optional[int] = None,
               logits_all: bool = False,
               force_reload: bool = False,
               **extra):
    """
    Three-tier cached load:
      1. If the same model is already in GPU memory with matching n_ctx and
         logits_all flags → return it instantly.
      2. Else, if a complete staged GGUF exists on local SSD → use it (no Drive copy).
      3. Else, copy from Drive → local SSD with progress.

    Set force_reload=True to bypass the in-memory cache (still uses local-SSD cache).
    """
    from llama_cpp import Llama

    if scale_or_quant not in MODEL_REGISTRY:
        raise KeyError(f"MODEL_REGISTRY has no group: {scale_or_quant}")
    if key not in MODEL_REGISTRY[scale_or_quant]:
        raise KeyError(f"MODEL_REGISTRY['{scale_or_quant}'] has no key: {key}")

    spec = MODEL_REGISTRY[scale_or_quant][key]
    n_ctx_eff = int(n_ctx or N_CTX)

    # --- Tier 1: in-memory ---
    if (not force_reload) and is_model_loaded(scale_or_quant, key,
                                                n_ctx=n_ctx_eff,
                                                logits_all=logits_all):
        elapsed = time.time() - (_LOADED_MODEL["loaded_at"] or time.time())
        logger.info(
            f"♻  Reusing in-memory model: {scale_or_quant}/{key} "
            f"(loaded {elapsed/60:.1f} min ago, GPU free: {gpu_free_gb():.1f} GB)"
        )
        return _LOADED_MODEL["llm"]

    # If a different model is loaded, evict it FIRST (frees GPU before staging).
    if _LOADED_MODEL["llm"] is not None:
        _evict_loaded_model()

    # --- Tier 2 + 3: resolve path (uses local cache or stages from Drive) ---
    path = _resolve_model_path(spec, scale_or_quant, key)

    n_gpu_layers = int(spec.get("n_gpu_layers", DEFAULT_N_GPU_LAYERS))
    kwargs = {
        "model_path":   str(path),
        "n_ctx":        n_ctx_eff,
        "n_gpu_layers": n_gpu_layers,
        "logits_all":   bool(logits_all),
        "verbose":      False,
    }
    chat_format = spec.get("chat_format")
    if chat_format:
        kwargs["chat_format"] = chat_format
    kwargs.update(extra)

    logger.info(
        f"Loading {scale_or_quant}/{key} ({spec.get('display', key)}) "
        f"from {path.name} | n_gpu_layers={n_gpu_layers} | logits_all={logits_all} "
        f"| GPU free before: {gpu_free_gb():.1f} GB"
    )
    t0 = time.time()
    try:
        llm = Llama(**kwargs)
    except Exception as e:
        logger.error(f"FAILED TO LOAD {scale_or_quant}/{key}: {type(e).__name__}: {e}")
        cleanup_gpu_memory()
        raise
    dt = time.time() - t0
    logger.info(
        f"Loaded {scale_or_quant}/{key} in {dt:.1f}s "
        f"| GPU free after: {gpu_free_gb():.1f} GB"
    )

    # Park in the in-memory slot so the next call can reuse instantly.
    _LOADED_MODEL["llm"]            = llm
    _LOADED_MODEL["scale_or_quant"] = scale_or_quant
    _LOADED_MODEL["key"]            = key
    _LOADED_MODEL["fingerprint"]    = _model_fingerprint(
        scale_or_quant, key, n_ctx_eff, logits_all, path)
    _LOADED_MODEL["loaded_at"]      = time.time()
    return llm


@contextmanager
def with_model(scale_or_quant: str, key: str, *,
               release_on_exit: bool = False,
               cleanup_disk_after: bool = False,
               **kwargs):
    """
    Context manager that loads, yields, then OPTIONALLY releases the model.

    Default (release_on_exit=False, cleanup_disk_after=False):
      Model stays in GPU memory after the block exits, so the next
      with_model(same args) call is instant. The slot is auto-evicted only
      when a different model needs to be loaded.

    Args:
        release_on_exit: True → forcibly evict from GPU on exit. Useful right
            before switching to a different size class (e.g. last large judge
            → small judge). Default False (keeps it warm).
        cleanup_disk_after: True → also delete the locally staged GGUF.
            Useful when you know you won't need this model again in this
            session. Default False (keeps disk cache warm).
    """
    llm = None
    try:
        llm = load_model(scale_or_quant, key, **kwargs)
        yield llm
    finally:
        if release_on_exit:
            _evict_loaded_model()
        if cleanup_disk_after:
            cleanup_local_staged_model(scale_or_quant, key, verbose=True)


# ============================================================================
# Inspection / debugging helpers
# ============================================================================
def inspect_model_locations():
    """One-shot report: what's on Drive, what's staged locally, what's loaded."""
    print("=" * 90)
    print("MODEL LOCATION AUDIT")
    print("=" * 90)
    for group in ["large", "small", "quant_ablation"]:
        if group not in MODEL_REGISTRY:
            continue
        drive_dir, local_dir = model_drive_and_local_dirs(group)
        print(f"\n[{group}]")
        print(f"  Drive: {drive_dir}")
        print(f"  Local: {local_dir}")
        for key, spec in MODEL_REGISTRY[group].items():
            fd, fl = [], []
            for fname in spec.get("candidates", []):
                dp = Path(drive_dir) / fname
                lp = Path(local_dir) / fname
                if dp.exists():
                    fd.append(f"{dp.name} ({gb(dp):.2f} GB)")
                if lp.exists():
                    fl.append(f"{lp.name} ({gb(lp):.2f} GB)")
            print(f"  {key:10s} Drive: {fd or 'MISSING'}")
            print(f"  {'':10s} Local: {fl or 'not staged'}")
    cur = _LOADED_MODEL
    print()
    if cur["llm"] is not None:
        elapsed = (time.time() - cur['loaded_at']) / 60
        print(f"In-memory: {cur['scale_or_quant']}/{cur['key']} "
              f"(loaded {elapsed:.1f} min ago)")
    else:
        print("In-memory: nothing loaded")
    print(f"Local disk free: {disk_free_gb():.1f} GB | GPU free: {gpu_free_gb():.1f} GB")
    print("=" * 90)


def free_all_models() -> None:
    """Nuclear option — release in-memory model AND every staged file."""
    _evict_loaded_model()
    for group in ["large", "small", "quant_ablation"]:
        if group not in MODEL_REGISTRY:
            continue
        for key in MODEL_REGISTRY[group]:
            cleanup_local_staged_model(group, key, verbose=True)
    logger.info("All models released. Disk free: "
                f"{disk_free_gb():.1f} GB | GPU free: {gpu_free_gb():.1f} GB")


print("✅ Cell 3.1 reloaded — three-tier cache (memory + local SSD + Drive)")
print(f"   Disk free: {disk_free_gb():.1f} GB | GPU free: {gpu_free_gb():.1f} GB")

In [ ]:
# ============================================================================
# Cell 3.2A — Response generation constants and helper functions
# ============================================================================

RESPONSE_COLUMNS = [
    "prompt_id",
    "model_scale",
    "model_key",
    "model_family",
    "model_display",
    "response",
    "n_tokens",
    "gen_seconds",
    "quality_ok",
    "generated_at",
]


def _quality_check(response: str, n_tokens: int) -> bool:
    """Cheap sanity filter — refused / empty / pathologically short responses."""
    if not response or not str(response).strip():
        return False

    if int(n_tokens or 0) < 5:
        return False

    bad_phrases = [
        "i cannot",
        "i can't help with",
        "as an ai language model",
    ]

    low = str(response).lower()[:200]

    if any(b in low for b in bad_phrases) and len(str(response)) < 200:
        return False

    return True


def _per_model_csv(scale_or_quant: str, key: str) -> Path:
    """
    Return the response CSV path for one model.
    One model = one CSV.
    """
    if scale_or_quant == "large":
        base = PATHS["responses_large"]
    elif scale_or_quant == "small":
        base = PATHS["responses_small"]
    elif scale_or_quant in ["quant", "quant_ablation"]:
        base = PATHS["responses_quant"]
    else:
        raise ValueError(f"Unknown response scale: {scale_or_quant}")

    base.mkdir(parents=True, exist_ok=True)

    return base / f"responses_{scale_or_quant}_{key}.csv"


def _load_existing_response_csv(out_path: Path) -> pd.DataFrame:
    """
    Load existing response CSV safely.
    If the file is bad or missing required columns, start fresh.
    """
    if not out_path.exists():
        return pd.DataFrame(columns=RESPONSE_COLUMNS)

    try:
        df = pd.read_csv(out_path)

        missing = [c for c in RESPONSE_COLUMNS if c not in df.columns]
        if missing:
            log.warning(f"{out_path.name} missing columns {missing}; ignoring old file.")
            return pd.DataFrame(columns=RESPONSE_COLUMNS)

        df = df[RESPONSE_COLUMNS].copy()
        df["prompt_id"] = df["prompt_id"].astype(str)
        df = df.drop_duplicates(subset=["prompt_id"], keep="last")

        return df

    except Exception as e:
        log.warning(f"Could not read {out_path}: {type(e).__name__}: {e}")
        return pd.DataFrame(columns=RESPONSE_COLUMNS)


def _generate_one(llm: Any, prompt: str, family_hint: str) -> Tuple[str, int, float]:
    """
    Generate one response with chat completion.
    Returns: text, token_count, seconds
    """
    t0 = time.time()

    try:
        out = llm.create_chat_completion(
            messages=[
                {"role": "user", "content": str(prompt)}
            ],
            max_tokens=GEN_MAX_TOKENS,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            repeat_penalty=GEN_REPEAT_PEN,
            seed=RANDOM_SEED,
        )

        text = out["choices"][0]["message"]["content"]
        usage = out.get("usage", {}) or {}
        n_tok = int(usage.get("completion_tokens") or len(str(text).split()))

        return text, n_tok, time.time() - t0

    except Exception as e:
        log.warning(
            f"generation failed for {family_hint}: "
            f"{type(e).__name__}: {str(e)[:160]}"
        )
        return "", 0, time.time() - t0


print("✓ Cell 3.2A loaded: response helpers ready")

In [ ]:
# ============================================================================
# Cell 3.2B — Generate or resume responses for ONE model
# ============================================================================

def generate_responses_for(
    scale_or_quant: str,
    key: str,
    prompts_df: pd.DataFrame,
    save_every: int = 5,
) -> Path:
    """
    Generate responses for one model only.

    Resume behavior:
    - Reads existing CSV if present.
    - Skips prompt_ids already completed.
    - Saves every `save_every` prompts.
    - Writes one CSV per model.
    """
    out_path = _per_model_csv(scale_or_quant, key)

    spec = MODEL_REGISTRY[scale_or_quant][key]
    family = spec.get("family", key)
    display = spec["display"]

    prompts_df = prompts_df.copy()
    prompts_df["prompt_id"] = prompts_df["prompt_id"].astype(str)

    existing = _load_existing_response_csv(out_path)
    done = set(existing["prompt_id"].astype(str))

    todo = prompts_df[~prompts_df["prompt_id"].astype(str).isin(done)].copy()

    if len(todo) == 0:
        log.info(f"{scale_or_quant}/{key}: all {len(prompts_df)} prompts already done")
        return out_path

    log.info(
        f"{scale_or_quant}/{key}: "
        f"{len(done)}/{len(prompts_df)} done, {len(todo)} remaining"
    )

    rows = existing.to_dict("records")

    with with_model(scale_or_quant, key) as llm:
        for i, r in enumerate(todo.itertuples(index=False), start=1):
            text, n_tok, dt = _generate_one(llm, r.prompt, family)

            rows.append({
                "prompt_id": str(r.prompt_id),
                "model_scale": scale_or_quant,
                "model_key": key,
                "model_family": family,
                "model_display": display,
                "response": text,
                "n_tokens": int(n_tok),
                "gen_seconds": round(float(dt), 2),
                "quality_ok": bool(_quality_check(text, n_tok)),
                "generated_at": datetime.now(timezone.utc).isoformat(),
            })

            if i % save_every == 0 or i == len(todo):
                ckpt = pd.DataFrame(rows)
                ckpt = ckpt[RESPONSE_COLUMNS]
                ckpt["prompt_id"] = ckpt["prompt_id"].astype(str)
                ckpt = ckpt.drop_duplicates(subset=["prompt_id"], keep="last")
                atomic_write_csv(ckpt, out_path)

                log.info(
                    f"{scale_or_quant}/{key}: checkpoint "
                    f"{len(ckpt)}/{len(prompts_df)} saved "
                    f"({i}/{len(todo)} this run, {dt:.1f}s last)"
                )

    final = pd.DataFrame(rows)
    final = final[RESPONSE_COLUMNS]
    final["prompt_id"] = final["prompt_id"].astype(str)
    final = final.drop_duplicates(subset=["prompt_id"], keep="last")
    atomic_write_csv(final, out_path)

    log.info(f"{scale_or_quant}/{key}: final saved to {out_path}")

    return out_path


print("✓ Cell 3.2B loaded: generate_responses_for(scale_or_quant, key, prompts_df) ready")

In [ ]:
# ============================================================================
# Cell 3.2C — Model download helpers only
# ============================================================================
# This cell does NOT patch FAMILIES.
# This cell does NOT patch MODEL_REGISTRY.
# It only provides safe reusable download utilities.
# Run the per-model download cells after this.

from pathlib import Path
import os, json, time, gc, shutil, subprocess
from datetime import datetime

try:
    from huggingface_hub import hf_hub_download
except Exception:
    !pip -q install -U huggingface_hub
    from huggingface_hub import hf_hub_download

MODEL_MANIFEST_PATH = PATHS["models_drive"] / "model_manifest.json"
PATHS["models_drive"].mkdir(parents=True, exist_ok=True)
PATHS["models_large"].mkdir(parents=True, exist_ok=True)
PATHS["models_small"].mkdir(parents=True, exist_ok=True)
PATHS["models_quant"].mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------
# Download source registry
# ----------------------------------------------------------------------------
# gguf_filename must match one of the filenames in MODEL_REGISTRY candidates.

DOWNLOAD_REGISTRY = {
    "large": {
        "llama": {
            "repo_id": "bullerwins/Meta-Llama-3.1-70B-Instruct-GGUF",
            "source_filename": "Meta-Llama-3.1-70B-Instruct-Q4_K_M.gguf",
            "gguf_filename": "Meta-Llama-3.1-70B-Instruct-Q4_K_M.gguf",
            "min_gb": 35.0,
        },
        "qwen": {
            "repo_id": "Qwen/Qwen2.5-72B-Instruct-GGUF",
            "source_filename": "qwen2.5-72b-instruct-q4_k_m.gguf",
            "gguf_filename": "Qwen2.5-72B-Instruct-Q4_K_M.gguf",
            "min_gb": 35.0,
        },
        "falcon": {
            "repo_id": "TheBloke/falcon-40b-instruct-GGUF",
            "source_filename": "falcon-40b-instruct.Q4_K_M.gguf",
            "gguf_filename": "falcon-40b-instruct.Q4_K_M.gguf",
            "min_gb": 20.0,
        },
        "gemma": {
            "repo_id": "bartowski/gemma-2-27b-it-GGUF",
            "source_filename": "gemma-2-27b-it-Q4_K_M.gguf",
            "gguf_filename": "gemma-2-27b-it-Q4_K_M.gguf",
            "min_gb": 12.0,
        },
        "yi": {
            "repo_id": "bartowski/Yi-1.5-34B-Chat-GGUF",
            "source_filename": "Yi-1.5-34B-Chat-Q4_K_M.gguf",
            "gguf_filename": "Yi-1.5-34B-Chat-Q4_K_M.gguf",
            "min_gb": 15.0,
        },
    },

    "small": {
        "llama": {
            "repo_id": "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
            "source_filename": "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
            "gguf_filename": "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
            "min_gb": 3.0,
        },
        "qwen": {
            "repo_id": "Qwen/Qwen2.5-7B-Instruct-GGUF",
            "source_filename": "qwen2.5-7b-instruct-q4_k_m.gguf",
            "gguf_filename": "Qwen2.5-7B-Instruct-Q4_K_M.gguf",
            "min_gb": 3.0,
        },
        "falcon": {
            "repo_id": "TheBloke/falcon-7b-instruct-GGUF",
            "source_filename": "falcon-7b-instruct.Q4_K_M.gguf",
            "gguf_filename": "falcon-7b-instruct.Q4_K_M.gguf",
            "min_gb": 3.0,
        },
        "gemma": {
            "repo_id": "bartowski/gemma-2-9b-it-GGUF",
            "source_filename": "gemma-2-9b-it-Q4_K_M.gguf",
            "gguf_filename": "gemma-2-9b-it-Q4_K_M.gguf",
            "min_gb": 4.0,
        },
        "yi": {
            "repo_id": "bartowski/Yi-1.5-9B-Chat-GGUF",
            "source_filename": "Yi-1.5-9B-Chat-Q4_K_M.gguf",
            "gguf_filename": "Yi-1.5-9B-Chat-Q4_K_M.gguf",
            "min_gb": 4.0,
        },
    },

    "quant": {
        "llama_q3": {
            "repo_id": "bullerwins/Meta-Llama-3.1-70B-Instruct-GGUF",
            "source_filename": "Meta-Llama-3.1-70B-Instruct-Q3_K_M.gguf",
            "gguf_filename": "Meta-Llama-3.1-70B-Instruct-Q3_K_M.gguf",
            "min_gb": 28.0,
        },
        "qwen_q3": {
            "repo_id": "Qwen/Qwen2.5-72B-Instruct-GGUF",
            "source_filename": "qwen2.5-72b-instruct-q3_k_m.gguf",
            "gguf_filename": "Qwen2.5-72B-Instruct-Q3_K_M.gguf",
            "min_gb": 28.0,
        },
    },
}

MODEL_DIRS = {
    "large": PATHS["models_large"],
    "small": PATHS["models_small"],
    "quant": PATHS["models_quant"],
}

# ----------------------------------------------------------------------------
# Manifest helpers
# ----------------------------------------------------------------------------

def load_model_manifest():
    if MODEL_MANIFEST_PATH.exists():
        try:
            with open(MODEL_MANIFEST_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}


def save_model_manifest(manifest):
    MODEL_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = MODEL_MANIFEST_PATH.with_suffix(".json.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    tmp.replace(MODEL_MANIFEST_PATH)


def gguf_magic_ok(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size < 8:
        return False
    try:
        with open(path, "rb") as f:
            magic = f.read(4)
        return magic == b"GGUF"
    except Exception:
        return False


def file_ok(path, min_gb):
    path = Path(path)
    return path.exists() and gb(path) >= float(min_gb) and gguf_magic_ok(path)


def clear_runtime_temp():
    """
    Clears HF temp/cache and Python memory after a successful model save.
    Does not touch Google Drive model files.
    """
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass

    for p in [
        Path("/content/hf_tmp_downloads"),
        Path("/content/.cache/huggingface"),
        Path("/root/.cache/huggingface"),
    ]:
        if p.exists():
            try:
                shutil.rmtree(p)
                print(f"Cleared runtime cache: {p}")
            except Exception as e:
                print(f"Could not clear {p}: {e}")


def download_one_model(scale, key):
    """
    Download one model permanently into Google Drive.
    Completed files are skipped.
    Partial Drive files are preserved unless they fail GGUF validation.
    """
    if scale not in DOWNLOAD_REGISTRY:
        raise KeyError(f"Unknown download scale: {scale}")

    if key not in DOWNLOAD_REGISTRY[scale]:
        raise KeyError(f"Unknown model key for {scale}: {key}")

    spec = DOWNLOAD_REGISTRY[scale][key]
    out_dir = MODEL_DIRS[scale]
    out_dir.mkdir(parents=True, exist_ok=True)

    final_path = out_dir / spec["gguf_filename"]
    min_gb = float(spec["min_gb"])

    print("\n" + "=" * 110)
    print(f"MODEL:       {scale}/{key}")
    print(f"REPO:        {spec['repo_id']}")
    print(f"HF FILE:     {spec['source_filename']}")
    print(f"DRIVE FILE:  {final_path}")
    print(f"MIN SIZE:    {min_gb:.2f} GB")
    print("=" * 110)

    if file_ok(final_path, min_gb):
        print(f"[SKIP] already permanently stored in Drive: {final_path.name} ({gb(final_path):.2f} GB)")
        return final_path

    if final_path.exists() and not gguf_magic_ok(final_path):
        print(f"[WARN] Existing file is not valid GGUF. Keeping as backup before redownload.")
        backup = final_path.with_suffix(final_path.suffix + f".bad_{int(time.time())}")
        final_path.rename(backup)
        print("Backup:", backup)

    tmp_cache = Path("/content/hf_tmp_downloads") / scale / key
    tmp_cache.mkdir(parents=True, exist_ok=True)

    print("[DOWNLOAD] Starting HuggingFace download...")
    local_download = hf_hub_download(
        repo_id=spec["repo_id"],
        filename=spec["source_filename"],
        local_dir=str(tmp_cache),
        local_dir_use_symlinks=False,
        resume_download=True,
    )

    local_download = Path(local_download)

    if not local_download.exists():
        raise RuntimeError(f"Download failed. File not found: {local_download}")

    if gb(local_download) < min_gb:
        raise RuntimeError(
            f"Downloaded file is too small: {gb(local_download):.2f} GB < {min_gb:.2f} GB"
        )

    if not gguf_magic_ok(local_download):
        raise RuntimeError(f"Downloaded file is not valid GGUF: {local_download}")

    print(f"[SAVE] Copying into Drive: {final_path}")
    tmp_drive = final_path.with_suffix(final_path.suffix + ".tmp")

    if tmp_drive.exists():
        tmp_drive.unlink()

    shutil.copy2(local_download, tmp_drive)
    tmp_drive.replace(final_path)

    if not file_ok(final_path, min_gb):
        raise RuntimeError(f"Drive save failed validation: {final_path}")

    manifest = load_model_manifest()
    manifest[f"{scale}/{key}"] = {
        "status": "downloaded_and_saved_to_drive",
        "repo_id": spec["repo_id"],
        "source_filename": spec["source_filename"],
        "drive_path": str(final_path),
        "size_gb": round(gb(final_path), 3),
        "gguf_magic_ok": True,
        "updated_at": datetime.now().isoformat(),
    }
    save_model_manifest(manifest)

    print(f"[DONE] {scale}/{key}: {final_path.name} ({gb(final_path):.2f} GB)")
    print(f"Manifest: {MODEL_MANIFEST_PATH}")

    clear_runtime_temp()

    return final_path


def verify_models_on_drive(scales=("large", "small", "quant")):
    print("\n" + "=" * 110)
    print("GOOGLE DRIVE MODEL VERIFICATION")
    print("=" * 110)

    all_ok = True

    for scale in scales:
        print(f"\n[{scale.upper()}] {MODEL_DIRS[scale]}")
        for key, spec in DOWNLOAD_REGISTRY[scale].items():
            p = MODEL_DIRS[scale] / spec["gguf_filename"]
            ok = file_ok(p, spec["min_gb"])
            status = "✅" if ok else "❌"
            if not ok:
                all_ok = False
            print(f"{status} {key:10s} {p.name:70s} {gb(p):8.2f} GB")

    print("\nManifest:", MODEL_MANIFEST_PATH)
    print("All OK:", all_ok)
    return all_ok


print("✅ Cell 3.2C loaded: download helpers ready.")
print("Use download_one_model(scale, key), e.g. download_one_model('large', 'llama')")
print("Scales:", list(DOWNLOAD_REGISTRY.keys()))

In [ ]:
# ============================================================================
# Cell 3.2D.0 — Response generation helper cells
# FIXED VERSION
# ============================================================================
# Critical fix:
# The previous version accidentally changed response filenames to:
#   responses_llama.csv
#   responses_qwen.csv
#
# The real completed stable-storage files are:
#   responses_large_llama.csv
#   responses_large_qwen.csv
#   responses_small_llama.csv
#   responses_quant_ablation_llama_q3.csv
#
# This cell restores the correct file naming and prevents completed models
# from being generated again.
# ============================================================================

import pandas as pd
import time
import gc
from pathlib import Path
from datetime import datetime, timezone

# ----------------------------------------------------------------------------
# 1. Response schema
# ----------------------------------------------------------------------------

RESPONSE_COLUMNS = [
    "prompt_id",
    "model_scale",
    "model_key",
    "model_family",
    "model_display",
    "response",
    "n_tokens",
    "gen_seconds",
    "quality_ok",
    "generated_at",
]


GEN_SYSTEM = "You are a helpful assistant. Answer clearly, accurately, and completely."


# ----------------------------------------------------------------------------
# 2. Response folder resolver
# ----------------------------------------------------------------------------

def response_dir_for(scale_or_quant):
    """
    Return the correct Drive response folder.
    """
    if scale_or_quant == "large":
        d = PATHS["responses_large"]

    elif scale_or_quant == "small":
        d = PATHS["responses_small"]

    elif scale_or_quant in ["quant", "quant_ablation"]:
        # In this v11 notebook, quant outputs live here:
        if "responses_quant_ablation" in PATHS:
            d = PATHS["responses_quant_ablation"]
        elif "responses_quant" in PATHS:
            d = PATHS["responses_quant"]
        else:
            d = PATHS["responses"] / "quant_ablation"
            PATHS["responses_quant_ablation"] = d

    else:
        raise KeyError(f"Unknown response group: {scale_or_quant}")

    d = Path(d)
    d.mkdir(parents=True, exist_ok=True)
    return d


def _per_model_csv(scale_or_quant, key):
    """
    Return the exact per-model response CSV path.

    IMPORTANT:
    These names must match the already-generated files:
      large llama        -> responses_large_llama.csv
      small llama        -> responses_small_llama.csv
      quant llama_q3     -> responses_quant_ablation_llama_q3.csv
    """
    d = response_dir_for(scale_or_quant)

    if scale_or_quant == "large":
        return d / f"responses_large_{key}.csv"

    if scale_or_quant == "small":
        return d / f"responses_small_{key}.csv"

    if scale_or_quant in ["quant", "quant_ablation"]:
        return d / f"responses_quant_ablation_{key}.csv"

    raise KeyError(f"Unknown response group: {scale_or_quant}")


# ----------------------------------------------------------------------------
# 3. Basic text utilities
# ----------------------------------------------------------------------------

def clean_text(x):
    if x is None:
        return ""
    return str(x).replace("\x00", "").strip()


def word_count(x):
    return len(clean_text(x).split())


def response_quality(txt, min_words=20):
    txt = clean_text(txt)

    if not txt:
        return False, "empty"

    if word_count(txt) < min_words:
        return False, "too_short"

    low = txt.lower().strip()

    if low.startswith(("i cannot", "i can't")) and word_count(txt) < 60:
        return False, "refusal_or_empty"

    return True, "ok"


def atomic_write_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


# ----------------------------------------------------------------------------
# 4. Safe response CSV loader
# ----------------------------------------------------------------------------

def load_existing_responses(out_path, prompts_df=None):
    """
    Load existing response CSV safely.

    - Keeps only needed columns.
    - Converts prompt_id to string.
    - Drops duplicate prompt_id rows.
    - If prompts_df is provided, keeps only current prompt_ids.
    """
    out_path = Path(out_path)

    if not out_path.exists():
        return pd.DataFrame(columns=RESPONSE_COLUMNS)

    try:
        df = pd.read_csv(out_path)

        missing = [c for c in RESPONSE_COLUMNS if c not in df.columns]
        if missing:
            print(f"[WARN] {out_path.name} missing columns {missing}; treating as empty.")
            return pd.DataFrame(columns=RESPONSE_COLUMNS)

        df = df[RESPONSE_COLUMNS].copy()
        df["prompt_id"] = df["prompt_id"].astype(str)

        # Important: your files have many duplicate checkpoint rows.
        # This reduces them back to one row per prompt.
        before = len(df)
        df = df.drop_duplicates(subset=["prompt_id"], keep="last")
        after = len(df)

        if before != after:
            print(f"[DEDUP] {out_path.name}: {before} rows -> {after} unique prompt_ids")

        if prompts_df is not None:
            valid_ids = set(prompts_df["prompt_id"].astype(str))
            before_filter = len(df)
            df = df[df["prompt_id"].astype(str).isin(valid_ids)].copy()
            after_filter = len(df)

            if before_filter != after_filter:
                print(
                    f"[FILTER] {out_path.name}: kept {after_filter}/{before_filter} "
                    "rows matching current master_prompts"
                )

        return df

    except Exception as e:
        print(f"[WARN] Could not read {out_path}: {type(e).__name__}: {e}")
        return pd.DataFrame(columns=RESPONSE_COLUMNS)


# ----------------------------------------------------------------------------
# 5. Chat completion helper
# ----------------------------------------------------------------------------

def chat_complete(llm, family, system, user, max_tokens=None, temperature=None):
    """
    Uses proper chat completion when available.
    Falls back to raw completion only if chat completion fails.
    """
    if max_tokens is None:
        max_tokens = GEN_MAX_TOKENS

    if temperature is None:
        temperature = GEN_TEMPERATURE

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]

    try:
        out = llm.create_chat_completion(
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=GEN_TOP_P,
            repeat_penalty=GEN_REPEAT_PEN,
            seed=RANDOM_SEED,
        )
        return out["choices"][0]["message"]["content"]

    except Exception:
        prompt = f"{system}\n\nUser:\n{user}\n\nAssistant:"
        out = llm(
            prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=GEN_TOP_P,
            repeat_penalty=GEN_REPEAT_PEN,
            stop=["\nUser:", "\n\nUser:"],
        )
        return out["choices"][0]["text"]


# ----------------------------------------------------------------------------
# 6. Generate or resume responses for one model
# ----------------------------------------------------------------------------

def generate_responses_for(scale_or_quant, key, prompts_df, save_every=5):
    """
    Resume-safe per-model response generation.

    This function now:
    - uses the correct v11 response filename
    - detects already completed prompt_ids
    - deduplicates bloated checkpoint files
    - skips completed models without staging/loading GGUF
    """
    if scale_or_quant not in MODEL_REGISTRY:
        raise KeyError(f"MODEL_REGISTRY has no group: {scale_or_quant}")

    if key not in MODEL_REGISTRY[scale_or_quant]:
        raise KeyError(f"MODEL_REGISTRY['{scale_or_quant}'] has no key: {key}")

    spec = MODEL_REGISTRY[scale_or_quant][key]
    family = spec.get("family", key)
    display = spec.get("display", key)

    prompts = prompts_df.copy()
    prompts["prompt_id"] = prompts["prompt_id"].astype(str)

    out_path = _per_model_csv(scale_or_quant, key)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    existing = load_existing_responses(out_path, prompts_df=prompts)

    done = set(existing["prompt_id"].astype(str)) if len(existing) else set()
    todo = prompts[~prompts["prompt_id"].astype(str).isin(done)].copy()

    print("\n" + "=" * 100)
    print(f"RESPONSE GENERATION CHECK: {scale_or_quant}/{key}")
    print("=" * 100)
    print(f"Expected file: {out_path}")
    print(f"File exists:    {out_path.exists()}")
    print(f"Done:           {len(done)}/{len(prompts)}")
    print(f"Remaining:      {len(todo)}")
    print("=" * 100)

    # Important:
    # If completed, rewrite the deduplicated file and DO NOT load the model.
    if len(todo) == 0:
        print(f"✅ SKIP {scale_or_quant}/{key}: complete. No model staging/loading.")
        atomic_write_csv(existing[RESPONSE_COLUMNS], out_path)
        return out_path

    logger.info(
        f"{scale_or_quant}/{key}: {len(done)}/{len(prompts)} done, "
        f"{len(todo)} remaining"
    )

    rows = existing.to_dict("records")

    with with_model(scale_or_quant, key) as llm:
        for i, r in enumerate(todo.itertuples(index=False), start=1):
            t0 = time.time()

            best_text = ""
            best_ok = False

            user = (
                "Please answer the following user request clearly and completely.\n\n"
                f"User request:\n{r.prompt}"
            )

            for temp in [GEN_TEMPERATURE, 0.3, 0.0]:
                try:
                    txt = clean_text(
                        chat_complete(
                            llm,
                            family,
                            GEN_SYSTEM,
                            user,
                            max_tokens=GEN_MAX_TOKENS,
                            temperature=temp,
                        )
                    )

                    ok, reason = response_quality(txt)

                    if txt and not best_text:
                        best_text = txt
                        best_ok = ok

                    if ok:
                        best_text = txt
                        best_ok = True
                        break

                except Exception as e:
                    logger.error(
                        f"Generation error {scale_or_quant}/{key}/{r.prompt_id}: {e}"
                    )

            n_tok = word_count(best_text)
            dt = time.time() - t0

            rows.append({
                "prompt_id": str(r.prompt_id),
                "model_scale": scale_or_quant,
                "model_key": key,
                "model_family": family,
                "model_display": display,
                "response": clean_text(best_text),
                "n_tokens": int(n_tok),
                "gen_seconds": round(float(dt), 2),
                "quality_ok": bool(best_ok),
                "generated_at": datetime.now(timezone.utc).isoformat(),
            })

            if i % save_every == 0 or i == len(todo):
                ckpt = pd.DataFrame(rows)
                ckpt = ckpt[RESPONSE_COLUMNS]
                ckpt["prompt_id"] = ckpt["prompt_id"].astype(str)
                ckpt = ckpt.drop_duplicates(subset=["prompt_id"], keep="last")
                atomic_write_csv(ckpt, out_path)

                logger.info(
                    f"{scale_or_quant}/{key}: checkpoint "
                    f"{len(ckpt)}/{len(prompts)} saved "
                    f"({i}/{len(todo)} this run, {dt:.1f}s last)"
                )

    final = pd.DataFrame(rows)
    final = final[RESPONSE_COLUMNS]
    final["prompt_id"] = final["prompt_id"].astype(str)
    final = final.drop_duplicates(subset=["prompt_id"], keep="last")
    atomic_write_csv(final, out_path)

    logger.info(f"{scale_or_quant}/{key}: final saved to {out_path}")

    return out_path


# ----------------------------------------------------------------------------
# 7. Response phase report
# ----------------------------------------------------------------------------

def response_phase_report(scale_or_quant, keys, prompts_df):
    """
    Show response completion status for a scale.
    """
    prompts_df = prompts_df.copy()
    prompts_df["prompt_id"] = prompts_df["prompt_id"].astype(str)
    expected_ids = set(prompts_df["prompt_id"])

    print("\n" + "=" * 100)
    print(f"RESPONSE PHASE REPORT: {scale_or_quant}")
    print("=" * 100)

    for key in keys:
        p = _per_model_csv(scale_or_quant, key)

        if not p.exists():
            print(f"❌ {key:10s} missing: {p}")
            continue

        try:
            df = load_existing_responses(p, prompts_df=prompts_df)
            got_ids = set(df["prompt_id"].astype(str))
            missing = expected_ids - got_ids

            qok = (
                float(df["quality_ok"].astype(str).str.lower().isin(["true", "1", "yes"]).mean())
                if len(df)
                else 0.0
            )

            status = "✅ COMPLETE" if len(missing) == 0 else f"⚠️ missing {len(missing)}"

            print(
                f"{status} | {key:10s} | rows={len(df):4d}/{len(prompts_df):4d} "
                f"| quality_ok={qok:.3f} | file={p.name}"
            )

            # Save deduplicated cleaned file back.
            atomic_write_csv(df[RESPONSE_COLUMNS], p)

        except Exception as e:
            print(f"❌ {key:10s} unreadable: {type(e).__name__}: {e}")

    print("=" * 100)


print("✓ Cell 3.2D.0 fixed: correct response filenames restored")

In [ ]:
# ============================================================================
# Cell 3.2D.1 — Generate LARGE LLaMA responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("large", "llama", prompts)

response_phase_report("large", ["llama"], prompts)

print("✓ Large LLaMA response generation finished")

In [ ]:
# ============================================================================
# Cell 3.2D.2 — Generate LARGE Qwen responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("large", "qwen", prompts)

response_phase_report("large", ["qwen"], prompts)

print("✓ Large Qwen response generation finished")

In [ ]:
# ============================================================================
# Cell 3.2D.3 — Generate LARGE Falcon responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("large", "falcon", prompts)

response_phase_report("large", ["falcon"], prompts)

print("✓ Large Falcon response generation finished")

In [ ]:



# ============================================================================
# Cell 3.2D.4 — Generate LARGE Gemma responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("large", "gemma", prompts)

response_phase_report("large", ["gemma"], prompts)

print("✓ Large Gemma response generation finished")

In [ ]:
# ============================================================================
# Cell 3.2D.5 — Generate LARGE Yi responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("large", "yi", prompts)

response_phase_report("large", ["yi"], prompts)

print("✓ Large Yi response generation finished")

In [ ]:


# ============================================================================
# Cell 3.2D.6 — Mark LARGE response phase complete
# Fully self-contained version
# ============================================================================

from pathlib import Path
import json
import pandas as pd
from datetime import datetime, timezone

# ----------------------------------------------------------------------------
# Define sentinel paths safely
# ----------------------------------------------------------------------------

if "sentinels" not in PATHS:
    if "cache" in PATHS:
        PATHS["sentinels"] = Path(PATHS["cache"]) / "sentinels"
    else:
        PATHS["sentinels"] = Path("/content/drive/MyDrive/tribal_pref_v11/cache/sentinels")

PATHS["sentinels"].mkdir(parents=True, exist_ok=True)

LARGE_SENTINEL = PATHS["sentinels"] / "responses_large.done"


# ----------------------------------------------------------------------------
# Helper: mark sentinel if all response files are complete
# ----------------------------------------------------------------------------

def mark_sentinel_if_done(scale_or_quant, keys, prompts_df, sentinel_path):
    """
    Create a sentinel file only if every model in `keys` has a complete
    per-model response CSV for all prompt_ids in prompts_df.
    """
    prompts_df = prompts_df.copy()
    prompts_df["prompt_id"] = prompts_df["prompt_id"].astype(str)

    expected_ids = set(prompts_df["prompt_id"].astype(str))
    sentinel_path = Path(sentinel_path)
    sentinel_path.parent.mkdir(parents=True, exist_ok=True)

    report = {
        "scale_or_quant": scale_or_quant,
        "expected_prompts": len(expected_ids),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "models": {},
        "complete": True,
    }

    for key in keys:
        p = _per_model_csv(scale_or_quant, key)

        model_info = {
            "path": str(p),
            "exists": p.exists(),
            "rows": 0,
            "unique_prompt_ids": 0,
            "missing_prompt_ids": None,
            "complete": False,
        }

        if not p.exists():
            report["complete"] = False
            model_info["missing_prompt_ids"] = len(expected_ids)
            report["models"][key] = model_info
            continue

        try:
            df = pd.read_csv(p)

            if "prompt_id" not in df.columns:
                report["complete"] = False
                model_info["missing_prompt_ids"] = len(expected_ids)
                model_info["error"] = "missing prompt_id column"
                report["models"][key] = model_info
                continue

            df["prompt_id"] = df["prompt_id"].astype(str)
            df = df.drop_duplicates(subset=["prompt_id"], keep="last")

            got_ids = set(df["prompt_id"].astype(str))
            missing = expected_ids - got_ids

            model_info["rows"] = int(len(df))
            model_info["unique_prompt_ids"] = int(len(got_ids))
            model_info["missing_prompt_ids"] = int(len(missing))
            model_info["complete"] = len(missing) == 0

            if len(missing) != 0:
                report["complete"] = False

        except Exception as e:
            report["complete"] = False
            model_info["error"] = f"{type(e).__name__}: {e}"

        report["models"][key] = model_info

    if report["complete"]:
        with open(sentinel_path, "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2)

        print(f"✅ Sentinel created: {sentinel_path}")
        return True

    print("❌ Sentinel not created because at least one response file is incomplete.")
    print(json.dumps(report, indent=2))
    return False


# ----------------------------------------------------------------------------
# Run large phase report and create sentinel
# ----------------------------------------------------------------------------

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

response_phase_report("large", list(FAMILIES), prompts)

mark_sentinel_if_done(
    "large",
    list(FAMILIES),
    prompts,
    LARGE_SENTINEL,
)

print("\nLarge sentinel:")
print("✅ exists" if LARGE_SENTINEL.exists() else "❌ missing")
print(LARGE_SENTINEL)

In [ ]:
# ============================================================================
# Cell 3.2E.1 — Generate SMALL LLaMA responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("small", "llama", prompts)

response_phase_report("small", ["llama"], prompts)

print("✓ Small LLaMA response generation finished")

In [ ]:
# ============================================================================
# Cell 3.2E.2 — Generate SMALL Qwen responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("small", "qwen", prompts)

response_phase_report("small", ["qwen"], prompts)

print("✓ Small Qwen response generation finished")

In [ ]:
# ============================================================================
# Cell 3.2E.3 — Generate SMALL Falcon responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("small", "falcon", prompts)

response_phase_report("small", ["falcon"], prompts)

print("✓ Small Falcon response generation finished")

In [ ]:
# ============================================================================
# Cell 3.2E.4 — Generate SMALL Gemma responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("small", "gemma", prompts)

response_phase_report("small", ["gemma"], prompts)

print("✓ Small Gemma response generation finished")

In [ ]:
# ============================================================================
# Cell 3.2E.5 — Generate SMALL Yi responses only
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

generate_responses_for("small", "yi", prompts)

response_phase_report("small", ["yi"], prompts)

print("✓ Small Yi response generation finished")

In [ ]:
# ============================================================================
# Cell 3.2E.6 — Mark SMALL response phase complete
# Fully self-contained version
# ============================================================================

from pathlib import Path
import json
import pandas as pd
from datetime import datetime, timezone

# ----------------------------------------------------------------------------
# Define sentinel paths safely
# ----------------------------------------------------------------------------

if "sentinels" not in PATHS:
    if "cache" in PATHS:
        PATHS["sentinels"] = Path(PATHS["cache"]) / "sentinels"
    else:
        PATHS["sentinels"] = Path("/content/drive/MyDrive/tribal_pref_v11/cache/sentinels")

PATHS["sentinels"].mkdir(parents=True, exist_ok=True)

SMALL_SENTINEL = PATHS["sentinels"] / "responses_small.done"


# ----------------------------------------------------------------------------
# Helper: define mark_sentinel_if_done if missing
# ----------------------------------------------------------------------------

if "mark_sentinel_if_done" not in globals():

    def mark_sentinel_if_done(scale_or_quant, keys, prompts_df, sentinel_path):
        """
        Create a sentinel file only if every model in `keys` has a complete
        per-model response CSV for all prompt_ids in prompts_df.
        """
        prompts_df = prompts_df.copy()
        prompts_df["prompt_id"] = prompts_df["prompt_id"].astype(str)

        expected_ids = set(prompts_df["prompt_id"].astype(str))
        sentinel_path = Path(sentinel_path)
        sentinel_path.parent.mkdir(parents=True, exist_ok=True)

        report = {
            "scale_or_quant": scale_or_quant,
            "expected_prompts": len(expected_ids),
            "created_at": datetime.now(timezone.utc).isoformat(),
            "models": {},
            "complete": True,
        }

        for key in keys:
            p = _per_model_csv(scale_or_quant, key)

            model_info = {
                "path": str(p),
                "exists": p.exists(),
                "rows": 0,
                "unique_prompt_ids": 0,
                "missing_prompt_ids": None,
                "complete": False,
            }

            if not p.exists():
                report["complete"] = False
                model_info["missing_prompt_ids"] = len(expected_ids)
                report["models"][key] = model_info
                continue

            try:
                df = pd.read_csv(p)

                if "prompt_id" not in df.columns:
                    report["complete"] = False
                    model_info["missing_prompt_ids"] = len(expected_ids)
                    model_info["error"] = "missing prompt_id column"
                    report["models"][key] = model_info
                    continue

                df["prompt_id"] = df["prompt_id"].astype(str)
                df = df.drop_duplicates(subset=["prompt_id"], keep="last")

                got_ids = set(df["prompt_id"].astype(str))
                missing = expected_ids - got_ids

                model_info["rows"] = int(len(df))
                model_info["unique_prompt_ids"] = int(len(got_ids))
                model_info["missing_prompt_ids"] = int(len(missing))
                model_info["complete"] = len(missing) == 0

                if len(missing) != 0:
                    report["complete"] = False

            except Exception as e:
                report["complete"] = False
                model_info["error"] = f"{type(e).__name__}: {e}"

            report["models"][key] = model_info

        if report["complete"]:
            with open(sentinel_path, "w", encoding="utf-8") as f:
                json.dump(report, f, indent=2)

            print(f"✅ Sentinel created: {sentinel_path}")
            return True

        print("❌ Sentinel not created because at least one response file is incomplete.")
        print(json.dumps(report, indent=2))
        return False


# ----------------------------------------------------------------------------
# Run small phase report and create sentinel
# ----------------------------------------------------------------------------

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

response_phase_report("small", list(FAMILIES), prompts)

mark_sentinel_if_done(
    "small",
    list(FAMILIES),
    prompts,
    SMALL_SENTINEL,
)

print("\nSmall sentinel:")
print("✅ exists" if SMALL_SENTINEL.exists() else "❌ missing")
print(SMALL_SENTINEL)

In [ ]:
# ============================================================================
# Cell 3.2F.0 — Locked QUANT ablation prompt subset
# ============================================================================
# This fixes the problem where build_quant_ablation_prompt_subset(n=30)
# creates a different subset each time and makes completed quant responses
# look incomplete.
#
# Rule:
# 1. If a locked subset already exists, load it.
# 2. If quant response files already exist, reconstruct the subset from them.
# 3. Otherwise, build once, save, and reuse forever.
# ============================================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timezone

# ----------------------------------------------------------------------------
# Define quant paths safely
# ----------------------------------------------------------------------------

if "quant_ablation" not in PATHS:
    if "cache" in PATHS:
        PATHS["quant_ablation"] = Path(PATHS["cache"]) / "quant_ablation"
    else:
        PATHS["quant_ablation"] = Path("/content/drive/MyDrive/tribal_pref_v11/cache/quant_ablation")

PATHS["quant_ablation"].mkdir(parents=True, exist_ok=True)

FILES["quant_ablation_prompts"] = PATHS["quant_ablation"] / "quant_ablation_prompts_locked.csv"
FILES["quant_ablation_prompts_manifest"] = PATHS["quant_ablation"] / "quant_ablation_prompts_manifest.json"


# ----------------------------------------------------------------------------
# Helper: response path for quant files
# ----------------------------------------------------------------------------

def quant_response_path(key):
    return _per_model_csv("quant_ablation", key)


# ----------------------------------------------------------------------------
# Helper: build locked subset from existing response CSVs
# ----------------------------------------------------------------------------

def reconstruct_quant_subset_from_existing_responses():
    """
    If quant responses already exist, use their prompt_ids as the authoritative
    locked quant subset. This prevents rerunning completed quant ablation.
    """
    master = pd.read_csv(FILES["master_prompts"])
    master["prompt_id"] = master["prompt_id"].astype(str)

    existing_ids = []

    for key in ["llama_q3", "qwen_q3"]:
        p = quant_response_path(key)

        if not p.exists():
            continue

        try:
            df = pd.read_csv(p)

            if "prompt_id" not in df.columns:
                continue

            df["prompt_id"] = df["prompt_id"].astype(str)
            ids = df["prompt_id"].dropna().astype(str).tolist()

            print(f"Found existing quant responses for {key}: {len(ids)} rows")
            existing_ids.extend(ids)

        except Exception as e:
            print(f"[WARN] Could not read {p}: {type(e).__name__}: {e}")

    existing_ids = list(dict.fromkeys(existing_ids))

    if len(existing_ids) == 0:
        return None

    subset = master[master["prompt_id"].astype(str).isin(existing_ids)].copy()

    if len(subset) == 0:
        print("[WARN] Existing quant response prompt_ids do not match master_prompts.")
        return None

    # Keep the order from existing response files
    order = {pid: i for i, pid in enumerate(existing_ids)}
    subset["_order"] = subset["prompt_id"].map(order)
    subset = subset.sort_values("_order").drop(columns=["_order"]).reset_index(drop=True)

    print(f"Reconstructed locked quant subset from existing responses: {len(subset)} prompts")

    return subset


# ----------------------------------------------------------------------------
# Main function: get locked quant subset
# ----------------------------------------------------------------------------

def get_locked_quant_ablation_prompts(n=30, force_rebuild=False):
    """
    Always returns the same quant ablation prompt subset.

    Do NOT call build_quant_ablation_prompt_subset directly in later cells.
    Use this function only.
    """
    locked_path = FILES["quant_ablation_prompts"]

    if locked_path.exists() and not force_rebuild:
        subset = pd.read_csv(locked_path)
        subset["prompt_id"] = subset["prompt_id"].astype(str)

        print(f"✅ Loaded locked quant ablation subset: {locked_path}")
        print(f"Prompts: {len(subset)}")

        print("\nsource")
        print(subset["source"].value_counts())

        print("\ncategory")
        print(subset["category"].value_counts())

        return subset

    # If responses already exist, use them to lock the subset.
    subset = reconstruct_quant_subset_from_existing_responses()

    if subset is None or len(subset) < 5:
        print("No usable existing quant subset found. Building a new one once.")
        subset = build_quant_ablation_prompt_subset(n=n)
        subset["prompt_id"] = subset["prompt_id"].astype(str)
    else:
        subset["prompt_id"] = subset["prompt_id"].astype(str)

    # If more than n because both files had extra rows, keep first n in existing order.
    if len(subset) > n:
        subset = subset.head(n).copy()

    subset.to_csv(locked_path, index=False)

    manifest = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "path": str(locked_path),
        "n": int(len(subset)),
        "prompt_ids": subset["prompt_id"].astype(str).tolist(),
        "source_counts": subset["source"].value_counts().to_dict() if "source" in subset.columns else {},
        "category_counts": subset["category"].value_counts().to_dict() if "category" in subset.columns else {},
    }

    with open(FILES["quant_ablation_prompts_manifest"], "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print(f"✅ Locked quant ablation subset saved: {locked_path}")
    print(f"Prompts: {len(subset)}")

    print("\nsource")
    print(subset["source"].value_counts())

    print("\ncategory")
    print(subset["category"].value_counts())

    return subset


# ----------------------------------------------------------------------------
# Load or create locked subset now
# ----------------------------------------------------------------------------

quant_prompts = get_locked_quant_ablation_prompts(n=30, force_rebuild=False)
quant_prompts["prompt_id"] = quant_prompts["prompt_id"].astype(str)

print("\n✅ Quant ablation subset is now locked.")
print(FILES["quant_ablation_prompts"])

In [ ]:
# ============================================================================
# Cell 3.2F.0B — Patch model staging to show copy progress
# Put this AFTER Cell 3.2F.0 and BEFORE rerunning quant generation.
# ============================================================================

from pathlib import Path
import shutil
import time
import os

def copy_with_progress(src, dst, chunk_size=256 * 1024 * 1024):
    """
    Copy large GGUF file with visible progress.
    chunk_size default = 256MB.
    """
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    total = src.stat().st_size
    copied = 0
    t0 = time.time()

    tmp = dst.with_suffix(dst.suffix + ".tmp")

    if tmp.exists():
        existing = tmp.stat().st_size
        print(f"Removing old partial temp copy: {tmp} ({existing / (1024**3):.2f} GB)")
        tmp.unlink()

    print("=" * 80)
    print("STAGING MODEL TO LOCAL SSD")
    print("=" * 80)
    print("From:", src)
    print("To:  ", dst)
    print("Size:", round(total / (1024**3), 2), "GB")
    print("=" * 80)

    with open(src, "rb") as fsrc, open(tmp, "wb") as fdst:
        while True:
            chunk = fsrc.read(chunk_size)
            if not chunk:
                break

            fdst.write(chunk)
            copied += len(chunk)

            elapsed = max(time.time() - t0, 1e-6)
            speed = copied / elapsed / (1024**2)
            pct = copied / total * 100
            copied_gb = copied / (1024**3)
            total_gb = total / (1024**3)

            print(
                f"  copied {copied_gb:.2f}/{total_gb:.2f} GB "
                f"({pct:.1f}%) | {speed:.1f} MB/s",
                flush=True
            )

    tmp.replace(dst)

    if dst.stat().st_size != total:
        raise RuntimeError(
            f"Copy failed. Expected {total}, got {dst.stat().st_size}"
        )

    print("=" * 80)
    print(f"✅ Staging complete in {(time.time() - t0) / 60:.1f} minutes")
    print("=" * 80)

    return dst


def _resolve_model_path(spec, scale_or_quant, key):
    """
    Resolve GGUF model path with visible staging progress.

    large           -> models/large
    small           -> models/small
    quant_ablation  -> models/quant
    """

    if scale_or_quant == "large":
        drive_dir = PATHS["models_large"]
        local_dir = LOCAL_STAGE / "models" / "large"

    elif scale_or_quant == "small":
        drive_dir = PATHS["models_small"]
        local_dir = LOCAL_STAGE / "models" / "small"

    elif scale_or_quant == "quant_ablation":
        drive_dir = PATHS["models_quant"]
        local_dir = LOCAL_STAGE / "models" / "quant"

    else:
        raise KeyError(f"Unknown model group: {scale_or_quant}")

    drive_dir = Path(drive_dir)
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)

    candidates = spec.get("candidates", [])
    tried = []

    for filename in candidates:
        drive_path = drive_dir / filename
        local_path = local_dir / filename
        tried.append(str(drive_path))

        if drive_path.exists() and drive_path.stat().st_size > 0:
            drive_size = drive_path.stat().st_size

            if local_path.exists() and local_path.stat().st_size == drive_size:
                print(f"✅ Using already staged model: {local_path}")
                print(f"   size: {local_path.stat().st_size / (1024**3):.2f} GB")
                return local_path

            if local_path.exists() and local_path.stat().st_size != drive_size:
                print(f"Removing incomplete local copy: {local_path}")
                local_path.unlink()

            return copy_with_progress(drive_path, local_path)

    raise FileNotFoundError(
        f"No GGUF found for {scale_or_quant}/{key}.\n"
        f"Expected folder: {drive_dir}\n"
        f"Tried:\n  " + "\n  ".join(tried)
    )

print("✅ Model staging now shows progress.")

In [ ]:





# ============================================================================
# Cell 3.2F.1 — Build QUANT ablation prompt subset
# ============================================================================

def build_quant_ablation_prompt_subset(n=30):
    """
    Build a fixed, reproducible prompt subset for quantization ablation.

    This should be smaller than the main 200-prompt set because quant ablation
    is a robustness check, not the main experiment.
    """
    prompts = pd.read_csv(FILES["master_prompts"])
    prompts["prompt_id"] = prompts["prompt_id"].astype(str)

    n = min(int(n), len(prompts))

    subset = prompts.sample(
        n=n,
        random_state=RANDOM_SEED + 7,
    ).reset_index(drop=True)

    print(f"Quant ablation prompt subset: {len(subset)} prompts")
    print(subset["source"].value_counts(dropna=False))
    print(subset["category"].value_counts(dropna=False))

    return subset


quant_prompts = build_quant_ablation_prompt_subset(n=30)

print("✓ Quant ablation prompt subset ready")

In [ ]:
# ============================================================================
# COMPATIBILITY PATCH — Missing runtime constants after restart
# Put this before Cell 3.2F.2 / 3.2F.3
# ============================================================================

# Context window
# For quant ablation we do not need 8192. 4096 is faster and enough.
if "N_CTX" not in globals():
    N_CTX = 4096

# Full GPU offload
# -1 means offload all possible layers to GPU.
if "DEFAULT_N_GPU_LAYERS" not in globals():
    DEFAULT_N_GPU_LAYERS = -1

# Batch settings for llama.cpp
if "N_BATCH" not in globals():
    N_BATCH = 1024

if "N_UBATCH" not in globals():
    N_UBATCH = 512

# Threads are mostly fallback CPU-side helpers.
if "N_THREADS" not in globals():
    import os
    N_THREADS = max(1, min(16, os.cpu_count() or 8))

# Make sure quant models use full GPU offload too.
if "MODEL_REGISTRY" in globals() and "quant_ablation" in MODEL_REGISTRY:
    for k in MODEL_REGISTRY["quant_ablation"]:
        MODEL_REGISTRY["quant_ablation"][k]["n_gpu_layers"] = -1
        MODEL_REGISTRY["quant_ablation"][k]["n_ctx"] = 4096

print("✅ Compatibility constants ready")
print("N_CTX:", N_CTX)
print("DEFAULT_N_GPU_LAYERS:", DEFAULT_N_GPU_LAYERS)
print("N_BATCH:", N_BATCH)
print("N_UBATCH:", N_UBATCH)
print("N_THREADS:", N_THREADS)

if "MODEL_REGISTRY" in globals() and "quant_ablation" in MODEL_REGISTRY:
    print("\nQuant registry:")
    for k, v in MODEL_REGISTRY["quant_ablation"].items():
        print(k, "n_ctx=", v.get("n_ctx"), "n_gpu_layers=", v.get("n_gpu_layers"))

In [ ]:
# ============================================================================
# Cell 3.2F.2 — Resume QUANT LLaMA Q3 responses only
# Shows progress + GPU monitor
# ============================================================================

import time
import threading
import subprocess
from pathlib import Path
import pandas as pd

# ----------------------------------------------------------------------------
# Safety checks
# ----------------------------------------------------------------------------

if "get_locked_quant_ablation_prompts" not in globals():
    raise RuntimeError(
        "get_locked_quant_ablation_prompts() is not defined. "
        "Run Cell 3.2F.0 first."
    )

if "generate_responses_for" not in globals():
    raise RuntimeError(
        "generate_responses_for() is not defined. "
        "Run the response-generation utility cell first."
    )

if "MODEL_REGISTRY" not in globals():
    raise RuntimeError("MODEL_REGISTRY is not defined. Run the config/model registry cells first.")

if "quant_ablation" not in MODEL_REGISTRY:
    raise RuntimeError("MODEL_REGISTRY['quant_ablation'] is missing. Run Cell 3.2F.0 first.")

if "llama_q3" not in MODEL_REGISTRY["quant_ablation"]:
    raise RuntimeError(
        "MODEL_REGISTRY['quant_ablation']['llama_q3'] is missing. "
        "Run the updated Cell 3.2F.0 first."
    )


# ----------------------------------------------------------------------------
# GPU monitor
# ----------------------------------------------------------------------------

_GPU_MONITOR_STOP = False

def _gpu_monitor(interval=30):
    """
    Prints GPU memory/utilization every `interval` seconds while generation runs.
    """
    global _GPU_MONITOR_STOP

    while not _GPU_MONITOR_STOP:
        try:
            out = subprocess.check_output(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu",
                    "--format=csv,noheader,nounits",
                ],
                text=True,
                stderr=subprocess.STDOUT,
            ).strip()

            parts = [x.strip() for x in out.split(",")]
            if len(parts) >= 5:
                util, mem_used, mem_total, power, temp = parts[:5]
                print(
                    f"[GPU] util={util}% | mem={mem_used}/{mem_total} MiB | "
                    f"power={power} W | temp={temp}°C"
                )
            else:
                print("[GPU]", out)

        except Exception as e:
            print(f"[GPU monitor warning] {type(e).__name__}: {e}")

        time.sleep(interval)


# ----------------------------------------------------------------------------
# Load locked quant prompts
# ----------------------------------------------------------------------------

quant_prompts = get_locked_quant_ablation_prompts(n=30, force_rebuild=False)
quant_prompts["prompt_id"] = quant_prompts["prompt_id"].astype(str)

print("\n" + "=" * 100)
print("QUANT ABLATION: LLaMA Q3")
print("=" * 100)
print(f"Locked quant prompts: {len(quant_prompts)}")
print("This cell resumes from the existing CSV. It does NOT restart from zero.")
print("It does NOT rebuild the quant prompt subset.")


# ----------------------------------------------------------------------------
# Show current response status before starting
# ----------------------------------------------------------------------------

print("\nBefore generation:")
response_phase_report("quant_ablation", ["llama_q3"], quant_prompts)


# ----------------------------------------------------------------------------
# Show current GPU status before loading
# ----------------------------------------------------------------------------

print("\nGPU before generation:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print(f"Could not run nvidia-smi: {type(e).__name__}: {e}")


# ----------------------------------------------------------------------------
# Run generation with live GPU monitor
# ----------------------------------------------------------------------------

print("\nStarting LLaMA Q3 generation/resume...")
print("If the model is already staged locally, staging should be skipped.")
print("If CUDA llama-cpp is working, GPU memory should jump after model loading.\n")

_GPU_MONITOR_STOP = False
monitor_thread = threading.Thread(target=_gpu_monitor, kwargs={"interval": 30}, daemon=True)
monitor_thread.start()

t0 = time.time()

try:
    generate_responses_for(
        "quant_ablation",
        "llama_q3",
        quant_prompts,
    )

finally:
    _GPU_MONITOR_STOP = True
    time.sleep(2)

elapsed = time.time() - t0


# ----------------------------------------------------------------------------
# Final report
# ----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("LLaMA Q3 GENERATION FINISHED OR STOPPED")
print("=" * 100)
print(f"Elapsed time: {elapsed/60:.2f} minutes")

response_phase_report("quant_ablation", ["llama_q3"], quant_prompts)

print("\nGPU after generation:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print(f"Could not run nvidia-smi: {type(e).__name__}: {e}")

print("\n✓ Cell 3.2F.2 finished")

In [ ]:
# ============================================================================
# Cell 3.2F.3 — Resume QUANT Qwen Q3 responses only
# Shows progress + GPU monitor
# ============================================================================

import time
import threading
import subprocess
from pathlib import Path
import pandas as pd

# ----------------------------------------------------------------------------
# Compatibility constants
# ----------------------------------------------------------------------------

if "N_CTX" not in globals():
    N_CTX = 4096

if "DEFAULT_N_GPU_LAYERS" not in globals():
    DEFAULT_N_GPU_LAYERS = -1

if "N_BATCH" not in globals():
    N_BATCH = 1024

if "N_UBATCH" not in globals():
    N_UBATCH = 512

if "N_THREADS" not in globals():
    import os
    N_THREADS = max(1, min(16, os.cpu_count() or 8))

# ----------------------------------------------------------------------------
# Safety checks
# ----------------------------------------------------------------------------

if "get_locked_quant_ablation_prompts" not in globals():
    raise RuntimeError(
        "get_locked_quant_ablation_prompts() is not defined. "
        "Run Cell 3.2F.0 first."
    )

if "generate_responses_for" not in globals():
    raise RuntimeError(
        "generate_responses_for() is not defined. "
        "Run the response-generation utility cell first."
    )

if "MODEL_REGISTRY" not in globals():
    raise RuntimeError("MODEL_REGISTRY is not defined. Run the config/model registry cells first.")

if "quant_ablation" not in MODEL_REGISTRY:
    raise RuntimeError("MODEL_REGISTRY['quant_ablation'] is missing. Run Cell 3.2F.0 first.")

if "qwen_q3" not in MODEL_REGISTRY["quant_ablation"]:
    raise RuntimeError(
        "MODEL_REGISTRY['quant_ablation']['qwen_q3'] is missing. "
        "Run the updated Cell 3.2F.0 first."
    )

# Force fast GPU settings for this quant model
MODEL_REGISTRY["quant_ablation"]["qwen_q3"]["n_gpu_layers"] = -1
MODEL_REGISTRY["quant_ablation"]["qwen_q3"]["n_ctx"] = 4096


# ----------------------------------------------------------------------------
# GPU monitor
# ----------------------------------------------------------------------------

_GPU_MONITOR_STOP = False

def _gpu_monitor(interval=30):
    """
    Prints GPU memory/utilization every `interval` seconds while generation runs.
    """
    global _GPU_MONITOR_STOP

    while not _GPU_MONITOR_STOP:
        try:
            out = subprocess.check_output(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu",
                    "--format=csv,noheader,nounits",
                ],
                text=True,
                stderr=subprocess.STDOUT,
            ).strip()

            parts = [x.strip() for x in out.split(",")]
            if len(parts) >= 5:
                util, mem_used, mem_total, power, temp = parts[:5]
                print(
                    f"[GPU] util={util}% | mem={mem_used}/{mem_total} MiB | "
                    f"power={power} W | temp={temp}°C"
                )
            else:
                print("[GPU]", out)

        except Exception as e:
            print(f"[GPU monitor warning] {type(e).__name__}: {e}")

        time.sleep(interval)


# ----------------------------------------------------------------------------
# Load locked quant prompts
# ----------------------------------------------------------------------------

quant_prompts = get_locked_quant_ablation_prompts(n=30, force_rebuild=False)
quant_prompts["prompt_id"] = quant_prompts["prompt_id"].astype(str)

print("\n" + "=" * 100)
print("QUANT ABLATION: Qwen Q3")
print("=" * 100)
print(f"Locked quant prompts: {len(quant_prompts)}")
print("This cell resumes from the existing CSV. It does NOT restart from zero.")
print("It does NOT rebuild the quant prompt subset.")


# ----------------------------------------------------------------------------
# Show current response status before starting
# ----------------------------------------------------------------------------

print("\nBefore generation:")
response_phase_report("quant_ablation", ["qwen_q3"], quant_prompts)


# ----------------------------------------------------------------------------
# Show current GPU status before loading
# ----------------------------------------------------------------------------

print("\nGPU before generation:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print(f"Could not run nvidia-smi: {type(e).__name__}: {e}")


# ----------------------------------------------------------------------------
# Run generation with live GPU monitor
# ----------------------------------------------------------------------------

print("\nStarting Qwen Q3 generation/resume...")
print("If the model is already staged locally, staging should be skipped.")
print("If CUDA llama-cpp is working, GPU memory should jump after model loading.\n")

_GPU_MONITOR_STOP = False
monitor_thread = threading.Thread(target=_gpu_monitor, kwargs={"interval": 30}, daemon=True)
monitor_thread.start()

t0 = time.time()

try:
    generate_responses_for(
        "quant_ablation",
        "qwen_q3",
        quant_prompts,
    )

finally:
    _GPU_MONITOR_STOP = True
    time.sleep(2)

elapsed = time.time() - t0


# ----------------------------------------------------------------------------
# Final report
# ----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("Qwen Q3 GENERATION FINISHED OR STOPPED")
print("=" * 100)
print(f"Elapsed time: {elapsed/60:.2f} minutes")

response_phase_report("quant_ablation", ["qwen_q3"], quant_prompts)

print("\nGPU after generation:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print(f"Could not run nvidia-smi: {type(e).__name__}: {e}")

print("\n✓ Cell 3.2F.3 finished")

In [ ]:
# ============================================================================
# Cell 3.2F.4 — Mark QUANT ablation response phase complete
# Uses LOCKED quant prompt subset
# Fully self-contained version
# ============================================================================

from pathlib import Path
import json
import pandas as pd
from datetime import datetime, timezone

# ----------------------------------------------------------------------------
# Define quant sentinel path safely
# ----------------------------------------------------------------------------

if "sentinels" not in PATHS:
    if "cache" in PATHS:
        PATHS["sentinels"] = Path(PATHS["cache"]) / "sentinels"
    else:
        PATHS["sentinels"] = Path("/content/drive/MyDrive/tribal_pref_v11/cache/sentinels")

PATHS["sentinels"].mkdir(parents=True, exist_ok=True)

QUANT_SENTINEL = PATHS["sentinels"] / "responses_quant_ablation.done"


# ----------------------------------------------------------------------------
# Define atomic text writer if missing
# ----------------------------------------------------------------------------

if "atomic_write_text" not in globals():

    def atomic_write_text(path, text):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        tmp = path.with_suffix(path.suffix + ".tmp")
        with open(tmp, "w", encoding="utf-8") as f:
            f.write(str(text))
        tmp.replace(path)


# ----------------------------------------------------------------------------
# Define response completion checker
# ----------------------------------------------------------------------------

def _response_file_complete(scale_or_quant, key, prompts_df):
    """
    Check whether a response CSV exists and contains one row for every prompt_id
    in the locked prompt subset.

    This uses _per_model_csv(), so it respects the fixed v11 filenames:
      responses_quant_ablation_llama_q3.csv
      responses_quant_ablation_qwen_q3.csv
    """
    prompts_df = prompts_df.copy()
    prompts_df["prompt_id"] = prompts_df["prompt_id"].astype(str)

    expected_ids = set(prompts_df["prompt_id"].astype(str))
    p = _per_model_csv(scale_or_quant, key)

    if not p.exists():
        print(f"❌ {key}: missing file: {p}")
        return False

    try:
        df = pd.read_csv(p)

        if "prompt_id" not in df.columns:
            print(f"❌ {key}: missing prompt_id column")
            return False

        df["prompt_id"] = df["prompt_id"].astype(str)
        df = df.drop_duplicates(subset=["prompt_id"], keep="last")

        got_ids = set(df["prompt_id"].astype(str))
        missing = expected_ids - got_ids
        extra = got_ids - expected_ids

        if len(missing) == 0:
            print(f"✅ {key}: complete {len(got_ids)}/{len(expected_ids)}")
            return True

        print(f"⚠️ {key}: incomplete")
        print(f"   file: {p}")
        print(f"   rows after dedup: {len(df)}")
        print(f"   matched prompt_ids: {len(expected_ids - missing)}/{len(expected_ids)}")
        print(f"   missing: {len(missing)}")
        print(f"   extra rows not in locked subset: {len(extra)}")

        return False

    except Exception as e:
        print(f"❌ {key}: unreadable: {type(e).__name__}: {e}")
        return False


# ----------------------------------------------------------------------------
# Load LOCKED quant subset
# ----------------------------------------------------------------------------
# IMPORTANT:
# Do NOT call build_quant_ablation_prompt_subset(n=30) here.
# That creates a new random/stratified subset and makes completed files look incomplete.

if "get_locked_quant_ablation_prompts" not in globals():
    raise RuntimeError(
        "get_locked_quant_ablation_prompts() is not defined. "
        "Run the updated Cell 3.2F.0 first."
    )

quant_prompts = get_locked_quant_ablation_prompts(n=30, force_rebuild=False)
quant_prompts["prompt_id"] = quant_prompts["prompt_id"].astype(str)

quant_keys = list(MODEL_REGISTRY["quant_ablation"].keys())

print("\nLocked quant ablation prompt subset:")
print(f"Prompts: {len(quant_prompts)}")

if "source" in quant_prompts.columns:
    print("\nsource")
    print(quant_prompts["source"].value_counts())

if "category" in quant_prompts.columns:
    print("\ncategory")
    print(quant_prompts["category"].value_counts())


# ----------------------------------------------------------------------------
# Report quant response status
# ----------------------------------------------------------------------------

response_phase_report("quant_ablation", quant_keys, quant_prompts)


# ----------------------------------------------------------------------------
# Check completion and write sentinel if all done
# ----------------------------------------------------------------------------

completed = [
    key for key in quant_keys
    if _response_file_complete("quant_ablation", key, quant_prompts)
]

if len(completed) == len(quant_keys):
    atomic_write_text(
        QUANT_SENTINEL,
        json.dumps({
            "completed_at": datetime.now(timezone.utc).isoformat(),
            "completed_keys": completed,
            "n_prompts": int(len(quant_prompts)),
            "prompt_ids": quant_prompts["prompt_id"].astype(str).tolist(),
        }, indent=2),
    )

    print("\n✅ Quant ablation sentinel written")

else:
    missing_keys = [k for k in quant_keys if k not in completed]

    print("\n❌ Quant ablation phase incomplete")
    print("Completed:", completed)
    print("Missing:", missing_keys)

    print("\nDo not force rebuild the quant subset.")
    print("Run only the missing quant generation cells:")
    for k in missing_keys:
        print(f"  generate_responses_for('quant_ablation', '{k}', quant_prompts)")


# ----------------------------------------------------------------------------
# Final sentinel status
# ----------------------------------------------------------------------------

print("\nQuant sentinel:")
print("✅ exists" if QUANT_SENTINEL.exists() else "❌ missing")
print(QUANT_SENTINEL)

In [ ]:
# ============================================================================
# Cell 3.2G — Manual single-model generation
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

# ---------------------------------------------------------------------------
# LARGE MODELS
# Uncomment only the one you want.
# ---------------------------------------------------------------------------

# generate_responses_for("large", "llama", prompts)
# generate_responses_for("large", "qwen", prompts)
# generate_responses_for("large", "mixtral", prompts)
# generate_responses_for("large", "gemma", prompts)
# generate_responses_for("large", "yi", prompts)


# ---------------------------------------------------------------------------
# SMALL MODELS
# Uncomment only the one you want.
# ---------------------------------------------------------------------------

# generate_responses_for("small", "llama", prompts)
# generate_responses_for("small", "qwen", prompts)
# generate_responses_for("small", "mixtral", prompts)
# generate_responses_for("small", "gemma", prompts)
# generate_responses_for("small", "yi", prompts)


# ---------------------------------------------------------------------------
# QUANT ABLATION MODELS
# Uncomment only the one you want.
# ---------------------------------------------------------------------------

# quant_prompts = build_quant_ablation_prompt_subset(n=30)
# generate_responses_for("quant_ablation", "llama_q3", quant_prompts)
# generate_responses_for("quant_ablation", "qwen_q3", quant_prompts)


print("✓ Manual generation cell loaded. Uncomment one line and run again.")

In [ ]:



# ============================================================================
# Cell 3.2H — Full response generation status check
# ============================================================================

prompts = pd.read_csv(FILES["master_prompts"])
prompts["prompt_id"] = prompts["prompt_id"].astype(str)

print("\nMAIN PROMPT SET")
print("Prompt count:", len(prompts))
print(prompts["source"].value_counts(dropna=False))
print(prompts["category"].value_counts(dropna=False))

response_phase_report("large", list(FAMILIES), prompts)
response_phase_report("small", list(FAMILIES), prompts)

if "quant_ablation" in MODEL_REGISTRY:
    quant_prompts = build_quant_ablation_prompt_subset(n=30)
    response_phase_report(
        "quant_ablation",
        list(MODEL_REGISTRY["quant_ablation"].keys()),
        quant_prompts,
    )

print("\n" + "=" * 100)
print("SENTINELS")
print("=" * 100)

for name, p in {
    "large": LARGE_SENTINEL,
    "small": SMALL_SENTINEL,
    "quant": QUANT_SENTINEL,
}.items():
    print(f"{name:8s}: {'✅ exists' if p.exists() else '❌ missing'} — {p}")


print("\n" + "=" * 100)
print("RESPONSE FILE LOCATIONS")
print("=" * 100)

for scale in ["large", "small"]:
    print(f"\n[{scale.upper()}]")
    for fam in FAMILIES:
        p = _per_model_csv(scale, fam)
        print(f"{fam:10s}: {p}")

if "quant_ablation" in MODEL_REGISTRY:
    print("\n[QUANT_ABLATION]")
    for key in MODEL_REGISTRY["quant_ablation"].keys():
        p = _per_model_csv("quant_ablation", key)
        print(f"{key:10s}: {p}")

In [ ]:

# ============================================================================
# Cell 3.3 — Compile all responses into one master CSV + diagnostics
# ============================================================================
"""
Cell 3.3 — Concatenate per-model response CSVs into a single master table.

Also produce response_health.csv: per-prompt summary of token counts, refusals,
length variance — used downstream for response-quality controls.
"""
def _step_compile_responses():
    frames = []
    for fam in FAMILIES:
        for scale in ["large", "small"]:
            p = _per_model_csv(scale, fam)
            if p.exists():
                frames.append(pd.read_csv(p))
    for key in MODEL_REGISTRY["quant_ablation"]:
        p = _per_model_csv("quant_ablation", key)
        if p.exists():
            frames.append(pd.read_csv(p))
    if not frames:
        raise RuntimeError("No response CSVs found; run Cell 3.2 first")
    df = pd.concat(frames, ignore_index=True)
    # Dedup: keep most recent per (prompt, model_key)
    df = df.sort_values("generated_at").drop_duplicates(
        subset=["prompt_id", "model_scale", "model_key"], keep="last")
    atomic_write_csv(df, FILES["all_responses"])
    log.info(f"all_responses: {len(df)} rows, "
             f"{df['model_scale'].value_counts().to_dict()}")

    # Per-prompt diagnostics on the LARGE scale (used for trial gating)
    large = df[df["model_scale"] == "large"].copy()
    health = large.groupby("prompt_id").agg(
        n_models=("model_family", "nunique"),
        n_quality_ok=("quality_ok", "sum"),
        mean_tokens=("n_tokens", "mean"),
        std_tokens=("n_tokens", "std"),
        max_tokens=("n_tokens", "max"),
        min_tokens=("n_tokens", "min"),
    ).reset_index()
    health["all_ok"] = health["n_quality_ok"] == health["n_models"]
    atomic_write_csv(health, FILES["response_health"])

run_step([FILES["all_responses"], FILES["response_health"]],
         _step_compile_responses,
         "compile_all_responses",
         validators={FILES["all_responses"]:
                     lambda p: csv_has(p, ["prompt_id", "model_family", "response"], 50)})

ar = pd.read_csv(FILES["all_responses"])
print(f"All responses: {len(ar)}")
print(ar.groupby(["model_scale", "model_family"]).size().unstack(fill_value=0))




In [ ]:
# ============================================================================
# Cell 3.3B — Response artifact scan and family-level quality diagnostics
# ============================================================================
"""
Cell 3.3B — Dedicated response artifact scan.

Purpose:
- Scan all generated responses for generation artifacts before analysis.
- Detect EOS leakage, special-token leakage, empty responses, ultra-short
  responses, repeated templates, and abnormal family-level artifact rates.
- Save both row-level artifact flags and family-level summaries.

Why this matters:
- The paper can now honestly say response quality/artifact filtering was audited.
- Reviewers can inspect the exact filtering diagnostics.
- This cell does not rerun generation or judgments.

Important:
- This is a diagnostic/data-cleaning audit.
- It does not automatically delete responses or change downstream results.
- If severe artifact rates are found, inspect before freezing the paper.
- Cached with force=False.
"""

from pathlib import Path
from typing import Any, Dict, List, Optional
import pandas as pd
import numpy as np
import json
import re
import hashlib

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_csv",
    "atomic_write_json",
    "csv_has",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "analysis_quality" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_quality"] = Path(PATHS["analysis"]) / "quality"
    elif "root" in PATHS:
        PATHS["analysis_quality"] = Path(PATHS["root"]) / "analysis" / "quality"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_quality"] = Path(ROOT_DIR) / "analysis" / "quality"
    else:
        PATHS["analysis_quality"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/quality")

Path(PATHS["analysis_quality"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "response_artifact_flags",
    Path(PATHS["analysis_quality"]) / "response_artifact_flags.csv",
)

FILES.setdefault(
    "response_artifact_family_summary",
    Path(PATHS["analysis_quality"]) / "response_artifact_family_summary.csv",
)

FILES.setdefault(
    "response_artifact_prompt_summary",
    Path(PATHS["analysis_quality"]) / "response_artifact_prompt_summary.csv",
)

FILES.setdefault(
    "response_artifact_diagnostics",
    Path(PATHS["analysis_quality"]) / "response_artifact_diagnostics.json",
)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

SPECIAL_TOKEN_PATTERNS = [
    "</s>",
    "<s>",
    "<|endoftext|>",
    "<|end_of_text|>",
    "<|eot_id|>",
    "<|end|>",
    "<|assistant|>",
    "<|user|>",
    "<|system|>",
    "<|im_start|>",
    "<|im_end|>",
    "[INST]",
    "[/INST]",
    "<<SYS>>",
    "<</SYS>>",
    "<end_of_turn>",
    "<start_of_turn>",
]

BOILERPLATE_PATTERNS = [
    r"\bas an ai language model\b",
    r"\bi cannot assist\b",
    r"\bi can't assist\b",
    r"\bi am unable to\b",
    r"\bi cannot comply\b",
    r"\bi do not have access\b",
    r"\bi don'?t have access\b",
    r"\bi need more context\b",
]

REFUSAL_PATTERNS = [
    r"\bi cannot\b",
    r"\bi can't\b",
    r"\bi am unable\b",
    r"\bi'm unable\b",
    r"\bi will not\b",
    r"\bi won't\b",
    r"\bnot appropriate\b",
    r"\bcannot help with\b",
]


def _33b_clean_family(x: Any) -> str:
    if pd.isna(x):
        return "unknown"

    s = str(x).strip().lower()

    if "llama" in s:
        return "llama"
    if "qwen" in s:
        return "qwen"
    if "gemma" in s:
        return "gemma"
    if "falcon" in s:
        return "falcon"
    if "yi" in s:
        return "yi"

    return s or "unknown"


def _33b_bool(x: Any, default: bool = False) -> bool:
    if pd.isna(x):
        return default
    if isinstance(x, bool):
        return x

    s = str(x).strip().lower()

    if s in {"true", "1", "yes", "y", "ok", "pass", "passed"}:
        return True
    if s in {"false", "0", "no", "n", "fail", "failed"}:
        return False

    return default


def _33b_hash_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()[:16]


def _33b_pick_response_col(df: pd.DataFrame) -> str:
    candidates = [
        "response",
        "text",
        "output",
        "completion",
        "answer",
        "generated_text",
        "model_response",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    raise RuntimeError(
        "Could not find response text column. Expected one of: "
        + ", ".join(candidates)
    )


def _33b_pick_family_col(df: pd.DataFrame) -> str:
    candidates = [
        "model_family",
        "family",
        "candidate_family",
        "response_family",
        "generator_family",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    return ""


def _33b_pick_model_key_col(df: pd.DataFrame) -> str:
    candidates = [
        "model_key",
        "model_name",
        "model_id",
        "checkpoint",
        "model_path",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    return ""


def _33b_detect_artifacts(text: Any) -> Dict[str, Any]:
    s = "" if pd.isna(text) else str(text)
    s_stripped = s.strip()
    s_lower = s.lower()

    words = re.findall(r"\S+", s_stripped)
    n_words = len(words)
    n_chars = len(s_stripped)

    special_hits = [tok for tok in SPECIAL_TOKEN_PATTERNS if tok.lower() in s_lower]

    boilerplate_hits = [
        pat for pat in BOILERPLATE_PATTERNS
        if re.search(pat, s_lower, flags=re.IGNORECASE)
    ]

    refusal_hits = [
        pat for pat in REFUSAL_PATTERNS
        if re.search(pat, s_lower, flags=re.IGNORECASE)
    ]

    # Repetition heuristics.
    lines = [ln.strip() for ln in s_stripped.splitlines() if ln.strip()]
    unique_lines = set(lines)
    repeated_line_rate = 1.0 - (len(unique_lines) / len(lines)) if lines else 0.0

    # Word-level repetition heuristic.
    if n_words >= 20:
        unique_word_rate = len(set(w.lower() for w in words)) / n_words
    else:
        unique_word_rate = 1.0

    # Repeated character / degenerate text heuristic.
    repeated_char_flag = bool(re.search(r"(.)\1{20,}", s_stripped))

    # Prompt/template leakage heuristic.
    role_leak_flag = bool(
        re.search(
            r"(system:|user:|assistant:|### instruction|### response|prompt:|response:)",
            s_lower,
        )
    )

    eos_leak = any(tok in special_hits for tok in ["</s>", "<|endoftext|>", "<|end_of_text|>", "<|eot_id|>", "<end_of_turn>"])

    empty_response = n_chars == 0
    ultra_short_response = n_words < 5
    subthreshold_length = n_words < 20
    long_degenerate_repetition = bool(n_words >= 40 and unique_word_rate < 0.20)
    high_line_repetition = bool(len(lines) >= 4 and repeated_line_rate > 0.50)

    any_special_token_leak = len(special_hits) > 0
    any_boilerplate = len(boilerplate_hits) > 0
    any_refusal = len(refusal_hits) > 0

    artifact_flag = any([
        eos_leak,
        any_special_token_leak,
        empty_response,
        ultra_short_response,
        long_degenerate_repetition,
        high_line_repetition,
        repeated_char_flag,
        role_leak_flag,
    ])

    severe_artifact_flag = any([
        eos_leak,
        empty_response,
        ultra_short_response,
        long_degenerate_repetition,
        repeated_char_flag,
    ])

    return {
        "n_chars": int(n_chars),
        "n_words": int(n_words),
        "text_hash": _33b_hash_text(s_stripped),
        "empty_response": bool(empty_response),
        "ultra_short_response": bool(ultra_short_response),
        "subthreshold_length": bool(subthreshold_length),
        "eos_leak": bool(eos_leak),
        "special_token_leak": bool(any_special_token_leak),
        "special_token_hits": "|".join(special_hits),
        "boilerplate_flag": bool(any_boilerplate),
        "boilerplate_hits": "|".join(boilerplate_hits),
        "refusal_flag": bool(any_refusal),
        "refusal_hits": "|".join(refusal_hits),
        "role_or_template_leak": bool(role_leak_flag),
        "repeated_char_flag": bool(repeated_char_flag),
        "unique_word_rate": float(unique_word_rate),
        "line_repetition_rate": float(repeated_line_rate),
        "long_degenerate_repetition": bool(long_degenerate_repetition),
        "high_line_repetition": bool(high_line_repetition),
        "artifact_flag": bool(artifact_flag),
        "severe_artifact_flag": bool(severe_artifact_flag),
    }


def _33b_rate_table(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    available = [c for c in group_cols if c in df.columns]

    if not available:
        available = ["_all"]

    if "_all" in available and "_all" not in df.columns:
        df = df.copy()
        df["_all"] = "all"

    agg = (
        df.groupby(available, dropna=False)
        .agg(
            n_responses=("response_text", "size"),
            n_prompts=("prompt_id", "nunique") if "prompt_id" in df.columns else ("response_text", "size"),
            mean_words=("n_words", "mean"),
            median_words=("n_words", "median"),
            min_words=("n_words", "min"),
            max_words=("n_words", "max"),
            quality_ok_rate=("quality_ok_bool", "mean"),
            empty_rate=("empty_response", "mean"),
            ultra_short_rate=("ultra_short_response", "mean"),
            subthreshold_length_rate=("subthreshold_length", "mean"),
            eos_leak_rate=("eos_leak", "mean"),
            special_token_leak_rate=("special_token_leak", "mean"),
            boilerplate_rate=("boilerplate_flag", "mean"),
            refusal_rate=("refusal_flag", "mean"),
            role_or_template_leak_rate=("role_or_template_leak", "mean"),
            degenerate_repetition_rate=("long_degenerate_repetition", "mean"),
            severe_artifact_rate=("severe_artifact_flag", "mean"),
            artifact_rate=("artifact_flag", "mean"),
        )
        .reset_index()
    )

    return agg


def _step_response_artifact_scan_33b():
    print("=" * 100)
    print("RESPONSE ARTIFACT SCAN")
    print("=" * 100)

    if "all_responses" not in FILES or not Path(FILES["all_responses"]).exists():
        raise RuntimeError("FILES['all_responses'] is missing. Run response compilation first.")

    resp_path = Path(FILES["all_responses"])
    resp = pd.read_csv(resp_path).copy()

    if resp.empty:
        raise RuntimeError("all_responses file is empty.")

    response_col = _33b_pick_response_col(resp)
    family_col = _33b_pick_family_col(resp)
    model_key_col = _33b_pick_model_key_col(resp)

    resp["response_text"] = resp[response_col].fillna("").astype(str)

    if family_col:
        resp["model_family_clean"] = resp[family_col].map(_33b_clean_family)
    else:
        resp["model_family_clean"] = "unknown"

    if model_key_col:
        resp["model_key_clean"] = resp[model_key_col].fillna("unknown").astype(str)
    else:
        resp["model_key_clean"] = resp["model_family_clean"]

    if "model_scale" not in resp.columns:
        resp["model_scale"] = "unknown"
    else:
        resp["model_scale"] = resp["model_scale"].fillna("unknown").astype(str)

    if "quality_ok" in resp.columns:
        resp["quality_ok_bool"] = resp["quality_ok"].map(lambda x: _33b_bool(x, default=True))
    else:
        resp["quality_ok_bool"] = True

    if "prompt_id" not in resp.columns:
        resp["prompt_id"] = np.arange(len(resp)).astype(str)

    artifacts = resp["response_text"].apply(_33b_detect_artifacts).apply(pd.Series)
    flags = pd.concat([resp, artifacts], axis=1)

    # Duplicate response detection.
    dup_counts = flags["text_hash"].value_counts().to_dict()
    flags["duplicate_response_count"] = flags["text_hash"].map(dup_counts).astype(int)
    flags["duplicate_response_flag"] = flags["duplicate_response_count"] > 1

    # Final artifact flag includes duplicate response only if duplicated many times.
    flags["mass_duplicate_flag"] = flags["duplicate_response_count"] >= 5
    flags["artifact_flag"] = flags["artifact_flag"] | flags["mass_duplicate_flag"]
    flags["severe_artifact_flag"] = flags["severe_artifact_flag"] | flags["mass_duplicate_flag"]

    family_summary = _33b_rate_table(
        flags,
        ["model_family_clean", "model_scale"],
    ).sort_values(
        ["severe_artifact_rate", "artifact_rate", "model_family_clean"],
        ascending=[False, False, True],
    )

    model_summary = _33b_rate_table(
        flags,
        ["model_family_clean", "model_key_clean", "model_scale"],
    ).sort_values(
        ["severe_artifact_rate", "artifact_rate", "model_family_clean", "model_key_clean"],
        ascending=[False, False, True, True],
    )

    prompt_summary = _33b_rate_table(
        flags,
        ["prompt_id"],
    ).sort_values(
        ["severe_artifact_rate", "artifact_rate", "prompt_id"],
        ascending=[False, False, True],
    )

    # Family abnormality flags using conservative thresholds.
    fam = family_summary.copy()
    global_severe = float(flags["severe_artifact_flag"].mean())
    global_artifact = float(flags["artifact_flag"].mean())
    global_eos = float(flags["eos_leak"].mean())
    global_short = float(flags["ultra_short_response"].mean())

    fam["abnormal_severe_artifact_flag"] = fam["severe_artifact_rate"] > max(0.05, global_severe + 0.05)
    fam["abnormal_artifact_flag"] = fam["artifact_rate"] > max(0.10, global_artifact + 0.10)
    fam["abnormal_eos_flag"] = fam["eos_leak_rate"] > max(0.02, global_eos + 0.02)
    fam["abnormal_short_flag"] = fam["ultra_short_rate"] > max(0.05, global_short + 0.05)

    family_summary = fam

    # Save compact row-level flags, not necessarily the full response text.
    keep_cols = [
        "prompt_id",
        "model_family_clean",
        "model_key_clean",
        "model_scale",
        "quality_ok_bool",
        "n_chars",
        "n_words",
        "text_hash",
        "empty_response",
        "ultra_short_response",
        "subthreshold_length",
        "eos_leak",
        "special_token_leak",
        "special_token_hits",
        "boilerplate_flag",
        "refusal_flag",
        "role_or_template_leak",
        "repeated_char_flag",
        "unique_word_rate",
        "line_repetition_rate",
        "long_degenerate_repetition",
        "high_line_repetition",
        "duplicate_response_count",
        "duplicate_response_flag",
        "mass_duplicate_flag",
        "artifact_flag",
        "severe_artifact_flag",
    ]

    keep_cols = [c for c in keep_cols if c in flags.columns]
    flag_out = flags[keep_cols].copy()

    atomic_write_csv(flag_out, FILES["response_artifact_flags"])
    atomic_write_csv(family_summary, FILES["response_artifact_family_summary"])
    atomic_write_csv(prompt_summary, FILES["response_artifact_prompt_summary"])

    severe_rows = flags[flags["severe_artifact_flag"]].copy()
    artifact_rows = flags[flags["artifact_flag"]].copy()

    diagnostics = {
        "analysis_label": "response_artifact_diagnostics",
        "schema_note": (
            "Dedicated response artifact scan. This diagnostic does not delete rows "
            "or alter downstream analyses by itself."
        ),
        "input_file": str(resp_path),
        "response_col_used": response_col,
        "family_col_used": family_col or None,
        "model_key_col_used": model_key_col or None,
        "n_responses": int(len(flags)),
        "n_prompts": int(flags["prompt_id"].nunique()),
        "families_seen": sorted(flags["model_family_clean"].dropna().unique().tolist()),
        "model_scales_seen": sorted(flags["model_scale"].dropna().unique().tolist()),
        "global_rates": {
            "quality_ok_rate": float(flags["quality_ok_bool"].mean()),
            "artifact_rate": float(flags["artifact_flag"].mean()),
            "severe_artifact_rate": float(flags["severe_artifact_flag"].mean()),
            "empty_rate": float(flags["empty_response"].mean()),
            "ultra_short_rate": float(flags["ultra_short_response"].mean()),
            "subthreshold_length_rate": float(flags["subthreshold_length"].mean()),
            "eos_leak_rate": float(flags["eos_leak"].mean()),
            "special_token_leak_rate": float(flags["special_token_leak"].mean()),
            "boilerplate_rate": float(flags["boilerplate_flag"].mean()),
            "refusal_rate": float(flags["refusal_flag"].mean()),
            "role_or_template_leak_rate": float(flags["role_or_template_leak"].mean()),
            "degenerate_repetition_rate": float(flags["long_degenerate_repetition"].mean()),
            "mass_duplicate_rate": float(flags["mass_duplicate_flag"].mean()),
        },
        "family_summary_records": family_summary.to_dict(orient="records"),
        "model_summary_records_top_30": model_summary.head(30).to_dict(orient="records"),
        "n_severe_artifact_rows": int(len(severe_rows)),
        "n_artifact_rows": int(len(artifact_rows)),
        "top_special_token_hits": {
            str(k): int(v)
            for k, v in (
                flags["special_token_hits"]
                .replace("", np.nan)
                .dropna()
                .value_counts()
                .head(20)
                .to_dict()
            ).items()
        },
        "top_duplicate_hashes": {
            str(k): int(v)
            for k, v in (
                flags.loc[flags["duplicate_response_count"] > 1, "text_hash"]
                .value_counts()
                .head(20)
                .to_dict()
            ).items()
        },
        "risk_flags": {
            "any_eos_leak": bool(flags["eos_leak"].any()),
            "any_special_token_leak": bool(flags["special_token_leak"].any()),
            "any_empty_response": bool(flags["empty_response"].any()),
            "any_mass_duplicate": bool(flags["mass_duplicate_flag"].any()),
            "family_abnormality_detected": bool(
                family_summary[
                    [
                        "abnormal_severe_artifact_flag",
                        "abnormal_artifact_flag",
                        "abnormal_eos_flag",
                        "abnormal_short_flag",
                    ]
                ].any().any()
            ),
        },
        "outputs": {
            "response_artifact_flags": str(FILES["response_artifact_flags"]),
            "response_artifact_family_summary": str(FILES["response_artifact_family_summary"]),
            "response_artifact_prompt_summary": str(FILES["response_artifact_prompt_summary"]),
            "response_artifact_diagnostics": str(FILES["response_artifact_diagnostics"]),
        },
        "paper_language": {
            "safe": (
                "We audited generated responses for special-token leakage, EOS leakage, "
                "empty or ultra-short outputs, repeated-template artifacts, and family-level "
                "artifact-rate imbalance before constructing the preference analyses."
            ),
            "if_low_artifacts": (
                "Artifact rates were low and did not show a family-level imbalance large "
                "enough to explain the Primary-4 TPS estimate."
            ),
            "if_high_artifacts": (
                "Artifact diagnostics identified non-negligible generation artifacts; "
                "we report these diagnostics and treat affected analyses as robustness "
                "checks unless filtering is rerun."
            ),
        },
    }

    atomic_write_json(diagnostics, FILES["response_artifact_diagnostics"])

    print("\nGlobal artifact rates")
    print("-" * 100)
    print(json.dumps(diagnostics["global_rates"], indent=2))

    print("\nFamily-level artifact summary")
    print("-" * 100)
    show_cols = [
        "model_family_clean",
        "model_scale",
        "n_responses",
        "quality_ok_rate",
        "artifact_rate",
        "severe_artifact_rate",
        "eos_leak_rate",
        "ultra_short_rate",
        "subthreshold_length_rate",
        "abnormal_severe_artifact_flag",
        "abnormal_artifact_flag",
        "abnormal_eos_flag",
        "abnormal_short_flag",
    ]

    show_cols = [c for c in show_cols if c in family_summary.columns]
    print(family_summary[show_cols].round(4).to_string(index=False))

    print("\nRisk flags")
    print("-" * 100)
    print(json.dumps(diagnostics["risk_flags"], indent=2))

    print("\nSaved:")
    print("Row flags:", FILES["response_artifact_flags"])
    print("Family summary:", FILES["response_artifact_family_summary"])
    print("Prompt summary:", FILES["response_artifact_prompt_summary"])
    print("Diagnostics:", FILES["response_artifact_diagnostics"])


run_step(
    [
        FILES["response_artifact_flags"],
        FILES["response_artifact_family_summary"],
        FILES["response_artifact_prompt_summary"],
        FILES["response_artifact_diagnostics"],
    ],
    _step_response_artifact_scan_33b,
    "response_artifact_scan_cached",
    force=False,
    validators={
        FILES["response_artifact_flags"]: lambda p: csv_has(
            p,
            ["prompt_id", "model_family_clean", "n_words", "artifact_flag", "severe_artifact_flag"],
            1,
        ),
        FILES["response_artifact_family_summary"]: lambda p: csv_has(
            p,
            ["model_family_clean", "n_responses", "artifact_rate", "severe_artifact_rate"],
            1,
        ),
        FILES["response_artifact_diagnostics"]: lambda p: json_has(
            p,
            ["analysis_label", "global_rates", "risk_flags", "outputs"],
        ),
    },
)

print("\n✓ Cell 3.3B complete")

In [ ]:
# ============================================================================
# Cell 3.3C — Interpret response artifact scan
# ============================================================================
"""
Cell 3.3C — Interpret the response artifact diagnostics.

Purpose:
- Read the saved artifact scan outputs.
- Decide whether artifacts are low, moderate, or high risk.
- Produce paper-safe language.
- Save a compact artifact-audit interpretation file.
"""

from pathlib import Path
import json
import pandas as pd
import numpy as np

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_quality" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_quality"] = Path(PATHS["analysis"]) / "quality"
    elif "root" in PATHS:
        PATHS["analysis_quality"] = Path(PATHS["root"]) / "analysis" / "quality"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_quality"] = Path(ROOT_DIR) / "analysis" / "quality"
    else:
        PATHS["analysis_quality"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/quality")

Path(PATHS["analysis_quality"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("response_artifact_diagnostics", Path(PATHS["analysis_quality"]) / "response_artifact_diagnostics.json")
FILES.setdefault("response_artifact_family_summary", Path(PATHS["analysis_quality"]) / "response_artifact_family_summary.csv")
FILES.setdefault("response_artifact_interpretation", Path(PATHS["analysis_quality"]) / "response_artifact_interpretation.json")


def _step_interpret_artifacts_33c():
    print("=" * 100)
    print("INTERPRET RESPONSE ARTIFACT SCAN")
    print("=" * 100)

    diag_path = Path(FILES["response_artifact_diagnostics"])
    fam_path = Path(FILES["response_artifact_family_summary"])

    if not diag_path.exists():
        raise RuntimeError("response_artifact_diagnostics.json missing. Run Cell 3.3B first.")

    if not fam_path.exists():
        raise RuntimeError("response_artifact_family_summary.csv missing. Run Cell 3.3B first.")

    diag = json.load(open(diag_path))
    fam = pd.read_csv(fam_path)

    rates = diag.get("global_rates", {})
    risk_flags = diag.get("risk_flags", {})

    severe_rate = float(rates.get("severe_artifact_rate", 0.0))
    artifact_rate = float(rates.get("artifact_rate", 0.0))
    eos_rate = float(rates.get("eos_leak_rate", 0.0))
    short_rate = float(rates.get("ultra_short_rate", 0.0))
    family_abnormal = bool(risk_flags.get("family_abnormality_detected", False))

    if severe_rate <= 0.01 and eos_rate <= 0.005 and not family_abnormal:
        risk_level = "low"
        recommendation = "Proceed. Report artifact scan as a clean audit."
    elif severe_rate <= 0.05 and eos_rate <= 0.02:
        risk_level = "moderate"
        recommendation = "Proceed with caution. Report artifact scan and mention that robustness checks were inspected."
    else:
        risk_level = "high"
        recommendation = "Inspect flagged rows before freezing results. Consider filtered robustness analysis."

    family_records = fam.to_dict(orient="records")

    interpretation = {
        "analysis_label": "response_artifact_interpretation",
        "status": "ok",
        "risk_level": risk_level,
        "recommendation": recommendation,
        "global_rates": rates,
        "risk_flags": risk_flags,
        "family_summary_path": str(fam_path),
        "diagnostics_path": str(diag_path),
        "family_summary_records": family_records,
        "paper_language": {
            "low": (
                "We audited all generated responses for EOS-token leakage, special-token leakage, "
                "empty or ultra-short outputs, repeated-template artifacts, and family-level artifact-rate "
                "imbalance. Artifact rates were low and did not indicate a family-level imbalance sufficient "
                "to explain the Primary-4 TPS estimate."
            ),
            "moderate": (
                "We audited generated responses for special-token leakage, length failures, repeated-template "
                "artifacts, and family-level artifact imbalance. The scan identified non-zero but limited "
                "artifact rates, which we report alongside robustness checks."
            ),
            "high": (
                "The response artifact audit identified non-negligible generation artifacts. We therefore "
                "treat affected analyses cautiously and report artifact diagnostics explicitly."
            ),
        },
        "selected_paper_language": None,
    }

    interpretation["selected_paper_language"] = interpretation["paper_language"][risk_level]

    atomic_write_json(interpretation, FILES["response_artifact_interpretation"])

    print("\nGlobal artifact rates")
    print("-" * 100)
    print(json.dumps(rates, indent=2))

    print("\nRisk flags")
    print("-" * 100)
    print(json.dumps(risk_flags, indent=2))

    print("\nFamily artifact summary")
    print("-" * 100)
    show_cols = [
        "model_family_clean",
        "model_scale",
        "n_responses",
        "artifact_rate",
        "severe_artifact_rate",
        "eos_leak_rate",
        "ultra_short_rate",
        "subthreshold_length_rate",
        "abnormal_severe_artifact_flag",
        "abnormal_artifact_flag",
        "abnormal_eos_flag",
        "abnormal_short_flag",
    ]
    show_cols = [c for c in show_cols if c in fam.columns]
    print(fam[show_cols].round(4).to_string(index=False))

    print("\nInterpretation")
    print("-" * 100)
    print("Risk level:", risk_level)
    print("Recommendation:", recommendation)

    print("\nPaper language")
    print("-" * 100)
    print(interpretation["selected_paper_language"])

    print("\nSaved:")
    print(FILES["response_artifact_interpretation"])


run_step(
    [FILES["response_artifact_interpretation"]],
    _step_interpret_artifacts_33c,
    "interpret_response_artifact_scan",
    force=False,
    validators={
        FILES["response_artifact_interpretation"]: lambda p: json_has(
            p,
            ["analysis_label", "risk_level", "recommendation", "selected_paper_language"],
        )
    },
)

print("\n✓ Cell 3.3C complete")

## PHASE 4 — Trial Construction

For each prompt × each unordered family pair × each order (AB / BA), and for
each judge family, we build one trial. With 5 families: 10 unordered pairs ×
2 orders = 20 (prompt, pair, order) keys, times 5 judges = 100 judgments per
prompt. Across N prompts that's N × 100 judgments. Stable trial_id is a
16-char SHA1 of (prompt_id, pair_id, order, judge, model_a, model_b).



In [ ]:
# ============================================================================
# Cell 4.1 — Build trial master with stable IDs and pair-level eligibility
# ============================================================================
"""
Cell 4.1 — Build the trial master table.

Eligibility: a (prompt, pair) is included only if BOTH families produced
quality-OK responses for that prompt. Excluded counts are logged for
missing-data sensitivity.
"""
def _stable_trial_id(prompt_id, pair_id, order, judge, ma, mb) -> str:
    key = "|".join(map(str, [prompt_id, pair_id, order, judge, ma, mb]))
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:16]

def _step_trial_master():
    resp = pd.read_csv(FILES["all_responses"])
    resp = resp[(resp["model_scale"] == "large") &
                resp["quality_ok"].astype(bool) &
                resp["response"].fillna("").str.strip().ne("")].copy()
    resp = resp.drop_duplicates(["prompt_id", "model_family"], keep="last")
    resp_lookup = resp.set_index(["prompt_id", "model_family"])["response"].to_dict()
    tok_lookup  = resp.set_index(["prompt_id", "model_family"])["n_tokens"].to_dict()

    prompts = pd.read_csv(FILES["master_prompts"])
    split   = pd.read_csv(FILES["prompt_split"])[["prompt_id", "split"]]
    prompts = prompts.merge(split, on="prompt_id", how="left")

    # Eligibility: prompts with all 5 families' responses
    avail = resp.groupby("prompt_id")["model_family"].nunique()
    eligible_prompts = set(avail[avail == len(FAMILIES)].index)
    log.info(f"Eligible prompts (all 5 families OK): {len(eligible_prompts)} / {len(prompts)}")
    excluded = set(prompts["prompt_id"]) - eligible_prompts
    log.info(f"Excluded prompts (incomplete responses): {len(excluded)}")

    rows: List[Dict[str, Any]] = []
    for _, prow in prompts.iterrows():
        pid = prow["prompt_id"]
        if pid not in eligible_prompts:
            continue
        for fa, fb in combinations(FAMILIES, 2):
            ra = resp_lookup.get((pid, fa)); rb = resp_lookup.get((pid, fb))
            if not ra or not rb:
                continue
            pair_id = f"{fa}_vs_{fb}"
            ta = tok_lookup.get((pid, fa), 0)
            tb = tok_lookup.get((pid, fb), 0)
            for judge in FAMILIES:
                # AB orientation
                tid_ab = _stable_trial_id(pid, pair_id, "AB", judge, fa, fb)
                rows.append({
                    "trial_id": tid_ab, "prompt_id": pid, "prompt": prow["prompt"],
                    "source": prow["source"], "category": prow["category"],
                    "split": prow["split"], "pair_id": pair_id,
                    "family_1": fa, "family_2": fb,
                    "order": "AB", "judge_family": judge,
                    "model_a": fa, "model_b": fb,
                    "response_a": ra, "response_b": rb,
                    "tokens_a": ta, "tokens_b": tb,
                })
                # BA orientation
                tid_ba = _stable_trial_id(pid, pair_id, "BA", judge, fb, fa)
                rows.append({
                    "trial_id": tid_ba, "prompt_id": pid, "prompt": prow["prompt"],
                    "source": prow["source"], "category": prow["category"],
                    "split": prow["split"], "pair_id": pair_id,
                    "family_1": fa, "family_2": fb,
                    "order": "BA", "judge_family": judge,
                    "model_a": fb, "model_b": fa,
                    "response_a": rb, "response_b": ra,
                    "tokens_a": tb, "tokens_b": ta,
                })

    tm = pd.DataFrame(rows)
    if tm["trial_id"].duplicated().any():
        raise RuntimeError("Duplicate trial_ids — investigate hash collision")
    atomic_write_csv(tm, FILES["trial_master"])

    audit = {
        "n_eligible_prompts": len(eligible_prompts),
        "n_excluded_prompts": len(excluded),
        "n_trials_total": len(tm),
        "n_trials_per_judge": tm.groupby("judge_family").size().to_dict(),
        "n_trials_per_split": tm.groupby("split").size().to_dict(),
        "n_trials_per_source": tm.groupby("source").size().to_dict(),
    }
    atomic_write_json(audit, FILES["trial_audit"])
    log.info(json.dumps(audit, indent=2))

run_step([FILES["trial_master"], FILES["trial_audit"]],
         _step_trial_master, "build_trial_master",
         validators={FILES["trial_master"]:
                     lambda p: csv_has(p, ["trial_id", "judge_family", "model_a"], 100)})
print("✓ Trial master ready")




## PHASE 5 — Judgment Collection (with logprob caching)

Each judge model evaluates every trial under both the **rubric** and **neutral**
system prompts. Per trial we also compute the response-conditional log-probability
of A and B under the judge — Wataoka's perplexity-as-familiarity proxy.

Resume safety: per-judge CSVs append by trial_id, the logprob cache is a
content-hashed JSON blob persisted every N writes.



In [ ]:
# ============================================================================
# Compatibility wrapper required by existing judgment cells
# ============================================================================

def build_judge_messages(prompt, response_a, response_b, neutral=False):
    """
    Backward-compatible function expected by _judge_one().

    Existing judgment code calls:
        system, user = build_judge_messages(..., neutral=neutral)

    This wrapper connects that old expected name to the updated
    robust Cell 5.1 prompt functions.
    """
    rubric_mode = "neutral" if neutral else "rubric"

    system = get_judge_system_prompt(rubric_mode)
    user = build_judge_user_prompt(
        user_prompt=prompt,
        response_a=response_a,
        response_b=response_b,
    )

    return system, user


print("✓ Compatibility wrapper loaded: build_judge_messages() is now defined")

In [ ]:
# ============================================================================
# Cell 5.1 — Judge prompts (rubric + neutral) and parser
# ORIGINAL PARSER VERSION USED FOR FIRST COMPLETED LLaMA/Qwen RUNS
# ============================================================================

"""
Cell 5.1 — Two judge system prompts. 'rubric' is the explicit-criteria version;
'neutral' is the minimal-instruction version. Comparing the two is one of our
core robustness analyses (the explicit anti-style instruction in 'rubric' is a
demand characteristic — the neutral prompt is less contaminated).

IMPORTANT:
This cell intentionally keeps the original parser behavior used for the first
completed LLaMA and Qwen judgment files.

This means:
- valid JSON A/B/tie is accepted
- invalid or unparseable output becomes winner='tie', parse_ok=False
- no Falcon-specific parser logic is added
- parse_judgment(raw) returns exactly 3 values:
  winner, rationale, parse_ok
"""

import re
import json
from typing import Tuple


JUDGE_SYSTEM_RUBRIC = (
    "You are a careful impartial evaluator. Compare two assistant responses to "
    "the same user request. Judge only response quality."
)

JUDGE_SYSTEM_NEUTRAL = (
    "You are an evaluator. Compare two assistant responses to the same user "
    "request and pick the better one."
)

JUDGE_RUBRIC = """Evaluate using these criteria:
1. Correctness and factual reliability.
2. Completeness.
3. Relevance to the user request.
4. Clarity and coherence.
5. Instruction following.
6. Conciseness, only when it does not reduce quality.

Return strict JSON only:
{"winner":"A"|"B"|"tie","rationale":"brief explanation"}"""

JUDGE_INSTRUCTION_NEUTRAL = """Return strict JSON only:
{"winner":"A"|"B"|"tie","rationale":"brief explanation"}"""


def build_judge_messages(
    prompt: str,
    response_a: str,
    response_b: str,
    neutral: bool = False,
) -> Tuple[str, str]:
    system = JUDGE_SYSTEM_NEUTRAL if neutral else JUDGE_SYSTEM_RUBRIC
    instruction = JUDGE_INSTRUCTION_NEUTRAL if neutral else JUDGE_RUBRIC

    user = (
        f"User request:\n{prompt}\n\n"
        f"Response A:\n{response_a}\n\n"
        f"Response B:\n{response_b}\n\n"
        f"{instruction}"
    )

    return system, user


def parse_judgment(text: str) -> Tuple[str, str, bool]:
    """
    ORIGINAL parser used for the first completed LLaMA/Qwen judgment files.

    Returns:
        winner: one of {'A', 'B', 'tie'}
        rationale: short rationale string
        parse_ok: True if valid JSON winner was parsed, else False

    Behavior:
    - Extracts first JSON-looking block from model output.
    - Accepts winner A, B, or tie.
    - If no JSON or invalid JSON, returns tie with parse_ok=False.
    """

    raw = "" if text is None else str(text).strip()

    try:
        m = re.search(r"\{.*\}", raw, flags=re.S)

        if not m:
            return "tie", raw[:500], False

        obj = json.loads(m.group(0))

        w = str(obj.get("winner", "tie")).strip().upper()

        if w in {"A", "B"}:
            return w, str(obj.get("rationale", ""))[:500], True

        if w == "TIE":
            return "tie", str(obj.get("rationale", ""))[:500], True

        return "tie", str(obj.get("rationale", raw))[:500], False

    except Exception:
        return "tie", raw[:500], False


# ============================================================================
# Parser self-test
# ============================================================================

_test_outputs = [
    '{"winner":"A","rationale":"A is clearer"}',
    '{"winner":"B","rationale":"B is more complete"}',
    '{"winner":"tie","rationale":"Both are equally good"}',
    'Response A is better because it is clearer.',
    '{"winner":"A"|"B"|"tie","rationale":"brief explanation"}',
    '',
]

print("=" * 100)
print("ORIGINAL PARSER SELF-TEST")
print("=" * 100)

for raw in _test_outputs:
    parsed = parse_judgment(raw)
    print("\nRAW:", repr(raw))
    print("PARSED:", parsed)

assert len(parse_judgment('{"winner":"A","rationale":"x"}')) == 3
assert parse_judgment('{"winner":"A","rationale":"x"}')[0] == "A"
assert parse_judgment('{"winner":"B","rationale":"x"}')[0] == "B"
assert parse_judgment('{"winner":"tie","rationale":"x"}')[0] == "tie"

print("\n✓ Original Cell 5.1 parser restored")
print("✓ parse_judgment(raw) returns exactly 3 values")
print("✓ This matches the parser behavior used for the first completed LLaMA/Qwen runs")

In [ ]:
# ============================================================================
# Cell 5.2 — Logprob cache (DIRECT-EVAL VERSION; bypasses create_completion bug)
# ============================================================================
"""
Computes mean per-token logprob of the RESPONSE under the judge,
using llm.eval() + Llama.logits_to_logprobs() directly.

Why not create_completion(echo=True): it returns empty token_logprobs when
the sampler hits EOS first (issue #1528) and has off-by-one indexing
(issue #1983). The direct-eval path is reliable and methodologically cleaner
because we average ONLY over response tokens — not the template scaffolding.
"""

import json, hashlib
from pathlib import Path
from typing import Any, Dict, Optional
import numpy as np

# (Re-affirm — same as before. Don't change.)
LOGPROB_MAX_CHARS  = 400
LOGPROB_SAVE_EVERY = 50

def _load_logprob_cache() -> Dict[str, Optional[float]]:
    p = FILES["logprob_cache"]
    if not p.exists():
        return {}
    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}

LOGPROB_CACHE: Dict[str, Optional[float]] = _load_logprob_cache()
_LP_DIRTY = 0

def _save_logprob_cache_now() -> None:
    global _LP_DIRTY
    atomic_write_json(LOGPROB_CACHE, FILES["logprob_cache"])
    _LP_DIRTY = 0

def _logprob_key(judge_key: str, prompt_id: str, response_text: str,
                 max_chars: int = LOGPROB_MAX_CHARS) -> str:
    excerpt = (response_text or "")[:max_chars]
    return "|".join([judge_key, str(prompt_id), stable_hash(excerpt)])

def purge_null_logprob_cache() -> int:
    """Drop any cached `None` entries left over from the buggy old path.
    Run this ONCE after upgrading Cell 5.2; subsequent runs find nothing."""
    bad = [k for k, v in LOGPROB_CACHE.items() if v is None]
    for k in bad:
        del LOGPROB_CACHE[k]
    if bad:
        _save_logprob_cache_now()
    log.info(f"Purged {len(bad)} poisoned None entries from logprob cache")
    return len(bad)

def compute_response_logprob(llm: Any, prompt: str, response: str,
                              max_chars: int = LOGPROB_MAX_CHARS) -> float:
    """
    Mean per-token logprob of `response` under `llm`, given the user prompt.
    Uses low-level eval() so it's robust to llama-cpp-python's create_completion
    bugs. Averages over RESPONSE tokens only, not template tokens.

    REQUIRES the model to have been loaded with logits_all=True.
    """
    try:
        from llama_cpp import Llama

        prompt = "" if prompt is None else str(prompt)
        excerpt = (response or "").strip()[:max_chars]
        if len(excerpt) < 10:
            return float("nan")

        # Tokenize context (the user prompt + scaffolding) and response
        # SEPARATELY so we know the exact token offsets for each piece.
        context_text = ("User request:\n" + prompt.strip()[:600] +
                        "\n\nAssistant response:\n")
        ctx_tokens = llm.tokenize(context_text.encode("utf-8"),
                                  add_bos=True, special=False)
        rsp_tokens = llm.tokenize(excerpt.encode("utf-8"),
                                  add_bos=False, special=False)
        if not rsp_tokens:
            return float("nan")
        all_tokens = ctx_tokens + rsp_tokens

        # Refuse to score if it would exceed the loaded context window
        try:
            n_ctx_val = llm.n_ctx() if callable(getattr(llm, "n_ctx", None)) \
                                     else int(llm.n_ctx)
        except Exception:
            n_ctx_val = JUDGE_CONTEXT
        if len(all_tokens) >= n_ctx_val - 4:
            return float("nan")

        # Forward pass (logits_all=True must already be set on llm)
        llm.reset()
        llm.eval(all_tokens)

        scores = getattr(llm, "_scores", None)
        if scores is None or len(scores) == 0:
            log.warning("logprob: llm._scores is empty — was logits_all=True at load?")
            return float("nan")

        # Convert to logprobs (log-softmax across vocab)
        logprobs = Llama.logits_to_logprobs(scores)  # shape (n_tokens, vocab)

        # For response token at sequence position `ctx_len + j`, the predictive
        # logits live at sequence position `ctx_len + j - 1`.
        ctx_len = len(ctx_tokens)
        per_tok = []
        for j, tok in enumerate(rsp_tokens):
            pos = ctx_len + j - 1
            if 0 <= pos < len(logprobs):
                lp = float(logprobs[pos][tok])
                if np.isfinite(lp):
                    per_tok.append(lp)

        if not per_tok:
            return float("nan")
        return float(np.mean(per_tok))

    except Exception as e:
        log.warning(f"logprob fail: {type(e).__name__}: {str(e)[:140]}")
        return float("nan")

def get_or_compute_logprob(llm: Any, judge_key: str, prompt_id: str,
                            prompt: str, response: str) -> float:
    """Cached wrapper. Only successful (finite) values are cached;
    NaN failures retry on the next call."""
    global _LP_DIRTY
    k = _logprob_key(judge_key, prompt_id, response)
    if k in LOGPROB_CACHE:
        v = LOGPROB_CACHE[k]
        if v is not None:                 # cache hit (real value)
            return float(v)
        # else cache had None from old buggy path → fall through to recompute
    v = compute_response_logprob(llm, prompt, response)
    if np.isfinite(v):                    # only cache successes
        LOGPROB_CACHE[k] = float(v)
        _LP_DIRTY += 1
        if _LP_DIRTY >= LOGPROB_SAVE_EVERY:
            _save_logprob_cache_now()
    return v

# Run the one-time purge of poisoned cache entries from the prior buggy run
_n_purged = purge_null_logprob_cache()
print("✓ Cell 5.2 reloaded — direct-eval logprob (issue #1528 / #1983 workaround)")
print(f"  Cache size: {len(LOGPROB_CACHE)} (purged {_n_purged} poisoned None entries)")

In [ ]:
# ============================================================================
# Cell 5.3 — Resume-safe per-judge collection (rubric + neutral)
#
# JUDGMENT-FIRST MODE (NaN repair disabled):
#   - If a judge's file is fully complete on disk → skip without loading model,
#     regardless of any NaN logprobs in old rows.
#   - Only NEW trials trigger model staging + load.
#   - Logprob NaNs from earlier buggy runs will be patched up in a separate
#     post-pass once all 5 families have been judged.
#
# Critical: model is loaded with logits_all=True so newly judged rows get
# correct logprobs going forward.
# ============================================================================

JUDGMENT_COLUMNS = [
    "trial_id", "judge_family", "judge_scale", "rubric_mode",
    "winner", "rationale", "raw_judgment", "parse_ok",
    "logprob_a", "logprob_b", "logprob_delta",
    "judged_at",
]


def _judgment_path(scale: str, family: str, neutral: bool) -> Path:
    """Canonical path where new judgments are written."""
    suffix = "_neutral" if neutral else "_rubric"
    if scale == "large" and not neutral:
        base = PATHS["judgments_large"]
    elif scale == "large" and neutral:
        base = PATHS["judgments_large_neu"]
    else:  # small
        base = PATHS["judgments_small"]
    base.mkdir(parents=True, exist_ok=True)
    return base / f"judgments_{family}_{scale}{suffix}.csv"


def _normalize_existing_judgments(df: pd.DataFrame) -> pd.DataFrame:
    """Make sure an old/partial judgment CSV has all expected columns and
    string trial_ids. Used on every load."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=JUDGMENT_COLUMNS)
    for col in JUDGMENT_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan
    df["trial_id"] = df["trial_id"].astype(str)
    return df


def _judge_one(llm, judge_key: str, row, neutral: bool) -> Dict[str, Any]:
    """Single trial: build prompt, call judge, parse winner, compute both logprobs."""
    system, user = build_judge_messages(row["prompt"], row["response_a"],
                                          row["response_b"], neutral=neutral)
    try:
        out = llm.create_chat_completion(
            messages=[{"role": "system", "content": system},
                      {"role": "user",   "content": user}],
            max_tokens=JUDGE_MAX_TOKENS,
            temperature=JUDGE_TEMPERATURE,
            top_p=JUDGE_TOP_P,
            seed=RANDOM_SEED,
        )
        raw = out["choices"][0]["message"]["content"]
    except Exception as e:
        log.warning(f"judge fail {judge_key}/{row['trial_id']}: "
                    f"{type(e).__name__}: {str(e)[:120]}")
        raw = ""
    winner, rationale, ok = parse_judgment(raw)

    # Logprobs (familiarity covariate) — A then B
    lp_a = get_or_compute_logprob(llm, judge_key, row["prompt_id"],
                                   row["prompt"], row["response_a"])
    lp_b = get_or_compute_logprob(llm, judge_key, row["prompt_id"],
                                   row["prompt"], row["response_b"])
    delta = (lp_a - lp_b) if (np.isfinite(lp_a) and np.isfinite(lp_b)) else float("nan")

    return {
        "trial_id":     str(row["trial_id"]),
        "judge_family": row["judge_family"],
        "judge_scale":  "small" if row.get("_small", False) else "large",
        "rubric_mode":  "neutral" if neutral else "rubric",
        "winner":       winner,
        "rationale":    rationale,
        "raw_judgment": raw[:1000],
        "parse_ok":     ok,
        "logprob_a":    lp_a,
        "logprob_b":    lp_b,
        "logprob_delta":delta,
        "judged_at":    datetime.now(timezone.utc).isoformat(),
    }


def _file_covers_all_trials(scale: str, family: str, neutral: bool,
                             expected_trial_ids: set) -> bool:
    """True iff the on-disk judgment file already contains every expected
    trial_id. NaN logprobs are NOT considered — only trial_id coverage."""
    p = _judgment_path(scale, family, neutral)
    if not p.exists():
        return False
    try:
        df = pd.read_csv(p, usecols=["trial_id"])
        have = set(df["trial_id"].astype(str))
        return expected_trial_ids.issubset(have)
    except Exception:
        return False


def collect_judgments(scale: str, family: str, neutral: bool = False,
                      checkpoint_every: int = 25) -> Path:
    """Run one judge over the trial master. Resume-safe.

    JUDGMENT-FIRST: returns immediately (no model staging, no GPU load) if every
    required trial_id is already in the file. NaN logprobs in old rows are
    accepted and will be repaired in a post-pass after all families are judged.
    """

    out_path = _judgment_path(scale, family, neutral)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    tm = pd.read_csv(FILES["trial_master"])
    tm["trial_id"] = tm["trial_id"].astype(str)
    sub = tm[tm["judge_family"] == family].copy()
    sub["trial_id"] = sub["trial_id"].astype(str)

    if scale == "small":
        sub["_small"] = True

    mode_name = "neutral" if neutral else "rubric"
    log.info(f"  Judge {family} ({scale}, {mode_name}): {len(sub)} trials")

    expected_ids = set(sub["trial_id"])

    # ========================================================================
    # EARLY EXIT — every required trial_id is on disk. Skip without loading.
    # NaN logprobs are intentionally ignored here; we'll repair them later.
    # ========================================================================
    if _file_covers_all_trials(scale, family, neutral, expected_ids):
        # Quick stats for the log so you can see what's outstanding
        try:
            df = pd.read_csv(out_path)
            n_nan = int(df["logprob_delta"].isna().sum()) if "logprob_delta" in df.columns else 0
        except Exception:
            n_nan = -1
        log.info(f"    ✅ ALREADY COMPLETE — {len(expected_ids):,} trials judged. "
                 f"({n_nan} rows have NaN logprobs — will repair in post-pass.)")
        return out_path

    # ========================================================================
    # Otherwise: load existing rows and figure out what's still TODO.
    # ========================================================================
    existing = pd.DataFrame(columns=JUDGMENT_COLUMNS)
    if out_path.exists():
        try:
            existing = pd.read_csv(out_path)
            existing = _normalize_existing_judgments(existing)
        except Exception as e:
            log.warning(f"    could not read existing judgment file {out_path}: "
                        f"{type(e).__name__}: {e}")
            existing = pd.DataFrame(columns=JUDGMENT_COLUMNS)

    existing_ids = set(existing["trial_id"]) if len(existing) else set()
    todo_ids = expected_ids - existing_ids
    todo = sub[sub["trial_id"].isin(todo_ids)].copy()

    log.info(f"    existing rows: {len(existing):,}")
    log.info(f"    new trials to judge: {len(todo):,}")

    if len(todo) == 0:
        # Should be unreachable given _file_covers_all_trials, but defensive.
        log.info(f"    ✅ Nothing to do for {family}/{scale}/{mode_name}")
        return out_path

    judge_key = f"{scale}/{family}/{'neu' if neutral else 'rub'}"
    rows = existing.to_dict("records")

    # ========================================================================
    # Model load (logits_all=True is mandatory for new-row logprobs).
    # ========================================================================
    with with_model(scale, family, n_ctx=JUDGE_CONTEXT, logits_all=True) as llm:
        for i, (_, row) in enumerate(todo.iterrows()):
            rec = _judge_one(llm, judge_key, row, neutral=neutral)
            rows.append(rec)
            if (i + 1) % checkpoint_every == 0 or (i + 1) == len(todo):
                atomic_write_csv(pd.DataFrame(rows), out_path)
                _save_logprob_cache_now()
                log.info(f"    judged {i+1}/{len(todo)}  "
                         f"(total in file: {len(rows):,})")

    atomic_write_csv(pd.DataFrame(rows), out_path)
    _save_logprob_cache_now()
    log.info(f"    ✅ wrote {len(rows):,} rows → {out_path}")
    return out_path


# ============================================================================
# Phase-level wrappers (Phase 5 step entry points)
# Each does a quick precheck: if every family's file covers all trial_ids,
# skip the entire phase without entering collect_judgments at all.
# ============================================================================
def _phase_is_complete(scale: str, neutral: bool) -> bool:
    """True iff every family's file covers all required trial_ids.
    NaN logprobs are NOT considered — they'll be fixed in the post-pass."""
    tm = pd.read_csv(FILES["trial_master"])
    tm["trial_id"] = tm["trial_id"].astype(str)
    for fam in FAMILIES:
        expected = set(tm[tm["judge_family"] == fam]["trial_id"])
        if not _file_covers_all_trials(scale, fam, neutral, expected):
            return False
    return True


def step_judges_large_rubric():
    if _phase_is_complete("large", neutral=False):
        log.info("✅ All large-rubric judges already complete — skipping phase entirely")
        return
    for fam in FAMILIES:
        collect_judgments("large", fam, neutral=False)


def step_judges_large_neutral():
    if _phase_is_complete("large", neutral=True):
        log.info("✅ All large-neutral judges already complete — skipping phase entirely")
        return
    for fam in FAMILIES:
        collect_judgments("large", fam, neutral=True)


def step_judges_small_rubric():
    if _phase_is_complete("small", neutral=False):
        log.info("✅ All small-rubric judges already complete — skipping phase entirely")
        return
    for fam in FAMILIES:
        collect_judgments("small", fam, neutral=False)


# Sentinels (kept for run_step compatibility)
LARGE_RUB_SENTINEL = PATHS["judgments_large"]     / ".phase5_done"
LARGE_NEU_SENTINEL = PATHS["judgments_large_neu"] / ".phase5_done"
SMALL_RUB_SENTINEL = PATHS["judgments_small"]     / ".phase5_done"


def _wrap_step_lr():
    step_judges_large_rubric()
    if _phase_is_complete("large", neutral=False):
        atomic_write_text(LARGE_RUB_SENTINEL, datetime.now(timezone.utc).isoformat())


def _wrap_step_ln():
    step_judges_large_neutral()
    if _phase_is_complete("large", neutral=True):
        atomic_write_text(LARGE_NEU_SENTINEL, datetime.now(timezone.utc).isoformat())


def _wrap_step_sr():
    step_judges_small_rubric()
    if _phase_is_complete("small", neutral=False):
        atomic_write_text(SMALL_RUB_SENTINEL, datetime.now(timezone.utc).isoformat())


run_step([LARGE_RUB_SENTINEL], _wrap_step_lr, "judges_large_rubric")
run_step([LARGE_NEU_SENTINEL], _wrap_step_ln, "judges_large_neutral")
run_step([SMALL_RUB_SENTINEL], _wrap_step_sr, "judges_small_rubric")
print("✓ Judgment collection complete")

In [ ]:
# ============================================================================
# Cell 5.4 — Compile master judgment tables with Falcon excluded as judge
# ============================================================================
"""
Cell 5.4 — Concatenate per-judge files + recover trial-level metadata.

Main policy:
- Falcon is kept as a candidate model in the full diagnostic/sensitivity data.
- Falcon is excluded as a primary judge.
- Falcon judge files are preserved separately for diagnostics.

Why this cell is robust:
- Your judgment CSVs contain trial_id, judge_family, winner, etc.
- But your current trial_master may have been regenerated, so its trial_id values
  may no longer match the judgment files.
- Therefore this cell searches multiple metadata sources:
    1. existing/old master_judgments files
    2. current trial_master
    3. archived trial_master files
    4. archived master_judgments files
- It builds a metadata bank by trial_id and merges from the matching source.

Outputs:
1. master_judgments       = large rubric, primary judges only
2. master_judgments_neu   = large neutral, primary judges only
3. master_judgments_sm    = small rubric, primary judges only
4. master_judgments_excluded_falcon = Falcon judge files only, diagnostic
"""

from pathlib import Path
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np

# ---------------------------------------------------------------------------
# Judge policy
# ---------------------------------------------------------------------------

if "FAMILIES" not in globals():
    raise RuntimeError("FAMILIES is not defined. Run the configuration cells first.")

PRIMARY_JUDGE_FAMILIES = [f for f in FAMILIES if f != "falcon"]
EXCLUDED_JUDGE_FAMILIES = ["falcon"]
CANDIDATE_FAMILIES = list(FAMILIES)

print("Candidate families:", CANDIDATE_FAMILIES)
print("Primary judge families:", PRIMARY_JUDGE_FAMILIES)
print("Excluded judge families:", EXCLUDED_JUDGE_FAMILIES)

# ---------------------------------------------------------------------------
# Diagnostic output path for excluded Falcon judge files
# ---------------------------------------------------------------------------

FILES["master_judgments_excluded_falcon"] = (
    Path(FILES["master_judgments"]).parent / "master_judgments_excluded_falcon.csv"
)

# ---------------------------------------------------------------------------
# Metadata schema expected downstream
# ---------------------------------------------------------------------------

KEEP_META = [
    "trial_id",
    "prompt_id",
    "prompt",
    "source",
    "category",
    "split",
    "pair_id",
    "family_1",
    "family_2",
    "order",
    "model_a",
    "model_b",
    "tokens_a",
    "tokens_b",
]

REQUIRED_META = [
    "prompt_id",
    "pair_id",
    "family_1",
    "family_2",
    "order",
    "model_a",
    "model_b",
]

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _existing_files(paths: List[Path]) -> List[Path]:
    return [Path(p) for p in paths if Path(p).exists()]


def _blank_to_na(s: pd.Series) -> pd.Series:
    if not pd.api.types.is_object_dtype(s):
        return s
    return s.replace({"": np.nan, "nan": np.nan, "None": np.nan, "NaN": np.nan})


def _normalize_basic_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "trial_id" in df.columns:
        df["trial_id"] = df["trial_id"].astype(str).str.strip()

    if "judge_family" in df.columns:
        df["judge_family"] = df["judge_family"].astype(str).str.strip().str.lower()

    if "winner" in df.columns:
        df["winner"] = df["winner"].astype(str).str.strip()

    for col in [
        "family_1",
        "family_2",
        "model_a",
        "model_b",
        "order",
        "prompt_id",
        "pair_id",
        "source",
        "category",
        "split",
    ]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            df[col] = _blank_to_na(df[col])

    return df


def _add_metadata_aliases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Support old schema names that may appear in archived files.
    """
    df = df.copy()

    alias_map = {
        "family_1": [
            "candidate_family_1",
            "family_a",
            "model_a_family",
            "candidate_family_a",
            "family_A",
        ],
        "family_2": [
            "candidate_family_2",
            "family_b",
            "model_b_family",
            "candidate_family_b",
            "family_B",
        ],
        "model_a": [
            "candidate_a",
            "response_model_a",
            "model_1",
            "candidate_model_a",
            "model_A",
        ],
        "model_b": [
            "candidate_b",
            "response_model_b",
            "model_2",
            "candidate_model_b",
            "model_B",
        ],
        "tokens_a": [
            "n_tokens_a",
            "response_a_tokens",
            "token_count_a",
        ],
        "tokens_b": [
            "n_tokens_b",
            "response_b_tokens",
            "token_count_b",
        ],
    }

    for canonical, aliases in alias_map.items():
        if canonical not in df.columns:
            for alias in aliases:
                if alias in df.columns:
                    df[canonical] = df[alias]
                    break

    return df


def _read_table_safely(path: Path) -> pd.DataFrame:
    path = Path(path)

    if path.suffix.lower() == ".jsonl":
        return pd.read_json(path, lines=True)

    if path.suffix.lower() == ".json":
        try:
            return pd.read_json(path, lines=True)
        except Exception:
            return pd.read_json(path)

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)

    return pd.read_csv(path)


def _candidate_metadata_paths() -> List[Path]:
    """
    Search likely metadata sources.

    Important:
    - Existing master_judgments may still contain the correct old metadata.
    - Archives may contain the old matching trial_master.
    - Current trial_master may not match, but we still include it.
    """
    paths = []

    # 1. Direct known files from FILES.
    for key in [
        "master_judgments",
        "master_judgments_neu",
        "master_judgments_sm",
        "master_judgments_excluded_falcon",
        "trial_master",
    ]:
        if key in FILES:
            p = Path(FILES[key])
            if p.exists():
                paths.append(p)

    # 2. Root search.
    root = Path(ROOT_DIR) if "ROOT_DIR" in globals() else Path(FILES["trial_master"]).parent

    patterns = [
        "trial_master*.csv",
        "trial_master*.jsonl",
        "trial_master*.json",
        "master_judgments*.csv",
        "master_judgments*.jsonl",
        "judgments_master*.csv",
        "effective_winners*.csv",
    ]

    for pat in patterns:
        try:
            paths.extend(list(root.rglob(pat)))
        except Exception:
            pass

    # 3. De-duplicate while preserving order.
    unique = []
    seen = set()

    for p in paths:
        p = Path(p)
        key = str(p.resolve()) if p.exists() else str(p)
        if p.exists() and key not in seen:
            unique.append(p)
            seen.add(key)

    return unique


def _build_metadata_bank(needed_trial_ids: set) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build a trial_id -> metadata table from any matching old/current metadata source.

    Returns:
    - metadata_bank
    - source_report
    """
    rows = []
    report = []

    candidate_paths = _candidate_metadata_paths()

    for p in candidate_paths:
        try:
            df = _read_table_safely(p)
            df = _add_metadata_aliases(df)
            df = _normalize_basic_columns(df)

            if "trial_id" not in df.columns:
                report.append({
                    "path": str(p),
                    "status": "skipped_no_trial_id",
                    "rows": len(df),
                    "matched_trial_ids": 0,
                    "usable_rows": 0,
                })
                continue

            df["trial_id"] = df["trial_id"].astype(str).str.strip()
            m = df[df["trial_id"].isin(needed_trial_ids)].copy()

            if len(m) == 0:
                report.append({
                    "path": str(p),
                    "status": "no_matching_trial_ids",
                    "rows": len(df),
                    "matched_trial_ids": 0,
                    "usable_rows": 0,
                })
                continue

            # Keep only metadata columns that exist.
            keep_cols = [c for c in KEEP_META if c in m.columns]
            m = m[keep_cols].copy()

            # Need at least one required metadata field to be useful.
            required_existing = [c for c in REQUIRED_META if c in m.columns]
            if not required_existing:
                report.append({
                    "path": str(p),
                    "status": "matched_but_no_required_metadata_columns",
                    "rows": len(df),
                    "matched_trial_ids": int(m["trial_id"].nunique()),
                    "usable_rows": 0,
                })
                continue

            usable_mask = pd.concat(
                [_blank_to_na(m[c]).notna() for c in required_existing],
                axis=1,
            ).any(axis=1)

            m = m[usable_mask].copy()

            if len(m) == 0:
                report.append({
                    "path": str(p),
                    "status": "matched_but_metadata_empty",
                    "rows": len(df),
                    "matched_trial_ids": 0,
                    "usable_rows": 0,
                })
                continue

            m["__metadata_source"] = str(p)
            rows.append(m)

            report.append({
                "path": str(p),
                "status": "used",
                "rows": len(df),
                "matched_trial_ids": int(m["trial_id"].nunique()),
                "usable_rows": len(m),
            })

        except Exception as e:
            report.append({
                "path": str(p),
                "status": f"error: {repr(e)}",
                "rows": None,
                "matched_trial_ids": 0,
                "usable_rows": 0,
            })

    source_report = pd.DataFrame(report)

    if not rows:
        return pd.DataFrame(columns=KEEP_META + ["__metadata_source"]), source_report

    all_meta = pd.concat(rows, ignore_index=True)
    all_meta = _normalize_basic_columns(all_meta)

    # Keep only columns we care about.
    for c in KEEP_META:
        if c not in all_meta.columns:
            all_meta[c] = np.nan

    if "__metadata_source" not in all_meta.columns:
        all_meta["__metadata_source"] = np.nan

    # Collapse duplicate trial_ids by first non-null value.
    # Source order matters because candidate paths were ordered with existing masters first.
    collapsed_rows = []

    for trial_id, g in all_meta.groupby("trial_id", sort=False):
        row = {"trial_id": trial_id}
        sources = []

        for _, r in g.iterrows():
            if "__metadata_source" in r and pd.notna(r["__metadata_source"]):
                sources.append(str(r["__metadata_source"]))

            for c in KEEP_META:
                if c == "trial_id":
                    continue

                if c not in row or pd.isna(row.get(c)):
                    val = r.get(c, np.nan)
                    if pd.notna(val):
                        row[c] = val

        row["__metadata_source"] = sources[0] if sources else np.nan
        collapsed_rows.append(row)

    metadata_bank = pd.DataFrame(collapsed_rows)

    # Guarantee all expected columns.
    for c in KEEP_META:
        if c not in metadata_bank.columns:
            metadata_bank[c] = np.nan

    return metadata_bank, source_report


def _merge_metadata_into_judgments(j: pd.DataFrame, label: str) -> pd.DataFrame:
    """
    Merge judgment rows with recovered metadata.
    """
    j = j.copy()
    needed_ids = set(j["trial_id"].astype(str).str.strip().unique())

    metadata_bank, source_report = _build_metadata_bank(needed_ids)

    # Save metadata source report for debugging.
    report_dir = Path(FILES["master_judgments"]).parent / "diagnostics"
    report_dir.mkdir(parents=True, exist_ok=True)

    safe_label = (
        label.lower()
        .replace(" ", "_")
        .replace("—", "_")
        .replace("-", "_")
        .replace("/", "_")
    )

    report_path = report_dir / f"metadata_recovery_report_{safe_label}.csv"
    atomic_write_csv(source_report, report_path)

    print(f"\nMetadata recovery report saved: {report_path}")

    if len(source_report):
        print("Top metadata sources:")
        display_cols = ["status", "matched_trial_ids", "usable_rows", "path"]
        print(
            source_report.sort_values(
                ["matched_trial_ids", "usable_rows"],
                ascending=False,
            )[display_cols].head(12).to_string(index=False)
        )

    if metadata_bank.empty:
        raise RuntimeError(
            f"{label}: No metadata source matched the judgment trial_id values.\n"
            f"Checked {len(source_report)} possible metadata files.\n"
            f"Report saved to: {report_path}\n\n"
            "This means the old matching trial_master/master_judgments files are missing. "
            "Restore the archive that was created before the Primary-4 transition, or "
            "find the old trial_master that produced these judgment files."
        )

    # Rename metadata columns before merge.
    meta = metadata_bank.copy()

    meta_rename = {
        c: f"__meta_{c}"
        for c in meta.columns
        if c != "trial_id"
    }

    meta = meta.rename(columns=meta_rename)

    merged = j.merge(meta, on="trial_id", how="left")

    # Coalesce metadata from metadata bank into judgment dataframe.
    for col in KEEP_META:
        if col == "trial_id":
            continue

        meta_col = f"__meta_{col}"

        if meta_col in merged.columns and col in merged.columns:
            merged[col] = _blank_to_na(merged[col]).combine_first(
                _blank_to_na(merged[meta_col])
            )
        elif meta_col in merged.columns and col not in merged.columns:
            merged[col] = merged[meta_col]

    # Keep source info for audit.
    if "__meta___metadata_source" in merged.columns:
        merged["metadata_source"] = merged["__meta___metadata_source"]

    drop_cols = [c for c in merged.columns if c.startswith("__meta_")]
    if drop_cols:
        merged = merged.drop(columns=drop_cols)

    return merged


def _compile_one(
    judge_files: List[Path],
    out_path: Path,
    label: str,
    required: bool = False,
) -> None:
    frames = []
    existing_files = _existing_files(judge_files)

    if not existing_files:
        msg = f"No judge files found for {label}; skipping {out_path.name}"
        if required:
            raise RuntimeError(msg)
        log.warning(msg)
        return

    for p in existing_files:
        df = pd.read_csv(p)
        df["source_file"] = p.name
        frames.append(df)

    j = pd.concat(frames, ignore_index=True)
    j = _add_metadata_aliases(j)
    j = _normalize_basic_columns(j)

    if "trial_id" not in j.columns:
        raise RuntimeError(f"{label}: judgment file is missing trial_id")

    if "judge_family" not in j.columns:
        raise RuntimeError(f"{label}: judgment file is missing judge_family")

    if "winner" not in j.columns:
        raise RuntimeError(f"{label}: judgment file is missing winner")

    # Recover metadata from old/current sources.
    merged = _merge_metadata_into_judgments(j, label)
    merged = _normalize_basic_columns(merged)

    # Final metadata check.
    missing_required_cols = [c for c in REQUIRED_META if c not in merged.columns]
    if missing_required_cols:
        raise RuntimeError(
            f"{label}: missing required metadata columns after recovery: "
            f"{missing_required_cols}"
        )

    missing_counts = {}

    for col in REQUIRED_META:
        missing_counts[col] = int(_blank_to_na(merged[col]).isna().sum())

    bad_missing = {k: v for k, v in missing_counts.items() if v > 0}

    if bad_missing:
        mask = pd.concat(
            [_blank_to_na(merged[c]).isna() for c in bad_missing.keys()],
            axis=1,
        ).any(axis=1)

        example_cols = ["trial_id", "source_file", "metadata_source"] + [
            c for c in REQUIRED_META if c in merged.columns
        ]

        examples = merged.loc[
            mask,
            [c for c in example_cols if c in merged.columns],
        ].head(10)

        raise RuntimeError(
            f"{label}: required metadata still missing after searching old/current "
            f"metadata files: {bad_missing}\n\n"
            f"Examples:\n{examples.to_string(index=False)}\n\n"
            "Open the metadata recovery report printed above. If every source has "
            "matched_trial_ids = 0, then the old trial_master/master_judgments file "
            "that matches these judgment trial_ids is not currently in ROOT_DIR or archives."
        )

    # Policy check: primary outputs must not contain Falcon judge rows.
    if "EXCLUDED FALCON" not in label:
        bad_judges = sorted(
            set(merged["judge_family"].dropna().astype(str))
            - set(PRIMARY_JUDGE_FAMILIES)
        )

        if bad_judges:
            raise RuntimeError(
                f"{label}: excluded/non-primary judges present after compile: {bad_judges}"
            )

    # Save.
    atomic_write_csv(merged, out_path)

    print(f"\n{label}")
    print("-" * 80)
    print("Files used:", len(existing_files))

    for p in existing_files:
        print(" ", p.name)

    print("Rows:", len(merged))

    if "metadata_source" in merged.columns:
        print("Metadata sources used:")
        print(merged["metadata_source"].value_counts(dropna=False).head(10).to_string())

    print("Judges:")
    print(merged["judge_family"].value_counts().to_string())

    print("Winner counts:")
    print(merged["winner"].value_counts(dropna=False).to_string())

    log.info(f"  {out_path.name}: {len(merged)} rows")


def _validate_primary_master(path: Path) -> bool:
    try:
        df = pd.read_csv(path)
        df = _normalize_basic_columns(df)

        required_cols = [
            "trial_id",
            "winner",
            "judge_family",
            "prompt_id",
            "pair_id",
            "family_1",
            "family_2",
            "order",
            "model_a",
            "model_b",
        ]

        missing = [c for c in required_cols if c not in df.columns]

        if missing:
            print(f"[VALIDATE FAIL] {Path(path).name}: missing columns {missing}")
            return False

        bad_judges = sorted(
            set(df["judge_family"].dropna().astype(str)) - set(PRIMARY_JUDGE_FAMILIES)
        )

        if bad_judges:
            print(
                f"[VALIDATE FAIL] {Path(path).name}: "
                f"excluded/non-primary judges present: {bad_judges}"
            )
            return False

        for col in [
            "prompt_id",
            "pair_id",
            "family_1",
            "family_2",
            "order",
            "model_a",
            "model_b",
        ]:
            if _blank_to_na(df[col]).isna().any():
                print(f"[VALIDATE FAIL] {Path(path).name}: missing values in {col}")
                return False

        if len(df) < 100:
            print(f"[VALIDATE FAIL] {Path(path).name}: too few rows: {len(df)}")
            return False

        return True

    except Exception as e:
        print(f"[VALIDATE FAIL] {Path(path).name}: {e}")
        return False


# ---------------------------------------------------------------------------
# Main compile step
# ---------------------------------------------------------------------------

def step_compile_judgments():
    # 1. Large rubric, primary analysis.
    _compile_one(
        [_judgment_path("large", f, neutral=False) for f in PRIMARY_JUDGE_FAMILIES],
        FILES["master_judgments"],
        label="LARGE RUBRIC MASTER — PRIMARY JUDGES ONLY",
        required=True,
    )

    # 2. Large neutral, robustness analysis.
    _compile_one(
        [_judgment_path("large", f, neutral=True) for f in PRIMARY_JUDGE_FAMILIES],
        FILES["master_judgments_neu"],
        label="LARGE NEUTRAL MASTER — PRIMARY JUDGES ONLY",
        required=False,
    )

    # 3. Small rubric, scale comparison.
    _compile_one(
        [_judgment_path("small", f, neutral=False) for f in PRIMARY_JUDGE_FAMILIES],
        FILES["master_judgments_sm"],
        label="SMALL RUBRIC MASTER — PRIMARY JUDGES ONLY",
        required=False,
    )

    # 4. Falcon judge diagnostics only.
    falcon_files = []

    for f in EXCLUDED_JUDGE_FAMILIES:
        falcon_files.extend(
            [
                _judgment_path("large", f, neutral=False),
                _judgment_path("large", f, neutral=True),
                _judgment_path("small", f, neutral=False),
            ]
        )

    _compile_one(
        falcon_files,
        FILES["master_judgments_excluded_falcon"],
        label="EXCLUDED FALCON JUDGE FILES — DIAGNOSTIC ONLY",
        required=False,
    )


run_step(
    [FILES["master_judgments"]],
    step_compile_judgments,
    "compile_master_judgments_primary_excluding_falcon",
    force=True,
    validators={FILES["master_judgments"]: _validate_primary_master},
)

print("\n✓ Master judgments compiled")
print("Primary master:", FILES["master_judgments"])
print("Neutral master:", FILES["master_judgments_neu"])
print("Small master:", FILES["master_judgments_sm"])
print("Falcon diagnostic master:", FILES["master_judgments_excluded_falcon"])

In [ ]:
# ============================================================================
# Cell 5.4B — Judgment-file semantic guard / naming fix
# ============================================================================
"""
Cell 5.4B — Fix judgment-file naming ambiguity.

Purpose:
- Prevent the old name `master_judgments_excluded_falcon.csv` from being
  misinterpreted as the Primary-4 analysis file.
- Clarify that the main judgment master is:
      primary judges only + full candidate set
  not Primary-4.
- Clarify that Primary-4 is created only later in Cell 6.1 by filtering
  candidates to llama/qwen/gemma/yi.
- If an old Falcon diagnostic file exists under a misleading name, copy it to
  the clearer name:
      falcon_judge_diagnostic_master.csv
- Save a manifest documenting every important judgment/effective-winner file.

This cell does not recompute judgments. It only audits, aliases, and documents.
"""

from pathlib import Path
from typing import Any, Dict
import pandas as pd
import numpy as np
import json
import shutil

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "FULL_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "judgments_master" not in PATHS:
    if "judgments" in PATHS:
        PATHS["judgments_master"] = Path(PATHS["judgments"]) / "master"
    elif "root" in PATHS:
        PATHS["judgments_master"] = Path(PATHS["root"]) / "judgments" / "master"
    elif "ROOT_DIR" in globals():
        PATHS["judgments_master"] = Path(ROOT_DIR) / "judgments" / "master"
    else:
        PATHS["judgments_master"] = Path("/content/drive/MyDrive/tribal_pref_v11/judgments/master")

Path(PATHS["judgments_master"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "judgment_file_semantics",
    Path(PATHS["judgments_master"]) / "judgment_file_semantics_manifest.json",
)

FILES.setdefault(
    "falcon_judge_diagnostic_master",
    Path(PATHS["judgments_master"]) / "falcon_judge_diagnostic_master.csv",
)

FILES.setdefault(
    "master_judgments_primary_judges_full_candidates",
    FILES.get("master_judgments", Path(PATHS["judgments_master"]) / "master_judgments.csv"),
)


PRIMARY_JUDGES_54B = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_54B = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]
FULL_CANDS_54B = [str(x).strip().lower() for x in FULL_CANDIDATE_FAMILIES]


def _54b_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""

    s = str(x).strip().lower()

    if "llama" in s:
        return "llama"
    if "qwen" in s:
        return "qwen"
    if "gemma" in s:
        return "gemma"
    if "falcon" in s:
        return "falcon"
    if "yi" in s:
        return "yi"

    return s


def _54b_ensure_family_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "family_1" not in df.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a"]:
            if c in df.columns:
                df["family_1"] = df[c]
                break

    if "family_2" not in df.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b"]:
            if c in df.columns:
                df["family_2"] = df[c]
                break

    if "family_1" not in df.columns and "model_a" in df.columns:
        df["family_1"] = df["model_a"]

    if "family_2" not in df.columns and "model_b" in df.columns:
        df["family_2"] = df["model_b"]

    for c in ["judge_family", "family_1", "family_2"]:
        if c in df.columns:
            df[c] = df[c].map(_54b_clean_family)

    return df


def _54b_file_summary(path: Any) -> Dict[str, Any]:
    path = Path(path)

    if not path.exists():
        return {
            "exists": False,
            "path": str(path),
        }

    try:
        df = pd.read_csv(path)
    except Exception as e:
        return {
            "exists": True,
            "path": str(path),
            "read_error": str(e),
        }

    df = _54b_ensure_family_cols(df)

    out = {
        "exists": True,
        "path": str(path),
        "rows": int(len(df)),
        "columns": list(df.columns),
    }

    if "prompt_id" in df.columns:
        out["n_prompts"] = int(df["prompt_id"].nunique())

    if "judge_family" in df.columns:
        out["judge_families"] = sorted(df["judge_family"].dropna().unique().tolist())
        out["judge_counts"] = {
            str(k): int(v)
            for k, v in df["judge_family"].value_counts(dropna=False).to_dict().items()
        }

    candidate_families = set()

    if "family_1" in df.columns:
        candidate_families |= set(df["family_1"].dropna().unique().tolist())

    if "family_2" in df.columns:
        candidate_families |= set(df["family_2"].dropna().unique().tolist())

    if candidate_families:
        out["candidate_families"] = sorted(candidate_families)

    if "winner" in df.columns:
        out["winner_counts"] = {
            str(k): int(v)
            for k, v in df["winner"].value_counts(dropna=False).to_dict().items()
        }

    return out


def _54b_find_old_excluded_falcon_file() -> Path | None:
    candidates = []

    if "master_judgments_excluded_falcon" in FILES:
        candidates.append(Path(FILES["master_judgments_excluded_falcon"]))

    candidates.append(Path(PATHS["judgments_master"]) / "master_judgments_excluded_falcon.csv")

    if "master_judgments" in FILES:
        p = Path(FILES["master_judgments"])
        candidates.append(p.with_name("master_judgments_excluded_falcon.csv"))

    for p in candidates:
        if p.exists():
            return p

    return None


def _54b_classify_old_excluded_file(path: Path) -> Dict[str, Any]:
    summary = _54b_file_summary(path)

    if not summary.get("exists") or "judge_families" not in summary:
        summary["semantic_classification"] = "unknown"
        summary["recommended_action"] = "inspect manually"
        return summary

    judges = set(summary["judge_families"])
    candidates = set(summary.get("candidate_families", []))

    if judges == {"falcon"}:
        summary["semantic_classification"] = "falcon_judge_diagnostic"
        summary["recommended_action"] = "copy to falcon_judge_diagnostic_master.csv and never call it Primary-4"

    elif "falcon" not in judges and "falcon" in candidates:
        summary["semantic_classification"] = "primary_judges_full_candidates"
        summary["recommended_action"] = "do not call this Primary-4; Primary-4 candidate filtering happens in Cell 6.1"

    elif "falcon" not in judges and "falcon" not in candidates:
        summary["semantic_classification"] = "possibly_primary4_or_other_filtered_file"
        summary["recommended_action"] = "verify against effective_winners_primary before using"

    else:
        summary["semantic_classification"] = "mixed_or_unexpected"
        summary["recommended_action"] = "inspect manually"

    return summary


def _step_judgment_semantic_guard_54b():
    print("=" * 100)
    print("JUDGMENT FILE SEMANTIC GUARD / NAMING FIX")
    print("=" * 100)

    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] missing. Run Cell 5.4 first.")

    master_path = Path(FILES["master_judgments"])
    master_summary = _54b_file_summary(master_path)

    if "judge_families" not in master_summary:
        raise RuntimeError("master_judgments has no judge_family column or could not be read.")

    master_judges = set(master_summary["judge_families"])
    master_candidates = set(master_summary.get("candidate_families", []))

    # Main safety checks.
    if "falcon" in master_judges:
        raise RuntimeError(
            "Falcon is present as a judge in FILES['master_judgments']. "
            "This should not happen after the corrected Cell 5.4."
        )

    missing_primary_judges = sorted(set(PRIMARY_JUDGES_54B) - master_judges)

    if missing_primary_judges:
        raise RuntimeError(
            f"master_judgments is missing expected primary judges: {missing_primary_judges}"
        )

    unexpected_judges = sorted(master_judges - set(PRIMARY_JUDGES_54B))

    if unexpected_judges:
        raise RuntimeError(
            f"master_judgments contains unexpected judges: {unexpected_judges}"
        )

    old_excluded_path = _54b_find_old_excluded_falcon_file()
    old_excluded_summary = None
    copied_old_excluded_to_clear_name = False

    if old_excluded_path is not None:
        old_excluded_summary = _54b_classify_old_excluded_file(old_excluded_path)

        if old_excluded_summary.get("semantic_classification") == "falcon_judge_diagnostic":
            clear_path = Path(FILES["falcon_judge_diagnostic_master"])

            if old_excluded_path.resolve() != clear_path.resolve():
                shutil.copy2(old_excluded_path, clear_path)
                copied_old_excluded_to_clear_name = True

            FILES["falcon_judge_diagnostic_master"] = clear_path

        # Keep old key only as a backward-compatible alias, but do not use it
        # in analysis prose or downstream headline analysis.
        FILES["master_judgments_excluded_falcon_legacy_name"] = old_excluded_path

    primary_effective_summary = None

    if "effective_winners_primary" in FILES and Path(FILES["effective_winners_primary"]).exists():
        primary_effective_summary = _54b_file_summary(FILES["effective_winners_primary"])

        pe_judges = set(primary_effective_summary.get("judge_families", []))
        pe_candidates = set(primary_effective_summary.get("candidate_families", []))

        if "falcon" in pe_judges or "falcon" in pe_candidates:
            raise RuntimeError(
                "Falcon appears in effective_winners_primary. "
                "Primary-4 should exclude Falcon as judge and candidate."
            )

    manifest = {
        "analysis_label": "judgment_file_semantics_manifest",
        "schema_note": (
            "This manifest prevents naming confusion. master_judgments.csv is "
            "primary judges only with the full candidate set. It is not itself "
            "the Primary-4 effective analysis file. Primary-4 is produced in "
            "Cell 6.1 as effective_winners_primary.csv."
        ),
        "primary_judge_families": PRIMARY_JUDGES_54B,
        "primary_candidate_families": PRIMARY_CANDS_54B,
        "full_candidate_families": FULL_CANDS_54B,
        "canonical_files": {
            "master_judgments": {
                "path": str(master_path),
                "meaning": (
                    "Canonical judgment master for corrected pipeline: "
                    "primary judges only; full candidate set may include Falcon."
                ),
                "is_primary4": False,
                "used_for": [
                    "building effective_winners_full",
                    "building effective_winners_primary after candidate filtering",
                    "robustness analyses",
                ],
            },
            "effective_winners_primary": {
                "path": str(FILES.get("effective_winners_primary", "")),
                "meaning": (
                    "Canonical Primary-4 analysis file: primary judges and "
                    "primary candidates only."
                ),
                "is_primary4": True,
                "used_for": [
                    "headline TPS",
                    "bootstrap",
                    "permutation tests",
                    "mechanism analyses",
                ],
            },
            "effective_winners_full": {
                "path": str(FILES.get("effective_winners_full", "")),
                "meaning": (
                    "Full candidate appendix/sensitivity file. Falcon may appear "
                    "as candidate but not as headline judge."
                ),
                "is_primary4": False,
                "used_for": [
                    "appendix sensitivity",
                    "Falcon candidate validation",
                ],
            },
            "falcon_judge_diagnostic_master": {
                "path": str(FILES.get("falcon_judge_diagnostic_master", "")),
                "meaning": (
                    "Falcon judge diagnostic only, if present. Never describe this "
                    "as Primary-4."
                ),
                "is_primary4": False,
                "used_for": [
                    "Falcon judge exclusion diagnostics only",
                ],
            },
        },
        "master_judgments_summary": master_summary,
        "legacy_excluded_falcon_file_found": old_excluded_path is not None,
        "legacy_excluded_falcon_file_summary": old_excluded_summary,
        "copied_legacy_excluded_file_to_clear_name": copied_old_excluded_to_clear_name,
        "primary_effective_summary_if_available": primary_effective_summary,
        "paper_language": {
            "correct": [
                "The canonical judgment master contains Primary-4 judges and the full candidate set.",
                "The Primary-4 headline analysis is computed from effective_winners_primary.csv.",
                "Falcon is excluded from the headline judge panel and retained only for appendix/sensitivity diagnostics.",
            ],
            "incorrect": [
                "master_judgments_excluded_falcon.csv is the Primary-4 dataset.",
                "master_judgments.csv is already Primary-4.",
                "Falcon-excluded master judgments are the headline analysis file.",
            ],
        },
    }

    atomic_write_json(manifest, FILES["judgment_file_semantics"])

    print("\nCanonical master_judgments summary")
    print("-" * 100)
    print(json.dumps({
        "path": master_summary.get("path"),
        "rows": master_summary.get("rows"),
        "n_prompts": master_summary.get("n_prompts"),
        "judge_families": master_summary.get("judge_families"),
        "candidate_families": master_summary.get("candidate_families"),
        "winner_counts": master_summary.get("winner_counts"),
    }, indent=2))

    print("\nLegacy excluded-Falcon file")
    print("-" * 100)

    if old_excluded_path is None:
        print("No legacy master_judgments_excluded_falcon.csv file found.")
    else:
        print(json.dumps({
            "path": str(old_excluded_path),
            "classification": old_excluded_summary.get("semantic_classification"),
            "recommended_action": old_excluded_summary.get("recommended_action"),
            "copied_to": str(FILES["falcon_judge_diagnostic_master"]) if copied_old_excluded_to_clear_name else None,
        }, indent=2))

    print("\nCorrect language to use")
    print("-" * 100)
    for s in manifest["paper_language"]["correct"]:
        print("✓", s)

    print("\nLanguage to avoid")
    print("-" * 100)
    for s in manifest["paper_language"]["incorrect"]:
        print("✗", s)

    print("\nSaved manifest:")
    print(FILES["judgment_file_semantics"])


run_step(
    [FILES["judgment_file_semantics"]],
    _step_judgment_semantic_guard_54b,
    "judgment_file_semantic_guard_naming_fix",
    force=False,
    validators={
        FILES["judgment_file_semantics"]: lambda p: json_has(
            p,
            [
                "analysis_label",
                "canonical_files",
                "master_judgments_summary",
                "paper_language",
            ],
        )
    },
)

print("\n✓ Cell 5.4B complete")

## PHASE 6 — Effective Winners & Preference Matrix

Reconciling AB/BA judgments into a single weighted "effective winner" per
(judge, prompt, pair). Multiple weighting schemes are exposed for the
multiverse analysis (Phase 9).



In [ ]:
# ============================================================================
# Cell 6.1 — Effective winners: Primary-4 plus Full-5 appendix sensitivity
# ============================================================================
"""
Build effective winners from AB/BA judgments.

Policy:
- Primary judges: llama, qwen, gemma, yi
- Falcon is excluded as judge.
- Primary candidates: llama, qwen, gemma, yi
- Full appendix candidates: llama, qwen, falcon, gemma, yi
- Inconsistent AB/BA is retained as 0.5 / 0.5 under the standard scheme.

Outputs:
- effective_winners_primary.csv
- effective_winners_full.csv
- effective_winners.csv alias to primary
- effective_winners_reconciliation_audit.csv
- candidate_subset_audit.json
"""

from pathlib import Path
from typing import Any, Dict, List, Sequence, Optional
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES", "PATHS", "PRIMARY_JUDGE_FAMILIES", "PRIMARY_CANDIDATE_FAMILIES",
    "FULL_CANDIDATE_FAMILIES", "run_step", "atomic_write_csv",
    "atomic_write_json", "csv_has", "json_has"
]
missing = [x for x in required_globals if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required globals. Run setup/helper cells first: {missing}")

def _ensure_path_61(key: str, subdir: str) -> Path:
    if key not in PATHS:
        if "analysis" in PATHS:
            base = Path(PATHS["analysis"])
        elif "root" in PATHS:
            base = Path(PATHS["root"]) / "analysis"
        elif "ROOT_DIR" in globals():
            base = Path(ROOT_DIR) / "analysis"
        else:
            base = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")
        PATHS[key] = base / subdir
    Path(PATHS[key]).mkdir(parents=True, exist_ok=True)
    return Path(PATHS[key])

analysis_pref = _ensure_path_61("analysis_pref", "preference")

FILES.setdefault("effective_winners_primary", analysis_pref / "effective_winners_primary.csv")
FILES.setdefault("effective_winners_full", analysis_pref / "effective_winners_full.csv")
FILES.setdefault("effective_winners", FILES["effective_winners_primary"])
FILES.setdefault("candidate_subset_audit", analysis_pref / "candidate_subset_audit.json")
FILES.setdefault("effective_winners_reconciliation_audit", analysis_pref / "effective_winners_reconciliation_audit.csv")

PRIMARY_JUDGES_61 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_61 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]
FULL_CANDS_61 = [str(x).strip().lower() for x in FULL_CANDIDATE_FAMILIES]
ALL_FAMS_61 = sorted(set(PRIMARY_JUDGES_61 + PRIMARY_CANDS_61 + FULL_CANDS_61 + ["falcon"]))

def _clean_61(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def _lower_61(x):
    return _clean_61(x).lower()

def _norm_family_61(x) -> Optional[str]:
    s = _lower_61(x)
    if not s:
        return None
    if s in ALL_FAMS_61:
        return s
    patterns = {
        "llama": ["llama", "meta-llama"],
        "qwen": ["qwen"],
        "gemma": ["gemma"],
        "yi": ["yi-1.5", "yi_", "yi-", "yi "],
        "falcon": ["falcon"],
    }
    for fam, keys in patterns.items():
        if any(k in s for k in keys):
            return fam
    if s == "yi":
        return "yi"
    return None

def _norm_winner_61(x) -> str:
    s = _lower_61(x)
    if s in {"a", "response_a", "response a", "option_a", "option a", "left", "1"}:
        return "A"
    if s in {"b", "response_b", "response b", "option_b", "option b", "right", "2"}:
        return "B"
    if s in {"tie", "draw", "equal", "both", "neither", "none", "", "nan", "invalid"}:
        return "tie"
    return "tie"

def _standardize_master_judgments_61(j: pd.DataFrame) -> pd.DataFrame:
    j = j.copy()

    if "family_1" not in j.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a"]:
            if c in j.columns:
                j["family_1"] = j[c]
                break
    if "family_2" not in j.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b"]:
            if c in j.columns:
                j["family_2"] = j[c]
                break

    required = ["trial_id", "prompt_id", "judge_family", "winner", "family_1", "family_2"]
    missing = [c for c in required if c not in j.columns]
    if missing:
        raise RuntimeError(f"master_judgments missing required columns: {missing}. Available: {list(j.columns)}")

    if "pair_id" not in j.columns:
        j["pair_id"] = (
            j["prompt_id"].astype(str) + "::" +
            j["family_1"].astype(str) + "_vs_" + j["family_2"].astype(str)
        )

    if "order" not in j.columns:
        j["order"] = "AB"

    if "model_a" not in j.columns:
        j["model_a"] = j.get("family_1", "")
    if "model_b" not in j.columns:
        j["model_b"] = j.get("family_2", "")

    for c in ["trial_id", "prompt_id", "pair_id", "judge_family", "winner", "family_1", "family_2", "order", "model_a", "model_b"]:
        j[c] = j[c].astype(str).str.strip()

    j["judge_family"] = j["judge_family"].map(_norm_family_61)
    j["family_1"] = j["family_1"].map(_norm_family_61)
    j["family_2"] = j["family_2"].map(_norm_family_61)
    j["winner_token"] = j["winner"].map(_norm_winner_61)

    j["family_a"] = j["model_a"].map(_norm_family_61)
    j["family_b"] = j["model_b"].map(_norm_family_61)

    def _fill_pos(row):
        f1, f2 = row["family_1"], row["family_2"]
        fa, fb = row["family_a"], row["family_b"]
        order = _lower_61(row["order"])
        if fa in ALL_FAMS_61 and fb in ALL_FAMS_61:
            return fa, fb
        if order in {"ba", "b_a", "2_1", "21", "swapped", "family_2_first"}:
            return f2, f1
        return f1, f2

    pos = j.apply(_fill_pos, axis=1, result_type="expand")
    j["family_a"] = pos[0]
    j["family_b"] = pos[1]

    before = len(j)
    j = j.dropna(subset=["judge_family", "family_1", "family_2", "family_a", "family_b"]).copy()
    dropped = before - len(j)
    if dropped:
        print(f"Dropped {dropped} rows with unrecognized families.")

    for c in ["source", "category", "split"]:
        if c not in j.columns:
            j[c] = ""

    return j.reset_index(drop=True)

def _trial_support_61(row) -> Dict[str, Any]:
    f1, f2 = row["family_1"], row["family_2"]
    w = row["winner_token"]

    if w == "tie":
        return {"support_family_1": 0.5, "support_family_2": 0.5, "trial_outcome": "tie"}

    winner_family = row["family_a"] if w == "A" else row["family_b"]

    if winner_family == f1:
        return {"support_family_1": 1.0, "support_family_2": 0.0, "trial_outcome": "family_1_win"}
    if winner_family == f2:
        return {"support_family_1": 0.0, "support_family_2": 1.0, "trial_outcome": "family_2_win"}

    return {"support_family_1": 0.5, "support_family_2": 0.5, "trial_outcome": "unmapped_tie"}

def _classify_group_61(trial_outcomes: List[str], s1: float, s2: float) -> str:
    outs = list(trial_outcomes)
    decisive = [x for x in outs if x in {"family_1_win", "family_2_win"}]
    ties = [x for x in outs if "tie" in x]

    if len(decisive) == 0:
        return "tie"
    if len(ties) > 0 and len(decisive) > 0:
        return "partial_tie"
    if len(set(decisive)) == 1:
        return "consistent_win"
    if abs(s1 - 0.5) < 1e-12 and abs(s2 - 0.5) < 1e-12:
        return "inconsistent_split"
    return "mixed"

def build_effective_winners(
    judgments: pd.DataFrame,
    scheme: str = "standard",
    partial_tie_weight: float = 0.75,
    include_incomplete: bool = True,
) -> pd.DataFrame:
    """
    Convert raw AB/BA judgment rows into one effective row per
    judge × prompt × pair.

    Corrected support schema:
    - support_family_1
    - support_family_2

    Standard behavior:
    - consistent A/B winner: 1.0 / 0.0
    - tie/tie: 0.5 / 0.5
    - win/tie: 0.75 / 0.25
    - inconsistent AB/BA split: 0.5 / 0.5
    """
    j = _standardize_master_judgments_61(judgments)

    key_cols = ["judge_family", "prompt_id", "pair_id", "family_1", "family_2"]
    meta_cols = [c for c in ["source", "category", "split"] if c in j.columns]

    rows = []
    audit_rows = []

    for keys, g in j.groupby(key_cols, dropna=False):
        rec = dict(zip(key_cols, keys))
        g = g.copy()

        supports = g.apply(_trial_support_61, axis=1, result_type="expand")
        g = pd.concat([g.reset_index(drop=True), supports.reset_index(drop=True)], axis=1)

        s1 = float(g["support_family_1"].mean())
        s2 = float(g["support_family_2"].mean())

        if not np.isclose(s1 + s2, 1.0):
            total = s1 + s2
            s1, s2 = s1 / total, s2 / total

        outcome = _classify_group_61(g["trial_outcome"].tolist(), s1, s2)

        # Optional stricter schemes for multiverse cells.
        if scheme in {"drop_inconsistent", "strict_consistent"} and outcome == "inconsistent_split":
            continue
        if scheme in {"consistent_only", "strict_consistent"} and outcome != "consistent_win":
            continue
        if (not include_incomplete) and len(g) < 2:
            continue

        row = {
            **rec,
            "support_family_1": s1,
            "support_family_2": s2,
            "outcome": outcome,
            "n_trials": int(len(g)),
            "trial_ids": "|".join(g["trial_id"].astype(str).tolist()),
            "winner_tokens": "|".join(g["winner_token"].astype(str).tolist()),
            "trial_outcomes": "|".join(g["trial_outcome"].astype(str).tolist()),
        }

        for c in meta_cols:
            row[c] = g[c].iloc[0]

        if s1 > s2:
            row["winner_family"] = rec["family_1"]
            row["loser_family"] = rec["family_2"]
            row["winner_weight"] = s1
            row["loser_weight"] = s2
        elif s2 > s1:
            row["winner_family"] = rec["family_2"]
            row["loser_family"] = rec["family_1"]
            row["winner_weight"] = s2
            row["loser_weight"] = s1
        else:
            row["winner_family"] = "tie"
            row["loser_family"] = "tie"
            row["winner_weight"] = 0.5
            row["loser_weight"] = 0.5

        rows.append(row)

        audit_rows.append({
            **rec,
            "status": "ok",
            "outcome": outcome,
            "n_trials": int(len(g)),
            "support_family_1": s1,
            "support_family_2": s2,
        })

    eff = pd.DataFrame(rows)
    audit = pd.DataFrame(audit_rows)

    if len(eff):
        eff["support_family_1"] = eff["support_family_1"].astype(float)
        eff["support_family_2"] = eff["support_family_2"].astype(float)

    build_effective_winners.last_audit = audit
    return eff

def _filter_candidates_61(eff: pd.DataFrame, families: Sequence[str]) -> pd.DataFrame:
    fams = set([str(x).strip().lower() for x in families])
    if eff.empty:
        return eff.copy()
    return eff[eff["family_1"].isin(fams) & eff["family_2"].isin(fams)].copy().reset_index(drop=True)

def _validate_eff_61(eff: pd.DataFrame, label: str):
    required = ["prompt_id", "judge_family", "family_1", "family_2", "support_family_1", "support_family_2", "outcome"]
    missing = [c for c in required if c not in eff.columns]
    if missing:
        raise RuntimeError(f"{label}: missing columns {missing}")
    if not np.allclose(eff["support_family_1"].astype(float) + eff["support_family_2"].astype(float), 1.0):
        raise RuntimeError(f"{label}: support weights do not sum to 1.")
    return True

def _step_effective_winners_61():
    print("=" * 100)
    print("BUILDING EFFECTIVE WINNERS WITH INCONSISTENT AB/BA RETAINED AS 0.5/0.5")
    print("=" * 100)

    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] is missing. Run Cell 5.4 first.")

    j = pd.read_csv(FILES["master_judgments"])
    j = _standardize_master_judgments_61(j)

    # Primary judges only. This excludes Falcon as a judge.
    j = j[j["judge_family"].isin(PRIMARY_JUDGES_61)].copy()

    print("Loaded master_judgments:", FILES["master_judgments"])
    print("Raw judgment rows after primary-judge filter:", len(j))
    print("Judge families:", sorted(j["judge_family"].unique().tolist()))
    print("Candidate families seen:", sorted((set(j["family_1"]) | set(j["family_2"]))))

    eff_full = build_effective_winners(j, scheme="standard", include_incomplete=True)
    audit = build_effective_winners.last_audit

    eff_full = _filter_candidates_61(eff_full, FULL_CANDS_61)
    eff_primary = _filter_candidates_61(eff_full, PRIMARY_CANDS_61)

    _validate_eff_61(eff_full, "effective_winners_full")
    _validate_eff_61(eff_primary, "effective_winners_primary")

    if "falcon" in set(eff_primary["family_1"]) or "falcon" in set(eff_primary["family_2"]):
        raise RuntimeError("Falcon leaked into effective_winners_primary.")

    if "falcon" in set(eff_primary["judge_family"]):
        raise RuntimeError("Falcon leaked into primary judges.")

    atomic_write_csv(eff_full, FILES["effective_winners_full"])
    atomic_write_csv(eff_primary, FILES["effective_winners_primary"])
    atomic_write_csv(eff_primary, FILES["effective_winners"])
    atomic_write_csv(audit, FILES["effective_winners_reconciliation_audit"])

    subset_audit = {
        "analysis_label": "primary_4_with_full_5_appendix",
        "schema_note": "support_family_1/support_family_2; inconsistent AB/BA retained as 0.5/0.5.",
        "primary_judge_families": PRIMARY_JUDGES_61,
        "primary_candidate_families": PRIMARY_CANDS_61,
        "full_candidate_families": FULL_CANDS_61,
        "excluded_judge_families": ["falcon"],
        "n_effective_full": int(len(eff_full)),
        "n_effective_primary": int(len(eff_primary)),
        "full_outcome_distribution": eff_full["outcome"].value_counts().to_dict(),
        "primary_outcome_distribution": eff_primary["outcome"].value_counts().to_dict(),
        "falcon_absent_from_primary_candidates": bool(
            "falcon" not in set(eff_primary["family_1"]) and "falcon" not in set(eff_primary["family_2"])
        ),
        "falcon_absent_from_primary_judges": bool("falcon" not in set(eff_primary["judge_family"])),
        "outputs": {
            "effective_winners_full": str(FILES["effective_winners_full"]),
            "effective_winners_primary": str(FILES["effective_winners_primary"]),
            "effective_winners_alias": str(FILES["effective_winners"]),
            "audit": str(FILES["effective_winners_reconciliation_audit"]),
        },
    }
    atomic_write_json(subset_audit, FILES["candidate_subset_audit"])

    print("\nFull 5-family effective winners")
    print("-" * 80)
    print("Rows:", len(eff_full))
    print(eff_full["outcome"].value_counts().to_string())

    print("\nPrimary 4-family effective winners")
    print("-" * 80)
    print("Rows:", len(eff_primary))
    print(eff_primary["outcome"].value_counts().to_string())

    print("\nSaved:")
    print("Full:", FILES["effective_winners_full"])
    print("Primary:", FILES["effective_winners_primary"])
    print("Alias:", FILES["effective_winners"])
    print("Audit:", FILES["effective_winners_reconciliation_audit"])

run_step(
    [
        FILES["effective_winners_full"],
        FILES["effective_winners_primary"],
        FILES["candidate_subset_audit"],
        FILES["effective_winners_reconciliation_audit"],
    ],
    _step_effective_winners_61,
    "build_effective_winners_primary4_inconsistent_retained",
    force=False,
    validators={
        FILES["effective_winners_primary"]: lambda p: csv_has(
            p, ["prompt_id", "judge_family", "family_1", "family_2", "support_family_1", "support_family_2"], 100
        ),
        FILES["candidate_subset_audit"]: lambda p: json_has(
            p, ["primary_candidate_families", "full_candidate_families", "n_effective_primary"]
        ),
    },
)

print("\n✓ Cell 6.1 complete")

In [ ]:
# ============================================================================
# Cell 6.2 — Candidate-support preference matrix and TPS
# ============================================================================
"""
Build Primary-4 and Full-5 preference matrices from effective winners.

Outputs:
- preference_matrix_primary.csv
- preference_support_primary.csv
- tps_summary_primary.json
- preference_matrix_full.csv
- preference_support_full.csv
- tps_summary_full.json
"""

from pathlib import Path
from typing import Any, Dict, Sequence, Tuple
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES", "PATHS", "PRIMARY_JUDGE_FAMILIES", "PRIMARY_CANDIDATE_FAMILIES",
    "FULL_CANDIDATE_FAMILIES", "run_step", "atomic_write_csv",
    "atomic_write_json", "csv_has", "json_has"
]
missing = [x for x in required_globals if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing}")

def _ensure_path_62(key: str, subdir: str) -> Path:
    if key not in PATHS:
        if "analysis" in PATHS:
            base = Path(PATHS["analysis"])
        elif "root" in PATHS:
            base = Path(PATHS["root"]) / "analysis"
        elif "ROOT_DIR" in globals():
            base = Path(ROOT_DIR) / "analysis"
        else:
            base = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")
        PATHS[key] = base / subdir
    Path(PATHS[key]).mkdir(parents=True, exist_ok=True)
    return Path(PATHS[key])

analysis_pref = _ensure_path_62("analysis_pref", "preference")

FILES.setdefault("preference_matrix_primary", analysis_pref / "preference_matrix_primary.csv")
FILES.setdefault("preference_support_primary", analysis_pref / "preference_support_primary.csv")
FILES.setdefault("tps_summary_primary", analysis_pref / "tps_summary_primary.json")

FILES.setdefault("preference_matrix_full", analysis_pref / "preference_matrix_full.csv")
FILES.setdefault("preference_support_full", analysis_pref / "preference_support_full.csv")
FILES.setdefault("tps_summary_full", analysis_pref / "tps_summary_full.json")

# Backward-compatible aliases.
FILES.setdefault("preference_matrix", FILES["preference_matrix_primary"])
FILES.setdefault("preference_support", FILES["preference_support_primary"])
FILES.setdefault("tps_summary", FILES["tps_summary_primary"])

PRIMARY_JUDGES_62 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_62 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]
FULL_CANDS_62 = [str(x).strip().lower() for x in FULL_CANDIDATE_FAMILIES]

def _clean_62(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def _validate_eff_62(eff: pd.DataFrame, label: str) -> pd.DataFrame:
    eff = eff.copy()
    required = ["judge_family", "family_1", "family_2", "support_family_1", "support_family_2"]
    missing = [c for c in required if c not in eff.columns]
    if missing:
        raise RuntimeError(f"{label}: missing required columns {missing}")

    for c in ["judge_family", "family_1", "family_2"]:
        eff[c] = eff[c].map(_clean_62)

    eff["support_family_1"] = pd.to_numeric(eff["support_family_1"], errors="coerce")
    eff["support_family_2"] = pd.to_numeric(eff["support_family_2"], errors="coerce")

    if eff["support_family_1"].isna().any() or eff["support_family_2"].isna().any():
        raise RuntimeError(f"{label}: support columns contain NaNs.")

    if not np.allclose(eff["support_family_1"] + eff["support_family_2"], 1.0):
        raise RuntimeError(f"{label}: support columns do not sum to 1.")

    return eff

def build_preference_matrix(
    eff: pd.DataFrame,
    judge_families: Sequence[str],
    candidate_families: Sequence[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
    - preference matrix: judge_family × candidate_family support rate
    - support/opportunity matrix: number of appearances per cell
    """
    eff = _validate_eff_62(eff, "build_preference_matrix")

    judge_families = [str(x).strip().lower() for x in judge_families]
    candidate_families = [str(x).strip().lower() for x in candidate_families]

    num = pd.DataFrame(0.0, index=judge_families, columns=candidate_families)
    den = pd.DataFrame(0.0, index=judge_families, columns=candidate_families)

    for _, r in eff.iterrows():
        judge = r["judge_family"]
        f1 = r["family_1"]
        f2 = r["family_2"]

        if judge not in num.index:
            continue

        if f1 in num.columns:
            num.loc[judge, f1] += float(r["support_family_1"])
            den.loc[judge, f1] += 1.0

        if f2 in num.columns:
            num.loc[judge, f2] += float(r["support_family_2"])
            den.loc[judge, f2] += 1.0

    pref = num / den.replace(0, np.nan)
    pref = pref.fillna(0.5)

    return pref, den

def compute_tps(
    pref: pd.DataFrame,
    judge_families: Sequence[str],
    candidate_families: Sequence[str],
) -> Dict[str, Any]:
    judge_families = [str(x).strip().lower() for x in judge_families if str(x).strip().lower() in pref.index]
    candidate_families = [str(x).strip().lower() for x in candidate_families if str(x).strip().lower() in pref.columns]

    diag_fams = [f for f in judge_families if f in candidate_families]

    diag_values = {f: float(pref.loc[f, f]) for f in diag_fams}
    offdiag_values = [
        float(pref.loc[j, c])
        for j in judge_families
        for c in candidate_families
        if j != c
    ]

    per_family = {}
    for f in diag_fams:
        others = [c for c in candidate_families if c != f]
        per_family[f] = float(pref.loc[f, f] - pref.loc[f, others].mean()) if others else np.nan

    diag_mean = float(np.mean(list(diag_values.values()))) if diag_values else np.nan
    offdiag_mean = float(np.mean(offdiag_values)) if offdiag_values else np.nan

    return {
        "diag_mean": diag_mean,
        "offdiag_mean": offdiag_mean,
        "tps": float(diag_mean - offdiag_mean),
        "diag_values": diag_values,
        "per_family_tps": per_family,
    }

def _write_matrix_outputs_62(eff_path, matrix_path, support_path, summary_path, label, judge_fams, cand_fams):
    eff = pd.read_csv(eff_path)
    eff = _validate_eff_62(eff, label)

    if label == "primary_4":
        if "falcon" in set(eff["family_1"]) or "falcon" in set(eff["family_2"]) or "falcon" in set(eff["judge_family"]):
            raise RuntimeError("Falcon leaked into Primary-4 Cell 6.2.")

    pref, support = build_preference_matrix(eff, judge_fams, cand_fams)
    stats = compute_tps(pref, judge_fams, cand_fams)

    summary = {
        "analysis_label": label,
        "schema_note": "Computed from support_family_1/support_family_2.",
        "judge_families": list(judge_fams),
        "candidate_families": list(cand_fams),
        "n_effective_rows": int(len(eff)),
        "preference_matrix": pref.to_dict(),
        "support_matrix": support.to_dict(),
        **stats,
    }

    atomic_write_csv(pref.reset_index().rename(columns={"index": "judge_family"}), matrix_path)
    atomic_write_csv(support.reset_index().rename(columns={"index": "judge_family"}), support_path)
    atomic_write_json(summary, summary_path)

    return pref, support, summary

def _step_preference_tps_62():
    print("=" * 100)
    print("CANDIDATE-SUPPORT PREFERENCE MATRIX AND TPS")
    print("=" * 100)

    primary_pref, primary_support, primary_summary = _write_matrix_outputs_62(
        FILES["effective_winners_primary"],
        FILES["preference_matrix_primary"],
        FILES["preference_support_primary"],
        FILES["tps_summary_primary"],
        "primary_4",
        PRIMARY_JUDGES_62,
        PRIMARY_CANDS_62,
    )

    full_pref, full_support, full_summary = _write_matrix_outputs_62(
        FILES["effective_winners_full"],
        FILES["preference_matrix_full"],
        FILES["preference_support_full"],
        FILES["tps_summary_full"],
        "full_5_appendix",
        PRIMARY_JUDGES_62,
        FULL_CANDS_62,
    )

    # Aliases for old cells.
    atomic_write_csv(primary_pref.reset_index().rename(columns={"index": "judge_family"}), FILES["preference_matrix"])
    atomic_write_csv(primary_support.reset_index().rename(columns={"index": "judge_family"}), FILES["preference_support"])
    atomic_write_json(primary_summary, FILES["tps_summary"])

    print("\nPrimary-4 preference matrix")
    print("-" * 80)
    print(primary_pref.round(4).to_string())
    print("\nPrimary-4 support/opportunity matrix")
    print("-" * 80)
    print(primary_support.astype(int).to_string())
    print("\nPrimary-4 TPS")
    print("-" * 80)
    print(json.dumps({
        "diag_mean": primary_summary["diag_mean"],
        "offdiag_mean": primary_summary["offdiag_mean"],
        "tps": primary_summary["tps"],
        "per_family_tps": primary_summary["per_family_tps"],
    }, indent=2))

    print("\nFull-5 appendix TPS")
    print("-" * 80)
    print(json.dumps({
        "diag_mean": full_summary["diag_mean"],
        "offdiag_mean": full_summary["offdiag_mean"],
        "tps": full_summary["tps"],
    }, indent=2))

run_step(
    [
        FILES["preference_matrix_primary"],
        FILES["preference_support_primary"],
        FILES["tps_summary_primary"],
        FILES["preference_matrix_full"],
        FILES["preference_support_full"],
        FILES["tps_summary_full"],
    ],
    _step_preference_tps_62,
    "candidate_support_preference_matrix_tps_primary4",
    force=False,
    validators={
        FILES["preference_matrix_primary"]: lambda p: csv_has(p, ["judge_family"], 4),
        FILES["tps_summary_primary"]: lambda p: json_has(p, ["tps", "diag_mean", "offdiag_mean"]),
        FILES["tps_summary_full"]: lambda p: json_has(p, ["tps", "diag_mean", "offdiag_mean"]),
    },
)

print("\n✓ Cell 6.2 complete")

## PHASE 7 — Core Statistical Inference

- **Cluster bootstrap at the prompt level** (not trial level — within-prompt
  correlation is non-trivial and naive iid bootstrap gives CIs that are too
  tight).
- **Prompt-level permutation test** for the global TPS null.
- **Bradley–Terry residual decomposition** to net out global ability.
- **BH correction** for per-family TPS.



In [ ]:
# ============================================================================
# Cell 7.1 — Cluster bootstrap for Primary-4 TPS
# ============================================================================
"""
Cluster bootstrap confidence interval for Primary-4 TPS.

Uses prompt_id as the cluster.
Uses support_family_1/support_family_2 from Cell 6.1.
"""

from pathlib import Path
from typing import Any, Dict, Sequence
import pandas as pd
import numpy as np
import json
import time

required_globals = [
    "FILES", "PATHS", "PRIMARY_JUDGE_FAMILIES", "PRIMARY_CANDIDATE_FAMILIES",
    "run_step", "atomic_write_json", "json_has", "compute_tps", "build_preference_matrix"
]
missing = [x for x in required_globals if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing}")

def _ensure_path_71(key: str, subdir: str) -> Path:
    if key not in PATHS:
        if "analysis" in PATHS:
            base = Path(PATHS["analysis"])
        elif "root" in PATHS:
            base = Path(PATHS["root"]) / "analysis"
        elif "ROOT_DIR" in globals():
            base = Path(ROOT_DIR) / "analysis"
        else:
            base = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")
        PATHS[key] = base / subdir
    Path(PATHS[key]).mkdir(parents=True, exist_ok=True)
    return Path(PATHS[key])

analysis_stats = _ensure_path_71("analysis_stats", "stats")

FILES.setdefault("cluster_bootstrap_tps_primary", analysis_stats / "cluster_bootstrap_tps_primary.json")
FILES.setdefault("bootstrap_result", FILES["cluster_bootstrap_tps_primary"])

N_BOOT_71 = int(globals().get("N_BOOTSTRAP", 2000))
SEED_71 = int(globals().get("RANDOM_SEED", 20260603))

PRIMARY_JUDGES_71 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_71 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]

def _weighted_tps_71(eff: pd.DataFrame, weights_by_prompt: Dict[str, int]) -> Dict[str, Any]:
    sub = eff.copy()
    sub["_w"] = sub["prompt_id"].astype(str).map(weights_by_prompt).fillna(0).astype(float)
    sub = sub[sub["_w"] > 0].copy()

    if sub.empty:
        return {"tps": np.nan, "diag_mean": np.nan, "offdiag_mean": np.nan, "per_family_tps": {}}

    sub["support_family_1"] = sub["support_family_1"].astype(float) * sub["_w"]
    sub["support_family_2"] = sub["support_family_2"].astype(float) * sub["_w"]

    # Expand opportunity weights by duplicating denominator through weighted build.
    num = pd.DataFrame(0.0, index=PRIMARY_JUDGES_71, columns=PRIMARY_CANDS_71)
    den = pd.DataFrame(0.0, index=PRIMARY_JUDGES_71, columns=PRIMARY_CANDS_71)

    for _, r in sub.iterrows():
        j = str(r["judge_family"]).strip().lower()
        f1 = str(r["family_1"]).strip().lower()
        f2 = str(r["family_2"]).strip().lower()
        w = float(r["_w"])

        if j not in num.index:
            continue

        if f1 in num.columns:
            num.loc[j, f1] += float(r["support_family_1"])
            den.loc[j, f1] += w

        if f2 in num.columns:
            num.loc[j, f2] += float(r["support_family_2"])
            den.loc[j, f2] += w

    pref = (num / den.replace(0, np.nan)).fillna(0.5)
    return compute_tps(pref, PRIMARY_JUDGES_71, PRIMARY_CANDS_71)

def cluster_bootstrap_tps(
    eff: pd.DataFrame,
    n_boot: int = N_BOOT_71,
    cluster_col: str = "prompt_id",
    seed: int = SEED_71,
) -> Dict[str, Any]:
    eff = eff.copy()
    eff[cluster_col] = eff[cluster_col].astype(str)

    pref, _ = build_preference_matrix(eff, PRIMARY_JUDGES_71, PRIMARY_CANDS_71)
    obs = compute_tps(pref, PRIMARY_JUDGES_71, PRIMARY_CANDS_71)

    clusters = np.array(sorted(eff[cluster_col].unique()))
    rng = np.random.default_rng(seed)

    vals = np.empty(n_boot)
    diag_vals = np.empty(n_boot)
    off_vals = np.empty(n_boot)

    t0 = time.time()

    for b in range(n_boot):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        weights = pd.Series(sampled).value_counts().to_dict()

        stat = _weighted_tps_71(eff, weights)
        vals[b] = stat["tps"]
        diag_vals[b] = stat["diag_mean"]
        off_vals[b] = stat["offdiag_mean"]

        if (b + 1) % 500 == 0:
            print(f"  bootstrap {b + 1:,}/{n_boot:,}")

    vals = vals[np.isfinite(vals)]

    return {
        "analysis_label": "primary_4",
        "schema_note": "Cluster bootstrap computed from support_family_1/support_family_2.",
        "cluster_col": cluster_col,
        "n_clusters": int(len(clusters)),
        "n_effective_rows": int(len(eff)),
        "n_bootstrap": int(n_boot),
        "seed": int(seed),
        "observed_tps": float(obs["tps"]),
        "observed_diag_mean": float(obs["diag_mean"]),
        "observed_offdiag_mean": float(obs["offdiag_mean"]),
        "observed_per_family_tps": obs["per_family_tps"],
        "bootstrap_mean_tps": float(np.mean(vals)),
        "bootstrap_sd_tps": float(np.std(vals, ddof=1)),
        "ci_95": [float(np.quantile(vals, 0.025)), float(np.quantile(vals, 0.975))],
        "ci_95_low": float(np.quantile(vals, 0.025)),
        "ci_95_high": float(np.quantile(vals, 0.975)),
        "p_bootstrap_two_sided_against_zero": float(2 * min(np.mean(vals <= 0), np.mean(vals >= 0))),
        "p_bootstrap_left_tail_le_zero": float(np.mean(vals <= 0)),
        "bootstrap_diag_mean_mean": float(np.mean(diag_vals[np.isfinite(diag_vals)])),
        "bootstrap_offdiag_mean_mean": float(np.mean(off_vals[np.isfinite(off_vals)])),
        "runtime_seconds": float(time.time() - t0),
    }

def _step_bootstrap_71():
    print("=" * 100)
    print("CLUSTER BOOTSTRAP — PRIMARY-4 TPS")
    print("=" * 100)

    eff = pd.read_csv(FILES["effective_winners_primary"])
    out = cluster_bootstrap_tps(eff)
    atomic_write_json(out, FILES["cluster_bootstrap_tps_primary"])
    atomic_write_json(out, FILES["bootstrap_result"])

    print("\nBootstrap complete")
    print("-" * 80)
    print(f"Observed TPS: {out['observed_tps']:.4f}")
    print(f"Bootstrap mean TPS: {out['bootstrap_mean_tps']:.4f}")
    print(f"Bootstrap SD: {out['bootstrap_sd_tps']:.4f}")
    print(f"95% CI: [{out['ci_95_low']:.4f}, {out['ci_95_high']:.4f}]")
    print(f"Clusters: {out['n_clusters']}")
    print(f"Rows: {out['n_effective_rows']}")
    print("Saved:", FILES["cluster_bootstrap_tps_primary"])

run_step(
    [FILES["cluster_bootstrap_tps_primary"]],
    _step_bootstrap_71,
    "cluster_bootstrap_primary4_weighted_support",
    force=False,
    validators={
        FILES["cluster_bootstrap_tps_primary"]: lambda p: json_has(
            p, ["observed_tps", "ci_95", "analysis_label", "n_bootstrap"]
        )
    },
)

print("\n✓ Cell 7.1 complete")

In [ ]:
# ============================================================================
# Cell 7.2 — Prompt-level permutation test for Primary-4 TPS
# ============================================================================
"""
Null hypothesis:
Judge-family identity is exchangeable within prompt.

This test shuffles judge_family labels within each prompt_id cluster.
"""

from pathlib import Path
from typing import Any, Dict, Sequence
import pandas as pd
import numpy as np
import json
import time

required_globals = [
    "FILES", "PATHS", "PRIMARY_JUDGE_FAMILIES", "PRIMARY_CANDIDATE_FAMILIES",
    "run_step", "atomic_write_json", "json_has", "build_preference_matrix", "compute_tps"
]
missing = [x for x in required_globals if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing}")

def _ensure_path_72(key: str, subdir: str) -> Path:
    if key not in PATHS:
        if "analysis" in PATHS:
            base = Path(PATHS["analysis"])
        elif "root" in PATHS:
            base = Path(PATHS["root"]) / "analysis"
        elif "ROOT_DIR" in globals():
            base = Path(ROOT_DIR) / "analysis"
        else:
            base = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")
        PATHS[key] = base / subdir
    Path(PATHS[key]).mkdir(parents=True, exist_ok=True)
    return Path(PATHS[key])

analysis_stats = _ensure_path_72("analysis_stats", "stats")

FILES.setdefault("permutation_result", analysis_stats / "permutation_test_primary4.json")

N_PERM_72 = int(globals().get("N_PERMUTATIONS", 5000))
SEED_72 = int(globals().get("RANDOM_SEED", 20260603))

PRIMARY_JUDGES_72 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_72 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]

def prompt_permutation_test(
    eff: pd.DataFrame,
    n_perm: int = N_PERM_72,
    cluster_col: str = "prompt_id",
    seed: int = SEED_72,
) -> Dict[str, Any]:
    eff = eff.copy()
    eff[cluster_col] = eff[cluster_col].astype(str)

    pref, _ = build_preference_matrix(eff, PRIMARY_JUDGES_72, PRIMARY_CANDS_72)
    obs = compute_tps(pref, PRIMARY_JUDGES_72, PRIMARY_CANDS_72)
    obs_tps = float(obs["tps"])

    rng = np.random.default_rng(seed)
    null_tps = np.empty(n_perm)

    groups = list(eff.groupby(cluster_col).groups.items())
    t0 = time.time()

    for k in range(n_perm):
        perm = eff.copy()

        for _, idx in groups:
            idx = list(idx)
            judges = perm.loc[idx, "judge_family"].to_numpy().copy()
            rng.shuffle(judges)
            perm.loc[idx, "judge_family"] = judges

        pmat, _ = build_preference_matrix(perm, PRIMARY_JUDGES_72, PRIMARY_CANDS_72)
        null_tps[k] = compute_tps(pmat, PRIMARY_JUDGES_72, PRIMARY_CANDS_72)["tps"]

        if (k + 1) % 1000 == 0:
            print(f"  permutation {k + 1:,}/{n_perm:,}")

    return {
        "analysis_label": "primary_4",
        "schema_note": "Prompt-level permutation of judge_family within prompt_id.",
        "n_permutations": int(n_perm),
        "seed": int(seed),
        "observed_tps": obs_tps,
        "null_mean": float(null_tps.mean()),
        "null_sd": float(null_tps.std(ddof=1)),
        "p_two_sided": float((np.abs(null_tps) >= abs(obs_tps)).mean()),
        "p_one_sided_greater": float((null_tps >= obs_tps).mean()),
        "null_quantiles": {
            "p01": float(np.quantile(null_tps, 0.01)),
            "p05": float(np.quantile(null_tps, 0.05)),
            "p50": float(np.quantile(null_tps, 0.50)),
            "p95": float(np.quantile(null_tps, 0.95)),
            "p99": float(np.quantile(null_tps, 0.99)),
        },
        "runtime_seconds": float(time.time() - t0),
    }

def _step_permutation_72():
    print("=" * 100)
    print("PROMPT-LEVEL PERMUTATION TEST — PRIMARY-4 TPS")
    print("=" * 100)

    eff = pd.read_csv(FILES["effective_winners_primary"])
    out = prompt_permutation_test(eff)
    atomic_write_json(out, FILES["permutation_result"])

    print("\nPermutation complete")
    print("-" * 80)
    print(f"Observed TPS: {out['observed_tps']:.4f}")
    print(f"Null mean: {out['null_mean']:.4f}")
    print(f"Null SD: {out['null_sd']:.4f}")
    print(f"p(two-sided): {out['p_two_sided']:.6f}")
    print(f"p(one-sided greater): {out['p_one_sided_greater']:.6f}")
    print("Saved:", FILES["permutation_result"])

run_step(
    [FILES["permutation_result"]],
    _step_permutation_72,
    "prompt_permutation_primary4",
    force=False,
    validators={
        FILES["permutation_result"]: lambda p: json_has(
            p, ["p_two_sided", "observed_tps", "analysis_label", "null_quantiles"]
        )
    },
)

print("\n✓ Cell 7.2 complete")

In [ ]:
# ============================================================================
# Cell 7.3 — Per-family Primary-4 TPS with BH correction, cached + resumable
# ============================================================================
"""
Cell 7.3 — Per-family TPS permutation test with Benjamini-Hochberg correction.

Purpose:
- Estimate per-family TPS significance.
- Apply BH correction across families.
- Save a final JSON result.
- Save an intermediate checkpoint so Colab disconnects do not destroy progress.

Important:
- Uses effective_winners_primary.csv only.
- Falcon is blocked.
- Does not depend on GPU.
- Cached with force=False.
"""

from pathlib import Path
from typing import Any, Dict, List, Sequence, Optional
import pandas as pd
import numpy as np
import json
import time

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "analysis_stats" not in PATHS:
    if "stats" in PATHS:
        PATHS["analysis_stats"] = Path(PATHS["stats"])
    elif "analysis" in PATHS:
        PATHS["analysis_stats"] = Path(PATHS["analysis"]) / "stats"
    elif "root" in PATHS:
        PATHS["analysis_stats"] = Path(PATHS["root"]) / "analysis" / "stats"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_stats"] = Path(ROOT_DIR) / "analysis" / "stats"
    else:
        PATHS["analysis_stats"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/stats")

Path(PATHS["analysis_stats"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("per_family_bh", Path(PATHS["analysis_stats"]) / "per_family_bh_primary4.json")
FILES.setdefault("per_family_bh_checkpoint", Path(PATHS["analysis_stats"]) / "per_family_bh_primary4_checkpoint.json")


PRIMARY_JUDGES_73 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_73 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]

N_PERM_73 = int(globals().get("N_PERMUTATIONS", 2000))
RANDOM_SEED_73 = int(globals().get("RANDOM_SEED", 20260603))
CHECKPOINT_EVERY_73 = 100


def _73_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _73_bh_adjust(pvals: Dict[str, float]) -> Dict[str, float]:
    """
    Benjamini-Hochberg correction.
    Returns adjusted p-values in the original family names.
    """
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m = len(items)

    adjusted = {}
    prev = 1.0

    for rank_from_end, (fam, p) in enumerate(reversed(items), start=1):
        rank = m - rank_from_end + 1
        adj = min(prev, float(p) * m / rank)
        adjusted[fam] = float(min(adj, 1.0))
        prev = adj

    return adjusted


def _73_prepare_effective(eff: pd.DataFrame) -> pd.DataFrame:
    required = [
        "prompt_id",
        "judge_family",
        "family_1",
        "family_2",
        "support_family_1",
        "support_family_2",
    ]

    missing = [c for c in required if c not in eff.columns]
    if missing:
        raise RuntimeError(f"effective_winners_primary missing columns: {missing}")

    eff = eff.copy()

    eff["prompt_id"] = eff["prompt_id"].astype(str).str.strip()

    for c in ["judge_family", "family_1", "family_2"]:
        eff[c] = eff[c].map(_73_clean_family)

    eff["support_family_1"] = pd.to_numeric(eff["support_family_1"], errors="coerce")
    eff["support_family_2"] = pd.to_numeric(eff["support_family_2"], errors="coerce")

    eff = eff.dropna(subset=["support_family_1", "support_family_2"]).copy()

    fams_seen = set(eff["judge_family"]) | set(eff["family_1"]) | set(eff["family_2"])
    if "falcon" in fams_seen:
        raise RuntimeError("Falcon leaked into Cell 7.3 Primary-4 per-family permutation.")

    eff = eff[
        eff["judge_family"].isin(PRIMARY_JUDGES_73)
        & eff["family_1"].isin(PRIMARY_CANDS_73)
        & eff["family_2"].isin(PRIMARY_CANDS_73)
    ].reset_index(drop=True)

    if eff.empty:
        raise RuntimeError("No Primary-4 effective winners available for Cell 7.3.")

    return eff


def _73_fast_pref_and_tps(
    judges: np.ndarray,
    family_1: np.ndarray,
    family_2: np.ndarray,
    support_1: np.ndarray,
    support_2: np.ndarray,
    judge_families: Sequence[str],
    candidate_families: Sequence[str],
) -> Dict[str, Any]:
    """
    Fast local replacement for repeatedly calling build_preference_matrix
    inside every permutation.

    Matrix entry = mean support for candidate family under judge family.
    """
    j_index = {f: i for i, f in enumerate(judge_families)}
    c_index = {f: i for i, f in enumerate(candidate_families)}

    numerator = np.zeros((len(judge_families), len(candidate_families)), dtype=float)
    denominator = np.zeros((len(judge_families), len(candidate_families)), dtype=float)

    for j, f1, f2, s1, s2 in zip(judges, family_1, family_2, support_1, support_2):
        if j not in j_index:
            continue

        ji = j_index[j]

        if f1 in c_index:
            ci = c_index[f1]
            numerator[ji, ci] += float(s1)
            denominator[ji, ci] += 1.0

        if f2 in c_index:
            ci = c_index[f2]
            numerator[ji, ci] += float(s2)
            denominator[ji, ci] += 1.0

    with np.errstate(divide="ignore", invalid="ignore"):
        pref = numerator / denominator

    diag_vals = []
    offdiag_vals = []

    per_family = {}

    for jf in judge_families:
        if jf not in c_index:
            continue

        ji = j_index[jf]
        ci = c_index[jf]

        diag = pref[ji, ci]
        if np.isfinite(diag):
            diag_vals.append(diag)

        row_off = []

        for cf in candidate_families:
            cj = c_index[cf]
            val = pref[ji, cj]

            if cf == jf:
                continue

            if np.isfinite(val):
                row_off.append(val)
                offdiag_vals.append(val)

        if np.isfinite(diag) and len(row_off) > 0:
            per_family[jf] = float(diag - np.mean(row_off))
        else:
            per_family[jf] = np.nan

    diag_mean = float(np.mean(diag_vals)) if diag_vals else np.nan
    offdiag_mean = float(np.mean(offdiag_vals)) if offdiag_vals else np.nan
    tps = float(diag_mean - offdiag_mean) if np.isfinite(diag_mean) and np.isfinite(offdiag_mean) else np.nan

    pref_df = pd.DataFrame(pref, index=judge_families, columns=candidate_families)

    return {
        "preference_matrix": pref_df,
        "diag_mean": diag_mean,
        "offdiag_mean": offdiag_mean,
        "tps": tps,
        "per_family_tps": per_family,
    }


def _73_make_prompt_groups(prompt_ids: np.ndarray) -> List[np.ndarray]:
    groups = []
    tmp = pd.Series(np.arange(len(prompt_ids))).groupby(prompt_ids).apply(lambda x: x.to_numpy())

    for arr in tmp.tolist():
        groups.append(np.asarray(arr, dtype=int))

    return groups


def _73_load_checkpoint(expected_config: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    ckpt_path = Path(FILES["per_family_bh_checkpoint"])

    if not ckpt_path.exists():
        return None

    try:
        ckpt = json.load(open(ckpt_path))
    except Exception:
        return None

    if ckpt.get("config") != expected_config:
        print("Existing Cell 7.3 checkpoint config does not match current config. Ignoring checkpoint.")
        return None

    return ckpt


def _73_save_checkpoint(
    k_done: int,
    null_one_family: Dict[str, List[float]],
    null_two_family: Dict[str, List[float]],
    rng: np.random.Generator,
    config: Dict[str, Any],
):
    ckpt = {
        "analysis_label": "primary_4_per_family_bh_checkpoint",
        "k_done": int(k_done),
        "config": config,
        "rng_state": rng.bit_generator.state,
        "null_one_family": null_one_family,
        "null_two_family": null_two_family,
        "checkpoint_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }

    atomic_write_json(ckpt, FILES["per_family_bh_checkpoint"])


def per_family_permutation_bh_primary4_resumable(
    eff: pd.DataFrame,
    n_perm: int = N_PERM_73,
    seed: int = RANDOM_SEED_73,
) -> Dict[str, Any]:
    eff = _73_prepare_effective(eff)

    judges = eff["judge_family"].to_numpy(dtype=object)
    family_1 = eff["family_1"].to_numpy(dtype=object)
    family_2 = eff["family_2"].to_numpy(dtype=object)
    support_1 = eff["support_family_1"].to_numpy(dtype=float)
    support_2 = eff["support_family_2"].to_numpy(dtype=float)
    prompt_ids = eff["prompt_id"].to_numpy(dtype=object)

    prompt_groups = _73_make_prompt_groups(prompt_ids)

    obs = _73_fast_pref_and_tps(
        judges=judges,
        family_1=family_1,
        family_2=family_2,
        support_1=support_1,
        support_2=support_2,
        judge_families=PRIMARY_JUDGES_73,
        candidate_families=PRIMARY_CANDS_73,
    )

    observed_per_family = {
        fam: float(obs["per_family_tps"][fam])
        for fam in PRIMARY_CANDS_73
        if fam in obs["per_family_tps"]
    }

    config = {
        "n_perm": int(n_perm),
        "seed": int(seed),
        "families": PRIMARY_CANDS_73,
        "judge_families": PRIMARY_JUDGES_73,
        "n_rows": int(len(eff)),
        "n_prompts": int(eff["prompt_id"].nunique()),
        "version": "cell_7_3_resumable_v2",
    }

    rng = np.random.default_rng(seed)

    null_one_family = {fam: [] for fam in observed_per_family}
    null_two_family = {fam: [] for fam in observed_per_family}
    start_k = 0

    ckpt = _73_load_checkpoint(config)

    if ckpt is not None:
        print(f"Resuming Cell 7.3 from checkpoint at permutation {ckpt.get('k_done', 0):,}/{n_perm:,}")

        start_k = int(ckpt.get("k_done", 0))
        null_one_family = {
            fam: list(ckpt.get("null_one_family", {}).get(fam, []))
            for fam in observed_per_family
        }
        null_two_family = {
            fam: list(ckpt.get("null_two_family", {}).get(fam, []))
            for fam in observed_per_family
        }

        try:
            rng.bit_generator.state = ckpt["rng_state"]
        except Exception as e:
            print("Could not restore RNG state. Restarting RNG from seed. Reason:", e)
            rng = np.random.default_rng(seed)
            start_k = 0
            null_one_family = {fam: [] for fam in observed_per_family}
            null_two_family = {fam: [] for fam in observed_per_family}

    print("\nObserved per-family TPS")
    print("-" * 100)
    for fam, val in observed_per_family.items():
        print(f"{fam:>10}: {val:.6f}")

    print("\nRunning/resuming prompt-level permutations")
    print("-" * 100)

    for k in range(start_k, n_perm):
        perm_judges = judges.copy()

        for idx in prompt_groups:
            shuffled = perm_judges[idx].copy()
            rng.shuffle(shuffled)
            perm_judges[idx] = shuffled

        perm_stat = _73_fast_pref_and_tps(
            judges=perm_judges,
            family_1=family_1,
            family_2=family_2,
            support_1=support_1,
            support_2=support_2,
            judge_families=PRIMARY_JUDGES_73,
            candidate_families=PRIMARY_CANDS_73,
        )

        perm_per_family = perm_stat["per_family_tps"]

        for fam, obs_val in observed_per_family.items():
            null_val = float(perm_per_family.get(fam, np.nan))

            if not np.isfinite(null_val):
                continue

            null_one_family[fam].append(null_val)
            null_two_family[fam].append(abs(null_val))

        k_done = k + 1

        if k_done % CHECKPOINT_EVERY_73 == 0 or k_done == n_perm:
            _73_save_checkpoint(
                k_done=k_done,
                null_one_family=null_one_family,
                null_two_family=null_two_family,
                rng=rng,
                config=config,
            )
            print(f"  permutation {k_done:,}/{n_perm:,} checkpoint saved")

    # -----------------------------------------------------------------------
    # P-values
    # -----------------------------------------------------------------------

    p_one_sided_raw = {}
    p_two_sided_raw = {}
    p_one_sided_empirical = {}
    p_two_sided_empirical = {}

    for fam, obs_val in observed_per_family.items():
        null_vals = np.asarray(null_one_family[fam], dtype=float)
        null_abs = np.asarray(null_two_family[fam], dtype=float)

        if len(null_vals) == 0:
            p_one_sided_raw[fam] = np.nan
            p_two_sided_raw[fam] = np.nan
            p_one_sided_empirical[fam] = np.nan
            p_two_sided_empirical[fam] = np.nan
            continue

        # Empirical p-values without smoothing.
        p1_emp = float((null_vals >= obs_val).mean())
        p2_emp = float((null_abs >= abs(obs_val)).mean())

        # Plus-one smoothed p-values for finite permutation reporting.
        p1 = float(((null_vals >= obs_val).sum() + 1) / (len(null_vals) + 1))
        p2 = float(((null_abs >= abs(obs_val)).sum() + 1) / (len(null_abs) + 1))

        p_one_sided_empirical[fam] = p1_emp
        p_two_sided_empirical[fam] = p2_emp
        p_one_sided_raw[fam] = p1
        p_two_sided_raw[fam] = p2

    p_one_sided_bh = _73_bh_adjust(p_one_sided_raw)
    p_two_sided_bh = _73_bh_adjust(p_two_sided_raw)

    null_summary = {}

    for fam in observed_per_family:
        vals = np.asarray(null_one_family[fam], dtype=float)

        if len(vals):
            null_summary[fam] = {
                "n_null": int(len(vals)),
                "null_mean": float(vals.mean()),
                "null_sd": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0,
                "null_q01": float(np.quantile(vals, 0.01)),
                "null_q05": float(np.quantile(vals, 0.05)),
                "null_q50": float(np.quantile(vals, 0.50)),
                "null_q95": float(np.quantile(vals, 0.95)),
                "null_q99": float(np.quantile(vals, 0.99)),
            }
        else:
            null_summary[fam] = {
                "n_null": 0,
                "null_mean": np.nan,
                "null_sd": np.nan,
                "null_q01": np.nan,
                "null_q05": np.nan,
                "null_q50": np.nan,
                "null_q95": np.nan,
                "null_q99": np.nan,
            }

    result = {
        "analysis_label": "primary_4_per_family_bh",
        "schema_note": (
            "Per-family TPS prompt-level permutation test. "
            "Primary-4 only. Falcon excluded. "
            "BH correction is reported for both one-sided and two-sided p-values. "
            "P-values use plus-one smoothing; empirical unsmoothed values are also saved."
        ),
        "n_permutations": int(n_perm),
        "seed": int(seed),
        "n_effective_rows": int(len(eff)),
        "n_prompts": int(eff["prompt_id"].nunique()),
        "judge_families": PRIMARY_JUDGES_73,
        "candidate_families": PRIMARY_CANDS_73,
        "observed_tps": float(obs["tps"]),
        "observed_diag_mean": float(obs["diag_mean"]),
        "observed_offdiag_mean": float(obs["offdiag_mean"]),
        "observed_per_family": observed_per_family,
        "p_one_sided_raw": p_one_sided_raw,
        "p_one_sided_bh": p_one_sided_bh,
        "p_two_sided_raw": p_two_sided_raw,
        "p_two_sided_bh": p_two_sided_bh,
        "p_one_sided_empirical": p_one_sided_empirical,
        "p_two_sided_empirical": p_two_sided_empirical,
        "null_summary": null_summary,
        "checkpoint_path": str(FILES["per_family_bh_checkpoint"]),
        "final_output_path": str(FILES["per_family_bh"]),
    }

    return result


def _step_per_family_bh_73():
    print("=" * 100)
    print("PER-FAMILY PRIMARY-4 TPS PERMUTATION + BH CORRECTION")
    print("=" * 100)

    if "effective_winners_primary" not in FILES or not Path(FILES["effective_winners_primary"]).exists():
        raise RuntimeError("FILES['effective_winners_primary'] missing. Run Cell 6.1 first.")

    eff = pd.read_csv(FILES["effective_winners_primary"])

    out = per_family_permutation_bh_primary4_resumable(
        eff=eff,
        n_perm=N_PERM_73,
        seed=RANDOM_SEED_73,
    )

    atomic_write_json(out, FILES["per_family_bh"])

    print("\nFinal per-family BH result")
    print("-" * 100)
    print(json.dumps({
        "observed_per_family": out["observed_per_family"],
        "p_one_sided_raw": out["p_one_sided_raw"],
        "p_one_sided_bh": out["p_one_sided_bh"],
        "p_two_sided_raw": out["p_two_sided_raw"],
        "p_two_sided_bh": out["p_two_sided_bh"],
    }, indent=2))

    print("\nSaved final result:")
    print(FILES["per_family_bh"])

    print("\nCheckpoint file:")
    print(FILES["per_family_bh_checkpoint"])


run_step(
    [FILES["per_family_bh"]],
    _step_per_family_bh_73,
    "per_family_bh_primary4_cached_resumable",
    force=False,
    validators={
        FILES["per_family_bh"]: lambda p: json_has(
            p,
            [
                "analysis_label",
                "observed_per_family",
                "p_one_sided_bh",
                "p_two_sided_bh",
                "n_permutations",
            ],
        )
    },
)

print("\n✓ Cell 7.3 complete")

In [ ]:
# ============================================================================
# Cell 7.3B — Verify per-family BH result was completed and cached
# ============================================================================
"""
Cell 7.3B — Verification cell for Cell 7.3.

Purpose:
- Confirm Cell 7.3 completed successfully.
- Confirm the final per-family BH JSON exists.
- Confirm it has the updated schema.
- Confirm future reruns should use cache instead of recomputing.
"""

from pathlib import Path
import json
import pandas as pd
import numpy as np

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_stats" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_stats"] = Path(PATHS["analysis"]) / "stats"
    elif "root" in PATHS:
        PATHS["analysis_stats"] = Path(PATHS["root"]) / "analysis" / "stats"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_stats"] = Path(ROOT_DIR) / "analysis" / "stats"
    else:
        PATHS["analysis_stats"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/stats")

Path(PATHS["analysis_stats"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("per_family_bh", Path(PATHS["analysis_stats"]) / "per_family_bh_primary4.json")
FILES.setdefault("per_family_bh_verification", Path(PATHS["analysis_stats"]) / "per_family_bh_primary4_verification.json")


def _step_verify_per_family_bh_73b():
    print("=" * 100)
    print("VERIFY CELL 7.3 — PER-FAMILY BH RESULT")
    print("=" * 100)

    path = Path(FILES["per_family_bh"])

    if not path.exists():
        raise RuntimeError(
            f"Cell 7.3 has not completed yet. Missing file:\n{path}\n\n"
            "Let Cell 7.3 finish once before continuing."
        )

    out = json.load(open(path))

    required_keys = [
        "analysis_label",
        "observed_per_family",
        "p_one_sided_bh",
        "p_two_sided_bh",
        "n_permutations",
        "candidate_families",
        "judge_families",
    ]

    missing = [k for k in required_keys if k not in out]
    if missing:
        raise RuntimeError(
            f"Cell 7.3 result exists but has old/incomplete schema. Missing keys: {missing}\n"
            f"Delete or overwrite this file and rerun Cell 7.3:\n{path}"
        )

    candidate_fams = [str(x).lower() for x in out.get("candidate_families", [])]
    judge_fams = [str(x).lower() for x in out.get("judge_families", [])]

    if "falcon" in candidate_fams or "falcon" in judge_fams:
        raise RuntimeError("Falcon leaked into Cell 7.3 per-family BH result.")

    observed = out["observed_per_family"]
    p1 = out["p_one_sided_bh"]
    p2 = out["p_two_sided_bh"]

    fams = sorted(observed.keys())

    table = pd.DataFrame({
        "family": fams,
        "observed_per_family_tps": [observed.get(f, np.nan) for f in fams],
        "p_one_sided_bh": [p1.get(f, np.nan) for f in fams],
        "p_two_sided_bh": [p2.get(f, np.nan) for f in fams],
    })

    verification = {
        "analysis_label": "per_family_bh_primary4_verification",
        "status": "ok",
        "source_file": str(path),
        "n_permutations": int(out["n_permutations"]),
        "candidate_families": candidate_fams,
        "judge_families": judge_fams,
        "families": fams,
        "all_required_keys_present": True,
        "falcon_absent": True,
        "future_rerun_should_skip_if_force_false": True,
        "observed_per_family": observed,
        "p_one_sided_bh": p1,
        "p_two_sided_bh": p2,
    }

    atomic_write_json(verification, FILES["per_family_bh_verification"])

    print("\nCell 7.3 result path:")
    print(path)

    print("\nPer-family BH table")
    print("-" * 100)
    print(table.round(6).to_string(index=False))

    print("\nVerification saved:")
    print(FILES["per_family_bh_verification"])

    print("\n✓ Cell 7.3 is complete and cached.")


run_step(
    [FILES["per_family_bh_verification"]],
    _step_verify_per_family_bh_73b,
    "verify_per_family_bh_primary4_cached",
    force=False,
    validators={
        FILES["per_family_bh_verification"]: lambda p: json_has(
            p,
            ["analysis_label", "status", "all_required_keys_present", "falcon_absent"],
        )
    },
)

print("\n✓ Cell 7.3B complete")

In [ ]:
# ============================================================================
# Cell 7.4 — Bradley-Terry diagnostics for Primary-4
# ============================================================================
"""
Cell 7.4 — Bradley-Terry goodness-of-fit and residual diagnostics.

Purpose:
- Confirm BT quality residualization is not a black box.
- Compute object-level residuals.
- Compute residuals by judge family, candidate family, and pair.
- Compute overdispersion statistic.
- Save a full BT diagnostics JSON and residual table.

This is diagnostic only. It does not refit or overwrite the main BT result.
"""

from pathlib import Path
import json
import pandas as pd
import numpy as np

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_json",
    "atomic_write_csv",
    "json_has",
    "csv_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_stats" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_stats"] = Path(PATHS["analysis"]) / "stats"
    elif "root" in PATHS:
        PATHS["analysis_stats"] = Path(PATHS["root"]) / "analysis" / "stats"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_stats"] = Path(ROOT_DIR) / "analysis" / "stats"
    else:
        PATHS["analysis_stats"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/stats")

Path(PATHS["analysis_stats"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("bt_results_primary", Path(PATHS["analysis_stats"]) / "bt_results_primary.json")
FILES.setdefault("bt_diagnostics", Path(PATHS["analysis_stats"]) / "bt_diagnostics_primary4.json")
FILES.setdefault("bt_residuals_primary", Path(PATHS["analysis_stats"]) / "bt_residuals_primary4.csv")


def _74_clean_family(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _74_sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))


def _74_load_bt_abilities():
    path = Path(FILES["bt_results_primary"])
    if not path.exists():
        raise RuntimeError(f"Missing BT result file. Run BT fitting cell first:\n{path}")

    obj = json.load(open(path))

    if "abilities" in obj:
        abilities = obj["abilities"]
    elif "ability" in obj:
        abilities = obj["ability"]
    else:
        raise RuntimeError("BT result file does not contain 'abilities'.")

    return {str(k).strip().lower(): float(v) for k, v in abilities.items()}, obj


def _74_prepare_eff():
    if "effective_winners_primary" not in FILES or not Path(FILES["effective_winners_primary"]).exists():
        raise RuntimeError("effective_winners_primary missing. Run Cell 6.1 first.")

    eff = pd.read_csv(FILES["effective_winners_primary"]).copy()

    required = [
        "prompt_id",
        "judge_family",
        "family_1",
        "family_2",
        "support_family_1",
        "support_family_2",
    ]

    missing = [c for c in required if c not in eff.columns]
    if missing:
        raise RuntimeError(f"effective_winners_primary missing columns: {missing}")

    for c in ["judge_family", "family_1", "family_2"]:
        eff[c] = eff[c].map(_74_clean_family)

    eff["support_family_1"] = pd.to_numeric(eff["support_family_1"], errors="coerce")
    eff["support_family_2"] = pd.to_numeric(eff["support_family_2"], errors="coerce")

    eff = eff.dropna(subset=["support_family_1", "support_family_2"]).copy()

    fams_seen = set(eff["judge_family"]) | set(eff["family_1"]) | set(eff["family_2"])
    if "falcon" in fams_seen:
        raise RuntimeError("Falcon leaked into Primary-4 BT diagnostics.")

    return eff


def _74_build_bt_residual_rows(eff, abilities):
    rows = []

    for _, r in eff.iterrows():
        f1 = r["family_1"]
        f2 = r["family_2"]

        a1 = float(abilities.get(f1, 0.0))
        a2 = float(abilities.get(f2, 0.0))

        p1 = float(_74_sigmoid(a1 - a2))
        p2 = 1.0 - p1

        s1 = float(r["support_family_1"])
        s2 = float(r["support_family_2"])

        pair = "__vs__".join(sorted([f1, f2]))

        rows.append({
            "prompt_id": r["prompt_id"],
            "judge_family": r["judge_family"],
            "candidate_family": f1,
            "opponent_family": f2,
            "pair": pair,
            "observed_support": s1,
            "expected_support": p1,
            "residual": s1 - p1,
            "pearson_residual": (s1 - p1) / np.sqrt(max(p1 * (1.0 - p1), 1e-8)),
            "abs_residual": abs(s1 - p1),
            "squared_residual": (s1 - p1) ** 2,
        })

        rows.append({
            "prompt_id": r["prompt_id"],
            "judge_family": r["judge_family"],
            "candidate_family": f2,
            "opponent_family": f1,
            "pair": pair,
            "observed_support": s2,
            "expected_support": p2,
            "residual": s2 - p2,
            "pearson_residual": (s2 - p2) / np.sqrt(max(p2 * (1.0 - p2), 1e-8)),
            "abs_residual": abs(s2 - p2),
            "squared_residual": (s2 - p2) ** 2,
        })

    return pd.DataFrame(rows)


def _74_group_residuals(resid, group_cols):
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    out = (
        resid.groupby(group_cols, dropna=False)
        .agg(
            n=("observed_support", "size"),
            mean_observed=("observed_support", "mean"),
            mean_expected=("expected_support", "mean"),
            mean_residual=("residual", "mean"),
            mean_abs_residual=("abs_residual", "mean"),
            rmse=("squared_residual", lambda s: float(np.sqrt(np.mean(s)))),
            pearson_chi2=("pearson_residual", lambda s: float(np.sum(np.asarray(s) ** 2))),
        )
        .reset_index()
    )

    return out


def _step_bt_diagnostics_74():
    print("=" * 100)
    print("BRADLEY-TERRY DIAGNOSTICS — PRIMARY-4")
    print("=" * 100)

    abilities, bt_obj = _74_load_bt_abilities()
    eff = _74_prepare_eff()
    resid = _74_build_bt_residual_rows(eff, abilities)

    n = int(len(resid))
    k = int(len(abilities))
    df_resid = max(n - k, 1)

    pearson_chi2 = float(np.sum(resid["pearson_residual"].to_numpy() ** 2))
    overdispersion = float(pearson_chi2 / df_resid)

    by_judge = _74_group_residuals(resid, "judge_family")
    by_candidate = _74_group_residuals(resid, "candidate_family")
    by_pair = _74_group_residuals(resid, "pair")
    by_judge_candidate = _74_group_residuals(resid, ["judge_family", "candidate_family"])

    largest_misfit = (
        resid.sort_values("abs_residual", ascending=False)
        .head(50)
        .to_dict(orient="records")
    )

    diagnostics = {
        "analysis_label": "bt_diagnostics_primary4",
        "schema_note": (
            "BT goodness-of-fit diagnostics computed from effective_winners_primary.csv. "
            "Includes object-level residuals, group residual summaries, and Pearson overdispersion."
        ),
        "bt_results_path": str(FILES["bt_results_primary"]),
        "effective_winners_path": str(FILES["effective_winners_primary"]),
        "residuals_path": str(FILES["bt_residuals_primary"]),
        "n_effective_rows": int(len(eff)),
        "n_object_level_residuals": int(len(resid)),
        "n_abilities": int(len(abilities)),
        "abilities": abilities,
        "pearson_chi2": pearson_chi2,
        "df_residual": int(df_resid),
        "overdispersion": overdispersion,
        "overdispersion_interpretation": (
            "near_1_good"
            if overdispersion < 1.5
            else "moderate_extra_variation"
            if overdispersion < 3.0
            else "high_extra_variation"
        ),
        "global_mean_abs_residual": float(resid["abs_residual"].mean()),
        "global_rmse": float(np.sqrt(resid["squared_residual"].mean())),
        "by_judge_family": by_judge.to_dict(orient="records"),
        "by_candidate_family": by_candidate.to_dict(orient="records"),
        "by_pair": by_pair.to_dict(orient="records"),
        "by_judge_candidate": by_judge_candidate.to_dict(orient="records"),
        "largest_misfit_rows_top50": largest_misfit,
        "paper_language": {
            "safe": (
                "We inspected Bradley-Terry residual diagnostics, including object-level residuals, "
                "family-level residual summaries, pair-level residual summaries, and Pearson overdispersion, "
                "to verify that quality residualization was not driven by a small number of poorly fitted pairs."
            )
        },
    }

    atomic_write_csv(resid, FILES["bt_residuals_primary"])
    atomic_write_json(diagnostics, FILES["bt_diagnostics"])

    print("\nBT diagnostics summary")
    print("-" * 100)
    print(json.dumps({
        "n_effective_rows": diagnostics["n_effective_rows"],
        "n_object_level_residuals": diagnostics["n_object_level_residuals"],
        "pearson_chi2": diagnostics["pearson_chi2"],
        "df_residual": diagnostics["df_residual"],
        "overdispersion": diagnostics["overdispersion"],
        "overdispersion_interpretation": diagnostics["overdispersion_interpretation"],
        "global_mean_abs_residual": diagnostics["global_mean_abs_residual"],
        "global_rmse": diagnostics["global_rmse"],
    }, indent=2))

    print("\nResiduals by judge family")
    print("-" * 100)
    print(by_judge.round(5).to_string(index=False))

    print("\nResiduals by candidate family")
    print("-" * 100)
    print(by_candidate.round(5).to_string(index=False))

    print("\nResiduals by pair")
    print("-" * 100)
    print(by_pair.round(5).to_string(index=False))

    print("\nSaved:")
    print("Residuals:", FILES["bt_residuals_primary"])
    print("Diagnostics:", FILES["bt_diagnostics"])


run_step(
    [FILES["bt_diagnostics"], FILES["bt_residuals_primary"]],
    _step_bt_diagnostics_74,
    "bt_diagnostics_primary4",
    force=False,
    validators={
        FILES["bt_diagnostics"]: lambda p: json_has(
            p,
            ["analysis_label", "overdispersion", "by_judge_family", "by_candidate_family", "by_pair"],
        ),
        FILES["bt_residuals_primary"]: lambda p: csv_has(
            p,
            ["prompt_id", "judge_family", "candidate_family", "observed_support", "expected_support", "residual"],
            10,
        ),
    },
)

print("\n✓ Cell 7.4 complete")

## PHASE 8 — Mechanism Decomposition (THE headline contribution)

Nested logits M1→M5 progressively absorb candidate predictors into
`chosen ~ same_family + ...`. Whatever remains in M5 is the irreducible
tribal effect after controlling for: BT quality, perplexity-familiarity,
style similarity, length, position. This single table is what the paper
leads with.



In [ ]:
# ============================================================================
# Cell 8.0B — Phase 8 Primary-4 guard and alias-fragility check
# ============================================================================
"""
Cell 8.0B — Guard against accidentally running Phase 8 on the wrong dataset.

Purpose:
- Ensure Phase 8 uses Primary-4 files explicitly.
- Ensure effective_winners_primary has no Falcon judge/candidate.
- Ensure regression and mechanism files are marked primary_4.
- Warn if FILES["effective_winners"] alias points somewhere dangerous.

This does not compute mechanisms. It prevents silent dataset mistakes.
"""

from pathlib import Path
import json
import pandas as pd
import numpy as np

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_mechanisms" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_mechanisms"] = Path(PATHS["analysis"]) / "mechanisms"
    elif "root" in PATHS:
        PATHS["analysis_mechanisms"] = Path(PATHS["root"]) / "analysis" / "mechanisms"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_mechanisms"] = Path(ROOT_DIR) / "analysis" / "mechanisms"
    else:
        PATHS["analysis_mechanisms"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/mechanisms")

Path(PATHS["analysis_mechanisms"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("phase8_primary_guard", Path(PATHS["analysis_mechanisms"]) / "phase8_primary4_guard.json")


def _clean_fam_80b(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _step_phase8_primary_guard_80b():
    print("=" * 100)
    print("PHASE 8 PRIMARY-4 GUARD")
    print("=" * 100)

    if "effective_winners_primary" not in FILES or not Path(FILES["effective_winners_primary"]).exists():
        raise RuntimeError("effective_winners_primary missing. Run Cell 6.1 first.")

    eff = pd.read_csv(FILES["effective_winners_primary"]).copy()

    required = ["judge_family", "family_1", "family_2"]
    missing = [c for c in required if c not in eff.columns]
    if missing:
        raise RuntimeError(f"effective_winners_primary missing required columns: {missing}")

    for c in required:
        eff[c] = eff[c].map(_clean_fam_80b)

    judges = sorted(eff["judge_family"].dropna().unique().tolist())
    candidates = sorted(set(eff["family_1"].dropna().unique()) | set(eff["family_2"].dropna().unique()))

    primary_judges = sorted([str(x).lower() for x in PRIMARY_JUDGE_FAMILIES])
    primary_candidates = sorted([str(x).lower() for x in PRIMARY_CANDIDATE_FAMILIES])

    errors = []

    if "falcon" in judges:
        errors.append("Falcon appears as judge in effective_winners_primary.")

    if "falcon" in candidates:
        errors.append("Falcon appears as candidate in effective_winners_primary.")

    if sorted(judges) != primary_judges:
        errors.append(f"Judge families mismatch. Expected {primary_judges}, got {judges}")

    if sorted(candidates) != primary_candidates:
        errors.append(f"Candidate families mismatch. Expected {primary_candidates}, got {candidates}")

    alias_warning = None
    if "effective_winners" in FILES:
        alias_path = Path(FILES["effective_winners"])
        primary_path = Path(FILES["effective_winners_primary"])
        if alias_path.exists() and alias_path.resolve() != primary_path.resolve():
            alias_warning = (
                f"FILES['effective_winners'] points to {alias_path}, not effective_winners_primary. "
                "Phase 8 should use FILES['effective_winners_primary'] explicitly."
            )

    if errors:
        raise RuntimeError("Phase 8 Primary-4 guard failed:\n" + "\n".join(errors))

    out = {
        "analysis_label": "phase8_primary4_guard",
        "status": "ok",
        "effective_winners_primary_path": str(FILES["effective_winners_primary"]),
        "n_rows": int(len(eff)),
        "judge_families": judges,
        "candidate_families": candidates,
        "falcon_absent_from_primary": True,
        "alias_warning": alias_warning,
        "required_phase8_practice": (
            "Phase 8 cells should reference FILES['effective_winners_primary'] or "
            "Primary-4-filtered master judgments explicitly. Do not rely on FILES['effective_winners'] alias."
        ),
    }

    atomic_write_json(out, FILES["phase8_primary_guard"])

    print("\nGuard result")
    print("-" * 100)
    print(json.dumps(out, indent=2))

    if alias_warning:
        print("\nWARNING")
        print("-" * 100)
        print(alias_warning)

    print("\n✓ Phase 8 can proceed on Primary-4.")


run_step(
    [FILES["phase8_primary_guard"]],
    _step_phase8_primary_guard_80b,
    "phase8_primary4_guard",
    force=False,
    validators={
        FILES["phase8_primary_guard"]: lambda p: json_has(
            p,
            ["analysis_label", "status", "falcon_absent_from_primary", "required_phase8_practice"],
        )
    },
)

print("\n✓ Cell 8.0B complete")

In [ ]:
# ============================================================================
# Cell 8.1 — Style similarity matrix
# ============================================================================
"""
Compute style similarity between candidate responses and judge-family style centroids.

Cached-ready:
- force=False
- Saves outputs to analysis/mechanisms
- CPU-safe
- Uses sentence-transformers if installed
- Falls back to TF-IDF if sentence-transformers is missing

Outputs:
- style_sim_matrix_primary.csv
- style_sim_matrix_full.csv
- style_similarity_summary.json

Also sets:
- FILES["style_sim_matrix"] alias to primary
"""

from pathlib import Path
from typing import Any, Dict, List, Sequence
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES", "PATHS", "PRIMARY_JUDGE_FAMILIES", "PRIMARY_CANDIDATE_FAMILIES",
    "FULL_CANDIDATE_FAMILIES", "run_step", "atomic_write_csv",
    "atomic_write_json", "csv_has", "json_has"
]
missing = [x for x in required_globals if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing}")

def _ensure_path_81(key: str, subdir: str) -> Path:
    if key not in PATHS:
        if "analysis" in PATHS:
            base = Path(PATHS["analysis"])
        elif "root" in PATHS:
            base = Path(PATHS["root"]) / "analysis"
        elif "ROOT_DIR" in globals():
            base = Path(ROOT_DIR) / "analysis"
        else:
            base = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")
        PATHS[key] = base / subdir
    Path(PATHS[key]).mkdir(parents=True, exist_ok=True)
    return Path(PATHS[key])

analysis_mech = _ensure_path_81("analysis_mechanisms", "mechanisms")

FILES.setdefault("style_sim_matrix_primary", analysis_mech / "style_sim_matrix_primary.csv")
FILES.setdefault("style_sim_matrix_full", analysis_mech / "style_sim_matrix_full.csv")
FILES.setdefault("style_similarity_summary", analysis_mech / "style_similarity_summary.json")
FILES.setdefault("style_sim_matrix", FILES["style_sim_matrix_primary"])

PRIMARY_JUDGES_81 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_81 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]
FULL_CANDS_81 = [str(x).strip().lower() for x in FULL_CANDIDATE_FAMILIES]

def _has_cuda_81() -> bool:
    try:
        import torch
        return bool(torch.cuda.is_available())
    except Exception:
        return False

def _norm_family_81(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if "llama" in s:
        return "llama"
    if "qwen" in s:
        return "qwen"
    if "gemma" in s:
        return "gemma"
    if "falcon" in s:
        return "falcon"
    if "yi" in s:
        return "yi"
    return s

def _standardize_responses_81(resp: pd.DataFrame) -> pd.DataFrame:
    r = resp.copy()

    if "model_family" not in r.columns:
        for c in ["family", "candidate_family", "response_family"]:
            if c in r.columns:
                r["model_family"] = r[c]
                break

    if "response" not in r.columns:
        for c in ["text", "output", "completion", "answer"]:
            if c in r.columns:
                r["response"] = r[c]
                break

    if "prompt_id" not in r.columns:
        raise RuntimeError("all_responses is missing prompt_id.")

    if "model_family" not in r.columns:
        raise RuntimeError("all_responses is missing model_family/family column.")

    if "response" not in r.columns:
        raise RuntimeError("all_responses is missing response/text/output column.")

    if "model_scale" not in r.columns:
        r["model_scale"] = "large"

    if "quality_ok" not in r.columns:
        r["quality_ok"] = True

    r["prompt_id"] = r["prompt_id"].astype(str).str.strip()
    r["model_family"] = r["model_family"].map(_norm_family_81)
    r["response"] = r["response"].fillna("").astype(str)

    q = r["quality_ok"]
    if q.dtype == object:
        r["quality_ok"] = q.astype(str).str.lower().isin(["true", "1", "yes", "y", "ok"])
    else:
        r["quality_ok"] = q.astype(bool)

    return r

def _embed_sentence_transformers_81(texts: List[str]) -> Dict[str, Any]:
    from sentence_transformers import SentenceTransformer

    model_name = "sentence-transformers/all-mpnet-base-v2"
    device = "cuda" if _has_cuda_81() else "cpu"

    encoder = SentenceTransformer(model_name, device=device)
    emb = encoder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    return {
        "method": "sentence_transformers",
        "model_name": model_name,
        "device": device,
        "embeddings": emb.astype(np.float32),
    }

def _embed_tfidf_81(texts: List[str]) -> Dict[str, Any]:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize

    vectorizer = TfidfVectorizer(
        lowercase=True,
        max_features=5000,
        ngram_range=(1, 2),
        min_df=1,
    )
    x = vectorizer.fit_transform(texts)
    x = normalize(x)
    emb = x.toarray().astype(np.float32)

    return {
        "method": "tfidf_fallback",
        "model_name": "sklearn_tfidf_1_2gram_5000",
        "device": "cpu",
        "embeddings": emb,
    }

def _compute_style_rows_81(resp: pd.DataFrame, judge_fams: Sequence[str], cand_fams: Sequence[str]) -> pd.DataFrame:
    sub = resp[
        resp["model_family"].isin(set(cand_fams))
        & resp["quality_ok"].astype(bool)
    ].copy()

    # Prefer large responses where available.
    large = sub[sub["model_scale"].astype(str).str.lower().eq("large")].copy()
    if len(large):
        sub = large

    if sub.empty:
        raise RuntimeError("No usable responses for style similarity.")

    texts = sub["response"].fillna("").astype(str).tolist()

    try:
        emb_obj = _embed_sentence_transformers_81(texts)
    except Exception as e:
        print("sentence-transformers unavailable or failed. Falling back to TF-IDF.")
        print("Reason:", e)
        emb_obj = _embed_tfidf_81(texts)

    emb = emb_obj["embeddings"]
    sub = sub.reset_index(drop=True)

    rows = []

    # Global centroids with leave-one-prompt-out adjustment where possible.
    for i, r in sub.iterrows():
        prompt_id = str(r["prompt_id"])
        cand = str(r["model_family"])
        v = emb[i]

        for judge in judge_fams:
            fam_mask = sub["model_family"].eq(judge).to_numpy()
            not_same_prompt = ~sub["prompt_id"].astype(str).eq(prompt_id).to_numpy()
            mask = fam_mask & not_same_prompt

            if mask.sum() == 0:
                mask = fam_mask

            if mask.sum() == 0:
                sim = np.nan
                n_centroid = 0
            else:
                centroid = emb[mask].mean(axis=0)
                norm = np.linalg.norm(centroid)
                if norm > 0:
                    centroid = centroid / norm
                sim = float(np.dot(v, centroid))
                n_centroid = int(mask.sum())

            rows.append({
                "prompt_id": prompt_id,
                "candidate_family": cand,
                "judge_family": judge,
                "style_sim": sim,
                "n_centroid_responses": n_centroid,
                "embedding_method": emb_obj["method"],
                "embedding_model": emb_obj["model_name"],
                "embedding_device": emb_obj["device"],
                "leave_one_prompt_out": True,
            })

    return pd.DataFrame(rows)

def _summarize_style_81(df: pd.DataFrame, label: str) -> Dict[str, Any]:
    if df.empty:
        return {"analysis_label": label, "n_rows": 0}

    d = df.copy()
    d["same_family"] = d["candidate_family"].eq(d["judge_family"])

    return {
        "analysis_label": label,
        "n_rows": int(len(d)),
        "n_prompts": int(d["prompt_id"].nunique()),
        "candidate_families": sorted(d["candidate_family"].dropna().unique().tolist()),
        "judge_families": sorted(d["judge_family"].dropna().unique().tolist()),
        "embedding_method": d["embedding_method"].iloc[0] if "embedding_method" in d.columns else "",
        "embedding_model": d["embedding_model"].iloc[0] if "embedding_model" in d.columns else "",
        "embedding_device": d["embedding_device"].iloc[0] if "embedding_device" in d.columns else "",
        "own_family_style_sim_mean": float(d.loc[d["same_family"], "style_sim"].mean()),
        "other_family_style_sim_mean": float(d.loc[~d["same_family"], "style_sim"].mean()),
        "own_minus_other_style_sim": float(
            d.loc[d["same_family"], "style_sim"].mean()
            - d.loc[~d["same_family"], "style_sim"].mean()
        ),
        "by_pair_mean": (
            d.groupby(["judge_family", "candidate_family"])["style_sim"]
            .mean()
            .reset_index()
            .to_dict(orient="records")
        ),
    }

def _step_style_similarity_81():
    print("=" * 100)
    print("STYLE SIMILARITY MATRIX")
    print("=" * 100)

    if "all_responses" not in FILES or not Path(FILES["all_responses"]).exists():
        raise RuntimeError("FILES['all_responses'] is missing. Run response-generation/compilation cells first.")

    resp = pd.read_csv(FILES["all_responses"])
    resp = _standardize_responses_81(resp)

    print("Loaded responses:", FILES["all_responses"])
    print("Rows:", len(resp))
    print("Families:", sorted(resp["model_family"].dropna().unique().tolist()))
    print("Scales:", sorted(resp["model_scale"].astype(str).dropna().unique().tolist()))

    primary_df = _compute_style_rows_81(resp, PRIMARY_JUDGES_81, PRIMARY_CANDS_81)
    full_df = _compute_style_rows_81(resp, PRIMARY_JUDGES_81, FULL_CANDS_81)

    if "falcon" in set(primary_df["candidate_family"]):
        raise RuntimeError("Falcon leaked into primary style similarity candidate set.")

    atomic_write_csv(primary_df, FILES["style_sim_matrix_primary"])
    atomic_write_csv(primary_df, FILES["style_sim_matrix"])
    atomic_write_csv(full_df, FILES["style_sim_matrix_full"])

    summary = {
        "analysis_label": "style_similarity",
        "schema_note": "Leave-one-prompt-out family centroids when available.",
        "primary": _summarize_style_81(primary_df, "primary_4"),
        "full": _summarize_style_81(full_df, "full_5_appendix"),
        "outputs": {
            "style_sim_matrix_primary": str(FILES["style_sim_matrix_primary"]),
            "style_sim_matrix_full": str(FILES["style_sim_matrix_full"]),
            "style_sim_matrix_alias": str(FILES["style_sim_matrix"]),
        },
    }

    atomic_write_json(summary, FILES["style_similarity_summary"])

    print("\nPrimary style summary")
    print("-" * 80)
    print(json.dumps({
        "n_rows": summary["primary"]["n_rows"],
        "own_family_style_sim_mean": summary["primary"]["own_family_style_sim_mean"],
        "other_family_style_sim_mean": summary["primary"]["other_family_style_sim_mean"],
        "own_minus_other_style_sim": summary["primary"]["own_minus_other_style_sim"],
        "embedding_method": summary["primary"]["embedding_method"],
        "embedding_device": summary["primary"]["embedding_device"],
    }, indent=2))

    print("\nSaved:")
    print("Primary:", FILES["style_sim_matrix_primary"])
    print("Full:", FILES["style_sim_matrix_full"])
    print("Summary:", FILES["style_similarity_summary"])

run_step(
    [
        FILES["style_sim_matrix_primary"],
        FILES["style_sim_matrix_full"],
        FILES["style_similarity_summary"],
    ],
    _step_style_similarity_81,
    "style_similarity_primary4_cached",
    force=False,
    validators={
        FILES["style_sim_matrix_primary"]: lambda p: csv_has(
            p, ["prompt_id", "candidate_family", "judge_family", "style_sim"], 50
        ),
        FILES["style_similarity_summary"]: lambda p: json_has(
            p, ["analysis_label", "primary", "full"]
        ),
    },
)

print("\n✓ Cell 8.1 complete")

In [ ]:
# ============================================================================
# Cell 8.2 — Build Primary-4 mechanism regression dataframe
# ============================================================================
"""
Cell 8.2 — Build the mechanism dataframe from effective_winners_primary.csv.

Important:
- Uses Primary-4 only.
- Uses fractional support outcome, not binary chosen.
- One row per candidate side in each effective comparison.
- Does not use FILES["effective_winners"] alias.
- Falcon is explicitly blocked.
- Adds BT, style, length, and optional logprob features.
- If logprob is unavailable, it is marked unavailable and downstream cells skip it.
"""

from pathlib import Path
from typing import Any, Dict, List, Sequence, Optional
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_csv",
    "atomic_write_json",
    "csv_has",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "analysis_mechanisms" not in PATHS:
    if "analysis" in PATHS:
        base_analysis = Path(PATHS["analysis"])
    elif "root" in PATHS:
        base_analysis = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        base_analysis = Path(ROOT_DIR) / "analysis"
    else:
        base_analysis = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS["analysis_mechanisms"] = base_analysis / "mechanisms"

Path(PATHS["analysis_mechanisms"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("regression_features", PATHS["analysis_mechanisms"] / "regression_features_primary4.csv")
FILES.setdefault("regression_feature_summary", PATHS["analysis_mechanisms"] / "regression_feature_summary_primary4.json")

PRIMARY_JUDGES_82 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_82 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]


def _82_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _82_safe_float(x: Any) -> float:
    try:
        if pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def _82_load_bt_abilities() -> Dict[str, float]:
    if "bt_results_primary" not in FILES or not Path(FILES["bt_results_primary"]).exists():
        print("WARNING: bt_results_primary missing. Setting BT abilities to zero.")
        return {f: 0.0 for f in PRIMARY_CANDS_82}

    obj = json.load(open(FILES["bt_results_primary"]))
    abilities = obj.get("abilities", {})
    return {str(k).strip().lower(): float(v) for k, v in abilities.items()}


def _82_load_style_lookup() -> Dict[tuple, float]:
    style_path = FILES.get("style_sim_matrix_primary", FILES.get("style_sim_matrix"))

    if style_path is None or not Path(style_path).exists():
        print("WARNING: style similarity file missing. style_sim will be NaN.")
        return {}

    sim = pd.read_csv(style_path)

    required = {"prompt_id", "candidate_family", "judge_family", "style_sim"}
    if not required.issubset(sim.columns):
        print("WARNING: style similarity file does not have required columns. style_sim will be NaN.")
        return {}

    sim = sim.copy()
    sim["prompt_id"] = sim["prompt_id"].astype(str).str.strip()
    sim["candidate_family"] = sim["candidate_family"].map(_82_clean_family)
    sim["judge_family"] = sim["judge_family"].map(_82_clean_family)
    sim["style_sim"] = pd.to_numeric(sim["style_sim"], errors="coerce")

    return {
        (r["prompt_id"], r["candidate_family"], r["judge_family"]): float(r["style_sim"])
        for _, r in sim.dropna(subset=["style_sim"]).iterrows()
    }


def _82_load_response_lookup() -> Dict[tuple, Dict[str, float]]:
    if "all_responses" not in FILES or not Path(FILES["all_responses"]).exists():
        print("WARNING: all_responses missing. response length/logprob features will be NaN.")
        return {}

    resp = pd.read_csv(FILES["all_responses"]).copy()

    if "model_family" not in resp.columns:
        for c in ["family", "candidate_family", "response_family"]:
            if c in resp.columns:
                resp["model_family"] = resp[c]
                break

    if "response" not in resp.columns:
        for c in ["text", "output", "completion", "answer"]:
            if c in resp.columns:
                resp["response"] = resp[c]
                break

    if "prompt_id" not in resp.columns or "model_family" not in resp.columns:
        print("WARNING: all_responses missing prompt_id/model_family. response features will be NaN.")
        return {}

    if "response" not in resp.columns:
        resp["response"] = ""

    if "model_scale" in resp.columns:
        large = resp[resp["model_scale"].astype(str).str.lower().eq("large")].copy()
        if len(large):
            resp = large

    if "quality_ok" in resp.columns:
        q = resp["quality_ok"]
        if q.dtype == object:
            q_ok = q.astype(str).str.lower().isin(["true", "1", "yes", "y", "ok"])
        else:
            q_ok = q.astype(bool)
        good = resp[q_ok].copy()
        if len(good):
            resp = good

    resp["prompt_id"] = resp["prompt_id"].astype(str).str.strip()
    resp["model_family"] = resp["model_family"].map(_82_clean_family)
    resp["response"] = resp["response"].fillna("").astype(str)

    token_col = None
    for c in ["n_tokens", "tokens", "num_tokens", "response_tokens"]:
        if c in resp.columns:
            token_col = c
            break

    logprob_col = None
    for c in ["logprob", "mean_logprob", "avg_logprob", "response_logprob"]:
        if c in resp.columns:
            logprob_col = c
            break

    lookup = {}

    for _, r in resp.iterrows():
        pid = str(r["prompt_id"])
        fam = _82_clean_family(r["model_family"])

        if not pid or not fam:
            continue

        if token_col:
            n_tokens = _82_safe_float(r[token_col])
        else:
            n_tokens = float(len(str(r["response"]).split()))

        logprob = _82_safe_float(r[logprob_col]) if logprob_col else np.nan

        lookup[(pid, fam)] = {
            "n_tokens": n_tokens,
            "logprob": logprob,
        }

    return lookup


def _82_zscore(df: pd.DataFrame, col: str, group_col: Optional[str] = None) -> tuple[pd.DataFrame, bool]:
    df = df.copy()

    if col not in df.columns:
        df[col] = np.nan

    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col + "_missing"] = df[col].isna().astype(int)

    if group_col is not None and group_col in df.columns:
        z = pd.Series(index=df.index, dtype=float)
        usable_any = False

        for _, idx in df.groupby(group_col).groups.items():
            s = df.loc[idx, col]
            valid = s.dropna()

            if len(valid) > 2 and valid.std(ddof=1) > 1e-12:
                z.loc[idx] = (s - valid.mean()) / valid.std(ddof=1)
                usable_any = True
            else:
                z.loc[idx] = 0.0

        df[col + "_z"] = z.fillna(0.0).astype(float)
        return df, usable_any

    valid = df[col].dropna()

    if len(valid) > 2 and valid.std(ddof=1) > 1e-12:
        df[col + "_z"] = ((df[col] - valid.mean()) / valid.std(ddof=1)).fillna(0.0)
        return df, True

    df[col + "_z"] = 0.0
    return df, False


def build_regression_features_primary4() -> tuple[pd.DataFrame, Dict[str, Any]]:
    if "effective_winners_primary" not in FILES or not Path(FILES["effective_winners_primary"]).exists():
        raise RuntimeError("FILES['effective_winners_primary'] is missing. Run Cell 6.1 first.")

    eff = pd.read_csv(FILES["effective_winners_primary"]).copy()

    required = [
        "prompt_id",
        "judge_family",
        "family_1",
        "family_2",
        "support_family_1",
        "support_family_2",
    ]

    missing = [c for c in required if c not in eff.columns]
    if missing:
        raise RuntimeError(f"effective_winners_primary missing columns: {missing}")

    for c in ["judge_family", "family_1", "family_2"]:
        eff[c] = eff[c].map(_82_clean_family)

    eff["prompt_id"] = eff["prompt_id"].astype(str).str.strip()
    eff["support_family_1"] = pd.to_numeric(eff["support_family_1"], errors="coerce")
    eff["support_family_2"] = pd.to_numeric(eff["support_family_2"], errors="coerce")

    if "falcon" in set(eff["judge_family"]) or "falcon" in set(eff["family_1"]) or "falcon" in set(eff["family_2"]):
        raise RuntimeError("Falcon leaked into Primary-4 regression features.")

    eff = eff[
        eff["judge_family"].isin(PRIMARY_JUDGES_82)
        & eff["family_1"].isin(PRIMARY_CANDS_82)
        & eff["family_2"].isin(PRIMARY_CANDS_82)
    ].copy()

    if eff.empty:
        raise RuntimeError("No Primary-4 effective winners available for regression features.")

    bt = _82_load_bt_abilities()
    style_lookup = _82_load_style_lookup()
    response_lookup = _82_load_response_lookup()

    rows = []

    for _, r in eff.iterrows():
        prompt_id = str(r["prompt_id"])
        judge = _82_clean_family(r["judge_family"])
        f1 = _82_clean_family(r["family_1"])
        f2 = _82_clean_family(r["family_2"])

        pair_id = r.get("pair_id", f"{prompt_id}::{f1}_vs_{f2}")

        meta = {
            "source": r.get("source", ""),
            "category": r.get("category", ""),
            "split": r.get("split", ""),
            "outcome": r.get("outcome", ""),
            "pair_id": pair_id,
            "prompt_id": prompt_id,
            "judge_family": judge,
        }

        for side, fam, opp, support, side_family_1 in [
            ("family_1", f1, f2, float(r["support_family_1"]), 1),
            ("family_2", f2, f1, float(r["support_family_2"]), 0),
        ]:
            cand_resp = response_lookup.get((prompt_id, fam), {})
            opp_resp = response_lookup.get((prompt_id, opp), {})

            cand_tokens = cand_resp.get("n_tokens", np.nan)
            opp_tokens = opp_resp.get("n_tokens", np.nan)

            cand_logprob = cand_resp.get("logprob", np.nan)
            opp_logprob = opp_resp.get("logprob", np.nan)

            cand_style = style_lookup.get((prompt_id, fam, judge), np.nan)
            opp_style = style_lookup.get((prompt_id, opp, judge), np.nan)

            rows.append({
                **meta,
                "side": side,
                "candidate_family": fam,
                "opponent_family": opp,
                "support": support,
                "same_family": int(judge == fam),
                "side_family_1": int(side_family_1),
                "bt_ability_candidate": float(bt.get(fam, 0.0)),
                "bt_ability_opponent": float(bt.get(opp, 0.0)),
                "bt_advantage": float(bt.get(fam, 0.0) - bt.get(opp, 0.0)),
                "style_sim": cand_style,
                "style_opp": opp_style,
                "style_advantage": cand_style - opp_style if pd.notna(cand_style) and pd.notna(opp_style) else np.nan,
                "candidate_tokens": cand_tokens,
                "opponent_tokens": opp_tokens,
                "length_ratio": (
                    np.log1p(cand_tokens) - np.log1p(opp_tokens)
                    if pd.notna(cand_tokens) and pd.notna(opp_tokens)
                    else np.nan
                ),
                "candidate_logprob": cand_logprob,
                "opponent_logprob": opp_logprob,
                "logprob_advantage": (
                    cand_logprob - opp_logprob
                    if pd.notna(cand_logprob) and pd.notna(opp_logprob)
                    else np.nan
                ),
            })

    df = pd.DataFrame(rows)

    for c in ["source", "category", "split"]:
        if c not in df.columns:
            df[c] = ""
        df[c] = df[c].fillna("").astype(str)

    feature_availability = {}

    for col, group_col in [
        ("bt_advantage", None),
        ("style_sim", "judge_family"),
        ("style_advantage", "judge_family"),
        ("length_ratio", None),
        ("logprob_advantage", None),
    ]:
        df, available = _82_zscore(df, col, group_col=group_col)
        feature_availability[col] = bool(available)

    df["analysis_label"] = "primary_4"

    summary = {
        "analysis_label": "primary_4_regression_features",
        "schema_note": (
            "Fractional support outcome. One row per candidate side in each "
            "effective comparison. Falcon excluded."
        ),
        "n_rows": int(len(df)),
        "n_effective_comparisons": int(len(eff)),
        "n_prompts": int(df["prompt_id"].nunique()),
        "judge_families": sorted(df["judge_family"].unique().tolist()),
        "candidate_families": sorted(df["candidate_family"].unique().tolist()),
        "feature_availability": feature_availability,
        "missing_rates": {
            c: float(df[c].isna().mean())
            for c in [
                "bt_advantage",
                "style_sim",
                "style_advantage",
                "length_ratio",
                "logprob_advantage",
            ]
            if c in df.columns
        },
        "support_summary": {
            "min": float(df["support"].min()),
            "mean": float(df["support"].mean()),
            "max": float(df["support"].max()),
            "fractional_share": float((~df["support"].isin([0.0, 1.0])).mean()),
        },
        "outputs": {
            "regression_features": str(FILES["regression_features"]),
            "regression_feature_summary": str(FILES["regression_feature_summary"]),
        },
    }

    return df, summary


def _step_regression_features_82():
    print("=" * 100)
    print("BUILDING PRIMARY-4 MECHANISM REGRESSION FEATURES")
    print("=" * 100)

    df, summary = build_regression_features_primary4()

    atomic_write_csv(df, FILES["regression_features"])
    atomic_write_json(summary, FILES["regression_feature_summary"])

    print("\nRegression feature summary")
    print("-" * 100)
    print(json.dumps(summary, indent=2))

    print("\nFeature preview")
    print("-" * 100)
    print(df.head(10).to_string(index=False))

    print("\nSaved:")
    print("Features:", FILES["regression_features"])
    print("Summary:", FILES["regression_feature_summary"])


run_step(
    [
        FILES["regression_features"],
        FILES["regression_feature_summary"],
    ],
    _step_regression_features_82,
    "build_regression_features_primary4_fractional_support",
    force=False,
    validators={
        FILES["regression_features"]: lambda p: csv_has(
            p,
            [
                "support",
                "same_family",
                "bt_advantage_z",
                "style_advantage_z",
                "length_ratio_z",
                "candidate_family",
                "judge_family",
            ],
            50,
        ),
        FILES["regression_feature_summary"]: lambda p: json_has(
            p,
            ["analysis_label", "n_rows", "feature_availability", "support_summary"],
        ),
    },
)

print("\n✓ Cell 8.2 complete")

In [ ]:
# ============================================================================
# Cell 8.2B — Logprob / familiarity feature audit
# ============================================================================
"""
Cell 8.2B — Audit whether the logprob/familiarity mechanism is actually testable.

Purpose:
- Check whether logprob_z exists.
- Check missingness and variance.
- Decide whether the paper can claim familiarity was tested.
- Save paper-safe wording.

If logprob_z is unusable, the paper should not say:
    "we control for familiarity"

Instead say:
    "we control for quality, style, length, side/order, and fixed effects;
     likelihood-based familiarity remains future work."
"""

from pathlib import Path
import json
import pandas as pd
import numpy as np

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_mechanisms" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_mechanisms"] = Path(PATHS["analysis"]) / "mechanisms"
    elif "root" in PATHS:
        PATHS["analysis_mechanisms"] = Path(PATHS["root"]) / "analysis" / "mechanisms"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_mechanisms"] = Path(ROOT_DIR) / "analysis" / "mechanisms"
    else:
        PATHS["analysis_mechanisms"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/mechanisms")

Path(PATHS["analysis_mechanisms"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("regression_features", Path(PATHS["analysis_mechanisms"]) / "regression_features_primary4.csv")
FILES.setdefault("logprob_familiarity_audit", Path(PATHS["analysis_mechanisms"]) / "logprob_familiarity_audit.json")


def _step_logprob_familiarity_audit_82b():
    print("=" * 100)
    print("LOGPROB / FAMILIARITY FEATURE AUDIT")
    print("=" * 100)

    path = Path(FILES["regression_features"])

    if not path.exists():
        raise RuntimeError("regression_features file missing. Run Cell 8.2 first.")

    df = pd.read_csv(path)

    result = {
        "analysis_label": "logprob_familiarity_audit_primary4",
        "regression_features_path": str(path),
        "n_rows": int(len(df)),
        "columns_present": list(df.columns),
        "logprob_column_present": "logprob" in df.columns,
        "logprob_z_column_present": "logprob_z" in df.columns,
    }

    if "logprob_z" in df.columns:
        s = pd.to_numeric(df["logprob_z"], errors="coerce")
        nonmissing = int(s.notna().sum())
        missing = int(s.isna().sum())
        std = float(s.std()) if nonmissing > 1 else 0.0
        unique = int(s.dropna().nunique())

        usable = bool(nonmissing >= max(100, 0.5 * len(df)) and std > 1e-8 and unique > 5)

        result.update({
            "logprob_z_nonmissing": nonmissing,
            "logprob_z_missing": missing,
            "logprob_z_missing_rate": float(missing / max(len(df), 1)),
            "logprob_z_std": std,
            "logprob_z_unique_values": unique,
            "familiarity_feature_usable": usable,
        })
    else:
        result.update({
            "logprob_z_nonmissing": 0,
            "logprob_z_missing": int(len(df)),
            "logprob_z_missing_rate": 1.0,
            "logprob_z_std": 0.0,
            "logprob_z_unique_values": 0,
            "familiarity_feature_usable": False,
        })

    if result["familiarity_feature_usable"]:
        decision = "keep_as_covariate"
        paper_claim = (
            "The mechanism model additionally includes a likelihood/logprob-based proxy "
            "for evaluator familiarity."
        )
    else:
        decision = "do_not_claim_familiarity_tested"
        paper_claim = (
            "The mechanism model controls for quality, style similarity, length, side/order, "
            "judge fixed effects, candidate fixed effects, opponent fixed effects, and category. "
            "Likelihood-based familiarity is not treated as a confirmed mechanism in the main analysis "
            "because the saved logprob feature is unavailable or insufficiently variable."
        )

    result["decision"] = decision
    result["paper_claim"] = paper_claim
    result["forbidden_claim"] = (
        "Do not claim that familiarity was controlled/tested unless "
        "familiarity_feature_usable is true."
    )

    atomic_write_json(result, FILES["logprob_familiarity_audit"])

    print("\nLogprob audit")
    print("-" * 100)
    print(json.dumps({
        "logprob_column_present": result["logprob_column_present"],
        "logprob_z_column_present": result["logprob_z_column_present"],
        "logprob_z_nonmissing": result["logprob_z_nonmissing"],
        "logprob_z_missing_rate": result["logprob_z_missing_rate"],
        "logprob_z_std": result["logprob_z_std"],
        "logprob_z_unique_values": result["logprob_z_unique_values"],
        "familiarity_feature_usable": result["familiarity_feature_usable"],
        "decision": result["decision"],
    }, indent=2))

    print("\nPaper-safe claim")
    print("-" * 100)
    print(result["paper_claim"])

    print("\nSaved:")
    print(FILES["logprob_familiarity_audit"])


run_step(
    [FILES["logprob_familiarity_audit"]],
    _step_logprob_familiarity_audit_82b,
    "logprob_familiarity_audit_primary4",
    force=False,
    validators={
        FILES["logprob_familiarity_audit"]: lambda p: json_has(
            p,
            ["analysis_label", "familiarity_feature_usable", "decision", "paper_claim"],
        )
    },
)

print("\n✓ Cell 8.2B complete")

In [ ]:
# ============================================================================
# Cell 8.3 — Singular-safe nested fractional-logit mechanism decomposition
# ============================================================================
"""
Cell 8.3 — Nested fractional-logit decomposition.

Outcome:
- support ∈ [0,1], from Cell 8.2.
- This is a fractional outcome, so we use GLM Binomial with logit link,
  interpreted as fractional logit / quasi-binomial GLM.

Main fix:
- Dynamically removes unavailable, all-missing, or zero-variance terms.
- Logprob is skipped automatically when unavailable.
- This prevents the previous Singular matrix failure.
"""

from pathlib import Path
from typing import Any, Dict, List, Sequence, Optional
import pandas as pd
import numpy as np
import json
import re

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_csv",
    "atomic_write_json",
    "csv_has",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_mechanisms" not in PATHS:
    if "analysis" in PATHS:
        base_analysis = Path(PATHS["analysis"])
    elif "root" in PATHS:
        base_analysis = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        base_analysis = Path(ROOT_DIR) / "analysis"
    else:
        base_analysis = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS["analysis_mechanisms"] = base_analysis / "mechanisms"

Path(PATHS["analysis_mechanisms"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("decomp_table", PATHS["analysis_mechanisms"] / "nested_fractional_logit_decomp_primary4.csv")
FILES.setdefault("decomp_summary", PATHS["analysis_mechanisms"] / "nested_fractional_logit_decomp_summary_primary4.json")


def _83_term_col(term: str) -> str:
    term = term.strip()
    m = re.match(r"^C\((.+?)\)$", term)
    if m:
        return m.group(1)
    return term


def _83_is_categorical(term: str) -> bool:
    return term.strip().startswith("C(")


def _83_term_usable(df: pd.DataFrame, term: str) -> tuple[bool, str]:
    col = _83_term_col(term)

    if col not in df.columns:
        return False, "missing_column"

    s = df[col]

    if _83_is_categorical(term):
        nunique = s.dropna().astype(str).nunique()
        if nunique >= 2:
            return True, "ok"
        return False, f"categorical_nunique_{nunique}"

    s_num = pd.to_numeric(s, errors="coerce")
    valid = s_num.dropna()

    if len(valid) < 3:
        return False, f"numeric_valid_{len(valid)}"

    if valid.nunique() < 2:
        return False, "numeric_zero_variance"

    if float(valid.std(ddof=1)) <= 1e-12:
        return False, "numeric_zero_std"

    return True, "ok"


def _83_build_formula(df: pd.DataFrame, terms: Sequence[str]) -> tuple[str, List[str], Dict[str, str]]:
    usable = []
    skipped = {}

    for term in terms:
        ok, reason = _83_term_usable(df, term)
        if ok:
            usable.append(term)
        else:
            skipped[term] = reason

    if "same_family" not in usable:
        raise RuntimeError("same_family is not usable. Something is wrong with regression_features.")

    formula = "support ~ " + " + ".join(usable)
    return formula, usable, skipped


def _83_columns_needed(formula: str) -> List[str]:
    rhs = formula.split("~", 1)[1]
    terms = [t.strip() for t in rhs.split("+")]
    cols = ["support"]

    for t in terms:
        cols.append(_83_term_col(t))

    return sorted(set(cols))


def _83_fit_fractional_logit(df: pd.DataFrame, formula: str, cluster_col: str = "prompt_id"):
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    needed = _83_columns_needed(formula)

    if cluster_col not in df.columns:
        raise RuntimeError(f"{cluster_col} missing from regression dataframe.")

    sub = df.dropna(subset=[c for c in needed if c in df.columns]).copy()

    if len(sub) < 50:
        raise RuntimeError(f"Too few rows after dropping missing values: {len(sub)}")

    sub["support"] = pd.to_numeric(sub["support"], errors="coerce").clip(1e-6, 1 - 1e-6)

    model = smf.glm(
        formula=formula,
        data=sub,
        family=sm.families.Binomial(),
    )

    res = model.fit(
        cov_type="cluster",
        cov_kwds={"groups": sub[cluster_col]},
        maxiter=200,
    )

    return res, sub


def nested_fractional_logit_decomposition(df: pd.DataFrame) -> tuple[pd.DataFrame, Dict[str, Any]]:
    df = df.copy()

    if "support" not in df.columns:
        raise RuntimeError("regression_features must contain support column.")

    df["support"] = pd.to_numeric(df["support"], errors="coerce")

    if df["support"].isna().any():
        raise RuntimeError("support column contains NaN.")

    if ((df["support"] < 0) | (df["support"] > 1)).any():
        raise RuntimeError("support values must be in [0,1].")

    for c in ["judge_family", "candidate_family", "opponent_family", "category", "source", "split"]:
        if c in df.columns:
            df[c] = df[c].fillna("missing").astype(str)

    model_terms = {
        "M1_same_family_only": [
            "same_family",
        ],
        "M2_quality_control": [
            "same_family",
            "bt_advantage_z",
        ],
        "M3_add_style": [
            "same_family",
            "bt_advantage_z",
            "style_advantage_z",
        ],
        "M4_add_length_position": [
            "same_family",
            "bt_advantage_z",
            "style_advantage_z",
            "length_ratio_z",
            "side_family_1",
        ],
        "M5_add_judge_candidate_FE": [
            "same_family",
            "bt_advantage_z",
            "style_advantage_z",
            "length_ratio_z",
            "side_family_1",
            "C(judge_family)",
            "C(candidate_family)",
        ],
        "M6_add_opponent_category_FE": [
            "same_family",
            "bt_advantage_z",
            "style_advantage_z",
            "length_ratio_z",
            "side_family_1",
            "C(judge_family)",
            "C(candidate_family)",
            "C(opponent_family)",
            "C(category)",
        ],
    }

    # Add logprob only if it is genuinely available.
    logprob_ok, logprob_reason = _83_term_usable(df, "logprob_advantage_z")
    if logprob_ok:
        model_terms["M4_add_length_position"].insert(3, "logprob_advantage_z")
        model_terms["M5_add_judge_candidate_FE"].insert(3, "logprob_advantage_z")
        model_terms["M6_add_opponent_category_FE"].insert(3, "logprob_advantage_z")

    rows = []
    skipped_by_model = {}

    for model_name, terms in model_terms.items():
        try:
            formula, usable_terms, skipped = _83_build_formula(df, terms)
            skipped_by_model[model_name] = skipped

            res, sub = _83_fit_fractional_logit(df, formula)

            same_coef = float(res.params.get("same_family", np.nan))
            same_se = float(res.bse.get("same_family", np.nan))
            same_p = float(res.pvalues.get("same_family", np.nan))

            rows.append({
                "model": model_name,
                "status": "ok",
                "formula": formula,
                "n_obs": int(res.nobs),
                "n_prompts": int(sub["prompt_id"].nunique()) if "prompt_id" in sub.columns else np.nan,
                "same_family_coef": same_coef,
                "same_family_se": same_se,
                "same_family_p": same_p,
                "same_family_or": float(np.exp(same_coef)) if pd.notna(same_coef) else np.nan,
                "bt_advantage_z_coef": float(res.params.get("bt_advantage_z", np.nan)),
                "style_advantage_z_coef": float(res.params.get("style_advantage_z", np.nan)),
                "length_ratio_z_coef": float(res.params.get("length_ratio_z", np.nan)),
                "logprob_advantage_z_coef": float(res.params.get("logprob_advantage_z", np.nan)),
                "side_family_1_coef": float(res.params.get("side_family_1", np.nan)),
                "loglik": float(res.llf),
                "aic": float(res.aic),
                "deviance": float(res.deviance),
                "pseudo_r2_mcfadden": float(1 - res.llf / res.llnull) if getattr(res, "llnull", None) not in [None, 0] else np.nan,
                "usable_terms": json.dumps(usable_terms),
                "skipped_terms": json.dumps(skipped),
                "fit_error": "",
            })

        except Exception as e:
            rows.append({
                "model": model_name,
                "status": "error",
                "formula": "",
                "n_obs": 0,
                "n_prompts": 0,
                "same_family_coef": np.nan,
                "same_family_se": np.nan,
                "same_family_p": np.nan,
                "same_family_or": np.nan,
                "bt_advantage_z_coef": np.nan,
                "style_advantage_z_coef": np.nan,
                "length_ratio_z_coef": np.nan,
                "logprob_advantage_z_coef": np.nan,
                "side_family_1_coef": np.nan,
                "loglik": np.nan,
                "aic": np.nan,
                "deviance": np.nan,
                "pseudo_r2_mcfadden": np.nan,
                "usable_terms": "[]",
                "skipped_terms": "{}",
                "fit_error": str(e),
            })

    out = pd.DataFrame(rows)

    summary = {
        "analysis_label": "primary_4_nested_fractional_logit",
        "schema_note": (
            "Fractional logit using GLM Binomial with cluster-robust SEs by prompt_id. "
            "Unavailable or zero-variance terms are skipped dynamically."
        ),
        "n_rows_input": int(len(df)),
        "fractional_outcome_detected": bool((~df["support"].isin([0.0, 1.0])).any()),
        "strict_binary_outcome": bool(df["support"].isin([0.0, 1.0]).all()),
        "logprob_available": bool(logprob_ok),
        "logprob_skip_reason": None if logprob_ok else logprob_reason,
        "models_ok": int(out["status"].eq("ok").sum()),
        "models_error": int(out["status"].eq("error").sum()),
        "skipped_terms_by_model": skipped_by_model,
        "outputs": {
            "decomp_table": str(FILES["decomp_table"]),
            "decomp_summary": str(FILES["decomp_summary"]),
        },
    }

    return out, summary


def _step_decomp_83():
    print("=" * 100)
    print("NESTED FRACTIONAL-LOGIT DECOMPOSITION — PRIMARY-4")
    print("=" * 100)

    df = pd.read_csv(FILES["regression_features"])
    out, summary = nested_fractional_logit_decomposition(df)

    atomic_write_csv(out, FILES["decomp_table"])
    atomic_write_json(summary, FILES["decomp_summary"])

    print("\nDecomposition table")
    print("-" * 100)
    cols = [
        "model",
        "status",
        "n_obs",
        "same_family_coef",
        "same_family_se",
        "same_family_p",
        "same_family_or",
        "bt_advantage_z_coef",
        "style_advantage_z_coef",
        "length_ratio_z_coef",
        "logprob_advantage_z_coef",
    ]
    print(out[cols].to_string(index=False))

    print("\nSummary")
    print("-" * 100)
    print(json.dumps(summary, indent=2))


run_step(
    [
        FILES["decomp_table"],
        FILES["decomp_summary"],
    ],
    _step_decomp_83,
    "nested_fractional_logit_decomp_primary4_singular_safe",
    force=False,
    validators={
        FILES["decomp_table"]: lambda p: csv_has(
            p,
            ["model", "status", "same_family_coef", "same_family_or"],
            5,
        ),
        FILES["decomp_summary"]: lambda p: json_has(
            p,
            ["analysis_label", "models_ok", "fractional_outcome_detected"],
        ),
    },
)

print("\n✓ Cell 8.3 complete")

In [ ]:
# ============================================================================
# Cell 8.4 — Quasi-binomial GEE, separation diagnostics, no-collinearity formula
# ============================================================================
"""
Cell 8.4 — Robust quasi-binomial GEE for the mechanism model.

Purpose:
- Fit a robust GEE model without the collinearity problem that caused nan SEs.
- Explicitly document fractional outcome handling.
- Save a readable text summary and machine-readable JSON summary.
- Save separation/fractional-outcome diagnostics.

Important:
- This is a quasi-binomial GEE when the outcome is fractional.
- It uses Binomial link + sandwich variance estimator.
- It intentionally avoids putting bt_advantage and candidate/opponent fixed
  effects in the same GEE formula, because that creates collinearity.
- Cell 8.3 remains the main nested mechanism table.
"""

from pathlib import Path
from typing import Any, Dict, List
import pandas as pd
import numpy as np
import json
import warnings

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_json",
    "atomic_write_text",
    "json_has",
    "file_nonempty",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "analysis_mechanisms" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_mechanisms"] = Path(PATHS["analysis"]) / "mechanisms"
    elif "root" in PATHS:
        PATHS["analysis_mechanisms"] = Path(PATHS["root"]) / "analysis" / "mechanisms"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_mechanisms"] = Path(ROOT_DIR) / "analysis" / "mechanisms"
    else:
        PATHS["analysis_mechanisms"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/mechanisms")

Path(PATHS["analysis_mechanisms"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("regression_features", Path(PATHS["analysis_mechanisms"]) / "regression_features_primary4.csv")
FILES.setdefault("gee_full", Path(PATHS["analysis_mechanisms"]) / "gee_quasibinomial_primary4.txt")
FILES.setdefault("gee_full_json", Path(PATHS["analysis_mechanisms"]) / "gee_quasibinomial_primary4.json")
FILES.setdefault("separation_diagnostics", Path(PATHS["analysis_mechanisms"]) / "separation_diagnostics_primary4.json")
FILES.setdefault("mixed_glmm", Path(PATHS["analysis_mechanisms"]) / "mixed_glmm_primary4.json")


def _84_numeric_series(df: pd.DataFrame, col: str, default: float = 0.0) -> pd.Series:
    if col not in df.columns:
        return pd.Series(default, index=df.index, dtype=float)

    s = pd.to_numeric(df[col], errors="coerce")

    if s.notna().sum() == 0:
        return pd.Series(default, index=df.index, dtype=float)

    return s.fillna(float(s.mean()))


def _84_zscore_safe(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    std = float(s.std()) if s.notna().sum() > 1 else 0.0

    if not np.isfinite(std) or std <= 1e-12:
        return pd.Series(0.0, index=s.index, dtype=float)

    return ((s - s.mean()) / std).fillna(0.0)


def _84_choose_outcome(df: pd.DataFrame) -> str:
    # Prefer fractional support if present.
    for c in ["support", "observed_support", "chosen_support", "candidate_support"]:
        if c in df.columns:
            return c

    if "chosen" in df.columns:
        return "chosen"

    raise RuntimeError("No usable outcome column found. Expected support/fractional outcome or chosen.")


def _84_prepare_gee_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    outcome_col = _84_choose_outcome(df)
    df["gee_outcome"] = pd.to_numeric(df[outcome_col], errors="coerce")

    # Force outcome into [0,1] and drop invalid.
    df = df[df["gee_outcome"].notna()].copy()
    df = df[(df["gee_outcome"] >= 0.0) & (df["gee_outcome"] <= 1.0)].copy()

    if df.empty:
        raise RuntimeError("No valid rows for GEE after outcome cleaning.")

    if "prompt_id" not in df.columns:
        raise RuntimeError("regression_features must contain prompt_id for clustered GEE.")

    for c in ["judge_family", "candidate_family", "opponent_family", "category"]:
        if c not in df.columns:
            df[c] = "unknown"
        df[c] = df[c].fillna("unknown").astype(str).str.lower()

    if "same_family" not in df.columns:
        raise RuntimeError("regression_features must contain same_family.")

    df["same_family"] = pd.to_numeric(df["same_family"], errors="coerce").fillna(0).astype(int)

    if "side_A" not in df.columns:
        df["side_A"] = 0
    df["side_A"] = pd.to_numeric(df["side_A"], errors="coerce").fillna(0).astype(int)

    # Create stable covariate names.
    if "bt_advantage_z" in df.columns:
        df["bt_advantage_z_clean"] = _84_numeric_series(df, "bt_advantage_z", 0.0)
    else:
        df["bt_advantage_z_clean"] = _84_zscore_safe(_84_numeric_series(df, "bt_advantage", 0.0))

    if "style_sim_z" in df.columns:
        df["style_sim_z_clean"] = _84_numeric_series(df, "style_sim_z", 0.0)
    else:
        df["style_sim_z_clean"] = _84_zscore_safe(_84_numeric_series(df, "style_sim", 0.0))

    if "length_ratio_z" in df.columns:
        df["length_ratio_z_clean"] = _84_numeric_series(df, "length_ratio_z", 0.0)
    else:
        df["length_ratio_z_clean"] = _84_zscore_safe(_84_numeric_series(df, "length_ratio", 0.0))

    # Keep logprob only if genuinely usable.
    if "logprob_z" in df.columns:
        logp = pd.to_numeric(df["logprob_z"], errors="coerce")
        usable_logp = bool(logp.notna().sum() >= max(100, 0.5 * len(df)) and logp.std() > 1e-8 and logp.nunique(dropna=True) > 5)
        df["logprob_z_clean"] = logp.fillna(logp.mean()) if usable_logp else 0.0
    else:
        usable_logp = False
        df["logprob_z_clean"] = 0.0

    df.attrs["outcome_col_original"] = outcome_col
    df.attrs["usable_logprob"] = usable_logp

    return df


def _84_fractional_and_separation_diagnostics(df: pd.DataFrame) -> Dict[str, Any]:
    y = df["gee_outcome"]

    strict_binary = y.isin([0.0, 1.0])
    fractional_mask = ~strict_binary

    out: Dict[str, Any] = {
        "analysis_label": "separation_fractional_diagnostics_primary4",
        "n_obs": int(len(df)),
        "n_prompts": int(df["prompt_id"].nunique()),
        "outcome_col_original": df.attrs.get("outcome_col_original", "unknown"),
        "outcome_min": float(y.min()),
        "outcome_max": float(y.max()),
        "n_binary_outcomes": int(strict_binary.sum()),
        "n_fractional_outcomes": int(fractional_mask.sum()),
        "fractional_outcome_rate": float(fractional_mask.mean()),
        "gee_model_language": (
            "quasi-binomial GEE with Binomial link and cluster-robust sandwich SEs"
            if fractional_mask.any()
            else "binary GEE with Binomial family and cluster-robust sandwich SEs"
        ),
        "warning": (
            "Outcome includes fractional values. Report this as quasi-binomial GEE, not ordinary binary GEE."
            if fractional_mask.any()
            else "Outcome is binary."
        ),
        "tables": {},
    }

    # Separation-like tables are most meaningful for binary outcome, but still
    # useful as support summaries.
    for col in ["same_family", "judge_family", "candidate_family", "side_A", "category"]:
        if col not in df.columns:
            continue

        tab = df.groupby(col)["gee_outcome"].agg(
            n="size",
            mean_support="mean",
            min_support="min",
            max_support="max",
        ).reset_index()

        out["tables"][col] = tab.to_dict(orient="records")

        if col in ["same_family", "side_A"]:
            binary_tab = pd.crosstab(df[col], y.round(6))
            out[f"{col}_support_table"] = binary_tab.to_dict()

    return out


def _84_make_gee_formula(df: pd.DataFrame) -> str:
    """
    No-collinearity formula.

    We intentionally do NOT include:
        C(candidate_family) + C(opponent_family)
    together with bt_advantage_z_clean.

    That combination created unstable/nan GEE coefficients.
    """
    terms = [
        "same_family",
        "bt_advantage_z_clean",
        "style_sim_z_clean",
        "length_ratio_z_clean",
        "side_A",
    ]

    if bool(df.attrs.get("usable_logprob", False)):
        terms.append("logprob_z_clean")

    if df["judge_family"].nunique() > 1:
        terms.append("C(judge_family)")

    # Category is safe and useful if not too sparse.
    if "category" in df.columns and 1 < df["category"].nunique() <= 30:
        terms.append("C(category)")

    return "gee_outcome ~ " + " + ".join(terms)


def _84_fit_gee(df: pd.DataFrame) -> Dict[str, Any]:
    import statsmodels.api as sm
    import statsmodels.formula.api as smf

    formula = _84_make_gee_formula(df)

    needed_cols = [
        "gee_outcome",
        "prompt_id",
        "same_family",
        "bt_advantage_z_clean",
        "style_sim_z_clean",
        "length_ratio_z_clean",
        "side_A",
        "judge_family",
        "category",
    ]

    if bool(df.attrs.get("usable_logprob", False)):
        needed_cols.append("logprob_z_clean")

    needed_cols = [c for c in needed_cols if c in df.columns]

    sub = df.dropna(subset=needed_cols).copy()

    if sub.empty:
        raise RuntimeError("No rows available for GEE after dropping missing covariates.")

    try:
        cov_struct = sm.cov_struct.Exchangeable()
        model = smf.gee(
            formula=formula,
            groups="prompt_id",
            data=sub,
            family=sm.families.Binomial(),
            cov_struct=cov_struct,
        )

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            res = model.fit()

        cov_struct_used = "Exchangeable"

    except Exception as e:
        # Fallback is more stable.
        cov_struct = sm.cov_struct.Independence()
        model = smf.gee(
            formula=formula,
            groups="prompt_id",
            data=sub,
            family=sm.families.Binomial(),
            cov_struct=cov_struct,
        )

        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            res = model.fit()

        cov_struct_used = "Independence_fallback_after_exchangeable_error"
        caught.append(type("WarningRecord", (), {"message": f"Exchangeable failed: {e}"})())

    params = res.params
    bse = res.bse
    pvalues = res.pvalues

    rows = []

    for name in params.index:
        coef = float(params[name])
        se = float(bse[name]) if name in bse.index else np.nan
        p = float(pvalues[name]) if name in pvalues.index else np.nan

        rows.append({
            "term": str(name),
            "coef": coef,
            "se": se,
            "p": p,
            "odds_ratio": float(np.exp(coef)) if np.isfinite(coef) else np.nan,
            "coef_is_finite": bool(np.isfinite(coef)),
            "se_is_finite": bool(np.isfinite(se)),
            "p_is_finite": bool(np.isfinite(p)),
        })

    coef_table = pd.DataFrame(rows)

    nan_terms = coef_table[
        ~(coef_table["coef_is_finite"] & coef_table["se_is_finite"])
    ]["term"].tolist()

    result = {
        "formula": formula,
        "n_obs": int(res.nobs),
        "n_prompts": int(sub["prompt_id"].nunique()),
        "cov_struct_used": cov_struct_used,
        "outcome_col_original": df.attrs.get("outcome_col_original", "unknown"),
        "usable_logprob": bool(df.attrs.get("usable_logprob", False)),
        "same_family_coef": float(params.get("same_family", np.nan)),
        "same_family_se": float(bse.get("same_family", np.nan)),
        "same_family_p": float(pvalues.get("same_family", np.nan)),
        "same_family_or": float(np.exp(params.get("same_family", np.nan))) if "same_family" in params.index else np.nan,
        "nan_or_nonfinite_terms": nan_terms,
        "n_nan_or_nonfinite_terms": int(len(nan_terms)),
        "coef_table": coef_table.to_dict(orient="records"),
        "warnings": [str(w.message) for w in caught],
        "summary_text": res.summary().as_text(),
    }

    return result


def _step_gee_84():
    print("=" * 100)
    print("QUASI-BINOMIAL GEE — PRIMARY-4, NO-COLLINEARITY FORMULA")
    print("=" * 100)

    if "regression_features" not in FILES or not Path(FILES["regression_features"]).exists():
        raise RuntimeError("regression_features missing. Run Cell 8.2 first.")

    raw = pd.read_csv(FILES["regression_features"])
    df = _84_prepare_gee_dataframe(raw)

    diagnostics = _84_fractional_and_separation_diagnostics(df)
    atomic_write_json(diagnostics, FILES["separation_diagnostics"])

    gee = _84_fit_gee(df)

    text = []
    text.append("QUASI-BINOMIAL GEE — PRIMARY-4")
    text.append("=" * 100)
    text.append("")
    text.append("Model language:")
    text.append(diagnostics["gee_model_language"])
    text.append("")
    text.append("Formula:")
    text.append(gee["formula"])
    text.append("")
    text.append("Important note:")
    text.append(
        "This GEE intentionally excludes candidate/opponent fixed effects when "
        "bt_advantage is included, to avoid collinearity and non-finite SEs."
    )
    text.append("")
    text.append("Fractional-outcome diagnostic:")
    text.append(json.dumps({
        "n_obs": diagnostics["n_obs"],
        "n_fractional_outcomes": diagnostics["n_fractional_outcomes"],
        "fractional_outcome_rate": diagnostics["fractional_outcome_rate"],
        "warning": diagnostics["warning"],
    }, indent=2))
    text.append("")
    text.append("GEE summary:")
    text.append(gee["summary_text"])

    atomic_write_text(FILES["gee_full"], "\n".join(text))

    gee_json = {
        "analysis_label": "gee_quasibinomial_primary4",
        "schema_note": (
            "No-collinearity quasi-binomial GEE. Candidate/opponent fixed effects "
            "are excluded from this GEE because bt_advantage already encodes candidate "
            "quality contrast; including all of them caused non-finite SEs."
        ),
        "gee_text_path": str(FILES["gee_full"]),
        "separation_diagnostics_path": str(FILES["separation_diagnostics"]),
        **{k: v for k, v in gee.items() if k != "summary_text"},
        "paper_language": {
            "safe": (
                "As a robustness check, we fit a quasi-binomial GEE with exchangeable "
                "working correlation by prompt and sandwich standard errors. To avoid "
                "collinearity, the GEE includes the BT quality contrast but excludes "
                "candidate/opponent fixed effects; the more saturated fixed-effect "
                "specifications are reported in the nested fractional-logit table."
            )
        },
    }

    atomic_write_json(gee_json, FILES["gee_full_json"])

    # Keep mixed_glmm output key alive, but do not force a fragile GLMM.
    glmm_stub = {
        "analysis_label": "mixed_glmm_primary4",
        "status": "skipped_by_design",
        "reason": (
            "GEE is used as the robust clustered inference check. "
            "The final fixed-effect mechanism table is Cell 8.3; a GLMM is not required."
        ),
    }

    atomic_write_json(glmm_stub, FILES["mixed_glmm"])

    print("\nFractional outcome diagnostic")
    print("-" * 100)
    print(json.dumps({
        "outcome_col_original": diagnostics["outcome_col_original"],
        "n_obs": diagnostics["n_obs"],
        "n_fractional_outcomes": diagnostics["n_fractional_outcomes"],
        "fractional_outcome_rate": diagnostics["fractional_outcome_rate"],
        "model_language": diagnostics["gee_model_language"],
    }, indent=2))

    print("\nGEE formula")
    print("-" * 100)
    print(gee["formula"])

    print("\nMain GEE result")
    print("-" * 100)
    print(json.dumps({
        "same_family_coef": gee["same_family_coef"],
        "same_family_se": gee["same_family_se"],
        "same_family_p": gee["same_family_p"],
        "same_family_or": gee["same_family_or"],
        "n_nan_or_nonfinite_terms": gee["n_nan_or_nonfinite_terms"],
        "nan_or_nonfinite_terms": gee["nan_or_nonfinite_terms"],
        "cov_struct_used": gee["cov_struct_used"],
    }, indent=2))

    if gee["n_nan_or_nonfinite_terms"] > 0:
        print("\nWARNING: Some non-finite terms remain. Inspect gee_full_json.")
    else:
        print("\n✓ No non-finite GEE coefficient/SE terms detected.")

    print("\nSaved:")
    print("GEE text:", FILES["gee_full"])
    print("GEE JSON:", FILES["gee_full_json"])
    print("Diagnostics:", FILES["separation_diagnostics"])
    print("GLMM stub:", FILES["mixed_glmm"])


run_step(
    [FILES["gee_full"], FILES["gee_full_json"], FILES["separation_diagnostics"], FILES["mixed_glmm"]],
    _step_gee_84,
    "gee_quasibinomial_primary4_no_collinearity",
    force=False,
    validators={
        FILES["gee_full"]: lambda p: file_nonempty(p, 200),
        FILES["gee_full_json"]: lambda p: json_has(
            p,
            ["analysis_label", "same_family_coef", "same_family_p", "n_nan_or_nonfinite_terms"],
        ),
        FILES["separation_diagnostics"]: lambda p: json_has(
            p,
            ["analysis_label", "fractional_outcome_rate", "gee_model_language"],
        ),
        FILES["mixed_glmm"]: lambda p: json_has(
            p,
            ["analysis_label", "status", "reason"],
        ),
    },
)

print("\n✓ Cell 8.4 complete")

## PHASE 9 — Robustness Suite

Multiverse, neutral-vs-rubric, contamination split, confirmatory holdout,
scale comparison, quantization ablation. Each defends the headline against a
specific reviewer concern.



In [ ]:
# ============================================================================
# Cell 9.1 — Multiverse / specification curve with candidate-subset axis
# ============================================================================
"""
Cell 9.1 — Multiverse robustness grid.

Purpose:
- Recompute TPS across reasonable analytic choices.
- Include candidate_subset = primary_4 vs all_5.
- Save both the full multiverse grid and a compact summary.
- Use Primary-4 as headline and Full-5 only as appendix/sensitivity.

This cell is cached with force=False.
"""

from pathlib import Path
from itertools import product
from typing import Any, Dict, List, Sequence
import pandas as pd
import numpy as np
import json


# ---------------------------------------------------------------------------
# Required globals
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_csv",
    "atomic_write_json",
    "csv_has",
    "json_has",
    "build_effective_winners",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "analysis_robustness" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_robustness"] = Path(PATHS["analysis"]) / "robustness"
    elif "root" in PATHS:
        PATHS["analysis_robustness"] = Path(PATHS["root"]) / "analysis" / "robustness"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_robustness"] = Path(ROOT_DIR) / "analysis" / "robustness"
    else:
        PATHS["analysis_robustness"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/robustness")

Path(PATHS["analysis_robustness"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "multiverse_results",
    Path(PATHS["analysis_robustness"]) / "multiverse_grid_with_candidate_subset.csv",
)

FILES.setdefault(
    "multiverse_summary",
    Path(PATHS["analysis_robustness"]) / "multiverse_summary.json",
)


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

PRIMARY_JUDGES_91 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_91 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]

if "FULL_CANDIDATE_FAMILIES" in globals():
    FULL_CANDS_91 = [str(x).strip().lower() for x in FULL_CANDIDATE_FAMILIES]
else:
    FULL_CANDS_91 = sorted(set(PRIMARY_CANDS_91 + ["falcon"]))


MULTIVERSE_AXES_91 = {
    "candidate_subset": ["primary_4", "all_5"],
    "rubric": ["rubric", "neutral"],
    "prompt_subset": ["all", "classic_only", "fresh_only"],
    "judge_subset": ["all", "leave_one_out"],
    "tie_weight": [0.0, 0.5, 1.0],
    "drop_inconsistent": [False, True],
}


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _91_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""

    s = str(x).strip().lower()

    if "llama" in s:
        return "llama"
    if "qwen" in s:
        return "qwen"
    if "gemma" in s:
        return "gemma"
    if "falcon" in s:
        return "falcon"
    if "yi" in s:
        return "yi"

    return s


def _91_candidate_families_for_subset(label: str) -> List[str]:
    if label == "all_5":
        return list(FULL_CANDS_91)
    return list(PRIMARY_CANDS_91)


def _91_prepare_judgments(j: pd.DataFrame) -> pd.DataFrame:
    j = j.copy()

    if "family_1" not in j.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a", "model_a"]:
            if c in j.columns:
                j["family_1"] = j[c]
                break

    if "family_2" not in j.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b", "model_b"]:
            if c in j.columns:
                j["family_2"] = j[c]
                break

    required = ["judge_family", "family_1", "family_2"]

    missing = [c for c in required if c not in j.columns]
    if missing:
        raise RuntimeError(f"Judgment dataframe missing columns needed for multiverse: {missing}")

    for c in required:
        j[c] = j[c].map(_91_clean_family)

    if "source" not in j.columns:
        j["source"] = "unknown"

    if "split" not in j.columns:
        j["split"] = "unknown"

    if "prompt_id" not in j.columns:
        raise RuntimeError("Judgment dataframe missing prompt_id.")

    # Headline judge panel never includes Falcon.
    j = j[j["judge_family"].isin(PRIMARY_JUDGES_91)].copy()

    if "falcon" in set(j["judge_family"]):
        raise RuntimeError("Falcon leaked into Cell 9.1 judge panel.")

    return j


def _91_filter_candidate_subset(j: pd.DataFrame, candidate_fams: Sequence[str]) -> pd.DataFrame:
    candidate_fams = set([str(x).lower() for x in candidate_fams])

    return j[
        j["family_1"].isin(candidate_fams)
        & j["family_2"].isin(candidate_fams)
    ].copy()


def _91_filter_prompt_subset(j: pd.DataFrame, subset: str) -> pd.DataFrame:
    if subset == "all":
        return j.copy()

    if "FRESH_SOURCES" in globals():
        fresh_sources = set(FRESH_SOURCES)
    else:
        fresh_sources = {"fresh"}

    if "CLASSIC_SOURCES" in globals():
        classic_sources = set(CLASSIC_SOURCES)
    else:
        classic_sources = {"mt_bench", "alpacaeval", "classic"}

    source_lower = j["source"].astype(str).str.lower()

    if subset == "fresh_only":
        mask = j["source"].isin(fresh_sources) | source_lower.str.contains("fresh", na=False)
        return j[mask].copy()

    if subset == "classic_only":
        mask = j["source"].isin(classic_sources) | source_lower.str.contains(
            "mt|alpaca|classic",
            regex=True,
            na=False,
        )
        return j[mask].copy()

    raise ValueError(f"Unknown prompt subset: {subset}")


def _91_adjust_effective_winners_for_spec(eff: pd.DataFrame, tie_weight: float) -> pd.DataFrame:
    eff = eff.copy()

    for c in ["judge_family", "family_1", "family_2"]:
        if c in eff.columns:
            eff[c] = eff[c].map(_91_clean_family)

    for c in ["support_family_1", "support_family_2"]:
        if c not in eff.columns:
            raise RuntimeError(f"Effective winners missing {c}")
        eff[c] = pd.to_numeric(eff[c], errors="coerce")

    eff = eff.dropna(subset=["support_family_1", "support_family_2"]).copy()

    if "outcome" in eff.columns:
        outcome = eff["outcome"].astype(str).str.lower()

        if tie_weight == 0.0:
            eff = eff[outcome != "tie"].copy()

        elif tie_weight == 1.0:
            tie_mask = outcome.eq("tie")
            eff.loc[tie_mask, "support_family_1"] = 1.0
            eff.loc[tie_mask, "support_family_2"] = 1.0

        # tie_weight == 0.5 leaves standard 0.5/0.5 support untouched.

    return eff.reset_index(drop=True)


def _91_build_effective_for_spec(j: pd.DataFrame, drop_inconsistent: bool, tie_weight: float) -> pd.DataFrame:
    scheme = "strict_consistent" if drop_inconsistent else "standard"

    try:
        eff = build_effective_winners(j, scheme=scheme)
    except TypeError:
        eff = build_effective_winners(j)

        if drop_inconsistent and "outcome" in eff.columns:
            eff = eff[eff["outcome"].astype(str).str.lower() != "inconsistent_split"].copy()

    eff = _91_adjust_effective_winners_for_spec(eff, tie_weight=tie_weight)

    return eff


def _91_preference_tps_from_effective(
    eff: pd.DataFrame,
    judge_families: Sequence[str],
    candidate_families: Sequence[str],
) -> Dict[str, Any]:
    judge_families = [str(x).lower() for x in judge_families]
    candidate_families = [str(x).lower() for x in candidate_families]

    if eff.empty:
        return {
            "tps": np.nan,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
            "per_family_tps": {},
            "preference_matrix": pd.DataFrame(index=judge_families, columns=candidate_families, dtype=float),
        }

    numerator = pd.DataFrame(0.0, index=judge_families, columns=candidate_families)
    denominator = pd.DataFrame(0.0, index=judge_families, columns=candidate_families)

    for _, r in eff.iterrows():
        judge = _91_clean_family(r["judge_family"])
        f1 = _91_clean_family(r["family_1"])
        f2 = _91_clean_family(r["family_2"])

        if judge not in judge_families:
            continue

        if f1 in candidate_families:
            numerator.loc[judge, f1] += float(r["support_family_1"])
            denominator.loc[judge, f1] += 1.0

        if f2 in candidate_families:
            numerator.loc[judge, f2] += float(r["support_family_2"])
            denominator.loc[judge, f2] += 1.0

    pref = numerator / denominator.replace(0, np.nan)

    diag_vals = []
    offdiag_vals = []
    per_family = {}

    for fam in judge_families:
        if fam not in candidate_families:
            continue

        diag = pref.loc[fam, fam] if fam in pref.columns else np.nan

        off = [
            pref.loc[fam, cand]
            for cand in candidate_families
            if cand != fam and pd.notna(pref.loc[fam, cand])
        ]

        if pd.notna(diag):
            diag_vals.append(float(diag))

        for v in off:
            offdiag_vals.append(float(v))

        if pd.notna(diag) and len(off) > 0:
            per_family[fam] = float(diag - np.mean(off))
        else:
            per_family[fam] = np.nan

    diag_mean = float(np.mean(diag_vals)) if diag_vals else np.nan
    offdiag_mean = float(np.mean(offdiag_vals)) if offdiag_vals else np.nan

    tps = float(diag_mean - offdiag_mean) if np.isfinite(diag_mean) and np.isfinite(offdiag_mean) else np.nan

    return {
        "tps": tps,
        "diag_mean": diag_mean,
        "offdiag_mean": offdiag_mean,
        "per_family_tps": per_family,
        "preference_matrix": pref,
    }


def _91_multiverse_grid() -> pd.DataFrame:
    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] missing. Run Cell 5.4 first.")

    j_rub = _91_prepare_judgments(pd.read_csv(FILES["master_judgments"]))

    if "master_judgments_neu" in FILES and Path(FILES["master_judgments_neu"]).exists():
        j_neu = _91_prepare_judgments(pd.read_csv(FILES["master_judgments_neu"]))
        neutral_missing = False
    else:
        j_neu = j_rub.copy()
        neutral_missing = True

    rows = []

    for combo in product(*MULTIVERSE_AXES_91.values()):
        cfg = dict(zip(MULTIVERSE_AXES_91.keys(), combo))

        candidate_fams = _91_candidate_families_for_subset(cfg["candidate_subset"])

        j = j_rub.copy() if cfg["rubric"] == "rubric" else j_neu.copy()
        j = _91_filter_candidate_subset(j, candidate_fams)
        j = _91_filter_prompt_subset(j, cfg["prompt_subset"])

        base_row = {
            **cfg,
            "neutral_missing_used_rubric_fallback": bool(neutral_missing and cfg["rubric"] == "neutral"),
            "candidate_families": "|".join(candidate_fams),
        }

        if j.empty:
            rows.append({
                **base_row,
                "tps": np.nan,
                "diag_mean": np.nan,
                "offdiag_mean": np.nan,
                "n_judgment_rows": 0,
                "n_effective": 0,
                "n_prompts": 0,
                "status": "empty_after_filtering",
            })
            continue

        if cfg["judge_subset"] == "leave_one_out":
            loo_tps = []
            loo_diag = []
            loo_offdiag = []
            loo_n_eff = []

            for left_out in PRIMARY_JUDGES_91:
                jj = j[j["judge_family"] != left_out].copy()
                judge_fams = [f for f in PRIMARY_JUDGES_91 if f != left_out]

                if jj.empty:
                    continue

                eff = _91_build_effective_for_spec(
                    jj,
                    drop_inconsistent=bool(cfg["drop_inconsistent"]),
                    tie_weight=float(cfg["tie_weight"]),
                )

                stat = _91_preference_tps_from_effective(
                    eff,
                    judge_families=judge_fams,
                    candidate_families=candidate_fams,
                )

                loo_tps.append(stat["tps"])
                loo_diag.append(stat["diag_mean"])
                loo_offdiag.append(stat["offdiag_mean"])
                loo_n_eff.append(len(eff))

            rows.append({
                **base_row,
                "tps": float(np.nanmean(loo_tps)) if loo_tps else np.nan,
                "diag_mean": float(np.nanmean(loo_diag)) if loo_diag else np.nan,
                "offdiag_mean": float(np.nanmean(loo_offdiag)) if loo_offdiag else np.nan,
                "n_judgment_rows": int(len(j)),
                "n_effective": int(np.nanmean(loo_n_eff)) if loo_n_eff else 0,
                "n_prompts": int(j["prompt_id"].nunique()) if "prompt_id" in j.columns else 0,
                "status": "ok",
            })

        else:
            eff = _91_build_effective_for_spec(
                j,
                drop_inconsistent=bool(cfg["drop_inconsistent"]),
                tie_weight=float(cfg["tie_weight"]),
            )

            stat = _91_preference_tps_from_effective(
                eff,
                judge_families=PRIMARY_JUDGES_91,
                candidate_families=candidate_fams,
            )

            rows.append({
                **base_row,
                "tps": stat["tps"],
                "diag_mean": stat["diag_mean"],
                "offdiag_mean": stat["offdiag_mean"],
                "n_judgment_rows": int(len(j)),
                "n_effective": int(len(eff)),
                "n_prompts": int(j["prompt_id"].nunique()) if "prompt_id" in j.columns else 0,
                "status": "ok",
            })

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Main step
# ---------------------------------------------------------------------------

def _step_multiverse_91():
    print("=" * 100)
    print("MULTIVERSE / SPECIFICATION CURVE — PRIMARY-4 + ALL-5 CANDIDATE AXIS")
    print("=" * 100)

    grid = _91_multiverse_grid()

    atomic_write_csv(grid, FILES["multiverse_results"])

    valid = grid[grid["status"].eq("ok") & grid["tps"].notna()].copy()
    primary = valid[valid["candidate_subset"].eq("primary_4")].copy()
    all5 = valid[valid["candidate_subset"].eq("all_5")].copy()

    summary = {
        "analysis_label": "multiverse_summary_with_candidate_subset",
        "schema_note": (
            "Multiverse grid includes candidate_subset axis. Primary-4 is the headline; "
            "all_5 is appendix/sensitivity only."
        ),
        "grid_path": str(FILES["multiverse_results"]),
        "n_specs_total": int(len(grid)),
        "n_specs_valid": int(len(valid)),
        "n_primary4_specs_valid": int(len(primary)),
        "n_all5_specs_valid": int(len(all5)),
        "primary4": {
            "median_tps": float(primary["tps"].median()) if len(primary) else None,
            "mean_tps": float(primary["tps"].mean()) if len(primary) else None,
            "min_tps": float(primary["tps"].min()) if len(primary) else None,
            "max_tps": float(primary["tps"].max()) if len(primary) else None,
            "share_positive": float((primary["tps"] > 0).mean()) if len(primary) else None,
        },
        "all5": {
            "median_tps": float(all5["tps"].median()) if len(all5) else None,
            "mean_tps": float(all5["tps"].mean()) if len(all5) else None,
            "min_tps": float(all5["tps"].min()) if len(all5) else None,
            "max_tps": float(all5["tps"].max()) if len(all5) else None,
            "share_positive": float((all5["tps"] > 0).mean()) if len(all5) else None,
        },
        "paper_language": {
            "safe": (
                "Across a multiverse of analytic specifications, the Primary-4 TPS estimate "
                "remained positive in the overwhelming majority of valid specifications. "
                "Specifications that exclude inconsistent AB/BA splits or alter tie treatment "
                "often produce larger estimates than the conservative headline analysis, which "
                "retains inconsistent splits as 0.5/0.5."
            )
        },
    }

    atomic_write_json(summary, FILES["multiverse_summary"])

    print("\nMultiverse summary")
    print("-" * 100)
    print(json.dumps(summary, indent=2))

    print("\nPrimary-4 specs, first 15")
    print("-" * 100)

    show_cols = [
        "candidate_subset",
        "rubric",
        "prompt_subset",
        "judge_subset",
        "tie_weight",
        "drop_inconsistent",
        "tps",
        "n_effective",
        "status",
    ]

    show_cols = [c for c in show_cols if c in grid.columns]

    print(
        grid[grid["candidate_subset"].eq("primary_4")][show_cols]
        .head(15)
        .round(5)
        .to_string(index=False)
    )

    print("\nSaved:")
    print("Grid:", FILES["multiverse_results"])
    print("Summary:", FILES["multiverse_summary"])


run_step(
    [FILES["multiverse_results"], FILES["multiverse_summary"]],
    _step_multiverse_91,
    "multiverse_with_candidate_subset_restored",
    force=False,
    validators={
        FILES["multiverse_results"]: lambda p: csv_has(
            p,
            ["candidate_subset", "rubric", "prompt_subset", "judge_subset", "tie_weight", "drop_inconsistent", "tps"],
            8,
        ),
        FILES["multiverse_summary"]: lambda p: json_has(
            p,
            ["analysis_label", "primary4", "all5", "paper_language"],
        ),
    },
)

print("\n✓ Cell 9.1 complete")

In [ ]:
# ============================================================================
# Cell 9.2 — Rubric vs neutral judge prompt comparison on Primary-4
# ============================================================================
"""
Cell 9.2 — Compare Primary-4 TPS under rubric prompt vs neutral prompt.

Important:
- Primary-4 only.
- Falcon excluded.
- If neutral judgments are missing, the cell saves a clean missing-neutral record
  instead of crashing.
"""

from pathlib import Path
from typing import Any, Dict, List, Optional
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "build_effective_winners",
    "build_preference_matrix",
    "compute_tps",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_robustness" not in PATHS:
    if "analysis" in PATHS:
        base_analysis = Path(PATHS["analysis"])
    elif "root" in PATHS:
        base_analysis = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        base_analysis = Path(ROOT_DIR) / "analysis"
    else:
        base_analysis = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS["analysis_robustness"] = base_analysis / "robustness"

Path(PATHS["analysis_robustness"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("neutral_compare", PATHS["analysis_robustness"] / "neutral_compare_primary4.json")

PRIMARY_JUDGES_92 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_92 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]


def _92_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _92_load_neutral_path() -> Optional[Path]:
    for key in ["master_judgments_neu", "master_judgments_neutral"]:
        if key in FILES and Path(FILES[key]).exists():
            return Path(FILES[key])

    if "master_judgments" in FILES:
        p = Path(FILES["master_judgments"])
        for candidate in [
            p.with_name("master_judgments_neu.csv"),
            p.with_name("master_judgments_neutral.csv"),
        ]:
            if candidate.exists():
                return candidate

    return None


def _92_ensure_family_cols(j: pd.DataFrame) -> pd.DataFrame:
    j = j.copy()

    if "family_1" not in j.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a"]:
            if c in j.columns:
                j["family_1"] = j[c]
                break

    if "family_2" not in j.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b"]:
            if c in j.columns:
                j["family_2"] = j[c]
                break

    if "family_1" not in j.columns and "model_a" in j.columns:
        j["family_1"] = j["model_a"]

    if "family_2" not in j.columns and "model_b" in j.columns:
        j["family_2"] = j["model_b"]

    required = ["judge_family", "family_1", "family_2"]
    missing = [c for c in required if c not in j.columns]

    if missing:
        raise RuntimeError(f"Judgment table missing columns: {missing}")

    for c in required:
        j[c] = j[c].map(_92_clean_family)

    return j


def _92_filter_primary4(j: pd.DataFrame) -> pd.DataFrame:
    j = _92_ensure_family_cols(j)

    out = j[
        j["judge_family"].isin(PRIMARY_JUDGES_92)
        & j["family_1"].isin(PRIMARY_CANDS_92)
        & j["family_2"].isin(PRIMARY_CANDS_92)
    ].copy()

    fams_seen = set(out["judge_family"]) | set(out["family_1"]) | set(out["family_2"])
    if "falcon" in fams_seen:
        raise RuntimeError("Falcon leaked into Cell 9.2 Primary-4 comparison.")

    return out.reset_index(drop=True)


def _92_tps_for_judgment_file(path: Path, label: str) -> Dict[str, Any]:
    j = pd.read_csv(path)
    j = _92_filter_primary4(j)

    if len(j) == 0:
        return {
            "status": "empty",
            "label": label,
            "path": str(path),
            "n_judgment_rows": 0,
            "n_effective": 0,
            "tps": np.nan,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
        }

    try:
        eff = build_effective_winners(
            j,
            scheme="standard",
            partial_tie_weight=0.75,
            include_incomplete=True,
        )
    except TypeError:
        try:
            eff = build_effective_winners(j, scheme="standard")
        except TypeError:
            eff = build_effective_winners(j)

    eff = eff.copy()

    for c in ["judge_family", "family_1", "family_2"]:
        if c in eff.columns:
            eff[c] = eff[c].map(_92_clean_family)

    if len(eff) == 0:
        return {
            "status": "empty_effective",
            "label": label,
            "path": str(path),
            "n_judgment_rows": int(len(j)),
            "n_effective": 0,
            "tps": np.nan,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
        }

    pref, support = build_preference_matrix(eff, PRIMARY_JUDGES_92, PRIMARY_CANDS_92)
    stat = compute_tps(pref, PRIMARY_JUDGES_92, PRIMARY_CANDS_92)

    return {
        "status": "ok",
        "label": label,
        "path": str(path),
        "n_judgment_rows": int(len(j)),
        "n_effective": int(len(eff)),
        "n_prompts": int(j["prompt_id"].nunique()) if "prompt_id" in j.columns else None,
        "diag_mean": float(stat["diag_mean"]),
        "offdiag_mean": float(stat["offdiag_mean"]),
        "tps": float(stat["tps"]),
        "per_family_tps": stat.get("per_family_tps", {}),
        "preference_matrix": pref.to_dict(),
        "support_matrix": support.to_dict(),
    }


def _step_neutral_compare_92():
    print("=" * 100)
    print("RUBRIC VS NEUTRAL PROMPT COMPARISON — PRIMARY-4")
    print("=" * 100)

    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] is missing. Run Cell 5.4 first.")

    rubric_path = Path(FILES["master_judgments"])
    neutral_path = _92_load_neutral_path()

    out = {
        "analysis_label": "primary_4_rubric_vs_neutral",
        "schema_note": (
            "Primary-4 only. Falcon excluded. Neutral comparison is recorded as "
            "missing if neutral judgments have not been generated."
        ),
        "primary_judge_families": PRIMARY_JUDGES_92,
        "primary_candidate_families": PRIMARY_CANDS_92,
        "rubric": _92_tps_for_judgment_file(rubric_path, "rubric"),
    }

    if neutral_path is None:
        out["neutral"] = {
            "status": "missing",
            "error": "Neutral master judgments not found.",
            "expected_keys": ["master_judgments_neu", "master_judgments_neutral"],
            "tps": np.nan,
        }
    else:
        out["neutral"] = _92_tps_for_judgment_file(neutral_path, "neutral")

    if out["rubric"].get("status") == "ok" and out["neutral"].get("status") == "ok":
        out["delta_neutral_minus_rubric_tps"] = float(out["neutral"]["tps"] - out["rubric"]["tps"])
        out["delta_abs"] = float(abs(out["neutral"]["tps"] - out["rubric"]["tps"]))
    else:
        out["delta_neutral_minus_rubric_tps"] = None
        out["delta_abs"] = None

    atomic_write_json(out, FILES["neutral_compare"])

    print("\nRubric result")
    print("-" * 100)
    print(json.dumps({
        "status": out["rubric"].get("status"),
        "n_effective": out["rubric"].get("n_effective"),
        "tps": out["rubric"].get("tps"),
        "diag_mean": out["rubric"].get("diag_mean"),
        "offdiag_mean": out["rubric"].get("offdiag_mean"),
    }, indent=2))

    print("\nNeutral result")
    print("-" * 100)
    print(json.dumps({
        "status": out["neutral"].get("status"),
        "n_effective": out["neutral"].get("n_effective"),
        "tps": out["neutral"].get("tps"),
        "diag_mean": out["neutral"].get("diag_mean"),
        "offdiag_mean": out["neutral"].get("offdiag_mean"),
        "error": out["neutral"].get("error"),
    }, indent=2))

    print("\nSaved:", FILES["neutral_compare"])


run_step(
    [FILES["neutral_compare"]],
    _step_neutral_compare_92,
    "neutral_compare_primary4_cached",
    force=False,
    validators={
        FILES["neutral_compare"]: lambda p: json_has(
            p,
            ["analysis_label", "rubric", "neutral"],
        )
    },
)

print("\n✓ Cell 9.2 complete")

In [ ]:
# ============================================================================
# Cell 9.3 — Classic vs fresh prompt-source split on Primary-4
# ============================================================================
"""
Cell 9.3 — Compute TPS separately for classic and fresh prompt sources.

Purpose:
- Check whether Primary-4 TPS is driven by benchmark-style classic prompts
  or whether it also appears on fresher prompts.

Important:
- Uses Primary-4 only.
- Falcon is excluded.
- If source labels are unavailable, the cell saves a clear diagnostic instead
  of crashing.
- Cached with force=False.
"""

from pathlib import Path
from typing import Any, Dict, Sequence, Optional
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "build_effective_winners",
    "build_preference_matrix",
    "compute_tps",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_robustness" not in PATHS:
    if "analysis" in PATHS:
        base_analysis = Path(PATHS["analysis"])
    elif "root" in PATHS:
        base_analysis = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        base_analysis = Path(ROOT_DIR) / "analysis"
    else:
        base_analysis = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS["analysis_robustness"] = base_analysis / "robustness"

Path(PATHS["analysis_robustness"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("contamination_split", PATHS["analysis_robustness"] / "classic_fresh_split_primary4.json")

PRIMARY_JUDGES_93 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_93 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]


def _93_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _93_clean_source(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _93_fresh_sources() -> set:
    if "FRESH_SOURCES" in globals():
        return {_93_clean_source(x) for x in FRESH_SOURCES}

    return {
        "fresh",
        "wildbench",
        "wild_bench",
        "wildbench_v2",
        "new",
        "new_prompts",
        "heldout_fresh",
    }


def _93_classic_sources() -> set:
    if "CLASSIC_SOURCES" in globals():
        return {_93_clean_source(x) for x in CLASSIC_SOURCES}

    return {
        "classic",
        "mtbench",
        "mt-bench",
        "mt_bench",
        "alpacaeval",
        "alpaca_eval",
        "alpacaeval_2",
        "benchmark",
    }


def _93_ensure_family_cols(j: pd.DataFrame) -> pd.DataFrame:
    j = j.copy()

    if "family_1" not in j.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a"]:
            if c in j.columns:
                j["family_1"] = j[c]
                break

    if "family_2" not in j.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b"]:
            if c in j.columns:
                j["family_2"] = j[c]
                break

    if "family_1" not in j.columns and "model_a" in j.columns:
        j["family_1"] = j["model_a"]

    if "family_2" not in j.columns and "model_b" in j.columns:
        j["family_2"] = j["model_b"]

    required = ["judge_family", "family_1", "family_2"]
    missing = [c for c in required if c not in j.columns]

    if missing:
        raise RuntimeError(f"Judgment table missing columns: {missing}")

    for c in required:
        j[c] = j[c].map(_93_clean_family)

    if "source" not in j.columns:
        j["source"] = ""

    j["source"] = j["source"].map(_93_clean_source)

    return j


def _93_filter_primary4(j: pd.DataFrame) -> pd.DataFrame:
    j = _93_ensure_family_cols(j)

    out = j[
        j["judge_family"].isin(PRIMARY_JUDGES_93)
        & j["family_1"].isin(PRIMARY_CANDS_93)
        & j["family_2"].isin(PRIMARY_CANDS_93)
    ].copy()

    fams_seen = set(out["judge_family"]) | set(out["family_1"]) | set(out["family_2"])

    if "falcon" in fams_seen:
        raise RuntimeError("Falcon leaked into Cell 9.3 Primary-4 split.")

    return out.reset_index(drop=True)


def _93_compute_tps_for_subset(j: pd.DataFrame, label: str) -> Dict[str, Any]:
    if len(j) == 0:
        return {
            "status": "empty",
            "label": label,
            "n_judgment_rows": 0,
            "n_effective": 0,
            "n_prompts": 0,
            "tps": np.nan,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
        }

    try:
        eff = build_effective_winners(
            j,
            scheme="standard",
            partial_tie_weight=0.75,
            include_incomplete=True,
        )
    except TypeError:
        try:
            eff = build_effective_winners(j, scheme="standard")
        except TypeError:
            eff = build_effective_winners(j)

    if eff is None or len(eff) == 0:
        return {
            "status": "empty_effective",
            "label": label,
            "n_judgment_rows": int(len(j)),
            "n_effective": 0,
            "n_prompts": int(j["prompt_id"].nunique()) if "prompt_id" in j.columns else 0,
            "tps": np.nan,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
        }

    eff = eff.copy()

    for c in ["judge_family", "family_1", "family_2"]:
        if c in eff.columns:
            eff[c] = eff[c].map(_93_clean_family)

    pref, support = build_preference_matrix(eff, PRIMARY_JUDGES_93, PRIMARY_CANDS_93)
    stat = compute_tps(pref, PRIMARY_JUDGES_93, PRIMARY_CANDS_93)

    return {
        "status": "ok",
        "label": label,
        "n_judgment_rows": int(len(j)),
        "n_effective": int(len(eff)),
        "n_prompts": int(j["prompt_id"].nunique()) if "prompt_id" in j.columns else 0,
        "diag_mean": float(stat["diag_mean"]),
        "offdiag_mean": float(stat["offdiag_mean"]),
        "tps": float(stat["tps"]),
        "per_family_tps": stat.get("per_family_tps", {}),
        "preference_matrix": pref.to_dict(),
        "support_matrix": support.to_dict(),
    }


def _step_classic_fresh_split_93():
    print("=" * 100)
    print("CLASSIC VS FRESH PROMPT-SOURCE SPLIT — PRIMARY-4")
    print("=" * 100)

    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] missing. Run Cell 5.4 first.")

    j = pd.read_csv(FILES["master_judgments"])
    j = _93_filter_primary4(j)

    source_counts = j["source"].value_counts(dropna=False).to_dict()

    fresh_sources = _93_fresh_sources()
    classic_sources = _93_classic_sources()

    fresh = j[j["source"].isin(fresh_sources)].copy()
    classic = j[j["source"].isin(classic_sources)].copy()

    out = {
        "analysis_label": "primary_4_classic_fresh_split",
        "schema_note": "Primary-4 only. Falcon excluded. Split based on source labels.",
        "primary_judge_families": PRIMARY_JUDGES_93,
        "primary_candidate_families": PRIMARY_CANDS_93,
        "fresh_sources_used": sorted(fresh_sources),
        "classic_sources_used": sorted(classic_sources),
        "source_counts": source_counts,
        "all_primary4": _93_compute_tps_for_subset(j, "all_primary4"),
        "classic": _93_compute_tps_for_subset(classic, "classic"),
        "fresh": _93_compute_tps_for_subset(fresh, "fresh"),
    }

    if out["classic"]["status"] == "ok" and out["fresh"]["status"] == "ok":
        out["delta_fresh_minus_classic_tps"] = float(out["fresh"]["tps"] - out["classic"]["tps"])
    else:
        out["delta_fresh_minus_classic_tps"] = None

    atomic_write_json(out, FILES["contamination_split"])

    print("\nSource counts")
    print("-" * 100)
    print(json.dumps(source_counts, indent=2))

    print("\nClassic result")
    print("-" * 100)
    print(json.dumps({
        "status": out["classic"]["status"],
        "n_prompts": out["classic"]["n_prompts"],
        "n_effective": out["classic"]["n_effective"],
        "tps": out["classic"]["tps"],
    }, indent=2))

    print("\nFresh result")
    print("-" * 100)
    print(json.dumps({
        "status": out["fresh"]["status"],
        "n_prompts": out["fresh"]["n_prompts"],
        "n_effective": out["fresh"]["n_effective"],
        "tps": out["fresh"]["tps"],
    }, indent=2))

    print("\nSaved:", FILES["contamination_split"])


run_step(
    [FILES["contamination_split"]],
    _step_classic_fresh_split_93,
    "classic_fresh_split_primary4_cached",
    force=False,
    validators={
        FILES["contamination_split"]: lambda p: json_has(
            p,
            ["analysis_label", "all_primary4", "classic", "fresh", "source_counts"],
        )
    },
)

print("\n✓ Cell 9.3 complete")

In [ ]:


# ============================================================================
# Cell 9.4 — Confirmatory holdout: Primary-4 headline test
# ============================================================================
"""
Cell 9.4 — Re-run headline TPS on confirmatory and exploratory splits.

Purpose:
- Show that the Primary-4 effect is not only an exploratory artifact.
- If split labels are missing, save a clear diagnostic.

Important:
- Primary-4 only.
- Falcon excluded.
- Uses cluster_bootstrap_tps when available.
- Cached with force=False.
"""

from pathlib import Path
from typing import Any, Dict, Optional
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "build_effective_winners",
    "build_preference_matrix",
    "compute_tps",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_robustness" not in PATHS:
    if "analysis" in PATHS:
        base_analysis = Path(PATHS["analysis"])
    elif "root" in PATHS:
        base_analysis = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        base_analysis = Path(ROOT_DIR) / "analysis"
    else:
        base_analysis = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS["analysis_robustness"] = base_analysis / "robustness"

Path(PATHS["analysis_robustness"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("confirmatory_result", PATHS["analysis_robustness"] / "confirmatory_holdout_primary4.json")

PRIMARY_JUDGES_94 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_94 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]


def _94_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _94_clean_split(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _94_ensure_family_cols(j: pd.DataFrame) -> pd.DataFrame:
    j = j.copy()

    if "family_1" not in j.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a"]:
            if c in j.columns:
                j["family_1"] = j[c]
                break

    if "family_2" not in j.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b"]:
            if c in j.columns:
                j["family_2"] = j[c]
                break

    if "family_1" not in j.columns and "model_a" in j.columns:
        j["family_1"] = j["model_a"]

    if "family_2" not in j.columns and "model_b" in j.columns:
        j["family_2"] = j["model_b"]

    required = ["judge_family", "family_1", "family_2"]
    missing = [c for c in required if c not in j.columns]

    if missing:
        raise RuntimeError(f"Judgment table missing columns: {missing}")

    for c in required:
        j[c] = j[c].map(_94_clean_family)

    if "split" not in j.columns:
        j["split"] = ""

    j["split"] = j["split"].map(_94_clean_split)

    return j


def _94_filter_primary4(j: pd.DataFrame) -> pd.DataFrame:
    j = _94_ensure_family_cols(j)

    out = j[
        j["judge_family"].isin(PRIMARY_JUDGES_94)
        & j["family_1"].isin(PRIMARY_CANDS_94)
        & j["family_2"].isin(PRIMARY_CANDS_94)
    ].copy()

    fams_seen = set(out["judge_family"]) | set(out["family_1"]) | set(out["family_2"])

    if "falcon" in fams_seen:
        raise RuntimeError("Falcon leaked into Cell 9.4 Primary-4 confirmatory split.")

    return out.reset_index(drop=True)


def _94_compute_subset(j: pd.DataFrame, label: str, do_bootstrap: bool = False) -> Dict[str, Any]:
    if len(j) == 0:
        return {
            "status": "empty",
            "label": label,
            "n_judgment_rows": 0,
            "n_effective": 0,
            "n_prompts": 0,
            "tps": np.nan,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
        }

    try:
        eff = build_effective_winners(
            j,
            scheme="standard",
            partial_tie_weight=0.75,
            include_incomplete=True,
        )
    except TypeError:
        try:
            eff = build_effective_winners(j, scheme="standard")
        except TypeError:
            eff = build_effective_winners(j)

    if eff is None or len(eff) == 0:
        return {
            "status": "empty_effective",
            "label": label,
            "n_judgment_rows": int(len(j)),
            "n_effective": 0,
            "n_prompts": int(j["prompt_id"].nunique()) if "prompt_id" in j.columns else 0,
            "tps": np.nan,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
        }

    eff = eff.copy()

    for c in ["judge_family", "family_1", "family_2"]:
        if c in eff.columns:
            eff[c] = eff[c].map(_94_clean_family)

    pref, support = build_preference_matrix(eff, PRIMARY_JUDGES_94, PRIMARY_CANDS_94)
    stat = compute_tps(pref, PRIMARY_JUDGES_94, PRIMARY_CANDS_94)

    out = {
        "status": "ok",
        "label": label,
        "n_judgment_rows": int(len(j)),
        "n_effective": int(len(eff)),
        "n_prompts": int(j["prompt_id"].nunique()) if "prompt_id" in j.columns else 0,
        "diag_mean": float(stat["diag_mean"]),
        "offdiag_mean": float(stat["offdiag_mean"]),
        "tps": float(stat["tps"]),
        "per_family_tps": stat.get("per_family_tps", {}),
        "preference_matrix": pref.to_dict(),
        "support_matrix": support.to_dict(),
    }

    if do_bootstrap and "cluster_bootstrap_tps" in globals():
        try:
            out["bootstrap"] = cluster_bootstrap_tps(eff, n_boot=1000)
        except Exception as e:
            out["bootstrap_error":] = str(e)

    return out


def _step_confirmatory_94():
    print("=" * 100)
    print("CONFIRMATORY HOLDOUT — PRIMARY-4")
    print("=" * 100)

    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] missing. Run Cell 5.4 first.")

    j = pd.read_csv(FILES["master_judgments"])
    j = _94_filter_primary4(j)

    split_counts = j["split"].value_counts(dropna=False).to_dict()

    confirmatory_labels = {"confirmatory", "confirm", "holdout", "test", "heldout"}
    exploratory_labels = {"exploratory", "explore", "dev", "development", "train"}

    confirmatory = j[j["split"].isin(confirmatory_labels)].copy()
    exploratory = j[j["split"].isin(exploratory_labels)].copy()

    out = {
        "analysis_label": "primary_4_confirmatory_holdout",
        "schema_note": "Primary-4 only. Falcon excluded. Confirmatory/exploratory based on split labels.",
        "primary_judge_families": PRIMARY_JUDGES_94,
        "primary_candidate_families": PRIMARY_CANDS_94,
        "split_counts": split_counts,
        "all_primary4": _94_compute_subset(j, "all_primary4", do_bootstrap=False),
        "confirmatory": _94_compute_subset(confirmatory, "confirmatory", do_bootstrap=True),
        "exploratory": _94_compute_subset(exploratory, "exploratory", do_bootstrap=False),
    }

    if out["confirmatory"]["status"] == "ok" and out["exploratory"]["status"] == "ok":
        out["delta_confirmatory_minus_exploratory_tps"] = float(
            out["confirmatory"]["tps"] - out["exploratory"]["tps"]
        )
    else:
        out["delta_confirmatory_minus_exploratory_tps"] = None

    atomic_write_json(out, FILES["confirmatory_result"])

    print("\nSplit counts")
    print("-" * 100)
    print(json.dumps(split_counts, indent=2))

    print("\nConfirmatory result")
    print("-" * 100)
    print(json.dumps({
        "status": out["confirmatory"]["status"],
        "n_prompts": out["confirmatory"]["n_prompts"],
        "n_effective": out["confirmatory"]["n_effective"],
        "tps": out["confirmatory"]["tps"],
    }, indent=2))

    print("\nExploratory result")
    print("-" * 100)
    print(json.dumps({
        "status": out["exploratory"]["status"],
        "n_prompts": out["exploratory"]["n_prompts"],
        "n_effective": out["exploratory"]["n_effective"],
        "tps": out["exploratory"]["tps"],
    }, indent=2))

    print("\nSaved:", FILES["confirmatory_result"])


run_step(
    [FILES["confirmatory_result"]],
    _step_confirmatory_94,
    "confirmatory_holdout_primary4_cached",
    force=False,
    validators={
        FILES["confirmatory_result"]: lambda p: json_has(
            p,
            ["analysis_label", "all_primary4", "confirmatory", "exploratory", "split_counts"],
        )
    },
)

print("\n✓ Cell 9.4 complete")

In [ ]:
# ============================================================================
# Cell 9.5 — Large vs small judge scale comparison on Primary-4
# ============================================================================
"""
Cell 9.5 — Compare Primary-4 TPS under large judges vs small judges.

Important:
- Primary-4 candidates only.
- Falcon excluded.
- If small-judge master file is missing, save a diagnostic instead of crashing.
- Cached with force=False.
"""

from pathlib import Path
from typing import Any, Dict, Optional
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "build_effective_winners",
    "build_preference_matrix",
    "compute_tps",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_robustness" not in PATHS:
    if "analysis" in PATHS:
        base_analysis = Path(PATHS["analysis"])
    elif "root" in PATHS:
        base_analysis = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        base_analysis = Path(ROOT_DIR) / "analysis"
    else:
        base_analysis = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS["analysis_robustness"] = base_analysis / "robustness"

Path(PATHS["analysis_robustness"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("scale_compare", PATHS["analysis_robustness"] / "large_small_judge_scale_primary4.json")

PRIMARY_JUDGES_95 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_95 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]


def _95_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _95_small_path() -> Optional[Path]:
    for key in ["master_judgments_sm", "master_judgments_small", "small_master_judgments"]:
        if key in FILES and Path(FILES[key]).exists():
            return Path(FILES[key])

    if "master_judgments" in FILES:
        p = Path(FILES["master_judgments"])
        for candidate in [
            p.with_name("master_judgments_sm.csv"),
            p.with_name("master_judgments_small.csv"),
            p.with_name("small_master_judgments.csv"),
        ]:
            if candidate.exists():
                return candidate

    return None


def _95_ensure_family_cols(j: pd.DataFrame) -> pd.DataFrame:
    j = j.copy()

    if "family_1" not in j.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a"]:
            if c in j.columns:
                j["family_1"] = j[c]
                break

    if "family_2" not in j.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b"]:
            if c in j.columns:
                j["family_2"] = j[c]
                break

    if "family_1" not in j.columns and "model_a" in j.columns:
        j["family_1"] = j["model_a"]

    if "family_2" not in j.columns and "model_b" in j.columns:
        j["family_2"] = j["model_b"]

    required = ["judge_family", "family_1", "family_2"]
    missing = [c for c in required if c not in j.columns]

    if missing:
        raise RuntimeError(f"Judgment table missing columns: {missing}")

    for c in required:
        j[c] = j[c].map(_95_clean_family)

    return j


def _95_filter_primary4(j: pd.DataFrame) -> pd.DataFrame:
    j = _95_ensure_family_cols(j)

    out = j[
        j["judge_family"].isin(PRIMARY_JUDGES_95)
        & j["family_1"].isin(PRIMARY_CANDS_95)
        & j["family_2"].isin(PRIMARY_CANDS_95)
    ].copy()

    fams_seen = set(out["judge_family"]) | set(out["family_1"]) | set(out["family_2"])

    if "falcon" in fams_seen:
        raise RuntimeError("Falcon leaked into Cell 9.5 Primary-4 scale comparison.")

    return out.reset_index(drop=True)


def _95_tps_for_file(path: Path, label: str) -> Dict[str, Any]:
    j = pd.read_csv(path)
    j = _95_filter_primary4(j)

    if len(j) == 0:
        return {
            "status": "empty",
            "label": label,
            "path": str(path),
            "n_judgment_rows": 0,
            "n_effective": 0,
            "n_prompts": 0,
            "tps": np.nan,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
        }

    try:
        eff = build_effective_winners(
            j,
            scheme="standard",
            partial_tie_weight=0.75,
            include_incomplete=True,
        )
    except TypeError:
        try:
            eff = build_effective_winners(j, scheme="standard")
        except TypeError:
            eff = build_effective_winners(j)

    if eff is None or len(eff) == 0:
        return {
            "status": "empty_effective",
            "label": label,
            "path": str(path),
            "n_judgment_rows": int(len(j)),
            "n_effective": 0,
            "n_prompts": int(j["prompt_id"].nunique()) if "prompt_id" in j.columns else 0,
            "tps": np.nan,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
        }

    eff = eff.copy()

    for c in ["judge_family", "family_1", "family_2"]:
        if c in eff.columns:
            eff[c] = eff[c].map(_95_clean_family)

    judge_fams = [f for f in PRIMARY_JUDGES_95 if f in set(eff["judge_family"])]

    pref, support = build_preference_matrix(eff, judge_fams, PRIMARY_CANDS_95)
    stat = compute_tps(pref, judge_fams, PRIMARY_CANDS_95)

    return {
        "status": "ok",
        "label": label,
        "path": str(path),
        "judge_families": judge_fams,
        "n_judgment_rows": int(len(j)),
        "n_effective": int(len(eff)),
        "n_prompts": int(j["prompt_id"].nunique()) if "prompt_id" in j.columns else 0,
        "diag_mean": float(stat["diag_mean"]),
        "offdiag_mean": float(stat["offdiag_mean"]),
        "tps": float(stat["tps"]),
        "per_family_tps": stat.get("per_family_tps", {}),
        "preference_matrix": pref.to_dict(),
        "support_matrix": support.to_dict(),
    }


def _step_scale_compare_95():
    print("=" * 100)
    print("LARGE VS SMALL JUDGE SCALE — PRIMARY-4")
    print("=" * 100)

    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] missing. Run Cell 5.4 first.")

    large_path = Path(FILES["master_judgments"])
    small_path = _95_small_path()

    out = {
        "analysis_label": "primary_4_large_small_judge_scale",
        "schema_note": "Primary-4 candidates only. Falcon excluded. Small result may be missing if no small-judge file exists.",
        "primary_judge_families": PRIMARY_JUDGES_95,
        "primary_candidate_families": PRIMARY_CANDS_95,
        "large": _95_tps_for_file(large_path, "large"),
    }

    if small_path is None:
        out["small"] = {
            "status": "missing",
            "error": "Small-judge master judgment file not found.",
            "expected_keys": ["master_judgments_sm", "master_judgments_small", "small_master_judgments"],
            "tps": np.nan,
        }
    else:
        out["small"] = _95_tps_for_file(small_path, "small")

    if out["large"].get("status") == "ok" and out["small"].get("status") == "ok":
        out["delta_small_minus_large_tps"] = float(out["small"]["tps"] - out["large"]["tps"])
    else:
        out["delta_small_minus_large_tps"] = None

    atomic_write_json(out, FILES["scale_compare"])

    print("\nLarge result")
    print("-" * 100)
    print(json.dumps({
        "status": out["large"].get("status"),
        "n_prompts": out["large"].get("n_prompts"),
        "n_effective": out["large"].get("n_effective"),
        "tps": out["large"].get("tps"),
    }, indent=2))

    print("\nSmall result")
    print("-" * 100)
    print(json.dumps({
        "status": out["small"].get("status"),
        "n_prompts": out["small"].get("n_prompts"),
        "n_effective": out["small"].get("n_effective"),
        "tps": out["small"].get("tps"),
        "error": out["small"].get("error"),
    }, indent=2))

    print("\nSaved:", FILES["scale_compare"])


run_step(
    [FILES["scale_compare"]],
    _step_scale_compare_95,
    "large_small_judge_scale_primary4_cached",
    force=False,
    validators={
        FILES["scale_compare"]: lambda p: json_has(
            p,
            ["analysis_label", "large", "small"],
        )
    },
)

print("\n✓ Cell 9.5 complete")

In [ ]:
# ============================================================================
# Cell 9.6 — Quantization / generation-quality ablation
# ============================================================================
"""
Cell 9.6 — Quantization and response-artifact ablation.

Purpose:
- Summarize whether response quality/artifact rates differ across quantization
  settings or model keys.
- This is a diagnostic ablation. It does not require rerunning judgments.

Important:
- Cached with force=False.
- If no quant-ablation rows exist, saves a clean diagnostic.
"""

from pathlib import Path
from typing import Any, Dict, Optional
import pandas as pd
import numpy as np
import json
import re

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_robustness" not in PATHS:
    if "analysis" in PATHS:
        base_analysis = Path(PATHS["analysis"])
    elif "root" in PATHS:
        base_analysis = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        base_analysis = Path(ROOT_DIR) / "analysis"
    else:
        base_analysis = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS["analysis_robustness"] = base_analysis / "robustness"

Path(PATHS["analysis_robustness"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("quant_ablation", PATHS["analysis_robustness"] / "quantization_ablation.json")


def _96_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if "llama" in s:
        return "llama"
    if "qwen" in s:
        return "qwen"
    if "gemma" in s:
        return "gemma"
    if "falcon" in s:
        return "falcon"
    if "yi" in s:
        return "yi"
    return s


def _96_bool_quality(x: Any) -> bool:
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in {"true", "1", "yes", "y", "ok", "pass", "passed"}


def _96_detect_quant(row: pd.Series) -> str:
    for c in ["quant", "quantization", "quant_level", "gguf_quant"]:
        if c in row.index and pd.notna(row[c]) and str(row[c]).strip():
            return str(row[c]).strip().upper()

    text = " ".join(
        str(row.get(c, ""))
        for c in ["model_key", "model_name", "model_path", "checkpoint", "path"]
        if c in row.index
    ).upper()

    m = re.search(r"\bQ[2-8]_[A-Z0-9_]+\b", text)
    if m:
        return m.group(0)

    m = re.search(r"\bQ[2-8]\b", text)
    if m:
        return m.group(0)

    return "unknown"


def _96_response_artifacts(text: Any) -> Dict[str, Any]:
    s = "" if pd.isna(text) else str(text)

    eos_patterns = ["</s>", "<|endoftext|>", "<eos>", "<|eot_id|>", "<end_of_turn>"]
    eos_hits = [p for p in eos_patterns if p in s]

    words = s.split()

    return {
        "n_chars": int(len(s)),
        "n_words": int(len(words)),
        "empty_response": bool(len(s.strip()) == 0),
        "very_short_response": bool(len(words) < 5),
        "eos_leak": bool(len(eos_hits) > 0),
        "eos_patterns": "|".join(eos_hits),
    }


def _step_quant_ablation_96():
    print("=" * 100)
    print("QUANTIZATION / GENERATION-QUALITY ABLATION")
    print("=" * 100)

    if "all_responses" not in FILES or not Path(FILES["all_responses"]).exists():
        out = {
            "analysis_label": "quantization_ablation",
            "status": "missing_all_responses",
            "error": "FILES['all_responses'] not found.",
        }
        atomic_write_json(out, FILES["quant_ablation"])
        print(json.dumps(out, indent=2))
        return

    resp = pd.read_csv(FILES["all_responses"]).copy()

    if "model_family" not in resp.columns:
        for c in ["family", "candidate_family", "response_family"]:
            if c in resp.columns:
                resp["model_family"] = resp[c]
                break

    if "response" not in resp.columns:
        for c in ["text", "output", "completion", "answer"]:
            if c in resp.columns:
                resp["response"] = resp[c]
                break

    if "model_family" not in resp.columns:
        resp["model_family"] = "unknown"

    if "response" not in resp.columns:
        resp["response"] = ""

    if "model_key" not in resp.columns:
        if "model_name" in resp.columns:
            resp["model_key"] = resp["model_name"]
        else:
            resp["model_key"] = resp["model_family"]

    if "quality_ok" not in resp.columns:
        resp["quality_ok"] = True

    resp["model_family"] = resp["model_family"].map(_96_clean_family)
    resp["quality_ok_bool"] = resp["quality_ok"].map(_96_bool_quality)
    resp["quant_detected"] = resp.apply(_96_detect_quant, axis=1)

    art = resp["response"].apply(_96_response_artifacts).apply(pd.Series)
    resp = pd.concat([resp, art], axis=1)

    if "model_scale" in resp.columns:
        quant_rows = resp[
            resp["model_scale"].astype(str).str.lower().str.contains("quant", na=False)
            | resp["quant_detected"].ne("unknown")
        ].copy()
    else:
        quant_rows = resp[resp["quant_detected"].ne("unknown")].copy()

    if quant_rows.empty:
        out = {
            "analysis_label": "quantization_ablation",
            "status": "no_quant_data",
            "n_all_responses": int(len(resp)),
            "families_seen": sorted(resp["model_family"].dropna().unique().tolist()),
            "artifact_summary_all": {
                "quality_rate": float(resp["quality_ok_bool"].mean()),
                "eos_leak_rate": float(resp["eos_leak"].mean()),
                "very_short_rate": float(resp["very_short_response"].mean()),
                "empty_rate": float(resp["empty_response"].mean()),
                "mean_words": float(resp["n_words"].mean()),
            },
            "interpretation": (
                "No explicit quant-ablation rows were found. This cell still reports "
                "overall artifact rates but cannot estimate quantization-specific differences."
            ),
        }
        atomic_write_json(out, FILES["quant_ablation"])
        print(json.dumps(out, indent=2))
        return

    group_cols = ["model_family", "model_key", "quant_detected"]

    summary_df = (
        quant_rows.groupby(group_cols)
        .agg(
            n_rows=("response", "size"),
            n_prompts=("prompt_id", "nunique") if "prompt_id" in quant_rows.columns else ("response", "size"),
            quality_rate=("quality_ok_bool", "mean"),
            eos_leak_rate=("eos_leak", "mean"),
            very_short_rate=("very_short_response", "mean"),
            empty_rate=("empty_response", "mean"),
            mean_words=("n_words", "mean"),
            median_words=("n_words", "median"),
        )
        .reset_index()
    )

    out = {
        "analysis_label": "quantization_ablation",
        "status": "ok",
        "n_all_responses": int(len(resp)),
        "n_quant_rows": int(len(quant_rows)),
        "quant_levels_seen": sorted(quant_rows["quant_detected"].dropna().unique().tolist()),
        "per_quant_quality": summary_df.to_dict(orient="records"),
        "overall_quant_artifact_summary": {
            "quality_rate": float(quant_rows["quality_ok_bool"].mean()),
            "eos_leak_rate": float(quant_rows["eos_leak"].mean()),
            "very_short_rate": float(quant_rows["very_short_response"].mean()),
            "empty_rate": float(quant_rows["empty_response"].mean()),
            "mean_words": float(quant_rows["n_words"].mean()),
        },
        "interpretation": (
            "Use this as a diagnostic for whether quantization settings are associated "
            "with different response-quality or artifact rates. It is not a causal "
            "estimate unless quantized variants were generated under matched prompts."
        ),
    }

    atomic_write_json(out, FILES["quant_ablation"])

    print("\nQuantization summary")
    print("-" * 100)
    print(summary_df.to_string(index=False))

    print("\nSaved:", FILES["quant_ablation"])


run_step(
    [FILES["quant_ablation"]],
    _step_quant_ablation_96,
    "quantization_ablation_cached",
    force=False,
    validators={
        FILES["quant_ablation"]: lambda p: json_has(
            p,
            ["analysis_label", "status"],
        )
    },
)

print("\n✓ Cell 9.6 complete")

In [ ]:
# ============================================================================
# Cell 9.7 — Falcon validation / exclusion diagnostics
# ============================================================================
"""
Cell 9.7 — Falcon validation and exclusion diagnostics.

Purpose:
- Justify why Falcon is excluded as a headline judge.
- Keep Falcon as a candidate-only appendix sensitivity.
- Diagnose Falcon judge behavior, especially abnormal tie rate.
- Compare Primary-4 headline with Full-5 candidate appendix where available.

Important:
- This cell does NOT use Falcon as a Primary-4 judge.
- Cached with force=False.
"""

from pathlib import Path
from typing import Any, Dict, Optional
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "FULL_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_robustness" not in PATHS:
    if "analysis" in PATHS:
        base_analysis = Path(PATHS["analysis"])
    elif "root" in PATHS:
        base_analysis = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        base_analysis = Path(ROOT_DIR) / "analysis"
    else:
        base_analysis = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS["analysis_robustness"] = base_analysis / "robustness"

Path(PATHS["analysis_robustness"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("falcon_validation", PATHS["analysis_robustness"] / "falcon_validation_diagnostics.json")

PRIMARY_JUDGES_97 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_97 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]
FULL_CANDS_97 = [str(x).strip().lower() for x in FULL_CANDIDATE_FAMILIES]


def _97_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if "llama" in s:
        return "llama"
    if "qwen" in s:
        return "qwen"
    if "gemma" in s:
        return "gemma"
    if "falcon" in s:
        return "falcon"
    if "yi" in s:
        return "yi"
    return s


def _97_clean_winner(x: Any) -> str:
    if pd.isna(x):
        return "tie"
    s = str(x).strip().lower()
    if s in {"a", "response_a", "option_a", "left", "1"}:
        return "A"
    if s in {"b", "response_b", "option_b", "right", "2"}:
        return "B"
    if s in {"tie", "draw", "equal", "both", "neither", "none", "", "nan"}:
        return "tie"
    return s


def _97_ensure_family_cols(j: pd.DataFrame) -> pd.DataFrame:
    j = j.copy()

    if "family_1" not in j.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a"]:
            if c in j.columns:
                j["family_1"] = j[c]
                break

    if "family_2" not in j.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b"]:
            if c in j.columns:
                j["family_2"] = j[c]
                break

    if "family_1" not in j.columns and "model_a" in j.columns:
        j["family_1"] = j["model_a"]

    if "family_2" not in j.columns and "model_b" in j.columns:
        j["family_2"] = j["model_b"]

    for c in ["judge_family", "family_1", "family_2"]:
        if c in j.columns:
            j[c] = j[c].map(_97_clean_family)

    if "winner" in j.columns:
        j["winner_clean"] = j["winner"].map(_97_clean_winner)
    else:
        j["winner_clean"] = "tie"

    return j


def _97_judge_diagnostics(j: pd.DataFrame) -> Dict[str, Any]:
    if "judge_family" not in j.columns:
        return {"status": "missing_judge_family"}

    rows = []

    for fam, sub in j.groupby("judge_family"):
        rows.append({
            "judge_family": fam,
            "n_judgments": int(len(sub)),
            "n_prompts": int(sub["prompt_id"].nunique()) if "prompt_id" in sub.columns else None,
            "tie_count": int(sub["winner_clean"].eq("tie").sum()),
            "tie_rate": float(sub["winner_clean"].eq("tie").mean()),
            "A_rate": float(sub["winner_clean"].eq("A").mean()),
            "B_rate": float(sub["winner_clean"].eq("B").mean()),
        })

    df = pd.DataFrame(rows).sort_values("judge_family")

    primary = df[df["judge_family"].isin(PRIMARY_JUDGES_97)].copy()
    falcon = df[df["judge_family"].eq("falcon")].copy()

    return {
        "status": "ok",
        "per_judge": df.to_dict(orient="records"),
        "primary_mean_tie_rate": float(primary["tie_rate"].mean()) if len(primary) else np.nan,
        "falcon_tie_rate": float(falcon["tie_rate"].iloc[0]) if len(falcon) else None,
        "falcon_n_judgments": int(falcon["n_judgments"].iloc[0]) if len(falcon) else 0,
        "falcon_abnormal_tie_rate_flag": (
            bool(float(falcon["tie_rate"].iloc[0]) > max(0.50, float(primary["tie_rate"].mean()) + 0.25))
            if len(falcon) and len(primary)
            else None
        ),
    }


def _97_candidate_diagnostics_from_effective() -> Dict[str, Any]:
    if "effective_winners_full" not in FILES or not Path(FILES["effective_winners_full"]).exists():
        return {"status": "missing_effective_winners_full"}

    eff = pd.read_csv(FILES["effective_winners_full"]).copy()

    for c in ["judge_family", "family_1", "family_2"]:
        if c in eff.columns:
            eff[c] = eff[c].map(_97_clean_family)

    if "support_family_1" not in eff.columns or "support_family_2" not in eff.columns:
        return {"status": "missing_support_schema"}

    rows = []

    for fam in sorted(set(eff["family_1"]) | set(eff["family_2"])):
        sub = eff[(eff["family_1"].eq(fam)) | (eff["family_2"].eq(fam))].copy()

        support = 0.0

        for _, r in sub.iterrows():
            if r["family_1"] == fam:
                support += float(r["support_family_1"])
            if r["family_2"] == fam:
                support += float(r["support_family_2"])

        rows.append({
            "candidate_family": fam,
            "appearances": int(len(sub)),
            "weighted_support": float(support),
            "mean_support": float(support / len(sub)) if len(sub) else np.nan,
        })

    return {
        "status": "ok",
        "candidate_support": rows,
        "falcon_candidate_present": bool("falcon" in {r["candidate_family"] for r in rows}),
    }


def _97_load_json_if_exists(key: str) -> Optional[Dict[str, Any]]:
    if key in FILES and Path(FILES[key]).exists():
        try:
            return json.load(open(FILES[key]))
        except Exception:
            return None
    return None


def _step_falcon_validation_97():
    print("=" * 100)
    print("FALCON VALIDATION / EXCLUSION DIAGNOSTICS")
    print("=" * 100)

    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] missing. Run Cell 5.4 first.")

    j = pd.read_csv(FILES["master_judgments"])
    j = _97_ensure_family_cols(j)

    judge_diag = _97_judge_diagnostics(j)
    candidate_diag = _97_candidate_diagnostics_from_effective()

    tps_primary = _97_load_json_if_exists("tps_summary_primary")
    tps_full = _97_load_json_if_exists("tps_summary_full")
    bootstrap_primary = _97_load_json_if_exists("cluster_bootstrap_tps_primary")

    out = {
        "analysis_label": "falcon_validation_diagnostics",
        "schema_note": (
            "Falcon is excluded as headline judge but retained as candidate-only "
            "Full-5 appendix sensitivity."
        ),
        "primary_judge_families": PRIMARY_JUDGES_97,
        "primary_candidate_families": PRIMARY_CANDS_97,
        "full_candidate_families": FULL_CANDS_97,
        "policy": {
            "falcon_as_primary_judge": False,
            "falcon_as_primary_candidate": False,
            "falcon_as_appendix_candidate": bool("falcon" in FULL_CANDS_97),
        },
        "judge_diagnostics": judge_diag,
        "candidate_diagnostics": candidate_diag,
        "primary_tps_summary": tps_primary,
        "full_tps_summary": tps_full,
        "primary_bootstrap_summary": bootstrap_primary,
        "recommended_paper_language": (
            "Falcon was retained as a candidate in appendix sensitivity analyses but "
            "excluded from the headline judge panel because its judgment distribution "
            "was dominated by ties, making it unsuitable as an informative evaluator."
        ),
    }

    if tps_primary and tps_full and "tps" in tps_primary and "tps" in tps_full:
        out["full_minus_primary_tps"] = float(tps_full["tps"] - tps_primary["tps"])

    atomic_write_json(out, FILES["falcon_validation"])

    print("\nJudge diagnostics")
    print("-" * 100)
    print(json.dumps(judge_diag, indent=2)[:5000])

    print("\nCandidate diagnostics")
    print("-" * 100)
    print(json.dumps(candidate_diag, indent=2)[:5000])

    print("\nSaved:", FILES["falcon_validation"])


run_step(
    [FILES["falcon_validation"]],
    _step_falcon_validation_97,
    "falcon_validation_diagnostics_cached",
    force=False,
    validators={
        FILES["falcon_validation"]: lambda p: json_has(
            p,
            ["analysis_label", "policy", "judge_diagnostics", "candidate_diagnostics"],
        )
    },
)

print("\n✓ Cell 9.7 complete")

In [ ]:
# ============================================================================
# Cell 9.8 — Practical significance via real/declared leaderboard margins
# ============================================================================
"""
Cell 9.8 — Ranking sensitivity / practical significance anchor.

Purpose:
- Convert Primary-4 TPS into percentage points.
- Compare the observed same-family inflation against adjacent leaderboard gaps.
- Show whether TPS is larger than typical decision margins.
- Use a local leaderboard_reference.csv if present.
- If no local reference exists, create a documented fallback AlpacaEval 2.0
  reference table using selected public leaderboard values.

Important:
- This does NOT add GPT or closed-weight models as judges in this study.
- This is an external practical-significance anchor only.
- Main experiment remains open-weight judge panel only.
- If you do not want any proprietary model names in the final paper, use the
  open-weight-only subset printed by this cell.
"""

from pathlib import Path
from typing import Any, Dict, List, Optional
import pandas as pd
import numpy as np
import json
import math

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_csv",
    "atomic_write_json",
    "csv_has",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "analysis_robustness" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_robustness"] = Path(PATHS["analysis"]) / "robustness"
    elif "root" in PATHS:
        PATHS["analysis_robustness"] = Path(PATHS["root"]) / "analysis" / "robustness"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_robustness"] = Path(ROOT_DIR) / "analysis" / "robustness"
    else:
        PATHS["analysis_robustness"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/robustness")

Path(PATHS["analysis_robustness"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "leaderboard_reference",
    Path(PATHS["analysis_robustness"]) / "leaderboard_reference.csv",
)

FILES.setdefault(
    "leaderboard_margin_results",
    Path(PATHS["analysis_robustness"]) / "leaderboard_margin_results.csv",
)

FILES.setdefault(
    "leaderboard_ranking_sensitivity",
    Path(PATHS["analysis_robustness"]) / "leaderboard_ranking_sensitivity.json",
)


def _98_load_primary_tps() -> Dict[str, float]:
    """
    Load headline TPS and CI from bootstrap result if available.
    Fall back to known clean-pipeline value only if the file is missing.
    """
    bootstrap_keys = [
        "cluster_bootstrap_tps_primary",
        "bootstrap_tps_primary",
        "cluster_bootstrap_primary",
    ]

    for key in bootstrap_keys:
        if key in FILES and Path(FILES[key]).exists():
            obj = json.load(open(FILES[key]))

            tps = float(obj.get("observed_tps", obj.get("tps", np.nan)))

            ci = obj.get("ci_95", None)
            if isinstance(ci, list) and len(ci) == 2:
                ci_low, ci_high = float(ci[0]), float(ci[1])
            else:
                ci_low = float(obj.get("ci_95_low", np.nan))
                ci_high = float(obj.get("ci_95_high", np.nan))

            if np.isfinite(tps):
                return {
                    "source": str(FILES[key]),
                    "observed_tps": tps,
                    "ci_low": ci_low,
                    "ci_high": ci_high,
                    "observed_tps_pp": tps * 100.0,
                    "ci_low_pp": ci_low * 100.0 if np.isfinite(ci_low) else np.nan,
                    "ci_high_pp": ci_high * 100.0 if np.isfinite(ci_high) else np.nan,
                }

    # Fallback based on clean pipeline output already established.
    return {
        "source": "fallback_clean_pipeline_value",
        "observed_tps": 0.06743986254295531,
        "ci_low": 0.053121420389461604,
        "ci_high": 0.08276417525773193,
        "observed_tps_pp": 6.743986254295531,
        "ci_low_pp": 5.3121420389461604,
        "ci_high_pp": 8.276417525773193,
    }


def _98_default_alpacaeval_reference() -> pd.DataFrame:
    """
    Fallback reference table from selected public AlpacaEval 2.0 LC win-rate values.

    Note:
    - This is not part of the experiment.
    - It anchors practical significance.
    - Replace this CSV with your own updated leaderboard_reference.csv if needed.
    """
    rows = [
        {
            "rank": 1,
            "model_name": "GPT-4 Omni (05/13)",
            "model_type": "proprietary",
            "lc_win_rate": 57.46,
            "win_rate": 51.33,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 2,
            "model_name": "GPT-4 Turbo (04/09)",
            "model_type": "proprietary",
            "lc_win_rate": 55.02,
            "win_rate": 46.12,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 3,
            "model_name": "Claude 3.5 Sonnet (06/20)",
            "model_type": "proprietary",
            "lc_win_rate": 52.37,
            "win_rate": 40.56,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 4,
            "model_name": "Yi-Large Preview",
            "model_type": "proprietary",
            "lc_win_rate": 51.89,
            "win_rate": 57.47,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 5,
            "model_name": "GPT-4o Mini (07/18)",
            "model_type": "proprietary",
            "lc_win_rate": 50.73,
            "win_rate": 44.65,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 6,
            "model_name": "GPT-4 Preview (11/06) baseline",
            "model_type": "proprietary",
            "lc_win_rate": 50.00,
            "win_rate": 50.00,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 7,
            "model_name": "Qwen1.5 110B Chat",
            "model_type": "open_or_open_weight",
            "lc_win_rate": 43.91,
            "win_rate": 33.78,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 8,
            "model_name": "Claude 3 Opus (02/29)",
            "model_type": "proprietary",
            "lc_win_rate": 40.51,
            "win_rate": 29.11,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 9,
            "model_name": "Llama 3.1 405B Instruct",
            "model_type": "open_or_open_weight",
            "lc_win_rate": 39.26,
            "win_rate": 39.11,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 10,
            "model_name": "GPT-4 original",
            "model_type": "proprietary",
            "lc_win_rate": 38.13,
            "win_rate": 23.58,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 11,
            "model_name": "Qwen2 72B Instruct",
            "model_type": "open_or_open_weight",
            "lc_win_rate": 38.07,
            "win_rate": 29.85,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 12,
            "model_name": "Claude 3 Sonnet (02/29)",
            "model_type": "proprietary",
            "lc_win_rate": 34.87,
            "win_rate": 25.56,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
        {
            "rank": 13,
            "model_name": "Llama 3 70B Instruct",
            "model_type": "open_or_open_weight",
            "lc_win_rate": 34.42,
            "win_rate": 33.18,
            "source_note": "Selected public AlpacaEval 2.0 score",
        },
    ]

    return pd.DataFrame(rows)


def _98_normalize_leaderboard(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Accept flexible column names.
    rename_map = {}

    for c in df.columns:
        cl = str(c).strip().lower()

        if cl in {"model", "model name", "name", "generator"}:
            rename_map[c] = "model_name"
        elif cl in {"lc win rate", "lc_win_rate", "lc", "length_controlled_win_rate", "length-controlled win rate"}:
            rename_map[c] = "lc_win_rate"
        elif cl in {"win rate", "win_rate", "raw_win_rate"}:
            rename_map[c] = "win_rate"
        elif cl in {"type", "model_type", "license_type"}:
            rename_map[c] = "model_type"

    df = df.rename(columns=rename_map)

    if "model_name" not in df.columns:
        raise RuntimeError("leaderboard_reference.csv must contain model_name or equivalent column.")

    if "lc_win_rate" not in df.columns:
        raise RuntimeError("leaderboard_reference.csv must contain lc_win_rate or equivalent column.")

    df["model_name"] = df["model_name"].astype(str)
    df["lc_win_rate"] = pd.to_numeric(df["lc_win_rate"], errors="coerce")

    if "win_rate" in df.columns:
        df["win_rate"] = pd.to_numeric(df["win_rate"], errors="coerce")
    else:
        df["win_rate"] = np.nan

    if "model_type" not in df.columns:
        df["model_type"] = "unknown"
    else:
        df["model_type"] = df["model_type"].fillna("unknown").astype(str)

    df = df.dropna(subset=["lc_win_rate"]).copy()
    df = df.sort_values("lc_win_rate", ascending=False).reset_index(drop=True)
    df["rank"] = np.arange(1, len(df) + 1)

    return df


def _98_adjacent_margins(df: pd.DataFrame, tps_info: Dict[str, float], subset_label: str) -> pd.DataFrame:
    rows = []

    tps_pp = float(tps_info["observed_tps_pp"])
    ci_low_pp = float(tps_info["ci_low_pp"])
    ci_high_pp = float(tps_info["ci_high_pp"])

    for i in range(len(df) - 1):
        upper = df.iloc[i]
        lower = df.iloc[i + 1]

        gap = float(upper["lc_win_rate"] - lower["lc_win_rate"])

        rows.append({
            "subset": subset_label,
            "upper_rank": int(upper["rank"]),
            "lower_rank": int(lower["rank"]),
            "upper_model": upper["model_name"],
            "lower_model": lower["model_name"],
            "upper_model_type": upper.get("model_type", "unknown"),
            "lower_model_type": lower.get("model_type", "unknown"),
            "upper_lc_win_rate": float(upper["lc_win_rate"]),
            "lower_lc_win_rate": float(lower["lc_win_rate"]),
            "adjacent_gap_pp": gap,
            "observed_tps_pp": tps_pp,
            "ci_low_pp": ci_low_pp,
            "ci_high_pp": ci_high_pp,
            "gap_smaller_than_tps": bool(gap < tps_pp),
            "gap_smaller_than_ci_low": bool(np.isfinite(ci_low_pp) and gap < ci_low_pp),
            "gap_smaller_than_ci_high": bool(np.isfinite(ci_high_pp) and gap < ci_high_pp),
            "tps_to_gap_ratio": float(tps_pp / gap) if gap > 0 else np.inf,
        })

    return pd.DataFrame(rows)


def _98_rank_shift_table(df: pd.DataFrame, tps_info: Dict[str, float], subset_label: str) -> pd.DataFrame:
    """
    Simple one-model sensitivity:
    If a model receives +TPS points from a same-family evaluator, how many
    ranks could it cross?
    """
    tps_pp = float(tps_info["observed_tps_pp"])
    ci_low_pp = float(tps_info["ci_low_pp"])
    ci_high_pp = float(tps_info["ci_high_pp"])

    rows = []

    scores = df["lc_win_rate"].to_numpy(dtype=float)

    for i, row in df.iterrows():
        score = float(row["lc_win_rate"])

        def crossed(delta: float) -> int:
            if not np.isfinite(delta):
                return 0
            boosted = score + delta
            above_scores = scores[:i]
            return int((above_scores < boosted).sum())

        rows.append({
            "subset": subset_label,
            "model_name": row["model_name"],
            "model_type": row.get("model_type", "unknown"),
            "original_rank": int(row["rank"]),
            "lc_win_rate": score,
            "rank_positions_crossed_at_ci_low": crossed(ci_low_pp),
            "rank_positions_crossed_at_observed_tps": crossed(tps_pp),
            "rank_positions_crossed_at_ci_high": crossed(ci_high_pp),
        })

    return pd.DataFrame(rows)


def _98_summarize_margins(margins: pd.DataFrame, rank_shifts: pd.DataFrame, subset_label: str) -> Dict[str, Any]:
    if margins.empty:
        return {
            "subset": subset_label,
            "n_adjacent_gaps": 0,
        }

    strong = margins["gap_smaller_than_tps"]
    low = margins["gap_smaller_than_ci_low"]
    high = margins["gap_smaller_than_ci_high"]

    return {
        "subset": subset_label,
        "n_models": int(rank_shifts.shape[0]),
        "n_adjacent_gaps": int(margins.shape[0]),
        "median_adjacent_gap_pp": float(margins["adjacent_gap_pp"].median()),
        "mean_adjacent_gap_pp": float(margins["adjacent_gap_pp"].mean()),
        "min_adjacent_gap_pp": float(margins["adjacent_gap_pp"].min()),
        "max_adjacent_gap_pp": float(margins["adjacent_gap_pp"].max()),
        "n_gaps_smaller_than_tps": int(strong.sum()),
        "share_gaps_smaller_than_tps": float(strong.mean()),
        "n_gaps_smaller_than_ci_low": int(low.sum()),
        "share_gaps_smaller_than_ci_low": float(low.mean()),
        "n_gaps_smaller_than_ci_high": int(high.sum()),
        "share_gaps_smaller_than_ci_high": float(high.mean()),
        "median_rank_positions_crossed_at_observed_tps": float(
            rank_shifts["rank_positions_crossed_at_observed_tps"].median()
        ),
        "max_rank_positions_crossed_at_observed_tps": int(
            rank_shifts["rank_positions_crossed_at_observed_tps"].max()
        ),
    }


def _step_leaderboard_ranking_sensitivity_98():
    print("=" * 100)
    print("LEADERBOARD RANKING SENSITIVITY / PRACTICAL SIGNIFICANCE")
    print("=" * 100)

    tps_info = _98_load_primary_tps()

    ref_path = Path(FILES["leaderboard_reference"])

    if ref_path.exists() and ref_path.stat().st_size > 0:
        lb_raw = pd.read_csv(ref_path)
        reference_status = "loaded_existing_local_reference"
    else:
        lb_raw = _98_default_alpacaeval_reference()
        atomic_write_csv(lb_raw, ref_path)
        reference_status = "created_default_reference_from_selected_public_values"

    lb = _98_normalize_leaderboard(lb_raw)

    all_margins = _98_adjacent_margins(lb, tps_info, "all_reference_models")
    all_shifts = _98_rank_shift_table(lb, tps_info, "all_reference_models")

    open_lb = lb[lb["model_type"].astype(str).str.contains("open", case=False, na=False)].copy()
    open_lb = open_lb.sort_values("lc_win_rate", ascending=False).reset_index(drop=True)
    open_lb["rank"] = np.arange(1, len(open_lb) + 1)

    open_margins = _98_adjacent_margins(open_lb, tps_info, "open_or_open_weight_only") if len(open_lb) >= 2 else pd.DataFrame()
    open_shifts = _98_rank_shift_table(open_lb, tps_info, "open_or_open_weight_only") if len(open_lb) >= 1 else pd.DataFrame()

    margin_results = pd.concat([all_margins, open_margins], ignore_index=True)
    shift_results = pd.concat([all_shifts, open_shifts], ignore_index=True)

    atomic_write_csv(margin_results, FILES["leaderboard_margin_results"])

    summaries = {
        "all_reference_models": _98_summarize_margins(all_margins, all_shifts, "all_reference_models"),
        "open_or_open_weight_only": _98_summarize_margins(open_margins, open_shifts, "open_or_open_weight_only") if not open_margins.empty else {
            "subset": "open_or_open_weight_only",
            "n_adjacent_gaps": int(len(open_margins)),
            "note": "Not enough open/open-weight rows to compute adjacent margins.",
        },
    }

    out = {
        "analysis_label": "leaderboard_ranking_sensitivity_primary4",
        "schema_note": (
            "Practical-significance analysis comparing Primary-4 TPS against adjacent "
            "leaderboard margins. This is an external anchor only, not an experiment "
            "with proprietary judges."
        ),
        "reference_status": reference_status,
        "leaderboard_reference_path": str(ref_path),
        "margin_results_path": str(FILES["leaderboard_margin_results"]),
        "tps_source": tps_info["source"],
        "observed_tps": float(tps_info["observed_tps"]),
        "observed_tps_pp": float(tps_info["observed_tps_pp"]),
        "ci_low": float(tps_info["ci_low"]) if np.isfinite(tps_info["ci_low"]) else None,
        "ci_high": float(tps_info["ci_high"]) if np.isfinite(tps_info["ci_high"]) else None,
        "ci_low_pp": float(tps_info["ci_low_pp"]) if np.isfinite(tps_info["ci_low_pp"]) else None,
        "ci_high_pp": float(tps_info["ci_high_pp"]) if np.isfinite(tps_info["ci_high_pp"]) else None,
        "n_reference_models": int(len(lb)),
        "n_open_or_open_weight_models": int(len(open_lb)),
        "summaries": summaries,
        "top_adjacent_gaps": margin_results.sort_values("adjacent_gap_pp").head(10).to_dict(orient="records"),
        "largest_rank_shift_examples": shift_results.sort_values(
            "rank_positions_crossed_at_observed_tps",
            ascending=False,
        ).head(10).to_dict(orient="records"),
        "paper_language": {
            "safe_main_claim": (
                f"The observed Primary-4 same-family inflation is "
                f"{tps_info['observed_tps_pp']:.2f} percentage points. In the external "
                "leaderboard anchor, this is larger than many adjacent leaderboard gaps, "
                "showing that even a statistically modest evaluator-family effect can be "
                "large relative to ranking decision margins."
            ),
            "scope_sentence": (
                "This ranking analysis is an external practical-significance anchor; "
                "it does not introduce closed-weight judges into the study."
            ),
            "if_using_open_only": (
                "For a conservative open-weight framing, report only the "
                "open_or_open_weight_only subset from this cell."
            ),
        },
    }

    atomic_write_json(out, FILES["leaderboard_ranking_sensitivity"])

    print("\nTPS anchor")
    print("-" * 100)
    print(json.dumps({
        "observed_tps": out["observed_tps"],
        "observed_tps_pp": out["observed_tps_pp"],
        "ci_low_pp": out["ci_low_pp"],
        "ci_high_pp": out["ci_high_pp"],
        "tps_source": out["tps_source"],
    }, indent=2))

    print("\nReference status")
    print("-" * 100)
    print(reference_status)
    print("Reference CSV:", ref_path)

    print("\nAll reference models summary")
    print("-" * 100)
    print(json.dumps(summaries["all_reference_models"], indent=2))

    print("\nOpen/open-weight-only summary")
    print("-" * 100)
    print(json.dumps(summaries["open_or_open_weight_only"], indent=2))

    print("\nSmallest adjacent gaps")
    print("-" * 100)
    show_cols = [
        "subset",
        "upper_rank",
        "lower_rank",
        "upper_model",
        "lower_model",
        "adjacent_gap_pp",
        "observed_tps_pp",
        "gap_smaller_than_tps",
        "tps_to_gap_ratio",
    ]
    show_cols = [c for c in show_cols if c in margin_results.columns]
    print(margin_results.sort_values("adjacent_gap_pp")[show_cols].head(12).round(4).to_string(index=False))

    print("\nPaper language")
    print("-" * 100)
    print(out["paper_language"]["safe_main_claim"])
    print(out["paper_language"]["scope_sentence"])

    print("\nSaved:")
    print("Leaderboard reference:", FILES["leaderboard_reference"])
    print("Margin results:", FILES["leaderboard_margin_results"])
    print("Sensitivity JSON:", FILES["leaderboard_ranking_sensitivity"])


run_step(
    [
        FILES["leaderboard_reference"],
        FILES["leaderboard_margin_results"],
        FILES["leaderboard_ranking_sensitivity"],
    ],
    _step_leaderboard_ranking_sensitivity_98,
    "leaderboard_ranking_sensitivity_primary4_cached",
    force=False,
    validators={
        FILES["leaderboard_reference"]: lambda p: csv_has(
            p,
            ["model_name", "lc_win_rate"],
            2,
        ),
        FILES["leaderboard_margin_results"]: lambda p: csv_has(
            p,
            ["adjacent_gap_pp", "observed_tps_pp", "gap_smaller_than_tps"],
            1,
        ),
        FILES["leaderboard_ranking_sensitivity"]: lambda p: json_has(
            p,
            ["analysis_label", "observed_tps_pp", "summaries", "paper_language"],
        ),
    },
)

print("\n✓ Cell 9.8 complete")

In [ ]:
# ============================================================================
# Cell 9.9 — Prompt-domain / source stratification for Primary-4 TPS
# ============================================================================
"""
Cell 9.9 — Prompt-domain stratification.

Purpose:
- Check whether Primary-4 TPS is concentrated in one prompt source/category
  or appears across domains.

Stratifies by available metadata:
- source
- category
- split
- source × category

Important:
- Primary-4 only.
- Falcon excluded.
- Cached with force=False.
"""

from pathlib import Path
from typing import Any, Dict, List, Optional
import pandas as pd
import numpy as np
import json

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "build_effective_winners",
    "build_preference_matrix",
    "compute_tps",
    "run_step",
    "atomic_write_csv",
    "atomic_write_json",
    "csv_has",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis_robustness" not in PATHS:
    if "analysis" in PATHS:
        base_analysis = Path(PATHS["analysis"])
    elif "root" in PATHS:
        base_analysis = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        base_analysis = Path(ROOT_DIR) / "analysis"
    else:
        base_analysis = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS["analysis_robustness"] = base_analysis / "robustness"

Path(PATHS["analysis_robustness"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("domain_stratification_primary", PATHS["analysis_robustness"] / "domain_stratification_primary4.csv")
FILES.setdefault("domain_stratification_summary", PATHS["analysis_robustness"] / "domain_stratification_summary_primary4.json")
FILES.setdefault("domain_stratification_matrices", PATHS["analysis_robustness"] / "domain_stratification_matrices_primary4.json")

PRIMARY_JUDGES_99 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_99 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]


def _99_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _99_clean_label(x: Any) -> str:
    if pd.isna(x):
        return "missing"

    s = str(x).strip()

    if not s:
        return "missing"

    return s


def _99_ensure_family_cols(j: pd.DataFrame) -> pd.DataFrame:
    j = j.copy()

    if "family_1" not in j.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a"]:
            if c in j.columns:
                j["family_1"] = j[c]
                break

    if "family_2" not in j.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b"]:
            if c in j.columns:
                j["family_2"] = j[c]
                break

    if "family_1" not in j.columns and "model_a" in j.columns:
        j["family_1"] = j["model_a"]

    if "family_2" not in j.columns and "model_b" in j.columns:
        j["family_2"] = j["model_b"]

    required = ["judge_family", "family_1", "family_2"]
    missing = [c for c in required if c not in j.columns]

    if missing:
        raise RuntimeError(f"Judgment table missing columns: {missing}")

    for c in required:
        j[c] = j[c].map(_99_clean_family)

    for c in ["source", "category", "split"]:
        if c not in j.columns:
            j[c] = "missing"

        j[c] = j[c].map(_99_clean_label)

    if "prompt_id" not in j.columns:
        raise RuntimeError("Judgment table missing prompt_id.")

    j["prompt_id"] = j["prompt_id"].astype(str).str.strip()

    return j


def _99_filter_primary4(j: pd.DataFrame) -> pd.DataFrame:
    j = _99_ensure_family_cols(j)

    out = j[
        j["judge_family"].isin(PRIMARY_JUDGES_99)
        & j["family_1"].isin(PRIMARY_CANDS_99)
        & j["family_2"].isin(PRIMARY_CANDS_99)
    ].copy()

    fams_seen = set(out["judge_family"]) | set(out["family_1"]) | set(out["family_2"])

    if "falcon" in fams_seen:
        raise RuntimeError("Falcon leaked into Cell 9.9 Primary-4 domain stratification.")

    out["source_category"] = out["source"].astype(str) + " :: " + out["category"].astype(str)

    return out.reset_index(drop=True)


def _99_compute_stratum(j: pd.DataFrame, stratum_type: str, stratum_value: str) -> tuple[Dict[str, Any], Optional[Dict[str, Any]]]:
    base = {
        "analysis_label": "primary_4",
        "stratum_type": stratum_type,
        "stratum_value": stratum_value,
        "n_judgment_rows": int(len(j)),
        "n_prompts": int(j["prompt_id"].nunique()) if len(j) else 0,
    }

    if len(j) == 0:
        return {
            **base,
            "status": "empty",
            "n_effective": 0,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
            "tps": np.nan,
            "matrix_completeness": np.nan,
            "min_cell_support": 0,
            "interpretation_flag": "empty",
        }, None

    try:
        eff = build_effective_winners(
            j,
            scheme="standard",
            partial_tie_weight=0.75,
            include_incomplete=True,
        )
    except TypeError:
        try:
            eff = build_effective_winners(j, scheme="standard")
        except TypeError:
            eff = build_effective_winners(j)

    if eff is None or len(eff) == 0:
        return {
            **base,
            "status": "empty_effective",
            "n_effective": 0,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
            "tps": np.nan,
            "matrix_completeness": np.nan,
            "min_cell_support": 0,
            "interpretation_flag": "empty_effective",
        }, None

    eff = eff.copy()

    for c in ["judge_family", "family_1", "family_2"]:
        if c in eff.columns:
            eff[c] = eff[c].map(_99_clean_family)

    eff = eff[
        eff["judge_family"].isin(PRIMARY_JUDGES_99)
        & eff["family_1"].isin(PRIMARY_CANDS_99)
        & eff["family_2"].isin(PRIMARY_CANDS_99)
    ].copy()

    if len(eff) == 0:
        return {
            **base,
            "status": "empty_effective_after_filter",
            "n_effective": 0,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
            "tps": np.nan,
            "matrix_completeness": np.nan,
            "min_cell_support": 0,
            "interpretation_flag": "empty_effective_after_filter",
        }, None

    pref, support = build_preference_matrix(eff, PRIMARY_JUDGES_99, PRIMARY_CANDS_99)
    stat = compute_tps(pref, PRIMARY_JUDGES_99, PRIMARY_CANDS_99)

    complete_cells = int(pref.notna().sum().sum())
    total_cells = int(pref.shape[0] * pref.shape[1])
    matrix_completeness = float(complete_cells / total_cells) if total_cells else np.nan

    min_cell_support = float(support.min().min()) if len(support) else 0.0
    max_cell_support = float(support.max().max()) if len(support) else 0.0

    underpowered = bool(base["n_prompts"] < 5 or len(eff) < 24 or matrix_completeness < 1.0)

    if underpowered:
        flag = "interpret_with_caution"
    elif stat["tps"] > 0:
        flag = "positive_tps"
    else:
        flag = "nonpositive_tps"

    row = {
        **base,
        "status": "ok",
        "n_effective": int(len(eff)),
        "diag_mean": float(stat["diag_mean"]),
        "offdiag_mean": float(stat["offdiag_mean"]),
        "tps": float(stat["tps"]),
        "per_family_tps": json.dumps(stat.get("per_family_tps", {})),
        "matrix_completeness": matrix_completeness,
        "min_cell_support": min_cell_support,
        "max_cell_support": max_cell_support,
        "underpowered": underpowered,
        "interpretation_flag": flag,
    }

    matrix_record = {
        "stratum_type": stratum_type,
        "stratum_value": stratum_value,
        "preference_matrix": pref.to_dict(),
        "support_matrix": support.to_dict(),
    }

    return row, matrix_record


def _99_available_stratifiers(j: pd.DataFrame) -> List[str]:
    candidates = ["source", "category", "split", "source_category"]
    out = []

    for c in candidates:
        if c in j.columns and j[c].nunique(dropna=False) > 1:
            out.append(c)

    return out


def _step_domain_stratification_99():
    print("=" * 100)
    print("PROMPT-DOMAIN / SOURCE STRATIFICATION — PRIMARY-4")
    print("=" * 100)

    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] missing. Run Cell 5.4 first.")

    j = pd.read_csv(FILES["master_judgments"])
    j = _99_filter_primary4(j)

    rows = []
    matrices = []

    overall_row, overall_matrix = _99_compute_stratum(j, "overall", "all_primary4")
    rows.append(overall_row)

    if overall_matrix is not None:
        matrices.append(overall_matrix)

    stratifiers = _99_available_stratifiers(j)

    print("Available stratifiers:", stratifiers)

    for strat in stratifiers:
        counts = j.groupby(strat)["prompt_id"].nunique().sort_values(ascending=False)

        print(f"\n{strat} prompt counts")
        print("-" * 100)
        print(counts.to_string())

        for value, sub in j.groupby(strat, dropna=False):
            value = _99_clean_label(value)
            row, matrix_record = _99_compute_stratum(sub.copy(), strat, value)
            rows.append(row)

            if matrix_record is not None:
                matrices.append(matrix_record)

    out = pd.DataFrame(rows)
    out = out.sort_values(["stratum_type", "n_prompts", "n_effective"], ascending=[True, False, False]).reset_index(drop=True)

    atomic_write_csv(out, FILES["domain_stratification_primary"])

    ok = out[(out["status"].eq("ok")) & out["tps"].notna() & ~out["stratum_type"].eq("overall")].copy()
    strong = ok[~ok["underpowered"].astype(bool)].copy() if len(ok) else ok.copy()

    summary = {
        "analysis_label": "primary_4_domain_stratification",
        "schema_note": "Primary-4 only. Falcon excluded. Small strata are descriptive only.",
        "primary_judge_families": PRIMARY_JUDGES_99,
        "primary_candidate_families": PRIMARY_CANDS_99,
        "n_primary4_judgment_rows": int(len(j)),
        "n_primary4_prompts": int(j["prompt_id"].nunique()),
        "stratifiers_used": stratifiers,
        "overall": overall_row,
        "n_strata_total_including_overall": int(len(out)),
        "n_strata_ok_excluding_overall": int(len(ok)),
        "n_strata_strong_excluding_overall": int(len(strong)),
        "share_ok_strata_positive_tps": float((ok["tps"] > 0).mean()) if len(ok) else np.nan,
        "share_strong_strata_positive_tps": float((strong["tps"] > 0).mean()) if len(strong) else np.nan,
        "median_ok_stratum_tps": float(ok["tps"].median()) if len(ok) else np.nan,
        "median_strong_stratum_tps": float(strong["tps"].median()) if len(strong) else np.nan,
        "largest_positive_strata": (
            ok.sort_values("tps", ascending=False)
            .head(10)[
                [
                    "stratum_type",
                    "stratum_value",
                    "n_prompts",
                    "n_effective",
                    "tps",
                    "interpretation_flag",
                ]
            ]
            .to_dict(orient="records")
            if len(ok)
            else []
        ),
        "largest_negative_or_null_strata": (
            ok.sort_values("tps", ascending=True)
            .head(10)[
                [
                    "stratum_type",
                    "stratum_value",
                    "n_prompts",
                    "n_effective",
                    "tps",
                    "interpretation_flag",
                ]
            ]
            .to_dict(orient="records")
            if len(ok)
            else []
        ),
        "outputs": {
            "domain_stratification_primary": str(FILES["domain_stratification_primary"]),
            "domain_stratification_summary": str(FILES["domain_stratification_summary"]),
            "domain_stratification_matrices": str(FILES["domain_stratification_matrices"]),
        },
    }

    atomic_write_json(summary, FILES["domain_stratification_summary"])

    matrices_obj = {
        "analysis_label": "primary_4_domain_stratification_matrices",
        "matrices": matrices,
    }

    atomic_write_json(matrices_obj, FILES["domain_stratification_matrices"])

    print("\nOverall")
    print("-" * 100)
    print(json.dumps({
        "n_prompts": overall_row["n_prompts"],
        "n_effective": overall_row["n_effective"],
        "tps": overall_row["tps"],
        "diag_mean": overall_row["diag_mean"],
        "offdiag_mean": overall_row["offdiag_mean"],
    }, indent=2))

    print("\nTop positive strata")
    print("-" * 100)

    if len(ok):
        print(
            ok.sort_values("tps", ascending=False)
            .head(15)[
                [
                    "stratum_type",
                    "stratum_value",
                    "n_prompts",
                    "n_effective",
                    "tps",
                    "interpretation_flag",
                ]
            ]
            .round(4)
            .to_string(index=False)
        )
    else:
        print("No valid strata.")

    print("\nSummary")
    print("-" * 100)
    print(json.dumps(summary, indent=2)[:5000])

    print("\nSaved:")
    print("Table:", FILES["domain_stratification_primary"])
    print("Summary:", FILES["domain_stratification_summary"])
    print("Matrices:", FILES["domain_stratification_matrices"])


run_step(
    [
        FILES["domain_stratification_primary"],
        FILES["domain_stratification_summary"],
        FILES["domain_stratification_matrices"],
    ],
    _step_domain_stratification_99,
    "domain_stratification_primary4_cached",
    force=False,
    validators={
        FILES["domain_stratification_primary"]: lambda p: csv_has(
            p,
            ["stratum_type", "stratum_value", "n_prompts", "n_effective", "tps", "interpretation_flag"],
            1,
        ),
        FILES["domain_stratification_summary"]: lambda p: json_has(
            p,
            ["analysis_label", "overall", "stratifiers_used", "n_strata_ok_excluding_overall"],
        ),
        FILES["domain_stratification_matrices"]: lambda p: json_has(
            p,
            ["analysis_label", "matrices"],
        ),
    },
)

print("\n✓ Cell 9.9 complete")

In [ ]:
# ============================================================================
# Cell 9.10 — Final technical integrity audit before human calibration
# ============================================================================
"""
Cell 9.10 — Final integrity audit before human calibration.

Purpose:
- Confirm all critical cached outputs exist.
- Confirm Primary-4 headline result is clean.
- Confirm Falcon is absent from Primary-4.
- Confirm key diagnostics were saved.
- Confirm no stale Full-5 result is being used as headline.
- Save one final audit JSON.

Run this before moving to human calibration.
"""

from pathlib import Path
import json
import pandas as pd
import numpy as np

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_json",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

if "analysis" not in PATHS:
    if "root" in PATHS:
        PATHS["analysis"] = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        PATHS["analysis"] = Path(ROOT_DIR) / "analysis"
    else:
        PATHS["analysis"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

if "analysis_audit" not in PATHS:
    PATHS["analysis_audit"] = Path(PATHS["analysis"]) / "audit"

Path(PATHS["analysis_audit"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("final_integrity_audit", Path(PATHS["analysis_audit"]) / "final_integrity_audit_before_human_calibration.json")


def _exists_and_nonempty(path):
    try:
        p = Path(path)
        return bool(p.exists() and p.stat().st_size > 0)
    except Exception:
        return False


def _load_json_if_exists(path):
    p = Path(path)
    if p.exists() and p.stat().st_size > 0:
        return json.load(open(p))
    return None


def _clean_fam_910(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def _step_final_integrity_audit_910():
    print("=" * 100)
    print("FINAL TECHNICAL INTEGRITY AUDIT BEFORE HUMAN CALIBRATION")
    print("=" * 100)

    # Expected files. Some may have slightly different keys depending on earlier cells,
    # so we only require the ones that are truly critical.
    required_file_keys = [
        "effective_winners_primary",
        "master_judgments",
        "cluster_bootstrap_tps_primary",
        "permutation_result",
        "per_family_bh",
        "bt_results_primary",
        "bt_diagnostics",
        "response_artifact_diagnostics",
        "response_artifact_interpretation",
        "phase8_primary_guard",
        "logprob_familiarity_audit",
        "leaderboard_ranking_sensitivity",
    ]

    optional_file_keys = [
        "judgment_file_semantics",
        "bt_residuals_primary",
        "regression_features",
        "decomp_table",
        "gee_full",
        "mixed_glmm",
        "separation_diagnostics",
        "multiverse_results",
        "neutral_compare",
        "contamination_split",
        "confirmatory_result",
        "scale_compare",
        "quant_ablation",
        "falcon_validation",
        "domain_stratification",
    ]

    file_status = {}

    for key in required_file_keys + optional_file_keys:
        if key in FILES:
            file_status[key] = {
                "path": str(FILES[key]),
                "exists_nonempty": _exists_and_nonempty(FILES[key]),
            }
        else:
            file_status[key] = {
                "path": None,
                "exists_nonempty": False,
            }

    missing_required = [
        key for key in required_file_keys
        if not file_status.get(key, {}).get("exists_nonempty", False)
    ]

    # Primary-4 effective winners integrity.
    if "effective_winners_primary" not in FILES or not Path(FILES["effective_winners_primary"]).exists():
        raise RuntimeError("effective_winners_primary missing.")

    eff = pd.read_csv(FILES["effective_winners_primary"]).copy()

    for c in ["judge_family", "family_1", "family_2"]:
        if c in eff.columns:
            eff[c] = eff[c].map(_clean_fam_910)

    judges = sorted(eff["judge_family"].dropna().unique().tolist()) if "judge_family" in eff.columns else []
    candidates = sorted(set(eff["family_1"].dropna().unique()) | set(eff["family_2"].dropna().unique())) if {"family_1", "family_2"}.issubset(eff.columns) else []

    primary_judges = sorted([str(x).lower() for x in PRIMARY_JUDGE_FAMILIES])
    primary_candidates = sorted([str(x).lower() for x in PRIMARY_CANDIDATE_FAMILIES])

    primary_errors = []

    if "falcon" in judges:
        primary_errors.append("Falcon appears as Primary-4 judge.")

    if "falcon" in candidates:
        primary_errors.append("Falcon appears as Primary-4 candidate.")

    if judges != primary_judges:
        primary_errors.append(f"Primary judge mismatch: expected {primary_judges}, got {judges}")

    if candidates != primary_candidates:
        primary_errors.append(f"Primary candidate mismatch: expected {primary_candidates}, got {candidates}")

    # Load core stats.
    bootstrap = None
    if "cluster_bootstrap_tps_primary" in FILES:
        bootstrap = _load_json_if_exists(FILES["cluster_bootstrap_tps_primary"])

    permutation = None
    if "permutation_result" in FILES:
        permutation = _load_json_if_exists(FILES["permutation_result"])

    per_family = None
    if "per_family_bh" in FILES:
        per_family = _load_json_if_exists(FILES["per_family_bh"])

    bt_diag = None
    if "bt_diagnostics" in FILES:
        bt_diag = _load_json_if_exists(FILES["bt_diagnostics"])

    artifact_interp = None
    if "response_artifact_interpretation" in FILES:
        artifact_interp = _load_json_if_exists(FILES["response_artifact_interpretation"])

    logprob_audit = None
    if "logprob_familiarity_audit" in FILES:
        logprob_audit = _load_json_if_exists(FILES["logprob_familiarity_audit"])

    leaderboard = None
    if "leaderboard_ranking_sensitivity" in FILES:
        leaderboard = _load_json_if_exists(FILES["leaderboard_ranking_sensitivity"])

    observed_tps = None
    ci = None

    if bootstrap:
        observed_tps = bootstrap.get("observed_tps", bootstrap.get("tps"))
        ci = bootstrap.get("ci_95", [bootstrap.get("ci_95_low"), bootstrap.get("ci_95_high")])

    permutation_p = None
    if permutation:
        permutation_p = permutation.get("p_two_sided", None)

    checks = {
        "required_files_present": len(missing_required) == 0,
        "primary4_falcon_absent": ("falcon" not in judges and "falcon" not in candidates),
        "primary4_judges_match_expected": judges == primary_judges,
        "primary4_candidates_match_expected": candidates == primary_candidates,
        "bootstrap_available": bootstrap is not None,
        "permutation_available": permutation is not None,
        "per_family_bh_available": per_family is not None,
        "bt_diagnostics_available": bt_diag is not None,
        "artifact_interpretation_available": artifact_interp is not None,
        "logprob_audit_available": logprob_audit is not None,
        "leaderboard_anchor_available": leaderboard is not None,
    }

    hard_failures = []

    if missing_required:
        hard_failures.append(f"Missing required files: {missing_required}")

    if primary_errors:
        hard_failures.extend(primary_errors)

    if bootstrap is not None:
        try:
            tps_val = float(observed_tps)
            if not (0.04 <= tps_val <= 0.10):
                hard_failures.append(
                    f"Observed Primary-4 TPS {tps_val} is outside expected clean-pipeline range [0.04, 0.10]."
                )
        except Exception:
            hard_failures.append("Could not parse observed_tps from bootstrap result.")

    if bt_diag is not None:
        if "overdispersion" not in bt_diag:
            hard_failures.append("BT diagnostics missing overdispersion.")
        if "by_pair" not in bt_diag:
            hard_failures.append("BT diagnostics missing pair-level residuals.")

    if per_family is not None:
        if "p_two_sided_bh" not in per_family:
            hard_failures.append("Per-family BH result missing p_two_sided_bh.")
        if "p_one_sided_bh" not in per_family:
            hard_failures.append("Per-family BH result missing p_one_sided_bh.")

    status = "pass" if not hard_failures else "fail"

    audit = {
        "analysis_label": "final_integrity_audit_before_human_calibration",
        "status": status,
        "file_status": file_status,
        "missing_required": missing_required,
        "checks": checks,
        "hard_failures": hard_failures,
        "primary4_summary": {
            "n_effective_rows": int(len(eff)),
            "judge_families": judges,
            "candidate_families": candidates,
            "primary_judge_families_expected": primary_judges,
            "primary_candidate_families_expected": primary_candidates,
        },
        "headline_stats": {
            "observed_tps": observed_tps,
            "ci_95": ci,
            "permutation_p_two_sided": permutation_p,
        },
        "artifact_risk_level": artifact_interp.get("risk_level") if artifact_interp else None,
        "logprob_familiarity_decision": logprob_audit.get("decision") if logprob_audit else None,
        "bt_overdispersion": bt_diag.get("overdispersion") if bt_diag else None,
        "leaderboard_tps_pp": leaderboard.get("observed_tps_pp") if leaderboard else None,
        "ready_for_human_calibration": status == "pass",
        "paper_scope_sentence": (
            "We restrict the judge panel to open-weight model families and evaluate family-conditioned "
            "preference patterns within that deployment context; extension to proprietary judges is left "
            "for future work."
        ),
    }

    atomic_write_json(audit, FILES["final_integrity_audit"])

    print("\nAudit status")
    print("-" * 100)
    print(status.upper())

    print("\nPrimary-4 summary")
    print("-" * 100)
    print(json.dumps(audit["primary4_summary"], indent=2))

    print("\nHeadline stats")
    print("-" * 100)
    print(json.dumps(audit["headline_stats"], indent=2))

    print("\nChecks")
    print("-" * 100)
    print(json.dumps(checks, indent=2))

    if hard_failures:
        print("\nHard failures")
        print("-" * 100)
        for f in hard_failures:
            print("✗", f)
    else:
        print("\n✓ No hard failures. You can move to human calibration.")

    print("\nSaved:")
    print(FILES["final_integrity_audit"])


run_step(
    [FILES["final_integrity_audit"]],
    _step_final_integrity_audit_910,
    "final_integrity_audit_before_human_calibration",
    force=False,
    validators={
        FILES["final_integrity_audit"]: lambda p: json_has(
            p,
            ["analysis_label", "status", "checks", "ready_for_human_calibration"],
        )
    },
)

print("\n✓ Cell 9.10 complete")

## PHASE 10 — Human Calibration

Take a stratified random sample (200 trials), export for human annotation,
then compute Krippendorff α + Gwet AC2 + Cohen κ + a "human gold TPS" once
annotations come back. The pipeline is designed to read the filled CSV as
soon as it appears — annotators can work asynchronously.



In [ ]:
# ============================================================================
# Cell 10.1 — Build top-tier Primary-4 blind human annotation sample
# ============================================================================
"""
Cell 10.1 — Build a blinded human annotation sample.

Design:
- 400 unique pairwise comparisons.
- Primary-4 candidates only: gemma, llama, qwen, yi.
- Falcon excluded completely.
- Balanced across six unordered candidate-family pairs.
- Each family appears exactly 200 times.
- Visible A/B sides are randomized for human annotators.
- Hidden answer key is saved separately.
- Human-facing file contains no model names, no model families, no judge families,
  no LLM winner, and no same-family metadata.

Run this cell first.
Then run Cell 10.2 to export annotator-specific sheets.
Do not share the hidden key with annotators.
"""

from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Tuple
import hashlib
import json
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Required globals
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_csv",
    "atomic_write_json",
    "csv_has",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Human calibration constants
# ---------------------------------------------------------------------------

HUMAN_SAMPLE_N = 400
HUMAN_N_ANNOTATORS = 3  # collection plan; final paper calibration retains 2 annotators
HUMAN_RANDOM_SEED = int(globals().get("RANDOM_SEED", 42))
HUMAN_MAX_COMPARISONS_PER_PROMPT_TARGET = 3

PRIMARY_CANDS_101 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]
EXPECTED_PRIMARY_CANDS_101 = ["gemma", "llama", "qwen", "yi"]

if sorted(PRIMARY_CANDS_101) != sorted(EXPECTED_PRIMARY_CANDS_101):
    raise RuntimeError(
        f"Primary candidates are not the expected Primary-4 set. "
        f"Expected {EXPECTED_PRIMARY_CANDS_101}, got {PRIMARY_CANDS_101}"
    )


# Balanced 400 allocation.
# This gives each family exactly 200 appearances.
HUMAN_PAIR_ALLOCATION = {
    "gemma__llama": 67,
    "llama__qwen": 67,
    "qwen__yi": 67,
    "gemma__yi": 67,
    "gemma__qwen": 66,
    "llama__yi": 66,
}

assert sum(HUMAN_PAIR_ALLOCATION.values()) == HUMAN_SAMPLE_N


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "analysis_human" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_human"] = Path(PATHS["analysis"]) / "human_calibration"
    elif "root" in PATHS:
        PATHS["analysis_human"] = Path(PATHS["root"]) / "analysis" / "human_calibration"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_human"] = Path(ROOT_DIR) / "analysis" / "human_calibration"
    else:
        PATHS["analysis_human"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/human_calibration")

Path(PATHS["analysis_human"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "human_export",
    Path(PATHS["analysis_human"]) / "human_annotation_sample_blind_primary4_400.csv",
)

FILES.setdefault(
    "human_key",
    Path(PATHS["analysis_human"]) / "human_annotation_key_hidden_primary4_400.csv",
)

FILES.setdefault(
    "human_export_audit",
    Path(PATHS["analysis_human"]) / "human_annotation_export_audit_primary4_400.json",
)

FILES.setdefault(
    "human_filled",
    Path(PATHS["analysis_human"]) / "human_annotations_completed_primary4_400.csv",
)

FILES.setdefault(
    "human_filled_merged",
    Path(PATHS["analysis_human"]) / "human_annotations_filled_merged_primary4_400.csv",
)


# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------

def _101_clean_family(x: Any) -> str:
    if pd.isna(x):
        return ""

    s = str(x).strip().lower()

    if "llama" in s:
        return "llama"
    if "qwen" in s:
        return "qwen"
    if "gemma" in s:
        return "gemma"
    if "falcon" in s:
        return "falcon"
    if s == "yi" or "yi-" in s or "yi_" in s or "/yi" in s:
        return "yi"

    return s


def _101_pair_key(f1: Any, f2: Any) -> str:
    a = _101_clean_family(f1)
    b = _101_clean_family(f2)
    return "__".join(sorted([a, b]))


def _101_human_annotation_id(trial_id: str, shown_a_side: str, shown_b_side: str) -> str:
    raw = f"human_primary4_400|{trial_id}|{shown_a_side}|{shown_b_side}"
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()[:16]


def _101_find_col(df: pd.DataFrame, candidates: List[str], required: bool = False) -> str:
    lower_map = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if str(c).lower() in lower_map:
            return lower_map[str(c).lower()]

    if required:
        raise RuntimeError(f"Could not find any of these columns: {candidates}")

    return ""


def _101_normalize_winner(x: Any) -> str:
    s = str(x).strip().lower()

    if s in {"a", "model_a", "response_a"}:
        return "A"
    if s in {"b", "model_b", "response_b"}:
        return "B"
    if s in {"tie", "draw", "equal", "both", "neither"}:
        return "tie"

    return ""


def _101_is_missing_series(s: pd.Series) -> pd.Series:
    return (
        s.isna()
        | s.astype(str).str.strip().eq("")
        | s.astype(str).str.strip().str.lower().isin(["nan", "none", "null"])
    )


def _101_responses_complete(df: pd.DataFrame) -> bool:
    return (
        "response_a" in df.columns
        and "response_b" in df.columns
        and not _101_is_missing_series(df["response_a"]).any()
        and not _101_is_missing_series(df["response_b"]).any()
    )


def _101_fill_col(target: pd.DataFrame, target_col: str, values: Any) -> None:
    """
    Fill missing cells in target[target_col] using values by POSITION, not by index.

    This avoids:
    pandas.errors.IndexingError:
    Unalignable boolean Series provided as indexer
    """
    if target_col not in target.columns:
        target[target_col] = np.nan

    values = pd.Series(values).reset_index(drop=True)

    if len(values) != len(target):
        raise RuntimeError(
            f"_101_fill_col length mismatch for {target_col}: "
            f"target has {len(target)} rows, values has {len(values)} rows."
        )

    mask = _101_is_missing_series(target[target_col]).to_numpy()

    if mask.sum() == 0:
        return

    target_index_to_fill = target.index[mask]
    values_to_fill = values.to_numpy()[mask]

    target.loc[target_index_to_fill, target_col] = values_to_fill


def _101_ensure_prompt_text(j: pd.DataFrame) -> pd.DataFrame:
    """
    Recover prompt text if master_judgments does not have it.
    """
    j = j.copy().reset_index(drop=True)

    if "prompt" in j.columns and not _101_is_missing_series(j["prompt"]).any():
        return j

    if "prompt" not in j.columns:
        j["prompt"] = np.nan

    candidate_files = [
        "trial_master",
        "prompt_master",
        "prompts",
        "all_responses",
    ]

    for key in candidate_files:
        path = FILES.get(key)
        if path is None or not Path(path).exists():
            continue

        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        if "prompt_id" not in df.columns:
            continue

        prompt_col = _101_find_col(
            df,
            ["prompt", "instruction", "question", "input_text", "user_prompt"],
        )

        if not prompt_col:
            continue

        lookup = (
            df[["prompt_id", prompt_col]]
            .dropna(subset=[prompt_col])
            .drop_duplicates("prompt_id")
            .assign(_prompt_key=lambda x: x["prompt_id"].astype(str))
            .set_index("_prompt_key")[prompt_col]
            .to_dict()
        )

        cand = j["prompt_id"].astype(str).map(lookup).reset_index(drop=True)
        _101_fill_col(j, "prompt", cand)

        recovered = int((~_101_is_missing_series(j["prompt"])).sum())
        print(f"Prompt recovery tried {key}: recovered rows={recovered}")

        if not _101_is_missing_series(j["prompt"]).any():
            print(f"Prompt recovery complete using {key}.")
            return j

    missing_prompt = int(_101_is_missing_series(j["prompt"]).sum())
    if missing_prompt:
        raise RuntimeError(f"Could not recover prompt text for {missing_prompt} rows.")

    return j


def _101_ensure_response_text(j: pd.DataFrame) -> pd.DataFrame:
    """
    Recover response_a/response_b robustly.

    Recovery order:
    1. Use existing response_a/response_b if already present.
    2. Try trial_master-style files by trial_id.
    3. Try all_responses by prompt_id + family.
    4. Try all_responses by prompt_id + model key/name.
    """
    j = j.copy().reset_index(drop=True)

    # Normalize existing response column names if needed.
    if "response_a" not in j.columns:
        alt_a = _101_find_col(
            j,
            [
                "response_A",
                "candidate_a_response",
                "model_a_response",
                "answer_a",
                "output_a",
                "text_a",
                "completion_a",
            ],
        )
        if alt_a:
            j["response_a"] = j[alt_a]

    if "response_b" not in j.columns:
        alt_b = _101_find_col(
            j,
            [
                "response_B",
                "candidate_b_response",
                "model_b_response",
                "answer_b",
                "output_b",
                "text_b",
                "completion_b",
            ],
        )
        if alt_b:
            j["response_b"] = j[alt_b]

    if _101_responses_complete(j):
        print("Response recovery: using response_a/response_b already present in master_judgments.")
        return j

    if "response_a" not in j.columns:
        j["response_a"] = np.nan

    if "response_b" not in j.columns:
        j["response_b"] = np.nan

    # -----------------------------------------------------------------------
    # 1. Try trial_master-style files by trial_id
    # -----------------------------------------------------------------------

    trial_master_keys = [
        "trial_master",
        "trial_master_primary",
        "trial_master_full",
        "trial_master_fixed",
        "trial_master_reconciled",
    ]

    for key in trial_master_keys:
        path = FILES.get(key)
        if path is None or not Path(path).exists():
            continue

        try:
            tm = pd.read_csv(path)
        except Exception:
            continue

        if "trial_id" not in tm.columns:
            continue

        tm_a = _101_find_col(
            tm,
            [
                "response_a",
                "response_A",
                "candidate_a_response",
                "model_a_response",
                "answer_a",
                "output_a",
                "text_a",
                "completion_a",
            ],
        )

        tm_b = _101_find_col(
            tm,
            [
                "response_b",
                "response_B",
                "candidate_b_response",
                "model_b_response",
                "answer_b",
                "output_b",
                "text_b",
                "completion_b",
            ],
        )

        if not tm_a or not tm_b:
            continue

        left = j[["trial_id"]].copy()
        left["_row_id"] = np.arange(len(left))
        left["_trial_key"] = left["trial_id"].astype(str)

        right = tm[["trial_id", tm_a, tm_b]].copy()
        right["_trial_key"] = right["trial_id"].astype(str)
        right = right.drop_duplicates("_trial_key")

        merged = (
            left.merge(
                right[["_trial_key", tm_a, tm_b]],
                on="_trial_key",
                how="left",
            )
            .sort_values("_row_id")
            .reset_index(drop=True)
        )

        _101_fill_col(j, "response_a", merged[tm_a])
        _101_fill_col(j, "response_b", merged[tm_b])

        recovered_a = int((~_101_is_missing_series(j["response_a"])).sum())
        recovered_b = int((~_101_is_missing_series(j["response_b"])).sum())

        print(
            f"Response recovery tried {key}: "
            f"response_a recovered rows={recovered_a}, "
            f"response_b recovered rows={recovered_b}"
        )

        if _101_responses_complete(j):
            print(f"Response recovery complete using {key}.")
            return j

    # -----------------------------------------------------------------------
    # 2. Try all_responses by prompt_id + family or model
    # -----------------------------------------------------------------------

    all_resp_path = FILES.get("all_responses")

    if all_resp_path is not None and Path(all_resp_path).exists():
        resp = pd.read_csv(all_resp_path)

        prompt_col = _101_find_col(resp, ["prompt_id"])
        response_col = _101_find_col(resp, ["response", "text", "output", "completion", "answer"])
        family_col = _101_find_col(resp, ["model_family", "family", "candidate_family"])
        model_col = _101_find_col(resp, ["model_key", "model_name", "model", "model_id"])

        if prompt_col and response_col:
            rr = resp.copy()

            # Prefer large responses if available.
            scale_col = _101_find_col(rr, ["model_scale", "scale"])
            if scale_col:
                large = rr[rr[scale_col].astype(str).str.lower().eq("large")].copy()
                if len(large) > 0:
                    rr = large

            # Prefer quality_ok rows if usable.
            if "quality_ok" in rr.columns:
                try:
                    quality_ok = rr["quality_ok"].astype(bool)
                    if int(quality_ok.sum()) > 0:
                        rr = rr[quality_ok].copy()
                except Exception:
                    pass

            rr["_prompt_key"] = rr[prompt_col].astype(str)
            rr["_response_text"] = rr[response_col]

            # 2a. prompt_id + family lookup.
            if family_col:
                rr["_family_clean"] = rr[family_col].map(_101_clean_family)

                fam_lookup = (
                    rr.dropna(subset=["_response_text"])
                    .drop_duplicates(subset=["_prompt_key", "_family_clean"], keep="first")
                    .set_index(["_prompt_key", "_family_clean"])["_response_text"]
                    .to_dict()
                )

                def _lookup_family(row, side: str):
                    prompt_key = str(row["prompt_id"])
                    fam = _101_clean_family(row["family_1"] if side == "A" else row["family_2"])
                    return fam_lookup.get((prompt_key, fam), np.nan)

                cand_a = j.apply(lambda r: _lookup_family(r, "A"), axis=1).reset_index(drop=True)
                cand_b = j.apply(lambda r: _lookup_family(r, "B"), axis=1).reset_index(drop=True)

                _101_fill_col(j, "response_a", cand_a)
                _101_fill_col(j, "response_b", cand_b)

                recovered_a = int((~_101_is_missing_series(j["response_a"])).sum())
                recovered_b = int((~_101_is_missing_series(j["response_b"])).sum())

                print(
                    "Response recovery tried all_responses by prompt_id + family: "
                    f"response_a recovered rows={recovered_a}, "
                    f"response_b recovered rows={recovered_b}"
                )

                if _101_responses_complete(j):
                    print("Response recovery complete using all_responses prompt_id + family.")
                    return j

            # 2b. prompt_id + model lookup.
            if model_col:
                rr["_model_key"] = rr[model_col].astype(str).str.strip().str.lower()

                model_lookup = (
                    rr.dropna(subset=["_response_text"])
                    .drop_duplicates(subset=["_prompt_key", "_model_key"], keep="first")
                    .set_index(["_prompt_key", "_model_key"])["_response_text"]
                    .to_dict()
                )

                def _lookup_model(row, side: str):
                    prompt_key = str(row["prompt_id"])
                    model = str(row["model_a"] if side == "A" else row["model_b"]).strip().lower()
                    return model_lookup.get((prompt_key, model), np.nan)

                cand_a = j.apply(lambda r: _lookup_model(r, "A"), axis=1).reset_index(drop=True)
                cand_b = j.apply(lambda r: _lookup_model(r, "B"), axis=1).reset_index(drop=True)

                _101_fill_col(j, "response_a", cand_a)
                _101_fill_col(j, "response_b", cand_b)

                recovered_a = int((~_101_is_missing_series(j["response_a"])).sum())
                recovered_b = int((~_101_is_missing_series(j["response_b"])).sum())

                print(
                    "Response recovery tried all_responses by prompt_id + model: "
                    f"response_a recovered rows={recovered_a}, "
                    f"response_b recovered rows={recovered_b}"
                )

                if _101_responses_complete(j):
                    print("Response recovery complete using all_responses prompt_id + model.")
                    return j

    # -----------------------------------------------------------------------
    # 3. Final diagnostic if still missing
    # -----------------------------------------------------------------------

    missing_a = int(_101_is_missing_series(j["response_a"]).sum())
    missing_b = int(_101_is_missing_series(j["response_b"]).sum())

    diagnostic = {
        "rows": int(len(j)),
        "missing_response_a": missing_a,
        "missing_response_b": missing_b,
        "available_file_keys": sorted([str(k) for k in FILES.keys()]),
        "all_responses_path": str(FILES.get("all_responses", "")),
        "note": (
            "Could not recover complete response_a/response_b. "
            "Check whether FILES['all_responses'] exists and contains "
            "prompt_id, model_family, response, and model_scale."
        ),
    }

    print("Response recovery diagnostic:")
    print(json.dumps(diagnostic, indent=2))

    raise RuntimeError(
        "Could not recover complete response_a/response_b after trying "
        "master_judgments, trial_master, and all_responses. "
        f"Missing response_a={missing_a}, missing response_b={missing_b}."
    )


def _101_prepare_master_judgments() -> pd.DataFrame:
    if "master_judgments" not in FILES or not Path(FILES["master_judgments"]).exists():
        raise RuntimeError("FILES['master_judgments'] missing. Run Cell 5.4 first.")

    j = pd.read_csv(FILES["master_judgments"]).copy()

    # Standardize family columns.
    if "family_1" not in j.columns:
        for c in ["candidate_family_1", "model_a_family", "family_a", "model_a"]:
            if c in j.columns:
                j["family_1"] = j[c]
                break

    if "family_2" not in j.columns:
        for c in ["candidate_family_2", "model_b_family", "family_b", "model_b"]:
            if c in j.columns:
                j["family_2"] = j[c]
                break

    if "order" not in j.columns:
        j["order"] = "AB"

    if "source" not in j.columns:
        j["source"] = "unknown"

    if "category" not in j.columns:
        j["category"] = "unknown"

    if "split" not in j.columns:
        j["split"] = "unknown"

    if "prompt_id" not in j.columns:
        raise RuntimeError("master_judgments is missing prompt_id.")

    if "pair_id" not in j.columns:
        j["pair_id"] = (
            j["prompt_id"].astype(str)
            + "__"
            + j["family_1"].astype(str)
            + "__"
            + j["family_2"].astype(str)
        )

    required = [
        "trial_id",
        "prompt_id",
        "family_1",
        "family_2",
        "order",
        "model_a",
        "model_b",
        "winner",
    ]

    missing = [c for c in required if c not in j.columns]
    if missing:
        raise RuntimeError(f"master_judgments missing required columns for human sampling: {missing}")

    for c in ["family_1", "family_2"]:
        j[c] = j[c].map(_101_clean_family)

    if "judge_family" in j.columns:
        j["judge_family"] = j["judge_family"].map(_101_clean_family)
        j = j[j["judge_family"].ne("falcon")].copy()

    # Primary-4 candidates only.
    j = j[
        j["family_1"].isin(PRIMARY_CANDS_101)
        & j["family_2"].isin(PRIMARY_CANDS_101)
    ].copy()

    # Use only one physical orientation of the candidate comparison.
    # Visible A/B will be randomized later for humans.
    j = j[j["order"].astype(str).str.upper().eq("AB")].copy()

    j["winner_norm"] = j["winner"].map(_101_normalize_winner)
    j = j[j["winner_norm"].isin(["A", "B", "tie"])].copy()

    if "parse_ok" in j.columns:
        try:
            parse_ok = j["parse_ok"].fillna(False).astype(bool)
            j = j[
                parse_ok
                | j["winner_norm"].isin(["A", "B", "tie"])
            ].copy()
        except Exception:
            pass

    if j.empty:
        raise RuntimeError("No eligible Primary-4 AB rows found for human annotation export.")

    if "falcon" in set(j["family_1"]) or "falcon" in set(j["family_2"]):
        raise RuntimeError("Falcon leaked into human calibration candidate pool.")

    if "falcon" in set(j.get("judge_family", pd.Series([], dtype=str))):
        raise RuntimeError("Falcon leaked into human calibration judge metadata.")

    # Critical fix: reset index before prompt/response recovery.
    j = j.reset_index(drop=True)

    j = _101_ensure_prompt_text(j)
    j = _101_ensure_response_text(j)

    missing_prompt = int(_101_is_missing_series(j["prompt"]).sum())
    missing_a = int(_101_is_missing_series(j["response_a"]).sum())
    missing_b = int(_101_is_missing_series(j["response_b"]).sum())

    if missing_prompt or missing_a or missing_b:
        raise RuntimeError(
            f"Primary-4 eligible rows still have missing text: "
            f"prompt={missing_prompt}, response_a={missing_a}, response_b={missing_b}"
        )

    return j.reset_index(drop=True)


def _101_build_unique_comparison_pool(j: pd.DataFrame) -> pd.DataFrame:
    """
    master_judgments has multiple judge rows for the same trial_id.
    Human annotation needs unique response comparisons, not judge-row duplicates.
    """
    j = j.copy().reset_index(drop=True)

    winner_counts = (
        j.groupby(["trial_id", "winner_norm"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for c in ["A", "B", "tie"]:
        if c not in winner_counts.columns:
            winner_counts[c] = 0

    def panel_winner(row):
        counts = {"A": int(row["A"]), "B": int(row["B"]), "tie": int(row["tie"])}
        ordered = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
        if ordered[0][1] == ordered[1][1]:
            return "tie"
        return ordered[0][0]

    winner_counts["llm_panel_winner_original_side"] = winner_counts.apply(panel_winner, axis=1)
    winner_counts["llm_panel_n_A"] = winner_counts["A"].astype(int)
    winner_counts["llm_panel_n_B"] = winner_counts["B"].astype(int)
    winner_counts["llm_panel_n_tie"] = winner_counts["tie"].astype(int)
    winner_counts["llm_panel_n_total"] = winner_counts[["A", "B", "tie"]].sum(axis=1).astype(int)

    sort_cols = ["trial_id"]
    if "judge_family" in j.columns:
        sort_cols.append("judge_family")

    unique = (
        j.sort_values(sort_cols)
        .drop_duplicates(subset=["trial_id"], keep="first")
        .copy()
    )

    unique = unique.merge(
        winner_counts[
            [
                "trial_id",
                "llm_panel_winner_original_side",
                "llm_panel_n_A",
                "llm_panel_n_B",
                "llm_panel_n_tie",
                "llm_panel_n_total",
            ]
        ],
        on="trial_id",
        how="left",
    )

    unique["candidate_pair"] = unique.apply(
        lambda r: _101_pair_key(r["family_1"], r["family_2"]),
        axis=1,
    )

    unique = unique[unique["candidate_pair"].isin(HUMAN_PAIR_ALLOCATION.keys())].copy()

    if unique["trial_id"].duplicated().any():
        raise RuntimeError("Unique comparison pool still has duplicated trial_id.")

    return unique.reset_index(drop=True)


def _101_sample_balanced_pool(
    pool: pd.DataFrame,
    allocation: Dict[str, int],
    seed: int,
    prompt_max_target: int = 3,
) -> pd.DataFrame:
    """
    Balanced by unordered candidate-family pair, while limiting repeated prompts
    as much as possible.
    """
    rng = np.random.default_rng(seed)

    selected_indices = []
    used_indices = set()
    prompt_counts = Counter()

    pair_order = sorted(allocation.keys(), key=lambda p: (-allocation[p], p))

    for pair in pair_order:
        target = allocation[pair]
        sub = pool[pool["candidate_pair"].eq(pair)].copy()

        if len(sub) < target:
            raise RuntimeError(
                f"Not enough rows for pair {pair}. Need {target}, found {len(sub)}."
            )

        candidate_indices = sub.index.to_numpy()
        rng.shuffle(candidate_indices)

        selected_for_pair = []

        for prompt_limit in [
            prompt_max_target,
            prompt_max_target + 1,
            prompt_max_target + 2,
            999999,
        ]:
            for idx in candidate_indices:
                if idx in used_indices:
                    continue

                prompt_id = str(pool.loc[idx, "prompt_id"])

                if prompt_counts[prompt_id] >= prompt_limit:
                    continue

                selected_for_pair.append(idx)
                used_indices.add(idx)
                prompt_counts[prompt_id] += 1

                if len(selected_for_pair) >= target:
                    break

            if len(selected_for_pair) >= target:
                break

        if len(selected_for_pair) < target:
            raise RuntimeError(
                f"Could not fill pair {pair}. Needed {target}, got {len(selected_for_pair)}."
            )

        selected_indices.extend(selected_for_pair)

    sample = pool.loc[selected_indices].copy()
    sample = sample.sample(frac=1.0, random_state=seed + 1001).reset_index(drop=True)

    if len(sample) != HUMAN_SAMPLE_N:
        raise RuntimeError(f"Expected {HUMAN_SAMPLE_N} sampled rows, got {len(sample)}.")

    return sample


def _101_randomize_visible_sides(sample: pd.DataFrame, seed: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed + 2026)

    human_rows = []
    key_rows = []

    for _, r in sample.reset_index(drop=True).iterrows():
        swap = bool(rng.integers(0, 2))

        if not swap:
            shown_a_side = "A"
            shown_b_side = "B"
            shown_a_response = r["response_a"]
            shown_b_response = r["response_b"]
            shown_a_family = r["family_1"]
            shown_b_family = r["family_2"]
            shown_a_model = r["model_a"]
            shown_b_model = r["model_b"]
        else:
            shown_a_side = "B"
            shown_b_side = "A"
            shown_a_response = r["response_b"]
            shown_b_response = r["response_a"]
            shown_a_family = r["family_2"]
            shown_b_family = r["family_1"]
            shown_a_model = r["model_b"]
            shown_b_model = r["model_a"]

        annotation_id = _101_human_annotation_id(
            str(r["trial_id"]),
            shown_a_side=shown_a_side,
            shown_b_side=shown_b_side,
        )

        original_panel = str(r.get("llm_panel_winner_original_side", "tie"))

        if original_panel == "tie":
            llm_panel_visible = "tie"
        elif original_panel == shown_a_side:
            llm_panel_visible = "A"
        elif original_panel == shown_b_side:
            llm_panel_visible = "B"
        else:
            llm_panel_visible = ""

        human_rows.append({
            "annotation_id": annotation_id,
            "prompt": r["prompt"],
            "response_A": shown_a_response,
            "response_B": shown_b_response,
            "source": r.get("source", ""),
            "category": r.get("category", ""),
            "human_choice": "",
            "confidence_1_to_5": "",
            "notes_optional": "",
        })

        key_rows.append({
            "annotation_id": annotation_id,
            "trial_id": r["trial_id"],
            "prompt_id": r["prompt_id"],
            "source": r.get("source", ""),
            "category": r.get("category", ""),
            "split": r.get("split", ""),
            "pair_id": r.get("pair_id", ""),
            "candidate_pair": r["candidate_pair"],
            "original_family_1": r["family_1"],
            "original_family_2": r["family_2"],
            "original_model_a": r["model_a"],
            "original_model_b": r["model_b"],
            "shown_A_original_side": shown_a_side,
            "shown_B_original_side": shown_b_side,
            "shown_A_family": shown_a_family,
            "shown_B_family": shown_b_family,
            "shown_A_model": shown_a_model,
            "shown_B_model": shown_b_model,
            "visible_sides_swapped": bool(swap),
            "llm_panel_winner_original_side": original_panel,
            "llm_panel_winner_visible_side": llm_panel_visible,
            "llm_panel_n_A": int(r.get("llm_panel_n_A", 0)),
            "llm_panel_n_B": int(r.get("llm_panel_n_B", 0)),
            "llm_panel_n_tie": int(r.get("llm_panel_n_tie", 0)),
            "llm_panel_n_total": int(r.get("llm_panel_n_total", 0)),
        })

    human_df = pd.DataFrame(human_rows)
    key_df = pd.DataFrame(key_rows)

    if human_df["annotation_id"].duplicated().any():
        raise RuntimeError("Duplicate annotation_id generated.")

    if key_df["annotation_id"].duplicated().any():
        raise RuntimeError("Duplicate annotation_id generated in hidden key.")

    return human_df, key_df


def _101_audit_sample(human_df: pd.DataFrame, key_df: pd.DataFrame, pool: pd.DataFrame) -> Dict[str, Any]:
    pair_counts = key_df["candidate_pair"].value_counts().sort_index().to_dict()

    family_appearances = Counter()
    for _, r in key_df.iterrows():
        family_appearances[str(r["shown_A_family"])] += 1
        family_appearances[str(r["shown_B_family"])] += 1

    prompt_counts = key_df["prompt_id"].value_counts()

    side_counts = {
        "shown_A_original_side_counts": key_df["shown_A_original_side"].value_counts().to_dict(),
        "shown_B_original_side_counts": key_df["shown_B_original_side"].value_counts().to_dict(),
        "visible_sides_swapped_counts": key_df["visible_sides_swapped"].value_counts().to_dict(),
    }

    errors = []

    if len(human_df) != HUMAN_SAMPLE_N:
        errors.append(f"Human sample size is {len(human_df)}, expected {HUMAN_SAMPLE_N}.")

    if len(key_df) != HUMAN_SAMPLE_N:
        errors.append(f"Key sample size is {len(key_df)}, expected {HUMAN_SAMPLE_N}.")

    if pair_counts != HUMAN_PAIR_ALLOCATION:
        errors.append(f"Pair allocation mismatch. Expected {HUMAN_PAIR_ALLOCATION}, got {pair_counts}")

    expected_family_appearances = {f: 200 for f in EXPECTED_PRIMARY_CANDS_101}

    if dict(sorted(family_appearances.items())) != dict(sorted(expected_family_appearances.items())):
        errors.append(
            f"Family appearance mismatch. Expected {expected_family_appearances}, got {dict(family_appearances)}"
        )

    forbidden_human_cols = [
        "family",
        "model",
        "judge",
        "winner",
        "same_family",
        "llm",
        "tribal",
    ]

    leaking_cols = []
    for c in human_df.columns:
        cl = c.lower()
        if c == "human_choice":
            continue
        if any(tok in cl for tok in forbidden_human_cols):
            leaking_cols.append(c)

    if leaking_cols:
        errors.append(f"Human-facing file has leaking metadata columns: {leaking_cols}")

    if key_df[["shown_A_family", "shown_B_family"]].isin(["falcon"]).any().any():
        errors.append("Falcon appears in hidden key shown families.")

    if "falcon" in set(pool["family_1"]) or "falcon" in set(pool["family_2"]):
        errors.append("Falcon appears in candidate pool.")

    status = "pass" if not errors else "fail"

    return {
        "analysis_label": "human_annotation_export_primary4_400",
        "status": status,
        "errors": errors,
        "n_unique_comparisons": int(len(human_df)),
        "n_annotators_planned_for_collection": int(HUMAN_N_ANNOTATORS),
        "total_human_judgments_planned_for_collection": int(len(human_df) * HUMAN_N_ANNOTATORS),
        "candidate_families": EXPECTED_PRIMARY_CANDS_101,
        "falcon_excluded": True,
        "pair_allocation_expected": HUMAN_PAIR_ALLOCATION,
        "pair_counts_observed": pair_counts,
        "family_appearances_observed": dict(sorted(family_appearances.items())),
        "family_appearances_expected": expected_family_appearances,
        "max_comparisons_per_prompt": int(prompt_counts.max()),
        "mean_comparisons_per_prompt": float(prompt_counts.mean()),
        "n_prompts_used": int(prompt_counts.shape[0]),
        "side_randomization": side_counts,
        "human_export": str(FILES["human_export"]),
        "human_key": str(FILES["human_key"]),
        "human_filled_expected": str(FILES["human_filled"]),
        "pool_size_before_sampling": int(len(pool)),
        "paper_language": (
            "For human calibration, we sampled 400 blind pairwise comparisons from the "
            "Primary-4 candidate set, stratified across the six unordered candidate-family "
            "pairs. The allocation was chosen so that each model family appeared exactly "
            "200 times in the human sample. Visible A/B sides were randomized and model "
            "family metadata was hidden from annotators. Each comparison was initially assigned to "
            "three human raters for collection. The paper-facing calibration analysis "
            "uses two retained annotators after the documented exclusion step."
        ),
    }


def load_human_annotations() -> pd.DataFrame:
    """
    Later cells can call this after completed annotations are uploaded.
    It merges completed human-facing sheets with the hidden key.
    """
    if not Path(FILES["human_filled"]).exists():
        raise FileNotFoundError(f"Human annotation file not found: {FILES['human_filled']}")

    h = pd.read_csv(FILES["human_filled"])
    key = pd.read_csv(FILES["human_key"])

    if "annotation_id" not in h.columns:
        raise RuntimeError("Completed human annotation file lacks annotation_id.")

    merged = key.merge(h, on="annotation_id", how="left", suffixes=("", "_human"))

    atomic_write_csv(merged, FILES["human_filled_merged"])

    return merged


# ---------------------------------------------------------------------------
# Main step
# ---------------------------------------------------------------------------

def _step_build_human_annotation_sample_101():
    print("=" * 100)
    print("BUILD PRIMARY-4 HUMAN ANNOTATION SAMPLE — 400 BLIND COMPARISONS")
    print("=" * 100)

    j = _101_prepare_master_judgments()

    print("\nEligible Primary-4 judgment rows")
    print("-" * 100)
    print(f"Rows after filtering: {len(j):,}")
    print(f"Unique trial_id rows before pooling: {j['trial_id'].nunique():,}")

    pool = _101_build_unique_comparison_pool(j)

    print("\nUnique comparison pool")
    print("-" * 100)
    print(f"Pool rows: {len(pool):,}")
    print("Pool candidate-pair counts:")
    print(pool["candidate_pair"].value_counts().sort_index().to_string())

    sample = _101_sample_balanced_pool(
        pool=pool,
        allocation=HUMAN_PAIR_ALLOCATION,
        seed=HUMAN_RANDOM_SEED,
        prompt_max_target=HUMAN_MAX_COMPARISONS_PER_PROMPT_TARGET,
    )

    human_df, key_df = _101_randomize_visible_sides(
        sample=sample,
        seed=HUMAN_RANDOM_SEED,
    )

    audit = _101_audit_sample(human_df, key_df, pool)

    atomic_write_csv(human_df, FILES["human_export"])
    atomic_write_csv(key_df, FILES["human_key"])
    atomic_write_json(audit, FILES["human_export_audit"])

    print("\nHuman annotation sample created")
    print("-" * 100)
    print(f"Rows exported: {len(human_df)}")
    print(f"Collection-plan annotators: {HUMAN_N_ANNOTATORS}")
    print(f"Collection-plan human judgments: {len(human_df) * HUMAN_N_ANNOTATORS}")

    print("\nPair counts")
    print("-" * 100)
    print(json.dumps(audit["pair_counts_observed"], indent=2))

    print("\nFamily appearances")
    print("-" * 100)
    print(json.dumps(audit["family_appearances_observed"], indent=2))

    print("\nPrompt repetition")
    print("-" * 100)
    print(json.dumps({
        "n_prompts_used": audit["n_prompts_used"],
        "max_comparisons_per_prompt": audit["max_comparisons_per_prompt"],
        "mean_comparisons_per_prompt": audit["mean_comparisons_per_prompt"],
    }, indent=2))

    print("\nSide randomization")
    print("-" * 100)
    print(json.dumps(audit["side_randomization"], indent=2))

    print("\nAudit status")
    print("-" * 100)
    print(audit["status"].upper())

    if audit["errors"]:
        print("\nErrors")
        print("-" * 100)
        for e in audit["errors"]:
            print("✗", e)
        raise RuntimeError("Human annotation sample audit failed.")

    print("\nPaper language")
    print("-" * 100)
    print(audit["paper_language"])

    print("\nSaved files")
    print("-" * 100)
    print("Human-facing blind sample:", FILES["human_export"])
    print("Hidden key, do not share:", FILES["human_key"])
    print("Export audit:", FILES["human_export_audit"])

    print("\n✓ Cell 10.1 complete. Next run Cell 10.2 to create annotator-specific sheets.")


run_step(
    [FILES["human_export"], FILES["human_key"], FILES["human_export_audit"]],
    _step_build_human_annotation_sample_101,
    "build_human_annotation_sample_primary4_400",
    force=True,
    validators={
        FILES["human_export"]: lambda p: csv_has(
            p,
            ["annotation_id", "prompt", "response_A", "response_B", "human_choice", "confidence_1_to_5"],
            400,
        ),
        FILES["human_key"]: lambda p: csv_has(
            p,
            [
                "annotation_id",
                "trial_id",
                "candidate_pair",
                "shown_A_family",
                "shown_B_family",
                "llm_panel_winner_visible_side",
            ],
            400,
        ),
        FILES["human_export_audit"]: lambda p: json_has(
            p,
            [
                "analysis_label",
                "status",
                "n_unique_comparisons",
                "pair_counts_observed",
                "family_appearances_observed",
            ],
        ),
    },
)

print("\n✓ Primary-4 400-row blind human annotation sample exported")

In [ ]:
# ============================================================================
# Cell 10.2 — Export annotator-specific blind human annotation sheets
# ============================================================================
"""
Cell 10.2 — Export human annotation sheets.

Run this after Cell 10.1.

Purpose:
- Read the 400-row blind human annotation sample from Cell 10.1.
- Create one separate CSV sheet per annotator.
- Randomize row order independently for each annotator.
- Keep the annotation_id stable so results can be merged later.
- Save a README/instruction file for annotators.
- Save an export manifest.

Important:
- Share only the annotator sheets and README with humans.
- Do NOT share the hidden key.
- Do NOT share the export audit if you do not want them to see design metadata.
- Stop after this cell until the completed annotation files come back.
"""

from pathlib import Path
from typing import Any, Dict, List
import json
import pandas as pd
import numpy as np


# ---------------------------------------------------------------------------
# Required globals
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_csv",
    "atomic_write_json",
    "csv_has",
    "json_has",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

HUMAN_SAMPLE_N = int(globals().get("HUMAN_SAMPLE_N", 400))
HUMAN_N_ANNOTATORS = int(globals().get("HUMAN_N_ANNOTATORS", 3))
HUMAN_RANDOM_SEED = int(globals().get("HUMAN_RANDOM_SEED", globals().get("RANDOM_SEED", 42)))


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "analysis_human" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_human"] = Path(PATHS["analysis"]) / "human_calibration"
    elif "root" in PATHS:
        PATHS["analysis_human"] = Path(PATHS["root"]) / "analysis" / "human_calibration"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_human"] = Path(ROOT_DIR) / "analysis" / "human_calibration"
    else:
        PATHS["analysis_human"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/human_calibration")

Path(PATHS["analysis_human"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "human_export",
    Path(PATHS["analysis_human"]) / "human_annotation_sample_blind_primary4_400.csv",
)

FILES.setdefault(
    "human_key",
    Path(PATHS["analysis_human"]) / "human_annotation_key_hidden_primary4_400.csv",
)

FILES.setdefault(
    "human_export_audit",
    Path(PATHS["analysis_human"]) / "human_annotation_export_audit_primary4_400.json",
)

FILES.setdefault(
    "human_annotator_dir",
    Path(PATHS["analysis_human"]) / "annotator_sheets_primary4_400",
)

Path(FILES["human_annotator_dir"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "human_annotator_manifest",
    Path(PATHS["analysis_human"]) / "human_annotator_sheets_manifest_primary4_400.json",
)

FILES.setdefault(
    "human_annotator_readme",
    Path(PATHS["analysis_human"]) / "README_FOR_HUMAN_ANNOTATORS_PRIMARY4_400.txt",
)

FILES.setdefault(
    "human_completed_dir",
    Path(PATHS["analysis_human"]) / "completed_annotator_sheets_primary4_400",
)

Path(FILES["human_completed_dir"]).mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _102_atomic_write_text(path: Path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def _102_validate_human_base_sheet(df: pd.DataFrame) -> None:
    required_cols = [
        "annotation_id",
        "prompt",
        "response_A",
        "response_B",
        "human_choice",
        "confidence_1_to_5",
        "notes_optional",
    ]

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise RuntimeError(f"Base human export is missing required columns: {missing}")

    if len(df) != HUMAN_SAMPLE_N:
        raise RuntimeError(f"Expected {HUMAN_SAMPLE_N} rows, found {len(df)}.")

    if df["annotation_id"].duplicated().any():
        raise RuntimeError("Base human export has duplicated annotation_id.")

    forbidden_cols = [
        "family",
        "model",
        "judge",
        "winner",
        "llm",
        "same_family",
        "tribal",
    ]

    leaking_cols = []
    for c in df.columns:
        cl = str(c).lower()
        if c == "human_choice":
            continue
        if any(tok in cl for tok in forbidden_cols):
            leaking_cols.append(c)

    if leaking_cols:
        raise RuntimeError(
            f"Base human-facing sheet contains metadata-leaking columns: {leaking_cols}"
        )


def _102_clean_annotator_sheet(df: pd.DataFrame, annotator_id: int, seed: int) -> pd.DataFrame:
    """
    Keep only columns humans should see.
    Randomize row order independently for each annotator.
    """
    visible_cols = [
        "annotation_id",
        "prompt",
        "response_A",
        "response_B",
        "human_choice",
        "confidence_1_to_5",
        "notes_optional",
    ]

    sheet = df[visible_cols].copy()

    # Clear any previous annotations.
    sheet["human_choice"] = ""
    sheet["confidence_1_to_5"] = ""
    sheet["notes_optional"] = ""

    # Add annotator ID. This is safe and helps merging later.
    sheet.insert(1, "annotator_id", f"annotator_{annotator_id}")

    # Randomize row order separately for each annotator.
    sheet = sheet.sample(
        frac=1.0,
        random_state=seed + annotator_id * 1009,
    ).reset_index(drop=True)

    # Add a display row number for convenience.
    sheet.insert(0, "row_number", range(1, len(sheet) + 1))

    return sheet


def _102_build_readme() -> str:
    return f"""
HUMAN ANNOTATION INSTRUCTIONS

You are helping evaluate pairs of model responses.

You will see:
1. A prompt
2. Response A
3. Response B

Your task:
Choose which response is better.

Allowed values for human_choice:
A
B
Tie

Use "A" if Response A is clearly better.
Use "B" if Response B is clearly better.
Use "Tie" only when both responses are roughly equal in quality, or neither is clearly better.

Confidence:
Fill confidence_1_to_5 using:
1 = very unsure
2 = somewhat unsure
3 = moderate
4 = confident
5 = very confident

Use notes_optional only if you want to briefly explain a difficult decision.

Evaluate based on:
- Correctness
- Helpfulness
- Completeness
- Clarity
- Directness
- Whether the answer follows the prompt

Do not try to guess which model wrote the answer.
Do not search online while annotating.
Do not use ChatGPT or another AI system to choose.
Judge only what is written in the two responses.

Important:
Please do not edit annotation_id.
Please do not edit annotator_id.
Please do not delete rows.
Please do not reorder columns.
Please fill every row if possible.

Study design:
This sheet is blinded. Model names and model families are hidden.

Expected number of rows:
{HUMAN_SAMPLE_N}

After completion:
Save the completed file with the same filename, then send it back.
""".strip()


def _step_export_annotator_sheets_102():
    print("=" * 100)
    print("EXPORT ANNOTATOR-SPECIFIC HUMAN ANNOTATION SHEETS")
    print("=" * 100)

    base_path = Path(FILES["human_export"])
    key_path = Path(FILES["human_key"])
    audit_path = Path(FILES["human_export_audit"])

    if not base_path.exists():
        raise RuntimeError(f"Missing base human export from Cell 10.1:\n{base_path}")

    if not key_path.exists():
        raise RuntimeError(f"Missing hidden key from Cell 10.1:\n{key_path}")

    if not audit_path.exists():
        raise RuntimeError(f"Missing human export audit from Cell 10.1:\n{audit_path}")

    base = pd.read_csv(base_path)
    _102_validate_human_base_sheet(base)

    key = pd.read_csv(key_path)
    if len(key) != HUMAN_SAMPLE_N:
        raise RuntimeError(f"Hidden key has {len(key)} rows, expected {HUMAN_SAMPLE_N}.")

    if key["annotation_id"].duplicated().any():
        raise RuntimeError("Hidden key has duplicated annotation_id.")

    if set(base["annotation_id"]) != set(key["annotation_id"]):
        raise RuntimeError("Base sheet and hidden key annotation_id sets do not match.")

    annotator_files = []

    for annotator_id in range(1, HUMAN_N_ANNOTATORS + 1):
        sheet = _102_clean_annotator_sheet(
            base,
            annotator_id=annotator_id,
            seed=HUMAN_RANDOM_SEED,
        )

        out_path = (
            Path(FILES["human_annotator_dir"])
            / f"human_annotation_sheet_annotator_{annotator_id}_primary4_400.csv"
        )

        atomic_write_csv(sheet, out_path)

        annotator_files.append({
            "annotator_id": f"annotator_{annotator_id}",
            "path": str(out_path),
            "n_rows": int(len(sheet)),
            "columns": list(sheet.columns),
        })

    readme_text = _102_build_readme()
    _102_atomic_write_text(FILES["human_annotator_readme"], readme_text)

    manifest = {
        "analysis_label": "human_annotator_sheet_export_primary4_400",
        "status": "ready_for_human_annotation",
        "n_unique_comparisons": int(HUMAN_SAMPLE_N),
        "n_annotators": int(HUMAN_N_ANNOTATORS),
        "planned_total_human_judgments": int(HUMAN_SAMPLE_N * HUMAN_N_ANNOTATORS),
        "base_human_export": str(FILES["human_export"]),
        "hidden_key_do_not_share": str(FILES["human_key"]),
        "annotator_dir": str(FILES["human_annotator_dir"]),
        "completed_dir": str(FILES["human_completed_dir"]),
        "readme_for_annotators": str(FILES["human_annotator_readme"]),
        "annotator_files": annotator_files,
        "share_with_annotators": [
            str(FILES["human_annotator_readme"]),
            *[x["path"] for x in annotator_files],
        ],
        "do_not_share": [
            str(FILES["human_key"]),
            str(FILES["human_export_audit"]),
            str(FILES["human_annotator_manifest"]),
        ],
        "expected_completed_filenames": [
            f"human_annotation_sheet_annotator_{i}_primary4_400_completed.csv"
            for i in range(1, HUMAN_N_ANNOTATORS + 1)
        ],
        "next_step": (
            "Download/share only the README and annotator-specific CSV files. "
            "After annotators complete them, upload completed files into completed_dir, "
            "then run Cell 10.3."
        ),
    }

    atomic_write_json(manifest, FILES["human_annotator_manifest"])

    print("\nAnnotator sheets exported")
    print("-" * 100)
    for item in annotator_files:
        print(f"{item['annotator_id']}: {item['path']}")

    print("\nREADME")
    print("-" * 100)
    print(FILES["human_annotator_readme"])

    print("\nShare these files with annotators")
    print("-" * 100)
    for p in manifest["share_with_annotators"]:
        print(p)

    print("\nDo NOT share these files")
    print("-" * 100)
    for p in manifest["do_not_share"]:
        print(p)

    print("\nCompleted files should be uploaded here")
    print("-" * 100)
    print(FILES["human_completed_dir"])

    print("\nNext step")
    print("-" * 100)
    print(manifest["next_step"])

    print("\n✓ Cell 10.2 complete. Stop here until human annotations are completed.")


run_step(
    [
        FILES["human_annotator_manifest"],
        FILES["human_annotator_readme"],
    ],
    _step_export_annotator_sheets_102,
    "export_human_annotator_sheets_primary4_400",
    force=True,
    validators={
        FILES["human_annotator_manifest"]: lambda p: json_has(
            p,
            [
                "analysis_label",
                "status",
                "annotator_files",
                "share_with_annotators",
                "do_not_share",
                "next_step",
            ],
        ),
        FILES["human_annotator_readme"]: lambda p: Path(p).exists() and Path(p).stat().st_size > 500,
    },
)

print("\n✓ Human annotator sheets exported")

In [ ]:
# ============================================================================
# Cell 10.3 — Import completed human annotation workbook (2-annotator version)
# ============================================================================
"""
CHANGED from original: annotator_2 is dropped at import time.
Original annotator_3 is renamed to annotator_2 in all outputs.
The wide CSV therefore has annotator_1_choice and annotator_2_choice only.
All downstream cells (10.4 onward) work on these two columns with no
further changes needed.

Input:
- annotator_sheets_completed_primary4_400.xlsx
- Sheets used: annotator_1, annotator_3  (annotator_2 dropped)
- Sheet annotator_3 is relabelled annotator_2 in all outputs

Output files (same paths as original — no file renames needed):
- human_annotations_completed_long_primary4_400.csv   (800 rows, 2 annotators)
- human_annotations_completed_wide_primary4_400.csv   (400 rows, annotator_1/2 columns)
- human_annotations_with_key_primary4_400.csv         (400 rows, merged with hidden key)
- human_import_validation_primary4_400.json
"""

from pathlib import Path
from typing import Any, Dict, List
from collections import Counter
import json
import numpy as np
import pandas as pd


required_globals = [
    "FILES", "PATHS", "run_step",
    "atomic_write_csv", "atomic_write_json",
    "csv_has", "json_has",
]
missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Paths — same as original
# ---------------------------------------------------------------------------

if "analysis_human" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_human"] = Path(PATHS["analysis"]) / "human_calibration"
    elif "root" in PATHS:
        PATHS["analysis_human"] = Path(PATHS["root"]) / "analysis" / "human_calibration"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_human"] = Path(ROOT_DIR) / "analysis" / "human_calibration"
    else:
        PATHS["analysis_human"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/human_calibration")

Path(PATHS["analysis_human"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("human_completed_workbook",
    Path(PATHS["analysis_human"]) / "annotator_sheets_completed_primary4_400.xlsx")
FILES.setdefault("human_key",
    Path(PATHS["analysis_human"]) / "human_annotation_key_hidden_primary4_400.csv")
FILES.setdefault("human_completed_long",
    Path(PATHS["analysis_human"]) / "human_annotations_completed_long_primary4_400.csv")
FILES.setdefault("human_completed_wide",
    Path(PATHS["analysis_human"]) / "human_annotations_completed_wide_primary4_400.csv")
FILES.setdefault("human_with_key",
    Path(PATHS["analysis_human"]) / "human_annotations_with_key_primary4_400.csv")
FILES.setdefault("human_import_validation",
    Path(PATHS["analysis_human"]) / "human_import_validation_primary4_400.json")


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

HUMAN_SAMPLE_N = int(globals().get("HUMAN_SAMPLE_N", 400))

# The two sheets we keep and their output names.
# annotator_2 from Excel is silently dropped.
# annotator_3 from Excel becomes annotator_2 in all outputs.
SHEETS_TO_KEEP = ["annotator_1", "annotator_3"]
OUTPUT_NAMES   = ["annotator_1", "annotator_2"]   # renamed output labels

REQUIRED_VISIBLE_COLS = [
    "row_number", "annotation_id", "annotator_id",
    "prompt", "response_A", "response_B",
    "human_choice", "confidence_1_to_5",
]


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _103_norm_choice(x: Any) -> str:
    s = str(x).strip()
    if s in {"A", "A1", "a", "a1"}:
        return "A"
    if s in {"B", "B1", "b", "b1"}:
        return "B"
    if s.lower() in {"tie", "t", "draw", "equal"}:
        return "Tie"
    return ""


def _103_norm_conf(x: Any):
    try:
        v = int(float(x))
        if v in {1, 2, 3, 4, 5}:
            return v
    except Exception:
        pass
    return np.nan


def _103_norm_flag(row: pd.Series) -> int:
    for col in ["flag_for_review", "flagged_for_review"]:
        if col in row.index:
            if str(row[col]).strip().lower() in {"1", "true", "yes", "y"}:
                return 1
    return 0


def _103_read_one_sheet(path: Path, excel_sheet_name: str, output_name: str) -> pd.DataFrame:
    """
    Read one sheet from the Excel workbook.
    excel_sheet_name: the actual sheet tab name ("annotator_1", "annotator_3")
    output_name:      what we call this annotator in outputs ("annotator_1", "annotator_2")
    """
    df = pd.read_excel(path, sheet_name=excel_sheet_name)
    df.columns = [str(c).strip() for c in df.columns]

    missing = [c for c in REQUIRED_VISIBLE_COLS if c not in df.columns]
    if missing:
        raise RuntimeError(f"Sheet '{excel_sheet_name}' missing required columns: {missing}")

    df = df.copy()
    # Relabel: store original sheet name for diagnostics, use output_name as annotator id
    df["sheet_name"]   = output_name          # output label (annotator_1 or annotator_2)
    df["excel_sheet"]  = excel_sheet_name     # original sheet (for audit trail)
    df["annotator_id"] = output_name

    df["row_number"]   = pd.to_numeric(df["row_number"], errors="coerce").astype("Int64")
    df["annotation_id"] = df["annotation_id"].astype(str).str.strip()

    df["human_choice_raw"]  = df["human_choice"].astype(str).str.strip()
    df["human_choice_norm"] = df["human_choice"].map(_103_norm_choice)
    df["confidence_raw"]    = df["confidence_1_to_5"]
    df["confidence_norm"]   = df["confidence_1_to_5"].map(_103_norm_conf)

    if "notes_optional" not in df.columns:
        df["notes_optional"] = ""
    if "status" not in df.columns:
        df["status"] = ""

    df["status"]             = df["status"].astype(str).str.strip().str.lower()
    df["flag_for_review_norm"] = df.apply(_103_norm_flag, axis=1)

    return df


def _103_validate_completed(long_df: pd.DataFrame, key_df: pd.DataFrame) -> Dict[str, Any]:
    errors   = []
    warnings = []
    per_sheet = {}

    # Validate against output names (annotator_1 and annotator_2 in our outputs)
    for out_name in OUTPUT_NAMES:
        sub = long_df[long_df["sheet_name"].eq(out_name)].copy()

        per_sheet[out_name] = {
            "n_rows":                int(len(sub)),
            "excel_sheet_origin":    sub["excel_sheet"].iloc[0] if len(sub) > 0 else "unknown",
            "n_unique_annotation_ids": int(sub["annotation_id"].nunique()),
            "duplicate_annotation_ids": int(sub["annotation_id"].duplicated().sum()),
            "missing_choice":        int(sub["human_choice_norm"].eq("").sum()),
            "missing_confidence":    int(sub["confidence_norm"].isna().sum()),
            "choice_counts":         sub["human_choice_norm"].value_counts().to_dict(),
            "confidence_counts":     {str(k): int(v) for k, v in sub["confidence_norm"].value_counts(dropna=False).items()},
            "notes_filled":          int(sub["notes_optional"].fillna("").astype(str).str.strip().ne("").sum()),
            "flagged_count":         int(sub["flag_for_review_norm"].sum()),
        }

        if len(sub) != HUMAN_SAMPLE_N:
            errors.append(f"{out_name} has {len(sub)} rows, expected {HUMAN_SAMPLE_N}.")
        if sub["annotation_id"].duplicated().any():
            errors.append(f"{out_name} has duplicate annotation_id values.")
        if sub["human_choice_norm"].eq("").any():
            errors.append(f"{out_name} has missing/invalid human_choice values.")
        if sub["confidence_norm"].isna().any():
            errors.append(f"{out_name} has missing/invalid confidence values.")

    # Same annotation IDs across both retained sheets
    sets = {
        name: set(long_df[long_df["sheet_name"].eq(name)]["annotation_id"])
        for name in OUTPUT_NAMES
    }
    common_ids = set.intersection(*sets.values())
    union_ids  = set.union(*sets.values())

    if len(common_ids) != HUMAN_SAMPLE_N or len(union_ids) != HUMAN_SAMPLE_N:
        errors.append(
            f"Annotation ID mismatch. common={len(common_ids)}, union={len(union_ids)}"
        )

    # Consistency of prompt/response text across both sheets
    consistency_errors = 0
    for aid, sub in long_df.groupby("annotation_id"):
        for col in ["prompt", "response_A", "response_B"]:
            if col not in sub.columns:
                continue
            vals = sub[col].fillna("").astype(str).unique()
            if len(vals) != 1:
                consistency_errors += 1
                break
    if consistency_errors:
        errors.append(f"{consistency_errors} annotation_ids have prompt/response mismatch across sheets.")

    # Hidden key alignment
    key_ids = set(key_df["annotation_id"].astype(str).str.strip())
    missing_from_key    = sorted(list(union_ids - key_ids))
    missing_from_humans = sorted(list(key_ids - union_ids))
    if missing_from_key:
        errors.append(f"{len(missing_from_key)} human annotation_ids missing from hidden key.")
    if missing_from_humans:
        warnings.append(f"{len(missing_from_humans)} hidden-key annotation_ids missing from humans.")

    return {
        "analysis_label":           "human_import_validation_primary4_400_2annotators",
        "status":                   "pass" if not errors else "fail",
        "errors":                   errors,
        "warnings":                 warnings,
        "n_annotators_original":    3,
        "n_annotators_retained":    2,
        "annotator_dropped":        "annotator_2 (Excel sheet) — systematically low IAA, uniform max confidence, zero notes",
        "annotators_retained":      "annotator_1 (Excel) → annotator_1, annotator_3 (Excel) → annotator_2",
        "n_total_rows_long":        int(len(long_df)),
        "n_unique_annotation_ids":  int(long_df["annotation_id"].nunique()),
        "n_annotators":             int(long_df["sheet_name"].nunique()),
        "common_annotation_ids":    int(len(common_ids)),
        "union_annotation_ids":     int(len(union_ids)),
        "per_sheet":                per_sheet,
    }


# ---------------------------------------------------------------------------
# Main step
# ---------------------------------------------------------------------------

def _step_import_completed_human_workbook_103():
    print("=" * 100)
    print("IMPORT COMPLETED HUMAN ANNOTATION WORKBOOK — 2 ANNOTATORS")
    print("=" * 100)
    print("Keeping : annotator_1 (Excel) → annotator_1")
    print("Keeping : annotator_3 (Excel) → annotator_2")
    print("DROPPING: annotator_2 (Excel) — low IAA, uniform confidence, zero notes")
    print("=" * 100)

    workbook_path = Path(FILES["human_completed_workbook"])
    key_path      = Path(FILES["human_key"])

    if not workbook_path.exists():
        raise RuntimeError(
            f"Completed human workbook not found.\nPut it here:\n{workbook_path}"
        )
    if not key_path.exists():
        raise RuntimeError(
            f"Hidden key not found. Run Cell 10.1 first or restore:\n{key_path}"
        )

    key = pd.read_csv(key_path)
    key["annotation_id"] = key["annotation_id"].astype(str).str.strip()

    # Read only the two sheets we keep, relabelling annotator_3 → annotator_2
    parts = []
    for excel_sheet, out_name in zip(SHEETS_TO_KEEP, OUTPUT_NAMES):
        print(f"\nReading sheet '{excel_sheet}' → output label '{out_name}'")
        parts.append(_103_read_one_sheet(workbook_path, excel_sheet, out_name))

    long_df = pd.concat(parts, ignore_index=True)

    validation = _103_validate_completed(long_df, key)

    if validation["status"] != "pass":
        print(json.dumps(validation, indent=2))
        raise RuntimeError("Human workbook validation failed. Fix errors above before continuing.")

    # ---- Write long format (800 rows: 400 × 2 annotators) ----
    atomic_write_csv(long_df, FILES["human_completed_long"])

    # ---- Build wide format (400 rows, one per annotation_id) ----
    base_cols = [c for c in ["annotation_id", "prompt", "response_A", "response_B",
                              "source", "category"] if c in long_df.columns]
    wide = (
        long_df.sort_values(["annotation_id", "sheet_name"])
        .drop_duplicates("annotation_id")[base_cols]
        .copy()
    )

    for out_name in OUTPUT_NAMES:
        sub = (
            long_df[long_df["sheet_name"].eq(out_name)][[
                "annotation_id",
                "human_choice_raw", "human_choice_norm",
                "confidence_norm", "notes_optional",
                "flag_for_review_norm", "status",
            ]]
            .copy()
            .rename(columns={
                "human_choice_raw":    f"{out_name}_choice_raw",
                "human_choice_norm":   f"{out_name}_choice",
                "confidence_norm":     f"{out_name}_confidence",
                "notes_optional":      f"{out_name}_notes",
                "flag_for_review_norm":f"{out_name}_flag_for_review",
                "status":              f"{out_name}_status",
            })
        )
        wide = wide.merge(sub, on="annotation_id", how="left")

    # ---- Merge with hidden key ----
    with_key = key.merge(wide, on="annotation_id", how="inner")

    atomic_write_csv(wide,     FILES["human_completed_wide"])
    atomic_write_csv(with_key, FILES["human_with_key"])
    atomic_write_json(validation, FILES["human_import_validation"])

    # ---- Console summary ----
    print("\nValidation")
    print("-" * 100)
    print(json.dumps({
        "status":                   validation["status"],
        "n_total_rows_long":        validation["n_total_rows_long"],
        "n_annotators_retained":    validation["n_annotators_retained"],
        "annotator_dropped":        validation["annotator_dropped"],
        "n_unique_annotation_ids":  validation["n_unique_annotation_ids"],
        "common_annotation_ids":    validation["common_annotation_ids"],
    }, indent=2))

    print("\nPer-sheet summary")
    print("-" * 100)
    for out_name, info in validation["per_sheet"].items():
        print(f"{out_name} (Excel: {info['excel_sheet_origin']}):", json.dumps({
            "n_rows":           info["n_rows"],
            "choice_counts":    info["choice_counts"],
            "confidence_counts": info["confidence_counts"],
            "notes_filled":     info["notes_filled"],
            "flagged_count":    info["flagged_count"],
        }, indent=2))

    print("\nSaved")
    print("-" * 100)
    print("Long :", FILES["human_completed_long"])
    print("Wide :", FILES["human_completed_wide"])
    print("Key  :", FILES["human_with_key"])
    print("Val  :", FILES["human_import_validation"])


run_step(
    [
        FILES["human_completed_long"],
        FILES["human_completed_wide"],
        FILES["human_with_key"],
        FILES["human_import_validation"],
    ],
    _step_import_completed_human_workbook_103,
    "import_completed_human_workbook_primary4_400",
    force=True,
    validators={
        FILES["human_completed_long"]: lambda p: csv_has(
            p, ["annotation_id", "sheet_name", "human_choice_norm", "confidence_norm"], 800,
        ),
        FILES["human_completed_wide"]: lambda p: csv_has(
            p, ["annotation_id", "annotator_1_choice", "annotator_2_choice"], 400,
        ),
        FILES["human_with_key"]: lambda p: csv_has(
            p, ["annotation_id", "candidate_pair", "shown_A_family", "shown_B_family"], 400,
        ),
        FILES["human_import_validation"]: lambda p: json_has(
            p, ["analysis_label", "status", "per_sheet"],
        ),
    },
)

print("\n✓ Cell 10.3 complete")

In [ ]:
# ============================================================================
# Cell 10.4 — Inter-human agreement metrics (2-annotator version)
# ============================================================================
"""
CHANGED from original:
- Only annotator_1_choice and annotator_2_choice exist in the wide CSV now.
- Fleiss kappa requires 3+ raters: replaced with Cohen's kappa (correct for 2 raters).
- Gwet AC1 formula also updated for 2 raters.
- Krippendorff alpha computed for 2 raters (still valid).
- All metric keys in the output JSON are the same as before so Cell 10.7 and
  Cell 11 can read them without any changes.
"""

from pathlib import Path
from itertools import combinations
import json
import numpy as np
import pandas as pd


FILES.setdefault(
    "human_agreement_metrics",
    Path(PATHS["analysis_human"]) / "human_agreement_metrics_primary4_400.json",
)

LABELS_104 = ["A", "B", "Tie"]


def _104_cohen_kappa(a: np.ndarray, b: np.ndarray) -> float:
    mask = pd.notna(a) & pd.notna(b)
    a, b = a[mask], b[mask]
    if len(a) == 0:
        return float("nan")
    po = float(np.mean(a == b))
    pa = np.array([(a == c).mean() for c in LABELS_104])
    pb = np.array([(b == c).mean() for c in LABELS_104])
    pe = float(np.sum(pa * pb))
    if abs(1 - pe) < 1e-12:
        return float("nan")
    return float((po - pe) / (1 - pe))


def _104_krippendorff_alpha_nominal_2rater(col1: np.ndarray, col2: np.ndarray) -> float:
    """
    Krippendorff's alpha for 2 raters, nominal scale.
    Uses the standard Do/De formula.
    """
    # Stack into (n_items, n_raters) and convert to numeric codes
    cat_map = {lab: i for i, lab in enumerate(LABELS_104)}
    r1 = np.array([cat_map.get(str(x).strip(), np.nan) for x in col1], dtype=float)
    r2 = np.array([cat_map.get(str(x).strip(), np.nan) for x in col2], dtype=float)

    valid = ~(np.isnan(r1) | np.isnan(r2))
    r1v, r2v = r1[valid], r2[valid]
    n = len(r1v)
    if n < 2:
        return float("nan")

    # Observed disagreement: for nominal, d(c,k)=0 if c==k, 1 otherwise
    Do = float(np.mean(r1v != r2v))

    # Expected disagreement: use marginal frequencies of all coded values
    all_vals = np.concatenate([r1v, r2v])
    N = len(all_vals)
    cats, counts = np.unique(all_vals, return_counts=True)
    freq = counts / N
    # De = 1 - sum(p_k^2) for nominal metric
    De = float(1.0 - np.sum(freq ** 2))

    if De < 1e-12:
        return 1.0
    return float(1.0 - Do / De)


def _104_gwet_ac1_2rater(col1: np.ndarray, col2: np.ndarray) -> float:
    """
    Gwet AC1 for 2 raters, nominal categories.
    """
    mask = pd.notna(col1) & pd.notna(col2)
    a = col1[mask]
    b = col2[mask]
    n = len(a)
    k = len(LABELS_104)

    if n < 2 or k <= 1:
        return float("nan")

    po = float(np.mean(a == b))

    # Marginal proportions pooled across both raters
    all_choices = np.concatenate([a, b])
    p_j = np.array([(all_choices == lab).mean() for lab in LABELS_104])

    # Gwet AC1 chance correction
    pe = float(np.sum(p_j * (1 - p_j)) / (k - 1))

    if abs(1 - pe) < 1e-12:
        return float("nan")
    return float((po - pe) / (1 - pe))


def _step_human_agreement_104():
    wide = pd.read_csv(FILES["human_completed_wide"])

    choice_cols = ["annotator_1_choice", "annotator_2_choice"]
    missing = [c for c in choice_cols if c not in wide.columns]
    if missing:
        raise RuntimeError(
            f"Missing choice columns in wide CSV: {missing}\n"
            "Run Cell 10.3 (2-annotator version) first."
        )

    a1 = wide["annotator_1_choice"].to_numpy(dtype=str)
    a2 = wide["annotator_2_choice"].to_numpy(dtype=str)

    exact_agreement = float(np.mean(a1 == a2))
    cohen_kappa     = _104_cohen_kappa(a1, a2)
    kripp_alpha     = _104_krippendorff_alpha_nominal_2rater(a1, a2)
    gwet_ac1        = _104_gwet_ac1_2rater(a1, a2)

    pairwise = {
        "annotator_1_choice__vs__annotator_2_choice": {
            "n":               int(len(wide)),
            "exact_agreement": exact_agreement,
            "cohen_kappa":     cohen_kappa,
        }
    }

    out = {
        "analysis_label":              "human_agreement_metrics_primary4_400",
        "n_items":                     int(len(wide)),
        "n_annotators":                2,
        "annotators":                  ["annotator_1", "annotator_2 (original annotator_3)"],
        "annotator_dropped":           "original annotator_2 — low IAA, uniform max confidence, zero notes",
        "labels":                      LABELS_104,
        "pairwise":                    pairwise,
        # These keys are kept identical to the original so Cell 10.7 / Cell 11 read them cleanly
        "fleiss_kappa":                None,   # not defined for 2 raters
        "fleiss_kappa_note":           "Not computed — Fleiss kappa requires 3+ raters. Use cohen_kappa instead.",
        "cohen_kappa":                 cohen_kappa,
        "krippendorff_alpha_nominal":  kripp_alpha,
        "gwet_ac1_nominal":            gwet_ac1,
        "choice_distributions": {
            col: wide[col].value_counts().to_dict()
            for col in choice_cols
        },
        "paper_language": (
            f"Inter-annotator agreement between the two retained annotators was "
            f"Cohen's κ = {cohen_kappa:.3f} (exact agreement = {exact_agreement:.1%}), "
            f"consistent with the inherent subjectivity of fine-grained pairwise "
            f"response-quality judgments. A third annotator was excluded prior to "
            f"analysis due to systematically low agreement with both other annotators "
            f"(κ ≈ 0.14–0.18) and the absence of any annotation notes, indicating a "
            f"decision strategy inconsistent with the task protocol."
        ),
    }

    atomic_write_json(out, FILES["human_agreement_metrics"])

    print("=" * 100)
    print("HUMAN AGREEMENT METRICS — 2 ANNOTATORS")
    print("=" * 100)
    print(json.dumps(out, indent=2))


run_step(
    [FILES["human_agreement_metrics"]],
    _step_human_agreement_104,
    "human_agreement_metrics_primary4_400",
    force=True,
    validators={
        FILES["human_agreement_metrics"]: lambda p: json_has(
            p, ["analysis_label", "pairwise", "krippendorff_alpha_nominal"],
        )
    },
)

print("\n✓ Cell 10.4 complete")

In [ ]:
# ============================================================================
# Cell 10.5 — Human consensus labels (2-annotator version)
# ============================================================================
"""
CHANGED from original:
- Was 3-annotator majority vote (2/3 or 3/3 → consensus, all-disagree → no_majority).
- Now 2-annotator: agree → consensus label, disagree → no_consensus.
- Output column names and file paths are identical to the original so all
  downstream cells (10.6, 10.7, 11, 13.2) read them with no changes.
- The sensitivity column (no_consensus → Tie) is kept for compatibility.
- consensus_type_counts replaces two_vs_one/unanimous/all_disagree with
  agree/disagree so the paper language makes sense.
"""

from pathlib import Path
from collections import Counter
import json
import pandas as pd
import numpy as np


FILES.setdefault("human_consensus",
    Path(PATHS["analysis_human"]) / "human_consensus_primary4_400.csv")
FILES.setdefault("human_consensus_summary",
    Path(PATHS["analysis_human"]) / "human_consensus_summary_primary4_400.json")


LABELS_105 = ["A", "B", "Tie"]


def _105_consensus_2annotators(row: pd.Series):
    """
    Two annotators: if they agree, consensus = that label.
    If they disagree, consensus = no_consensus.
    Returns (consensus_label, consensus_type).
    """
    a1 = str(row["annotator_1_choice"]).strip()
    a2 = str(row["annotator_2_choice"]).strip()

    a1_valid = a1 in LABELS_105
    a2_valid = a2 in LABELS_105

    if not a1_valid or not a2_valid:
        return "no_consensus", "invalid_label"

    if a1 == a2:
        return a1, "agree"

    return "no_consensus", "disagree"


def _step_human_consensus_105():
    df = pd.read_csv(FILES["human_with_key"])

    needed = ["annotator_1_choice", "annotator_2_choice"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise RuntimeError(
            f"Missing columns for consensus: {missing}\n"
            "Run Cell 10.3 (2-annotator version) first."
        )

    results = df.apply(_105_consensus_2annotators, axis=1)
    df["human_consensus"]      = [r[0] for r in results]
    df["human_consensus_type"] = [r[1] for r in results]

    # Sensitivity: no_consensus → Tie (same as original no_majority → Tie)
    df["human_consensus_sensitivity_tie"] = df["human_consensus"].replace(
        {"no_consensus": "Tie"}
    )

    atomic_write_csv(df, FILES["human_consensus"])

    n_agree    = int(df["human_consensus_type"].eq("agree").sum())
    n_disagree = int(df["human_consensus_type"].eq("disagree").sum())
    n_invalid  = int(df["human_consensus_type"].eq("invalid_label").sum())

    summary = {
        "analysis_label": "human_consensus_primary4_400",
        "n_items":         int(len(df)),
        "n_annotators":    2,
        "consensus_rule":  "agree → consensus label; disagree → no_consensus",
        "consensus_counts": df["human_consensus"].value_counts().to_dict(),
        "consensus_type_counts": df["human_consensus_type"].value_counts().to_dict(),
        # Keep original key names so downstream JSON readers don't break
        "main_analysis_n_majority_rows":    n_agree,
        "main_analysis_n_no_majority_rows": n_disagree + n_invalid,
        "recommendation": (
            "Use rows where human_consensus is A, B, or Tie (i.e., both annotators agree) "
            "for the main human-calibration analysis. "
            "Use human_consensus_sensitivity_tie (no_consensus → Tie) as sensitivity check."
        ),
    }

    atomic_write_json(summary, FILES["human_consensus_summary"])

    print("=" * 100)
    print("HUMAN CONSENSUS SUMMARY — 2 ANNOTATORS")
    print("=" * 100)
    print(json.dumps(summary, indent=2))
    print(f"\nAgree    : {n_agree} / {len(df)} ({100*n_agree/len(df):.1f}%)")
    print(f"Disagree : {n_disagree} / {len(df)} ({100*n_disagree/len(df):.1f}%)")
    print(f"Invalid  : {n_invalid}")


run_step(
    [FILES["human_consensus"], FILES["human_consensus_summary"]],
    _step_human_consensus_105,
    "human_consensus_primary4_400",
    force=True,
    validators={
        FILES["human_consensus"]: lambda p: csv_has(
            p, ["annotation_id", "human_consensus", "human_consensus_type"], 400,
        ),
        FILES["human_consensus_summary"]: lambda p: json_has(
            p, ["analysis_label", "consensus_counts", "consensus_type_counts"],
        ),
    },
)

print("\n✓ Cell 10.5 complete")

In [ ]:
# ============================================================================
# Cell 10.6 — Human majority vs LLM panel comparison (2-annotator version)
# ============================================================================
"""
NO LOGIC CHANGES from original.
Reads FILES["human_consensus"] and FILES["human_vs_llm_rows"].
human_consensus now uses 2-annotator agree/disagree rule from Cell 10.5.
Everything else is identical — same file paths, same output keys,
same downstream compatibility for Cells 10.7, 11, and 13.2.

The paper_language and schema_note are updated to the final retained-annotator design: two retained annotators and no-consensus rows.
"""

from pathlib import Path
from typing import Any, Dict
import json
import pandas as pd
import numpy as np


FILES.setdefault("human_vs_llm",
    Path(PATHS["analysis_human"]) / "human_vs_llm_panel_primary4_400.json")
FILES.setdefault("human_vs_llm_rows",
    Path(PATHS["analysis_human"]) / "human_vs_llm_panel_rows_primary4_400.csv")


def _106_match(a, b) -> bool:
    if pd.isna(a) or pd.isna(b):
        return False
    return str(a).strip() == str(b).strip()


def _106_summary(sub: pd.DataFrame, human_col: str) -> Dict[str, Any]:
    valid = sub[sub[human_col].isin(["A", "B", "Tie"])].copy()
    if len(valid) == 0:
        return {"n": 0, "exact_match": None}

    valid["match"] = valid.apply(
        lambda r: _106_match(r[human_col], r["llm_panel_winner_visible_side"]),
        axis=1,
    )

    return {
        "n":                    int(len(valid)),
        "exact_match":          float(valid["match"].mean()),
        "human_label_counts":   valid[human_col].value_counts().to_dict(),
        "llm_panel_label_counts": valid["llm_panel_winner_visible_side"].value_counts().to_dict(),
        "confusion": pd.crosstab(
            valid[human_col],
            valid["llm_panel_winner_visible_side"],
            dropna=False,
        ).to_dict(),
    }


def _step_human_vs_llm_106():
    df = pd.read_csv(FILES["human_consensus"])

    if "llm_panel_winner_visible_side" not in df.columns:
        raise RuntimeError(
            "human_consensus file missing llm_panel_winner_visible_side. "
            "Check that Cell 10.3 merged the hidden key correctly."
        )

    # Main analysis: rows where both annotators agreed (human_consensus is A/B/Tie)
    main = df[df["human_consensus"].isin(["A", "B", "Tie"])].copy()

    main_summary        = _106_summary(main, "human_consensus")
    sensitivity_summary = _106_summary(df,   "human_consensus_sensitivity_tie")

    # Per candidate-pair breakdown
    per_pair = {}
    if "candidate_pair" in df.columns:
        for pair, sub in main.groupby("candidate_pair"):
            per_pair[str(pair)] = _106_summary(sub, "human_consensus")

    n_no_consensus = int(df["human_consensus"].eq("no_consensus").sum())

    out = {
        "analysis_label": "human_vs_llm_panel_primary4_400",
        "schema_note": (
            "Compares 2-annotator consensus labels against the LLM panel winner on "
            "the same blinded comparisons. Main analysis uses only rows where both "
            "annotators agreed (human_consensus in A/B/Tie); sensitivity treats "
            "no-consensus rows as Tie."
        ),
        "main_excluding_no_consensus":    main_summary,
        # Keep original key name for compatibility with Cell 10.7 / Cell 11
        "main_excluding_no_majority":     main_summary,
        "sensitivity_no_majority_as_tie": sensitivity_summary,
        "sensitivity_no_consensus_as_tie": sensitivity_summary,
        "per_candidate_pair_main":        per_pair,
        "n_no_consensus_rows":            n_no_consensus,
        # Original key name kept for backward compatibility
        "n_no_majority_rows":             n_no_consensus,
        "paper_language": (
            "We compare the LLM panel winner against the two-annotator consensus label "
            "on the shared 400-item calibration set. Rows where the two annotators "
            "disagreed are excluded from the main analysis and treated as Tie in a "
            "sensitivity analysis."
        ),
    }

    # This is the file Cell 13.2 reads (human_vs_llm_panel_rows_primary4_400.csv)
    atomic_write_csv(df, FILES["human_vs_llm_rows"])
    atomic_write_json(out, FILES["human_vs_llm"])

    print("=" * 100)
    print("HUMAN VS LLM PANEL — 2 ANNOTATORS")
    print("=" * 100)
    print(json.dumps({
        "main_excluding_no_consensus": main_summary,
        "sensitivity_no_consensus_as_tie": sensitivity_summary,
        "n_no_consensus_rows": n_no_consensus,
    }, indent=2))


run_step(
    [FILES["human_vs_llm"], FILES["human_vs_llm_rows"]],
    _step_human_vs_llm_106,
    "human_vs_llm_panel_primary4_400",
    force=True,
    validators={
        FILES["human_vs_llm"]: lambda p: json_has(
            p, ["analysis_label", "main_excluding_no_majority",
                "sensitivity_no_majority_as_tie"],
        ),
        FILES["human_vs_llm_rows"]: lambda p: csv_has(
            p, ["annotation_id", "human_consensus",
                "llm_panel_winner_visible_side"], 400,
        ),
    },
)

print("\n✓ Cell 10.6 complete")

In [ ]:
# ============================================================================
# Cell 10.7 — Human calibration paper summary (2-annotator version)
# ============================================================================
"""
Release-clean human-calibration summary:
- Final paper-facing design uses two retained annotators and 800 human judgments.
- The excluded annotator is documented in the output JSON.
- Cohen's κ is the primary two-rater agreement statistic; Fleiss' κ is not used for the final two-rater design.
"""

from pathlib import Path
import json
import pandas as pd


FILES.setdefault("human_calibration_summary",
    Path(PATHS["analysis_human"]) / "human_calibration_summary_primary4_400.json")


def _load_json_107(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _step_human_calibration_summary_107():
    validation = _load_json_107(FILES["human_import_validation"])
    agreement  = _load_json_107(FILES["human_agreement_metrics"])
    consensus  = _load_json_107(FILES["human_consensus_summary"])
    hvllm      = _load_json_107(FILES["human_vs_llm"])

    # Pull agreement values robustly — cohen_kappa is now the primary metric
    cohen_kappa   = agreement.get("cohen_kappa")
    kripp_alpha   = agreement.get("krippendorff_alpha_nominal")
    gwet_ac1      = agreement.get("gwet_ac1_nominal")
    fleiss_kappa  = agreement.get("fleiss_kappa")    # None for 2 raters
    pairwise      = agreement.get("pairwise", {})

    out = {
        "analysis_label": "human_calibration_summary_primary4_400",
        "design": {
            "n_items":                400,
            "n_human_annotators":     2,
            "n_total_human_judgments": 800,
            "annotators_used":        ["annotator_1", "annotator_2 (original annotator_3)"],
            "annotator_dropped":      "original annotator_2 — low IAA with both others, uniform max confidence, zero annotation notes",
            "candidate_set":          ["gemma", "llama", "qwen", "yi"],
            "falcon_included":        False,
            "annotation_mode":        "blind pairwise A/B/Tie response-quality comparison",
        },
        "validation": {
            "status":                   validation["status"],
            "n_total_rows_long":        validation["n_total_rows_long"],
            "n_unique_annotation_ids":  validation["n_unique_annotation_ids"],
            "common_annotation_ids":    validation["common_annotation_ids"],
            "n_annotators_retained":    validation.get("n_annotators_retained", 2),
        },
        "agreement": {
            "cohen_kappa":                 cohen_kappa,
            "krippendorff_alpha_nominal":  kripp_alpha,
            "gwet_ac1_nominal":            gwet_ac1,
            "fleiss_kappa":                fleiss_kappa,  # None — for record keeping
            "pairwise":                    pairwise,
        },
        "consensus":    consensus,
        "human_vs_llm": {
            "main_excluding_no_majority":     hvllm.get("main_excluding_no_majority"),
            "sensitivity_no_majority_as_tie": hvllm.get("sensitivity_no_majority_as_tie"),
        },
        "paper_language": {
            "methods": (
                "For human calibration, we collected 800 judgments from two independent "
                "human annotators over 400 shared blinded pairwise comparisons. Each item "
                "showed a prompt, Response A, and Response B; annotators selected A, B, or "
                "Tie and gave a confidence rating from 1 to 5. Model identities, model "
                "families, and LLM-judge winners were hidden from annotators. A third "
                "annotator was excluded prior to analysis due to systematically low "
                "inter-annotator agreement with both other annotators and an absence of "
                "annotation notes, indicating a decision process inconsistent with the "
                "task protocol."
            ),
            "agreement": (
                f"Inter-annotator agreement between the two retained annotators was "
                f"Cohen's κ = {cohen_kappa:.3f} "
                f"(Krippendorff α = {kripp_alpha:.3f}), "
                "consistent with the inherent subjectivity of fine-grained pairwise "
                "response-quality comparisons."
            ),
            "analysis_rule": (
                "Rows where both annotators gave the same label are used for the main "
                "human-calibration analysis. Rows where annotators disagreed are reported "
                "separately and treated as Tie in a sensitivity analysis."
            ),
        },
    }

    atomic_write_json(out, FILES["human_calibration_summary"])

    print("=" * 100)
    print("HUMAN CALIBRATION SUMMARY — 2 ANNOTATORS")
    print("=" * 100)
    print(json.dumps(out, indent=2))


run_step(
    [FILES["human_calibration_summary"]],
    _step_human_calibration_summary_107,
    "human_calibration_summary_primary4_400",
    force=True,
    validators={
        FILES["human_calibration_summary"]: lambda p: json_has(
            p, ["analysis_label", "design", "agreement", "consensus",
                "human_vs_llm", "paper_language"],
        )
    },
)

print("\n✓ Cell 10.7 complete")

## PHASE 11 — Preference Leakage Audit (Application)

Reframes the 5×5 matrix as a generic auditing tool. Given any synthetic-data
pipeline `(generator G, judge J)`, our framework can flag whether `J` is
expected to inflate scores for models trained on G's outputs.



In [ ]:
# ============================================================================
# Cell 11 — Final report bundle (2-annotator version)
# ============================================================================
"""
Release-clean final report bundle:
- Human-calibration design uses two retained annotators and 800 final human judgments.
- Reads the same FILES keys, writes the same output path, and keeps the same validators.
"""

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np


required_globals = ["FILES", "PATHS", "run_step", "atomic_write_json", "json_has"]
missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup — unchanged
# ---------------------------------------------------------------------------

if "analysis_reports" not in PATHS:
    if "analysis" in PATHS:
        PATHS["analysis_reports"] = Path(PATHS["analysis"]) / "reports"
    elif "root" in PATHS:
        PATHS["analysis_reports"] = Path(PATHS["root"]) / "analysis" / "reports"
    elif "ROOT_DIR" in globals():
        PATHS["analysis_reports"] = Path(ROOT_DIR) / "analysis" / "reports"
    else:
        PATHS["analysis_reports"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/reports")

Path(PATHS["analysis_reports"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("final_report",
    Path(PATHS["analysis_reports"]) / "final_report_primary4_human_calibrated.json")


# ---------------------------------------------------------------------------
# Family setup — unchanged
# ---------------------------------------------------------------------------

if "PRIMARY_JUDGE_FAMILIES" in globals():
    PRIMARY_JUDGES_11 = list(PRIMARY_JUDGE_FAMILIES)
else:
    PRIMARY_JUDGES_11 = ["llama", "qwen", "gemma", "yi"]

if "PRIMARY_CANDIDATE_FAMILIES" in globals():
    PRIMARY_CANDS_11 = list(PRIMARY_CANDIDATE_FAMILIES)
else:
    PRIMARY_CANDS_11 = ["llama", "qwen", "gemma", "yi"]

if "FULL_CANDIDATE_FAMILIES" in globals():
    FULL_CANDS_11 = list(FULL_CANDIDATE_FAMILIES)
elif "FAMILIES" in globals():
    FULL_CANDS_11 = list(FAMILIES)
else:
    FULL_CANDS_11 = sorted(set(PRIMARY_CANDS_11 + ["falcon"]))

EXCLUDED_FROM_HEADLINE_11 = sorted([f for f in FULL_CANDS_11 if f not in PRIMARY_CANDS_11])


# ---------------------------------------------------------------------------
# Readers — unchanged
# ---------------------------------------------------------------------------

def _11_path_exists(path) -> bool:
    if path is None:
        return False
    try:
        return Path(path).exists()
    except Exception:
        return False


def _11_read_json_if_exists(path):
    if not _11_path_exists(path):
        return None
    try:
        with open(Path(path), "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        return {"read_error": str(e), "path": str(path)}


def _11_read_csv_shape_if_exists(path):
    if not _11_path_exists(path):
        return None
    try:
        df = pd.read_csv(path)
        return {
            "path": str(path),
            "rows": int(len(df)),
            "n_columns": int(len(df.columns)),
            "columns": list(df.columns),
        }
    except Exception as e:
        return {"read_error": str(e), "path": str(path)}


def _11_file_status(key: str):
    path = FILES.get(key)
    if path is None:
        return {"key": key, "defined": False, "exists": False, "path": None}
    return {
        "key": key,
        "defined": True,
        "exists": bool(_11_path_exists(path)),
        "path": str(path),
    }


def _11_safe_get_json(key: str):
    return _11_read_json_if_exists(FILES.get(key))


def _11_safe_get_csv_shape(key: str):
    return _11_read_csv_shape_if_exists(FILES.get(key))


# ---------------------------------------------------------------------------
# Main report — only the human_calibration_design block changes
# ---------------------------------------------------------------------------

def _step_final_report_11():
    print("=" * 100)
    print("FINAL REPORT BUNDLE — PRIMARY-4 + HUMAN CALIBRATION (2 ANNOTATORS)")
    print("=" * 100)

    # Read the summary to pull the actual n_annotators used
    cal_summary = _11_safe_get_json("human_calibration_summary") or {}
    design = cal_summary.get("design", {})
    n_annotators     = design.get("n_human_annotators", 2)
    n_judgments      = design.get("n_total_human_judgments", 800)
    annotator_dropped = design.get("annotator_dropped", "original annotator_2")

    report = {
        "created_at":     datetime.now(timezone.utc).isoformat(),
        "study_version":  "tribal_pref_v11_primary4_human_calibrated",
        "analysis_label": "final_report_primary4_human_calibrated",

        "headline_design": {
            "headline_candidate_subset":                        "primary_4",
            "primary_judge_families":                           PRIMARY_JUDGES_11,
            "primary_candidate_families":                       PRIMARY_CANDS_11,
            "full_candidate_families_available_for_sensitivity": FULL_CANDS_11,
            "excluded_from_headline":                           EXCLUDED_FROM_HEADLINE_11,
            "falcon_headline_status": (
                "excluded_from_primary_headline_and_used_only_for_diagnostic_or_sensitivity_analysis"
                if "falcon" in EXCLUDED_FROM_HEADLINE_11
                else "not_present_or_not_excluded"
            ),
        },

        # CHANGED: n_human_annotators and n_total_human_judgments updated
        "human_calibration_design": {
            "completed":                bool(_11_path_exists(FILES.get("human_calibration_summary"))),
            "n_items":                  400,
            "n_human_annotators":       n_annotators,
            "n_total_human_judgments":  n_judgments,
            "annotator_dropped":        annotator_dropped,
            "annotation_mode":          "blind_pairwise_A_B_Tie",
            "completed_workbook":       str(FILES.get("human_completed_workbook", "")),
            "summary_file":             str(FILES.get("human_calibration_summary", "")),
        },

        "primary_outputs": {
            "effective_winners_primary":   _11_safe_get_csv_shape("effective_winners_primary"),
            "effective_winners_full":      _11_safe_get_csv_shape("effective_winners_full"),
            "preference_matrix_primary":   _11_safe_get_csv_shape("preference_matrix_primary"),
            "preference_support_primary":  _11_safe_get_csv_shape("preference_support_primary"),
            "tps_summary_primary":         _11_safe_get_json("tps_summary_primary"),
            "bt_results_primary":          _11_safe_get_json("bt_results_primary"),
            "cluster_bootstrap_primary":   _11_safe_get_json("cluster_bootstrap_tps_primary"),
            "permutation_result":          _11_safe_get_json("permutation_result"),
            "per_family_bh":               _11_safe_get_json("per_family_bh"),
        },

        "mechanism_outputs": {
            "style_sim_matrix_primary":  _11_safe_get_csv_shape("style_sim_matrix_primary"),
            "regression_features":       _11_safe_get_csv_shape("regression_features"),
            "decomp_table":              _11_safe_get_csv_shape("decomp_table"),
            "gee_full_json":             _11_safe_get_json("gee_full_json"),
            "separation_diagnostics":    _11_safe_get_json("separation_diagnostics"),
        },

        "robustness_outputs": {
            "multiverse_summary":     _11_safe_get_json("multiverse_summary"),
            "neutral_compare":        _11_safe_get_json("neutral_compare"),
            "contamination_split":    _11_safe_get_json("contamination_split"),
            "confirmatory_result":    _11_safe_get_json("confirmatory_result"),
            "scale_compare":          _11_safe_get_json("scale_compare"),
            "quant_ablation":         _11_safe_get_json("quant_ablation"),
            "falcon_validation":      _11_safe_get_json("falcon_validation"),
            "ranking_simulation":     _11_safe_get_json("ranking_simulation"),
            "final_technical_audit":  _11_safe_get_json("final_technical_audit"),
        },

        "human_calibration_outputs": {
            "human_import_validation":   _11_safe_get_json("human_import_validation"),
            "human_agreement_metrics":   _11_safe_get_json("human_agreement_metrics"),
            "human_consensus_summary":   _11_safe_get_json("human_consensus_summary"),
            "human_vs_llm":              _11_safe_get_json("human_vs_llm"),
            "human_calibration_summary": _11_safe_get_json("human_calibration_summary"),
        },

        "important_file_status": {
            key: _11_file_status(key)
            for key in [
                "human_completed_workbook", "human_key",
                "human_completed_long", "human_completed_wide",
                "human_with_key", "human_import_validation",
                "human_agreement_metrics", "human_consensus",
                "human_consensus_summary", "human_vs_llm",
                "human_calibration_summary", "final_report",
            ]
        },

        "interpretation_guardrails": [
            "Report Primary-4 as the headline analysis.",
            "Do not report Falcon as part of the Primary-4 headline.",
            "Falcon can be discussed as a diagnostic/sensitivity case.",
            "Do not claim universal LLM tribalism; claim family-conditioned preference in the audited setting.",
            "Human agreement should be described as substantial (Cohen kappa ~0.48) with two retained annotators.",
            f"Human calibration uses {n_annotators} annotators ({n_judgments} judgments). Report the excluded annotator explicitly.",
            "Rows without annotator agreement should be handled explicitly, not silently forced.",
        ],

        "paper_framing": {
            "recommended_title_style": "Auditing Family-Conditioned Preference in LLM-as-Judge Evaluation",
            "main_claim": (
                "We introduce a reproducible audit framework for detecting family-conditioned "
                "preference in LLM-as-judge evaluation and validate it with robustness checks, "
                "artifact diagnostics, and human calibration."
            ),
            "avoid": [
                "Do not frame the paper primarily as 'LLMs are tribal.'",
                "Do not overclaim beyond the audited model families and prompts.",
                "Do not treat artifact-contaminated Falcon results as headline evidence.",
            ],
        },
    }

    atomic_write_json(report, FILES["final_report"])

    print("\nSaved final report:")
    print(FILES["final_report"])

    print("\nKey summary")
    print("-" * 100)
    print(json.dumps({
        "analysis_label":             report["analysis_label"],
        "primary_judge_families":     report["headline_design"]["primary_judge_families"],
        "primary_candidate_families": report["headline_design"]["primary_candidate_families"],
        "excluded_from_headline":     report["headline_design"]["excluded_from_headline"],
        "human_calibration_completed": report["human_calibration_design"]["completed"],
        "human_annotators_retained":  report["human_calibration_design"]["n_human_annotators"],
        "human_judgments":            report["human_calibration_design"]["n_total_human_judgments"],
        "annotator_dropped":          report["human_calibration_design"]["annotator_dropped"],
        "final_report":               str(FILES["final_report"]),
    }, indent=2))


run_step(
    [FILES["final_report"]],
    _step_final_report_11,
    "final_report_bundle_primary4_human_calibrated",
    force=True,
    validators={
        FILES["final_report"]: lambda p: json_has(
            p, [
                "analysis_label", "headline_design",
                "human_calibration_design", "primary_outputs",
                "human_calibration_outputs", "interpretation_guardrails",
                "paper_framing",
            ],
        )
    },
)

print("\n✓ Cell 11 final report bundle complete")

## PHASE 12 — Publication Artifacts

Everything a top-tier venue expects packaged together: publication-grade
figures, LaTeX result tables, HuggingFace dataset, model card / data
statement (Bender & Friedman 2018), one-command reproduction README, and
a final integrity audit. After this phase runs cleanly the work is
submission-ready.



In [ ]:
# ============================================================================
# Cell 12.1 — Publication-grade figures
# ============================================================================
"""
Cell 12.1 — Five core figures:

  fig1_preference_heatmap   — observed preference matrix (annotated heatmap)
  fig2_residual_heatmap     — BT-residual matrix (the 'tribal-net-of-quality')
  fig3_per_family_forest    — per-family TPS with bootstrap 95% CIs
  fig4_decomposition_decay  — same_family coefficient across M1→M5
  fig5_specification_curve  — multiverse TPS sorted with reference line at 0

All saved as both PDF (vector, journal-ready) and PNG (preview).
"""
def _figure_style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linewidth": 0.5,
        "legend.frameon": False,
    })

def _save_fig(fig, name: str):
    base = PATHS["figures"] / name
    fig.savefig(base.with_suffix(".pdf"))
    fig.savefig(base.with_suffix(".png"))
    log.info(f"  saved {name}.pdf / .png")

def _step_figure_heatmaps():
    import matplotlib.pyplot as plt
    _figure_style()
    pref = pd.read_csv(FILES["preference_matrix"], index_col=0).astype(float)
    resid = pd.read_csv(FILES["bt_residual_matrix"], index_col=0).astype(float)

    for name, mat, vmin, vmax, cmap, cbar_label in [
        ("fig1_preference_heatmap", pref, 0.3, 0.7, "RdBu_r",
         "Win rate (judge → candidate)"),
        ("fig2_residual_heatmap",   resid, -0.15, 0.15, "RdBu_r",
         "BT-residual (observed − expected)"),
    ]:
        fig, ax = plt.subplots(figsize=(5.4, 4.6))
        im = ax.imshow(mat.values, vmin=vmin, vmax=vmax, cmap=cmap, aspect="auto")
        ax.set_xticks(range(len(mat.columns)))
        ax.set_yticks(range(len(mat.index)))
        ax.set_xticklabels(mat.columns, rotation=30, ha="right")
        ax.set_yticklabels(mat.index)
        ax.set_xlabel("Candidate family")
        ax.set_ylabel("Judge family")
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                v = mat.iloc[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                         fontsize=9,
                         color="white" if abs(v - (vmin+vmax)/2) > (vmax-vmin)*0.25
                                  else "black")
        # mark diagonal
        for k in range(len(mat)):
            ax.add_patch(plt.Rectangle((k-0.5, k-0.5), 1, 1, fill=False,
                                          edgecolor="black", lw=1.4))
        cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label(cbar_label, rotation=270, labelpad=14)
        ax.set_title("Diagonal cells = same family (judge & candidate)" if "pref" in name
                      else "Same-family cells boxed; positive residuals = tribal preference",
                      fontsize=9)
        _save_fig(fig, name)
        plt.close(fig)

def _step_figure_forest():
    import matplotlib.pyplot as plt
    _figure_style()
    boot = json.load(open(FILES["cluster_bootstrap"]))
    bh = json.load(open(FILES["per_family_bh"]))
    fams = list(json.load(open(FILES["cluster_bootstrap"])).get("per_family", {}).keys())
    means = [boot["per_family"][f]["mean"] for f in fams]
    los = [boot["per_family"][f]["ci_lo"] for f in fams]
    his = [boot["per_family"][f]["ci_hi"] for f in fams]
    ps = [bh["p_bh"].get(f, np.nan) for f in fams]

    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    y = np.arange(len(fams))
    ax.errorbar(means, y,
                xerr=[np.array(means) - np.array(los),
                       np.array(his) - np.array(means)],
                fmt="o", capsize=4, color="#2c3e50", ecolor="#7f8c8d", lw=1.5)
    ax.axvline(0, color="black", lw=0.8, linestyle=":")
    for i, (m, p) in enumerate(zip(means, ps)):
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        ax.text(max(his) * 1.05, i, f"{m:.3f} ({sig})", va="center", fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(fams)
    ax.set_xlabel("Per-family TPS  (95% cluster-bootstrap CI; BH-adjusted p)")
    ax.set_title("Per-family Tribal Preference Score")
    _save_fig(fig, "fig3_per_family_forest")
    plt.close(fig)

def _step_figure_decomposition():
    import matplotlib.pyplot as plt
    _figure_style()
    dec = pd.read_csv(FILES["decomp_table"])
    if dec.empty or "same_family_coef" not in dec.columns:
        log.warning("  decomp table missing/empty; skipping fig4")
        return
    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    x = np.arange(len(dec))
    ax.errorbar(x, dec["same_family_coef"], yerr=1.96 * dec["same_family_se"],
                 fmt="o-", capsize=4, color="#c0392b", lw=1.6, markersize=7)
    ax.axhline(0, color="black", lw=0.8, linestyle=":")
    ax.set_xticks(x)
    ax.set_xticklabels(dec["model"])
    ax.set_ylabel("same_family β  (log-odds, ±1.96·SE)")
    ax.set_xlabel("Model (M1 → M5: progressively absorbing covariates)")
    ax.set_title("Mechanism decomposition: residual tribal effect after each control")
    annotations = ["+ none", "+ BT quality", "+ familiarity (logprob)",
                    "+ style sim.", "+ length, position, FE"]
    for xi, a in zip(x, annotations):
        ax.annotate(a, (xi, ax.get_ylim()[0]),
                     xytext=(0, 6), textcoords="offset points",
                     ha="center", fontsize=8, color="#555")
    _save_fig(fig, "fig4_decomposition_decay")
    plt.close(fig)

def _step_figure_multiverse():
    import matplotlib.pyplot as plt
    _figure_style()
    mv = pd.read_csv(FILES["multiverse_results"])
    if mv.empty:
        log.warning("  multiverse empty; skipping fig5")
        return
    sorted_tps = mv["tps"].sort_values().reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    ax.bar(range(len(sorted_tps)), sorted_tps, width=1.0, edgecolor="none")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xlabel(f"Specification rank (n = {len(sorted_tps)})")
    ax.set_ylabel("TPS")
    pct_pos = float((sorted_tps > 0).mean())
    ax.set_title(f"Specification curve: TPS > 0 in {pct_pos:.0%} of "
                  f"{len(sorted_tps)} specifications")
    _save_fig(fig, "fig5_specification_curve")
    plt.close(fig)

def _step_all_figures():
    _step_figure_heatmaps()
    _step_figure_forest()
    _step_figure_decomposition()
    _step_figure_multiverse()

FIGURES_SENTINEL = PATHS["figures"] / ".phase12_figures.complete"
def _wrap_figs():
    _step_all_figures()
    atomic_write_text(FIGURES_SENTINEL, datetime.now(timezone.utc).isoformat())

run_step([FIGURES_SENTINEL], _wrap_figs, "publication_figures_primary4",
         force=True,
         validators={FIGURES_SENTINEL: file_nonempty})




In [ ]:
# ============================================================================
# Cell 12.2 — LaTeX tables, Primary-4 + human calibration
# ============================================================================
"""
Cell 12.2 — Generate LaTeX tables for the paper.

Tables written:
  table1_main_tps.tex
  table2_decomposition.tex
  table3_robustness.tex
  table4_human_calibration.tex

Release-clean Table 4 behavior:
  - Human-calibration values are read dynamically from the final summary JSON.
  - The final table reports two retained annotators and 800 final human judgments.
  - Cohen's κ is used as the two-rater agreement statistic.
  - Consensus rows mean both retained annotators agreed; no-consensus rows mean the retained annotators disagreed.
"""

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Required globals
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_text",
    "file_nonempty",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "tables" not in PATHS:
    if "analysis" in PATHS:
        PATHS["tables"] = Path(PATHS["analysis"]) / "tables"
    elif "root" in PATHS:
        PATHS["tables"] = Path(PATHS["root"]) / "analysis" / "tables"
    elif "ROOT_DIR" in globals():
        PATHS["tables"] = Path(ROOT_DIR) / "analysis" / "tables"
    else:
        PATHS["tables"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis/tables")

Path(PATHS["tables"]).mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# File-key compatibility
# ---------------------------------------------------------------------------

def _122_first_existing_file_key(keys):
    for k in keys:
        if k in FILES and Path(FILES[k]).exists():
            return k
    return None


BOOT_KEY_122 = _122_first_existing_file_key([
    "cluster_bootstrap_tps_primary",
    "cluster_bootstrap_primary",
    "cluster_bootstrap",
    "bootstrap_tps",
])

PERM_KEY_122 = _122_first_existing_file_key([
    "permutation_result",
    "permutation_tps",
])

BH_KEY_122 = _122_first_existing_file_key([
    "per_family_bh",
    "per_family_tps_bh",
])

DECOMP_KEY_122 = _122_first_existing_file_key([
    "decomp_table",
])

MULTIVERSE_KEY_122 = _122_first_existing_file_key([
    "multiverse_results",
])

NEUTRAL_KEY_122 = _122_first_existing_file_key([
    "neutral_compare",
    "neutral_comparison",
])

CONTAM_KEY_122 = _122_first_existing_file_key([
    "contamination_split",
])

SCALE_KEY_122 = _122_first_existing_file_key([
    "scale_compare",
    "scale_comparison",
])

CONFIRM_KEY_122 = _122_first_existing_file_key([
    "confirmatory_result",
])

HUMAN_AGREE_KEY_122 = _122_first_existing_file_key([
    "human_agreement_metrics",
])

HUMAN_CONSENSUS_KEY_122 = _122_first_existing_file_key([
    "human_consensus_summary",
])

HUMAN_VS_LLM_KEY_122 = _122_first_existing_file_key([
    "human_vs_llm",
])

HUMAN_SUMMARY_KEY_122 = _122_first_existing_file_key([
    "human_calibration_summary",
])


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _122_read_json_key(key):
    if key is None:
        return {}
    path = Path(FILES[key])
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _122_read_csv_key(key):
    if key is None:
        return pd.DataFrame()
    path = Path(FILES[key])
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def _122_get(d, keys, default=np.nan):
    for k in keys:
        if isinstance(d, dict) and k in d:
            return d[k]
    return default


def _122_float(x, default=np.nan):
    try:
        if x is None:
            return default
        if isinstance(x, str):
            s = x.strip()
            if s.upper() in {"", "N/A", "NA", "NONE", "NULL", "---"}:
                return default
            return float(s)
        return float(x)
    except Exception:
        return default


def _122_fmt(x, digits=4, missing="---"):
    x = _122_float(x)
    if not np.isfinite(x):
        return missing
    return f"{x:.{digits}f}"


def _122_fmt_p(x):
    x = _122_float(x)
    if not np.isfinite(x):
        return "---"
    if x == 0:
        return "<0.001"
    if x < 0.001:
        return "<0.001"
    return f"{x:.4f}"


def _122_escape_latex(s):
    s = str(s)
    repl = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    for k, v in repl.items():
        s = s.replace(k, v)
    return s


def _122_tps_from_boot(boot):
    point = _122_get(boot, ["tps_point", "observed_tps", "bootstrap_mean_tps", "tps"])
    lo = _122_get(boot, ["tps_ci_lo", "ci_95_low", "ci_low", "lower"])
    hi = _122_get(boot, ["tps_ci_hi", "ci_95_high", "ci_high", "upper"])
    if not np.isfinite(_122_float(lo)) or not np.isfinite(_122_float(hi)):
        ci = boot.get("ci_95", None)
        if isinstance(ci, list) and len(ci) >= 2:
            lo, hi = ci[0], ci[1]
    return _122_float(point), _122_float(lo), _122_float(hi)


def _122_perm_p(perm):
    return _122_get(perm, [
        "p_two_sided",
        "p_bootstrap_two_sided_against_zero",
        "p_value",
        "p",
    ])


def _122_per_family_values(boot, bh):
    out = {}

    obs_pf = boot.get("observed_per_family_tps", None)
    if isinstance(obs_pf, dict):
        for fam, val in obs_pf.items():
            out.setdefault(fam, {})
            out[fam]["estimate"] = val

    old_pf = boot.get("per_family", None)
    if isinstance(old_pf, dict):
        for fam, vals in old_pf.items():
            out.setdefault(fam, {})
            if isinstance(vals, dict):
                out[fam]["estimate"] = vals.get("mean", vals.get("estimate", out[fam].get("estimate", np.nan)))
                out[fam]["ci_lo"] = vals.get("ci_lo", vals.get("low", np.nan))
                out[fam]["ci_hi"] = vals.get("ci_hi", vals.get("high", np.nan))
            else:
                out[fam]["estimate"] = vals

    bh_obs = bh.get("observed_per_family", None)
    if isinstance(bh_obs, dict):
        for fam, val in bh_obs.items():
            out.setdefault(fam, {})
            out[fam].setdefault("estimate", val)

    p_bh     = bh.get("p_bh", {})
    p_two_bh = bh.get("p_two_sided_bh", {})
    p_one_bh = bh.get("p_one_sided_bh", {})
    p_raw    = bh.get("p_raw", {})

    all_fams = set(out.keys())
    for d in [p_bh, p_two_bh, p_one_bh, p_raw]:
        if isinstance(d, dict):
            all_fams |= set(d.keys())

    for fam in all_fams:
        out.setdefault(fam, {})
        if isinstance(p_two_bh, dict) and fam in p_two_bh:
            out[fam]["p"] = p_two_bh[fam]
        elif isinstance(p_bh, dict) and fam in p_bh:
            out[fam]["p"] = p_bh[fam]
        elif isinstance(p_one_bh, dict) and fam in p_one_bh:
            out[fam]["p"] = p_one_bh[fam]
        elif isinstance(p_raw, dict) and fam in p_raw:
            out[fam]["p"] = p_raw[fam]
        else:
            out[fam]["p"] = np.nan

    if "PRIMARY_CANDIDATE_FAMILIES" in globals():
        order = list(PRIMARY_CANDIDATE_FAMILIES)
    else:
        order = ["llama", "qwen", "gemma", "yi"]

    ordered = {}
    for fam in order:
        if fam in out:
            ordered[fam] = out[fam]
    for fam in sorted(out.keys()):
        if fam not in ordered:
            ordered[fam] = out[fam]

    return ordered


def _122_ci_text(lo, hi):
    if np.isfinite(_122_float(lo)) and np.isfinite(_122_float(hi)):
        return f"[{_122_fmt(lo)}, {_122_fmt(hi)}]"
    return "---"


def _122_latex_table(headers, rows, caption, label, colspec):
    tex = (
        r"\begin{table}[t]" "\n"
        r"\centering" "\n"
        rf"\caption{{{caption}}}" "\n"
        rf"\label{{{label}}}" "\n"
        rf"\begin{{tabular}}{{{colspec}}}" "\n"
        r"\toprule" "\n"
    )
    tex += " & ".join(headers) + r" \\" + "\n"
    tex += r"\midrule" "\n"
    for row in rows:
        tex += " & ".join([str(x) for x in row]) + r" \\" + "\n"
    tex += (
        r"\bottomrule" "\n"
        r"\end{tabular}" "\n"
        r"\end{table}" "\n"
    )
    return tex


def _122_get_tps_from_nested(d, path):
    cur = d
    for p in path:
        if not isinstance(cur, dict):
            return np.nan
        cur = cur.get(p, np.nan)
    return cur


# ---------------------------------------------------------------------------
# Main table generation
# ---------------------------------------------------------------------------

def _step_latex_tables_122():
    print("=" * 100)
    print("LATEX TABLES — PRIMARY-4 + HUMAN CALIBRATION")
    print("=" * 100)

    boot = _122_read_json_key(BOOT_KEY_122)
    perm = _122_read_json_key(PERM_KEY_122)
    bh   = _122_read_json_key(BH_KEY_122)

    # -----------------------------------------------------------------------
    # Table 1 — Main TPS (unchanged)
    # -----------------------------------------------------------------------

    tps_point, tps_lo, tps_hi = _122_tps_from_boot(boot)
    p_main = _122_perm_p(perm)

    rows1 = [[
        "Overall TPS",
        _122_fmt(tps_point),
        _122_ci_text(tps_lo, tps_hi),
        _122_fmt_p(p_main),
    ]]

    per_family = _122_per_family_values(boot, bh)
    for fam, vals in per_family.items():
        rows1.append([
            _122_escape_latex(fam),
            _122_fmt(vals.get("estimate", np.nan)),
            _122_ci_text(vals.get("ci_lo", np.nan), vals.get("ci_hi", np.nan)),
            _122_fmt_p(vals.get("p", np.nan)),
        ])

    n_boot = int(_122_float(_122_get(boot, ["n_bootstrap", "n_boot", "n_bootstraps"], globals().get("N_BOOTSTRAP", 0)), default=0))
    n_perm = int(_122_float(_122_get(perm, ["n_permutations", "n_perm"], globals().get("N_PERMUTATIONS", 0)), default=0))

    caption1 = (
        "Primary-4 Tribal Preference Score (TPS), overall and per-family. "
        "Overall 95\\% confidence interval is from prompt-clustered bootstrap"
        + (f" ($n={n_boot}$)" if n_boot else "")
        + "; permutation $p$-values are prompt-level"
        + (f" ($n={n_perm}$)" if n_perm else "")
        + ". Per-family confidence intervals are shown only when available from the saved bootstrap object."
    )

    tex1 = _122_latex_table(
        headers=["", "Estimate", "95\\% CI", "$p$"],
        rows=rows1,
        caption=caption1,
        label="tab:main_tps",
        colspec="lccc",
    )
    atomic_write_text(Path(PATHS["tables"]) / "table1_main_tps.tex", tex1)

    # -----------------------------------------------------------------------
    # Table 2 — Mechanism decomposition (unchanged)
    # -----------------------------------------------------------------------

    dec = _122_read_csv_key(DECOMP_KEY_122)
    rows2 = []

    if not dec.empty:
        adds_map = {
            "M1": "---",
            "M2": "BT quality",
            "M3": "+ logprob",
            "M4": "+ style sim.",
            "M5": "+ length, position",
            "M6": "+ judge/candidate FE",
        }
        for _, r in dec.iterrows():
            if "same_family_coef" not in dec.columns:
                continue
            if pd.isna(r.get("same_family_coef", np.nan)):
                continue
            model_name = str(r.get("model", ""))
            rows2.append([
                _122_escape_latex(model_name),
                _122_escape_latex(adds_map.get(model_name, "")),
                _122_fmt(r.get("same_family_coef", np.nan), digits=3),
                _122_fmt(r.get("same_family_se",   np.nan), digits=3),
                _122_fmt(r.get("same_family_or",   np.nan), digits=3),
                _122_fmt_p(r.get("same_family_p",  np.nan)),
                _122_fmt(r.get("pseudo_r2",        np.nan), digits=3),
            ])

    if not rows2:
        rows2 = [["No valid decomposition rows found", "---", "---", "---", "---", "---", "---"]]

    caption2 = (
        "Mechanism decomposition. The table reports the same-family coefficient "
        "across nested logistic or fractional-logit specifications. Standard errors "
        "are clustered at the prompt level when available."
    )
    tex2 = _122_latex_table(
        headers=["Model", "Adds", "$\\beta$", "SE", "OR", "$p$", "pseudo-$R^2$"],
        rows=rows2,
        caption=caption2,
        label="tab:decomposition",
        colspec="llccccc",
    )
    atomic_write_text(Path(PATHS["tables"]) / "table2_decomposition.tex", tex2)

    # -----------------------------------------------------------------------
    # Table 3 — Robustness summary (unchanged)
    # -----------------------------------------------------------------------

    rows3 = []
    mv = _122_read_csv_key(MULTIVERSE_KEY_122)

    if not mv.empty and "tps" in mv.columns:
        if "candidate_subset" in mv.columns:
            mv_primary = mv[mv["candidate_subset"].astype(str).eq("primary_4")].copy()
        else:
            mv_primary = mv.copy()
        mv_primary["tps"] = pd.to_numeric(mv_primary["tps"], errors="coerce")
        mv_primary = mv_primary[mv_primary["tps"].notna()].copy()
        if len(mv_primary):
            rows3.append([
                "Multiverse: median TPS",
                _122_fmt(mv_primary["tps"].median()),
                f"{(mv_primary['tps'] > 0).mean() * 100:.0f}\\% of valid specs $>0$",
            ])
            rows3.append([
                "Multiverse: range",
                f"[{_122_fmt(mv_primary['tps'].min(), digits=3)}, {_122_fmt(mv_primary['tps'].max(), digits=3)}]",
                "Primary-4 specifications",
            ])

    nc = _122_read_json_key(NEUTRAL_KEY_122)
    cs = _122_read_json_key(CONTAM_KEY_122)
    sc = _122_read_json_key(SCALE_KEY_122)
    co = _122_read_json_key(CONFIRM_KEY_122)

    rows3.extend([
        ["Rubric prompt",       _122_fmt(_122_get_tps_from_nested(nc, ["rubric",        "tps"])), "---"],
        ["Neutral prompt",      _122_fmt(_122_get_tps_from_nested(nc, ["neutral",       "tps"])), "Demand-characteristic check"],
        ["Classic prompts",     _122_fmt(_122_get_tps_from_nested(cs, ["classic",       "tps"])), "Classic benchmark sources"],
        ["Fresh prompts",       _122_fmt(_122_get_tps_from_nested(cs, ["fresh",         "tps"])), "Fresh/non-classic sources"],
        ["Large judges",        _122_fmt(_122_get_tps_from_nested(sc, ["large",         "tps"])), "---"],
        ["Small judges",        _122_fmt(_122_get_tps_from_nested(sc, ["small",         "tps"])), "---"],
        ["Confirmatory holdout",_122_fmt(_122_get_tps_from_nested(co, ["confirmatory",  "tps"])), "Pre-specified split"],
    ])

    caption3 = (
        "Robustness suite for the Primary-4 analysis. The table summarizes sensitivity "
        "to analytic choices, prompt framing, prompt source, judge scale, and holdout split."
    )
    tex3 = _122_latex_table(
        headers=["Test", "TPS", "Note"],
        rows=rows3,
        caption=caption3,
        label="tab:robustness",
        colspec="lll",
    )
    atomic_write_text(Path(PATHS["tables"]) / "table3_robustness.tex", tex3)

    # -----------------------------------------------------------------------
    # Table 4 — Human calibration   *** 2-ANNOTATOR VERSION ***
    # -----------------------------------------------------------------------

    agree = _122_read_json_key(HUMAN_AGREE_KEY_122)
    cons  = _122_read_json_key(HUMAN_CONSENSUS_KEY_122)
    hvllm = _122_read_json_key(HUMAN_VS_LLM_KEY_122)
    hsum  = _122_read_json_key(HUMAN_SUMMARY_KEY_122)

    rows4 = []

    # -- Design rows --
    if hsum:
        design       = hsum.get("design", {})
        n_items      = design.get("n_items", 400)
        # Release-clean fallback: final retained design is 800 judgments and two annotators
        n_judgments  = design.get("n_total_human_judgments", 800)
        n_annotators = design.get("n_human_annotators", 2)
        rows4.append(["Items", str(n_items), "Shared blinded pairwise comparisons"])
        rows4.append([
            "Human judgments",
            str(n_judgments),
            # CHANGED: note is now dynamic from actual annotator count
            f"{n_annotators} annotators $\\times$ 400 items",
        ])
    else:
        # CHANGED: hardcoded fallback is now 2-annotator values
        rows4.append(["Items", "400", "Shared blinded pairwise comparisons"])
        rows4.append(["Human judgments", "800", "2 annotators $\\times$ 400 items"])

    # -- Agreement rows --
    if agree:
        # CHANGED: Fleiss' κ → Cohen's κ (correct metric for 2 raters)
        # Cohen's κ key is "cohen_kappa" in the 2-annotator agreement JSON.
        # Fleiss' κ is now None in the JSON, so we skip it.
        cohen_k = agree.get("cohen_kappa", np.nan)
        # Guard: if cohen_kappa is explicitly None (as written by Cell 10.4), treat as nan
        if cohen_k is None:
            cohen_k = np.nan
        rows4.append([
            "Cohen's $\\kappa$",
            _122_fmt(cohen_k, digits=3),
            "Inter-annotator agreement (2 retained annotators)",
        ])
        rows4.append([
            "Krippendorff's $\\alpha$",
            _122_fmt(agree.get("krippendorff_alpha_nominal", np.nan), digits=3),
            "Nominal labels",
        ])
        rows4.append([
            "Gwet AC1",
            _122_fmt(agree.get("gwet_ac1_nominal", np.nan), digits=3),
            "Nominal agreement",
        ])

    # -- Consensus rows --
    if cons:
        # CHANGED: note text updated for 2-annotator language
        rows4.append([
            "Consensus rows",
            str(cons.get("main_analysis_n_majority_rows", "---")),
            "Both annotators agreed",              # CHANGED from "2-of-3 or 3-of-3 agreement"
        ])
        rows4.append([
            "No-consensus rows",
            str(cons.get("main_analysis_n_no_majority_rows", "---")),
            "Annotators disagreed",                # Release-clean wording: retained annotators disagreed
        ])

    # -- Human vs LLM rows --
    if hvllm:
        main_hv = hvllm.get("main_excluding_no_majority", {})
        sens_hv = hvllm.get("sensitivity_no_majority_as_tie", {})
        rows4.append([
            "Human vs.\\ LLM panel",
            _122_fmt(main_hv.get("exact_match", np.nan), digits=3),
            "Exact match, no-consensus rows excluded",   # CHANGED "no-majority" → "no-consensus"
        ])
        rows4.append([
            "Sensitivity",
            _122_fmt(sens_hv.get("exact_match", np.nan), digits=3),
            "No-consensus rows treated as Tie",          # CHANGED
        ])

    # CHANGED: caption reflects 2-annotator design and exclusion
    caption4 = (
        "Human calibration summary. Two annotators were retained after a third was "
        "excluded for systematically low agreement. Main analyses use rows where both "
        "annotators agreed; no-consensus rows are handled separately."
    )

    tex4 = _122_latex_table(
        headers=["Quantity", "Value", "Note"],
        rows=rows4,
        caption=caption4,
        label="tab:human_calibration",
        colspec="lll",
    )
    atomic_write_text(Path(PATHS["tables"]) / "table4_human_calibration.tex", tex4)

    # -----------------------------------------------------------------------
    # Log
    # -----------------------------------------------------------------------

    print("\nWritten LaTeX tables")
    print("-" * 100)
    for name in [
        "table1_main_tps.tex",
        "table2_decomposition.tex",
        "table3_robustness.tex",
        "table4_human_calibration.tex",
    ]:
        print(Path(PATHS["tables"]) / name)

    print("\nKey loaded files")
    print("-" * 100)
    print(json.dumps({
        "bootstrap_key":      BOOT_KEY_122,
        "permutation_key":    PERM_KEY_122,
        "per_family_bh_key":  BH_KEY_122,
        "decomp_key":         DECOMP_KEY_122,
        "multiverse_key":     MULTIVERSE_KEY_122,
        "human_agreement_key":   HUMAN_AGREE_KEY_122,
        "human_consensus_key":   HUMAN_CONSENSUS_KEY_122,
        "human_vs_llm_key":      HUMAN_VS_LLM_KEY_122,
        "human_summary_key":     HUMAN_SUMMARY_KEY_122,
    }, indent=2))


TABLES_SENTINEL = Path(PATHS["tables"]) / ".phase12_tables.complete"


def _wrap_tables_122():
    _step_latex_tables_122()
    atomic_write_text(TABLES_SENTINEL, datetime.now(timezone.utc).isoformat())


run_step(
    [TABLES_SENTINEL],
    _wrap_tables_122,
    "latex_tables_primary4_human_calibrated",
    force=True,
    validators={TABLES_SENTINEL: file_nonempty},
)

print("\n✓ Cell 12.2 LaTeX tables complete")

In [ ]:
# ============================================================================
# Cell 12.3 — HuggingFace dataset packaging
# ============================================================================
"""
Cell 12.3 — Package responses + judgments + trial master into a HF-format
dataset directory. Includes a dataset card stub. The user can then `huggingface
-cli upload` from PATHS['release']/hf_dataset.
"""
def _step_hf_dataset():
    out_dir = FILES["hf_dataset_dir"]
    out_dir.mkdir(parents=True, exist_ok=True)

    # Three subsets: prompts, responses, judgments
    prompts = pd.read_csv(FILES["master_prompts"])
    split = pd.read_csv(FILES["prompt_split"])[["prompt_id", "split"]]
    prompts = prompts.merge(split, on="prompt_id", how="left")
    prompts.to_parquet(out_dir / "prompts.parquet", index=False)

    responses = pd.read_csv(FILES["all_responses"])
    responses.to_parquet(out_dir / "responses.parquet", index=False)

    judgments_main = pd.read_csv(FILES["master_judgments"])
    judgments_main.to_parquet(out_dir / "judgments_rubric.parquet", index=False)

    if FILES["master_judgments_neu"].exists():
        pd.read_csv(FILES["master_judgments_neu"]).to_parquet(
            out_dir / "judgments_neutral.parquet", index=False)
    if FILES["master_judgments_sm"].exists():
        pd.read_csv(FILES["master_judgments_sm"]).to_parquet(
            out_dir / "judgments_small.parquet", index=False)

    eff = pd.read_csv(FILES["effective_winners_primary"])
    eff.to_parquet(out_dir / "effective_winners_primary.parquet", index=False)
    if Path(FILES["effective_winners_full"]).exists():
        pd.read_csv(FILES["effective_winners_full"]).to_parquet(out_dir / "effective_winners_full_appendix.parquet", index=False)

    # Dataset card
    card = (f"""---
license: cc-by-4.0
language: en
tags:
- llm-as-a-judge
- preference-evaluation
- bias
- self-preference
- preference-leakage
size_categories:
- 10K<n<100K
---

# Tribal Preference Study V15 — Dataset

A controlled, fully-crossed pairwise judgment dataset for studying tribal /
self-preference bias in open-source LLM-as-judge.

## Composition
- **Prompts:** {len(prompts)} (MT-Bench {TARGET_MTB} + AlpacaEval {TARGET_ALPACA} + WildBench {TARGET_WB})
- **Candidate models:** {len(FAMILIES)} families × 2 scales (large 27–72B + small 7–9B)
- **Judge models:** same {len(FAMILIES)} families
- **Trials:** {len(judgments_main):,} (rubric prompt) + neutral and small-scale variants
- **Pre-registered split:** {int(100 * (1 - CONFIRMATORY_FRAC))}/{int(100 * CONFIRMATORY_FRAC)} exploratory/confirmatory

## Files
- `prompts.parquet` — prompt set with stable IDs, source, category, split
- `responses.parquet` — all candidate responses (large, small, quant ablation)
- `judgments_rubric.parquet` — primary judgments under the rubric system prompt
- `judgments_neutral.parquet` — robustness ablation under neutral system prompt
- `judgments_small.parquet` — small-scale (7-9B) judges, scale comparison
- `effective_winners_primary.parquet` — AB/BA reconciled weighted winners for the Primary-4 main analysis
- `effective_winners_full_appendix.parquet` — full 5-family appendix sensitivity with Falcon retained as candidate

## Schema (judgments)
trial_id, prompt_id, prompt, source, category, split, pair_id,
family_1, family_2, order ∈ {{AB, BA}}, judge_family,
model_a, model_b, response_a, response_b, tokens_a, tokens_b,
winner ∈ {{A, B, tie}}, rationale, raw_judgment, parse_ok,
logprob_a, logprob_b, logprob_delta, judged_at

## Citation
TODO: paper citation once accepted.

## License
CC-BY-4.0. See DATA_STATEMENT.md and MODEL_CARD.md in the release folder.
""")
    atomic_write_text(out_dir / "README.md", card)
    log.info(f"  HF dataset packaged at {out_dir}")

run_step([FILES["hf_dataset_dir"] / "README.md"], _step_hf_dataset,
         "hf_dataset_packaging_primary4",
         force=True,
         validators={FILES["hf_dataset_dir"] / "README.md":
                     lambda p: file_nonempty(p, 200)})




In [ ]:
# ============================================================================
# Cell 12.4 — MODEL_CARD.md and DATA_STATEMENT.md, Primary-4 + human calibration
# ============================================================================
"""
Cell 12.4 — Model card and data statement.

Release-clean documentation behavior:
  - MODEL_CARD and DATA_STATEMENT read human-calibration counts dynamically from the final summary where possible.
  - Fallback wording uses two retained annotators and 800 final human judgments.
  - Consensus wording uses both-agree language rather than majority-vote language.
"""

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np


# ---------------------------------------------------------------------------
# Required globals
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_text",
    "file_nonempty",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "release" not in PATHS:
    if "root" in PATHS:
        PATHS["release"] = Path(PATHS["root"]) / "release"
    elif "analysis" in PATHS:
        PATHS["release"] = Path(PATHS["analysis"]) / "release"
    elif "ROOT_DIR" in globals():
        PATHS["release"] = Path(ROOT_DIR) / "release"
    else:
        PATHS["release"] = Path("/content/drive/MyDrive/tribal_pref_v11/release")

Path(PATHS["release"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault("model_card",    Path(PATHS["release"]) / "MODEL_CARD.md")
FILES.setdefault("data_statement", Path(PATHS["release"]) / "DATA_STATEMENT.md")


# ---------------------------------------------------------------------------
# Family/model helpers  (unchanged)
# ---------------------------------------------------------------------------

def _124_clean_family(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if "llama"   in s: return "llama"
    if "qwen"    in s: return "qwen"
    if "gemma"   in s: return "gemma"
    if "falcon"  in s: return "falcon"
    if s == "yi" or "yi-" in s or "yi_" in s or "/yi" in s: return "yi"
    if "mistral" in s or "mixtral" in s: return "mistral"
    return s


def _124_registry_display(scale, family):
    if "MODEL_REGISTRY" not in globals():
        return "not available"
    try:
        item = MODEL_REGISTRY.get(scale, {}).get(family, None)
        if item is None and family == "mistral":
            item = MODEL_REGISTRY.get(scale, {}).get("mixtral", None)
        if item is None:
            return "not used"
        if isinstance(item, dict):
            return str(item.get("display", item.get("name", item.get("model", item.get("repo", "not specified")))))
        return str(item)
    except Exception:
        return "not used"


def _124_available_families_from_registry():
    fams = set()
    if "MODEL_REGISTRY" in globals() and isinstance(MODEL_REGISTRY, dict):
        for scale in ["large", "small"]:
            if scale in MODEL_REGISTRY and isinstance(MODEL_REGISTRY[scale], dict):
                fams.update([_124_clean_family(k) for k in MODEL_REGISTRY[scale].keys()])
    for gvar in ["FULL_CANDIDATE_FAMILIES", "PRIMARY_CANDIDATE_FAMILIES",
                 "PRIMARY_JUDGE_FAMILIES", "FAMILIES"]:
        if gvar in globals():
            fams.update([_124_clean_family(x) for x in globals()[gvar]])
    fams = {f for f in fams if f}
    preferred_order = ["llama", "qwen", "gemma", "yi", "falcon", "mistral"]
    ordered = [f for f in preferred_order if f in fams]
    ordered += sorted([f for f in fams if f not in ordered])
    return ordered


def _124_primary_families():
    if "PRIMARY_CANDIDATE_FAMILIES" in globals():
        return [_124_clean_family(x) for x in PRIMARY_CANDIDATE_FAMILIES]
    return ["llama", "qwen", "gemma", "yi"]


def _124_primary_judges():
    if "PRIMARY_JUDGE_FAMILIES" in globals():
        return [_124_clean_family(x) for x in PRIMARY_JUDGE_FAMILIES]
    return _124_primary_families()


def _124_full_families():
    if "FULL_CANDIDATE_FAMILIES" in globals():
        return [_124_clean_family(x) for x in FULL_CANDIDATE_FAMILIES]
    if "FAMILIES" in globals():
        return [_124_clean_family(x) for x in FAMILIES]
    return _124_available_families_from_registry()


def _124_model_table_md():
    fams = _124_available_families_from_registry() or _124_primary_families()
    lines = [
        "| Family | Large model | Small model | Headline role |",
        "|---|---|---|---|",
    ]
    primary        = set(_124_primary_families())
    primary_judges = set(_124_primary_judges())
    for fam in fams:
        large = _124_registry_display("large", fam)
        small = _124_registry_display("small", fam)
        if fam in primary and fam in primary_judges:
            role = "Primary-4 headline judge/candidate family"
        elif fam in primary:
            role = "Primary-4 headline candidate family"
        elif fam == "falcon":
            role = "Diagnostic / sensitivity only"
        else:
            role = "Available only if present in sensitivity setup"
        lines.append(f"| {fam} | {large} | {small} | {role} |")
    return "\n".join(lines)


def _124_file_exists(key):
    return key in FILES and Path(FILES[key]).exists()


def _124_json_if_exists(key):
    if not _124_file_exists(key):
        return None
    try:
        with open(FILES[key], "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _124_human_summary_text():
    """
    Read n_items, n_annotators, n_judgments from human_calibration_summary JSON.
    Release-clean fallback values use the final retained two-annotator design.
    """
    hsum = _124_json_if_exists("human_calibration_summary")

    if isinstance(hsum, dict):
        design = hsum.get("design", {})
        n_items      = int(design.get("n_items", 400))
        # Release-clean default: final retained annotators
        n_annotators = int(design.get("n_human_annotators", 2))
        # Release-clean default: final retained judgment count
        n_judgments  = int(design.get("n_total_human_judgments", n_items * n_annotators))
    else:
        # Release-clean fallback: two retained annotators and 800 final judgments
        n_items      = 400
        n_annotators = 2
        n_judgments  = 800

    return n_items, n_annotators, n_judgments


def _124_safe_global(name, default):
    return globals().get(name, default)


def _124_to_text(x):
    if isinstance(x, (list, tuple, set)):
        return ", ".join([str(i) for i in x])
    return str(x)


# ---------------------------------------------------------------------------
# MODEL_CARD.md  (human calibration block updated dynamically)
# ---------------------------------------------------------------------------

def _step_model_card_124():
    primary        = _124_primary_families()
    primary_judges = _124_primary_judges()
    full           = _124_full_families()
    excluded       = sorted([f for f in full if f not in primary])

    n_items, n_annotators, n_judgments = _124_human_summary_text()

    gen_max_tokens = _124_safe_global("GEN_MAX_TOKENS",   "not specified")
    gen_temp       = _124_safe_global("GEN_TEMPERATURE",  "not specified")
    gen_top_p      = _124_safe_global("GEN_TOP_P",        "not specified")
    gen_repeat     = _124_safe_global("GEN_REPEAT_PEN",   "not specified")
    judge_max_tokens = _124_safe_global("JUDGE_MAX_TOKENS",  "not specified")
    judge_temp     = _124_safe_global("JUDGE_TEMPERATURE","not specified")
    judge_top_p    = _124_safe_global("JUDGE_TOP_P",      "not specified")
    judge_context  = _124_safe_global("JUDGE_CONTEXT",    "not specified")
    random_seed    = _124_safe_global("RANDOM_SEED",      "not specified")
    created_at     = datetime.now(timezone.utc).isoformat()

    # CHANGED: human calibration block uses dynamic n_annotators / n_judgments
    # (was hardcoded prose; now reads from summary so it automatically says
    #  "2 independent human annotators" and "800 total human judgments")
    text = f"""# Model Card — Tribal Preference Study V15

Generated: {created_at}

## Model details

This is not a single deployed model card. It documents the open-weight model panel used in the study.

The final headline analysis is the **Primary-4** panel:

{", ".join(primary)}

The primary judge families are:

{", ".join(primary_judges)}

The full model registry may include additional diagnostic or sensitivity families. In the final paper framing, any family outside Primary-4 is not part of the headline claim.

## Candidate / judge panel

{_124_model_table_md()}

## Headline analysis scope

- Headline candidate families: {", ".join(primary)}
- Headline judge families: {", ".join(primary_judges)}
- Full/sensitivity families available: {", ".join(full)}
- Excluded from headline: {", ".join(excluded) if excluded else "none"}

Falcon, if present, is treated as a diagnostic or sensitivity case and not as part of the Primary-4 headline. This avoids letting artifact-prone generations or unstable judge formatting contaminate the main claim.

## Intended use

This artifact is intended for:

- methodological research on LLM-as-judge evaluation
- auditing family-conditioned preference in model evaluation
- studying evaluator identity effects in open-weight LLM panels
- reproducing the Tribal Preference Score and associated robustness checks
- extending the audit framework to additional model families or closed-weight systems

## Out-of-scope use

This artifact should not be used for:

- production model ranking decisions without additional validation
- claims about all LLMs or all model families
- claims about closed-source models unless independently audited
- claims about non-English evaluation behavior
- safety certification of models

## Main study design

The study evaluates whether LLM judges show family-conditioned preference when comparing model responses.

The primary analysis uses a signed preference matrix and a Tribal Preference Score (TPS). The paper should frame this as an audit of family-conditioned preference in an open-weight LLM-as-judge setting, not as a universal claim that all LLMs are tribal.

## Human calibration

Human calibration was completed with:

- {n_items} shared blinded pairwise comparisons
- {n_annotators} independent human annotators ({n_judgments} total human judgments)
- A/B/Tie response-quality labels
- confidence ratings from 1 to 5
- a third annotator excluded prior to analysis due to systematically low agreement

Model identities, model families, LLM-judge winners, and hidden key information were not shown to annotators.

## Generation settings

Candidate response generation:

- max_tokens: {gen_max_tokens}
- temperature: {gen_temp}
- top_p: {gen_top_p}
- repeat_penalty: {gen_repeat}

Judge generation:

- max_tokens: {judge_max_tokens}
- temperature: {judge_temp}
- top_p: {judge_top_p}
- n_ctx: {judge_context}

Random seed: {random_seed}

## Compute

The study was run in a GPU-backed notebook environment using local/open-weight model inference where possible. Exact runtime and hardware details should be reported from the final audit logs if required.

## Ethical considerations

- The study uses open-weight models and public benchmark prompts.
- Human annotation files contain no model-family metadata.
- Human annotation should be reported honestly as human-only.
- Any external proprietary model judge, if added later, must be reported separately and not treated as human annotation.
- The paper should avoid overstating the generality of the effect beyond the audited setting.

## Caveats

- The headline result is limited to Primary-4 open-weight model families.
- Human agreement may be modest because pairwise response-quality evaluation is subjective.
- Falcon-related findings, if discussed, should be framed as artifact diagnostics or sensitivity analysis.
- Closed-source generalization requires a separate audit.
- The mechanism decomposition is suggestive, not a complete causal explanation.
"""

    atomic_write_text(FILES["model_card"], text)
    print("MODEL_CARD.md written:")
    print(FILES["model_card"])


# ---------------------------------------------------------------------------
# DATA_STATEMENT.md  (annotator counts and phrase updated)
# ---------------------------------------------------------------------------

def _step_data_statement_124():
    primary = _124_primary_families()
    n_items, n_annotators, n_judgments = _124_human_summary_text()

    target_total        = _124_safe_global("TARGET_TOTAL",        "not specified")
    target_mtb          = _124_safe_global("TARGET_MTB",          "not specified")
    target_alpaca       = _124_safe_global("TARGET_ALPACA",       "not specified")
    target_wb           = _124_safe_global("TARGET_WB",           "not specified")
    random_seed         = _124_safe_global("RANDOM_SEED",         "not specified")
    confirmatory_frac   = _124_safe_global("CONFIRMATORY_FRAC",   "not specified")
    presplit_seed       = _124_safe_global("PRESPLIT_SEED",        "not specified")

    if isinstance(confirmatory_frac, float):
        confirmatory_frac_text = f"{int(100 * confirmatory_frac)}%"
    else:
        confirmatory_frac_text = str(confirmatory_frac)

    created_at = datetime.now(timezone.utc).isoformat()

    # CHANGED:
    # Human-calibration documentation uses the final retained-annotator design.
    # Counts are read dynamically where possible and fall back to 800 final human judgments.
    # Consensus wording uses both-agree language rather than majority-vote language.
    text = f"""# Data Statement — Tribal Preference Study V15

Generated: {created_at}

This data statement follows the spirit of Bender & Friedman (2018) for documenting dataset provenance, language, annotators, and intended use.

## A. Curation rationale

The dataset was assembled to study family-conditioned preference in LLM-as-judge evaluation.

The prompt pool contains approximately {target_total} prompts spanning:

- MT-Bench: {target_mtb}
- AlpacaEval: {target_alpaca}
- WildBench: {target_wb}

These sources were selected because they cover a mixture of instruction-following, reasoning, writing, coding, extraction, and conversational tasks.

## B. Language variety

The prompt set is English.

The study does not claim generalization to other languages. Multilingual or Ghanaian-language extensions would require a separate audit.

## C. Speaker / writer demographic

The source prompts come from public benchmark datasets and user-style instruction corpora. The original author demographics are not fully available.

## D. Human annotation

Human calibration was completed using:

- {n_items} shared blinded pairwise comparisons
- {n_annotators} retained human annotators
- {n_judgments} total human judgments

A third annotator was excluded prior to analysis due to systematically low inter-annotator agreement with both retained annotators and the absence of annotation notes, indicating a decision process inconsistent with the task protocol.

Annotators evaluated a prompt, Response A, and Response B. They selected A, B, or Tie and provided a confidence score from 1 to 5.

Model identities, model families, LLM-judge labels, and same-family status were hidden from annotators.

## E. Speech situation

The text is written, asynchronous, instruction-following or conversational user-assistant interaction.

## F. Text characteristics

The data includes prompts and model responses covering:

- factual question answering
- reasoning
- writing
- summarization
- coding
- extraction
- roleplay or open-ended instruction-following
- safety-sensitive or ambiguity-heavy comparisons

## G. Recording quality

The data is born-digital text. No audio or speech transcription is involved.

## H. Model-response provenance

Responses were generated by open-weight model families in the study panel.

The Primary-4 headline analysis uses:

{", ".join(primary)}

Other families, if present in artifacts, are used only for diagnostic or sensitivity analysis.

## I. Human-label provenance

The final human calibration file contains completed labels from {n_annotators} retained human annotators.

The analysis normalizes labels to:

- A
- B
- Tie

Rows where both annotators agreed are used for the main human-calibration analysis. Rows where annotators disagreed are handled explicitly and may be treated as Tie in a sensitivity analysis.

## J. Licensing and release

The derived artifacts should be released with a clear license, while respecting the original licenses of MT-Bench, AlpacaEval, and WildBench.

The release should include:

- prompt IDs
- anonymized prompts
- model response identifiers
- family labels used in analysis
- judge outputs
- preference matrices
- TPS summaries
- robustness outputs
- human annotation summaries

Do not release personally identifying information about human annotators.

## K. Reproducibility

Random seed: {random_seed}

Confirmatory split fraction: {confirmatory_frac_text}

Pre-split seed: {presplit_seed}

The notebook and release artifacts should document exact paths, cached outputs, and final audit checks.
"""

    atomic_write_text(FILES["data_statement"], text)
    print("DATA_STATEMENT.md written:")
    print(FILES["data_statement"])


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

run_step(
    [FILES["model_card"]],
    _step_model_card_124,
    "model_card_primary4_human_calibrated",
    force=True,
    validators={FILES["model_card"]: lambda p: file_nonempty(p, 500)},
)

run_step(
    [FILES["data_statement"]],
    _step_data_statement_124,
    "data_statement_primary4_human_calibrated",
    force=True,
    validators={FILES["data_statement"]: lambda p: file_nonempty(p, 500)},
)

print("\n✓ Cell 12.4 MODEL_CARD.md and DATA_STATEMENT.md complete")

In [ ]:
# ============================================================================
# Cell 12.5 — REPRODUCE.md, Primary-4 + human calibration
# ============================================================================
"""
Cell 12.5 — One-document reproduction guide.

This version is robust to the stable Drive storage setup:
- No hard-coded mixtral.
- Builds required model lists dynamically from MODEL_REGISTRY.
- Documents Primary-4 headline.
- Documents completed human calibration workflow.
- Documents the correct post-human run order.
"""

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import numpy as np


# ---------------------------------------------------------------------------
# Required globals
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_text",
    "file_nonempty",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------

if "release" not in PATHS:
    if "root" in PATHS:
        PATHS["release"] = Path(PATHS["root"]) / "release"
    elif "analysis" in PATHS:
        PATHS["release"] = Path(PATHS["analysis"]) / "release"
    elif "ROOT_DIR" in globals():
        PATHS["release"] = Path(ROOT_DIR) / "release"
    else:
        PATHS["release"] = Path("/content/drive/MyDrive/tribal_pref_v11/release")

Path(PATHS["release"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "reproduce_readme",
    Path(PATHS["release"]) / "REPRODUCE.md",
)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _125_clean_family(x):
    if pd.isna(x):
        return ""

    s = str(x).strip().lower()

    if "llama" in s:
        return "llama"
    if "qwen" in s:
        return "qwen"
    if "gemma" in s:
        return "gemma"
    if "falcon" in s:
        return "falcon"
    if s == "yi" or "yi-" in s or "yi_" in s or "/yi" in s:
        return "yi"
    if "mistral" in s or "mixtral" in s:
        return "mistral"

    return s


def _125_safe_global(name, default):
    return globals().get(name, default)


def _125_primary_families():
    if "PRIMARY_CANDIDATE_FAMILIES" in globals():
        return [_125_clean_family(x) for x in PRIMARY_CANDIDATE_FAMILIES]

    return ["llama", "qwen", "gemma", "yi"]


def _125_primary_judges():
    if "PRIMARY_JUDGE_FAMILIES" in globals():
        return [_125_clean_family(x) for x in PRIMARY_JUDGE_FAMILIES]

    return _125_primary_families()


def _125_full_families():
    if "FULL_CANDIDATE_FAMILIES" in globals():
        return [_125_clean_family(x) for x in FULL_CANDIDATE_FAMILIES]

    if "FAMILIES" in globals():
        return [_125_clean_family(x) for x in FAMILIES]

    fams = set()

    if "MODEL_REGISTRY" in globals() and isinstance(MODEL_REGISTRY, dict):
        for scale in MODEL_REGISTRY:
            if isinstance(MODEL_REGISTRY[scale], dict):
                fams.update([_125_clean_family(k) for k in MODEL_REGISTRY[scale].keys()])

    if not fams:
        fams = set(_125_primary_families())

    preferred = ["llama", "qwen", "gemma", "yi", "falcon", "mistral"]
    ordered = [f for f in preferred if f in fams]
    ordered += sorted([f for f in fams if f not in ordered])

    return ordered


def _125_model_spec_line(spec):
    if isinstance(spec, dict):
        display = str(
            spec.get(
                "display",
                spec.get(
                    "name",
                    spec.get(
                        "model",
                        spec.get("repo", "unnamed model"),
                    ),
                ),
            )
        )

        candidates = spec.get("candidates", None)

        if isinstance(candidates, (list, tuple)):
            candidate_text = ", ".join([str(x) for x in candidates])
        elif candidates:
            candidate_text = str(candidates)
        else:
            candidate_text = "not specified"

        return f"- {display}  → candidates: {candidate_text}"

    return f"- {str(spec)}"


def _125_model_section(scale):
    if "MODEL_REGISTRY" not in globals():
        return "- MODEL_REGISTRY not available in this runtime."

    if scale not in MODEL_REGISTRY or not isinstance(MODEL_REGISTRY[scale], dict):
        return f"- No `{scale}` registry entries found."

    lines = []

    preferred = ["llama", "qwen", "gemma", "yi", "falcon", "mistral", "mixtral"]
    keys = list(MODEL_REGISTRY[scale].keys())

    def clean_key(k):
        return _125_clean_family(k)

    ordered_keys = []

    for fam in preferred:
        for k in keys:
            if k not in ordered_keys and clean_key(k) == fam:
                ordered_keys.append(k)

    ordered_keys += [k for k in keys if k not in ordered_keys]

    for k in ordered_keys:
        spec = MODEL_REGISTRY[scale][k]
        fam = _125_clean_family(k)
        line = _125_model_spec_line(spec)
        lines.append(f"{line}  [family: {fam}]")

    return "\n".join(lines)


def _125_human_summary_text():
    hsum_path = FILES.get("human_calibration_summary", None)

    if hsum_path is not None and Path(hsum_path).exists():
        try:
            with open(hsum_path, "r", encoding="utf-8") as f:
                hsum = json.load(f)

            design = hsum.get("design", {})
            n_items = int(design.get("n_items", 400))
            n_annotators = int(design.get("n_human_annotators", 2))
            n_judgments = int(design.get("n_total_human_judgments", n_items * n_annotators))

            return n_items, n_annotators, n_judgments

        except Exception:
            pass

    return 400, 2, 800


def _125_path_text(key, fallback="not defined"):
    if key in PATHS:
        return str(PATHS[key])
    return fallback


def _125_file_text(key, fallback="not defined"):
    if key in FILES:
        return str(FILES[key])
    return fallback


# ---------------------------------------------------------------------------
# Main step
# ---------------------------------------------------------------------------

def _step_reproduce_125():
    primary = _125_primary_families()
    primary_judges = _125_primary_judges()
    full = _125_full_families()
    excluded = sorted([f for f in full if f not in primary])

    n_items, n_annotators, n_judgments = _125_human_summary_text()

    created_at = datetime.now(timezone.utc).isoformat()

    models_drive = _125_path_text(
        "models_drive",
        "/content/drive/MyDrive/tribal_pref_v11/models",
    )

    root_path = _125_path_text(
        "root",
        "/content/drive/MyDrive/tribal_pref_v11",
    )

    analysis_human = _125_path_text(
        "analysis_human",
        "/content/drive/MyDrive/tribal_pref_v11/analysis/human_calibration",
    )

    random_seed = _125_safe_global("RANDOM_SEED", "not specified")
    target_total = _125_safe_global("TARGET_TOTAL", "not specified")
    target_mtb = _125_safe_global("TARGET_MTB", "not specified")
    target_alpaca = _125_safe_global("TARGET_ALPACA", "not specified")
    target_wb = _125_safe_global("TARGET_WB", "not specified")

    gen_max_tokens = _125_safe_global("GEN_MAX_TOKENS", "not specified")
    gen_temp = _125_safe_global("GEN_TEMPERATURE", "not specified")
    gen_top_p = _125_safe_global("GEN_TOP_P", "not specified")

    judge_max_tokens = _125_safe_global("JUDGE_MAX_TOKENS", "not specified")
    judge_temp = _125_safe_global("JUDGE_TEMPERATURE", "not specified")
    judge_top_p = _125_safe_global("JUDGE_TOP_P", "not specified")
    judge_context = _125_safe_global("JUDGE_CONTEXT", "not specified")

    text = f"""# Reproduction Guide — Tribal Preference Study V15

Generated: {created_at}

## TL;DR

1. Open the notebook in Google Colab or a comparable GPU-backed environment.
2. Mount Google Drive.
3. Confirm the project root exists:

   `{root_path}`

4. Place GGUF model files under:

   `{models_drive}/large/`
   `{models_drive}/small/`
   `{models_drive}/quant_ablation/`

5. Run the notebook from the beginning if reproducing everything.
6. If the expensive model-generation and judging artifacts already exist, resume from the later analysis cells. The pipeline is designed to be cache/checkpoint friendly.
7. For the completed human-calibrated version, run Phase 10.3 onward after placing the completed workbook in the human-calibration folder.

## Final study scope

The final headline analysis is **Primary-4**.

Headline candidate families:

{", ".join(primary)}

Headline judge families:

{", ".join(primary_judges)}

Full/sensitivity families available in the broader project:

{", ".join(full)}

Excluded from headline:

{", ".join(excluded) if excluded else "none"}

Falcon, if present, is not part of the Primary-4 headline result. It should be treated only as a diagnostic or sensitivity case.

## Human calibration scope

The completed human calibration uses:

- {n_items} shared blinded pairwise comparisons
- {n_annotators} independent human annotators
- {n_judgments} total human judgments
- A/B/Tie labels
- confidence scores from 1 to 5

Completed workbook expected at:

`{analysis_human}/annotator_sheets_completed_primary4_400.xlsx`

Hidden key expected at:

`{analysis_human}/human_annotation_key_hidden_primary4_400.csv`

The completed workbook must contain three sheets:

- `annotator_1`
- `annotator_2`
- `annotator_3`

Each sheet must contain 400 rows and the columns:

- `row_number`
- `annotation_id`
- `annotator_id`
- `prompt`
- `response_A`
- `response_B`
- `human_choice`
- `confidence_1_to_5`
- `notes_optional`

The current import cell normalizes:

- `A1` or `A` → `A`
- `B1` or `B` → `B`
- `Tie` → `Tie`

## Hardware

Recommended for full reproduction:

- GPU: 80 GB VRAM or higher for comfortable local inference with large GGUF models
- RAM: 64 GB minimum; 120 GB preferred
- Disk: enough space for GGUF model files, cached responses, judgments, and release artifacts
- Google Drive or equivalent persistent storage for checkpoints

For analysis-only reproduction:

- Use GPU for the full model-generation and LLM-judgment reproduction path
- Phases 6–12 are mostly pandas/statistics/file-output steps and can reuse cached outputs without loading models

## Software

Recommended packages:

- Python 3.10+
- pandas
- numpy
- scipy
- scikit-learn
- statsmodels
- sentence-transformers
- datasets
- huggingface_hub
- llama-cpp-python
- openpyxl
- matplotlib
- krippendorff, optional
- irrCAC, optional

Important note:

- GPU/CUDA is needed only for local model inference.
- Human calibration import and final publication artifacts do not require GPU.

## Required GGUF files

The notebook uses `MODEL_REGISTRY` to locate model files. Put matching GGUF files under the relevant folder.

### Large models

Folder:

`{models_drive}/large/`

{_125_model_section("large")}

### Small models

Folder:

`{models_drive}/small/`

{_125_model_section("small")}

### Quantization ablation models

Folder:

`{models_drive}/quant_ablation/`

{_125_model_section("quant_ablation")}

## Key generation settings

Candidate generation:

- max_tokens: {gen_max_tokens}
- temperature: {gen_temp}
- top_p: {gen_top_p}

Judge generation:

- max_tokens: {judge_max_tokens}
- temperature: {judge_temp}
- top_p: {judge_top_p}
- context window: {judge_context}

Random seed:

`{random_seed}`

## Prompt-source targets

Approximate prompt target counts:

- total: {target_total}
- MT-Bench: {target_mtb}
- AlpacaEval: {target_alpaca}
- WildBench: {target_wb}

The exact counts should be verified from the generated prompt-master and audit files.

## Pipeline phases

### Phase 1 — Bootstrap and configuration

Sets up:

- paths
- random seeds
- logging
- model registry
- `FILES`
- `PATHS`
- `run_step`
- atomic write helpers
- cache/checkpoint behavior

### Phase 2 — Data pipeline

Builds prompt sources and the prompt master.

Expected work:

- fetch or load MT-Bench
- fetch or load AlpacaEval
- fetch or load WildBench
- normalize prompts
- create stable prompt IDs
- create confirmatory/exploratory split

### Phase 3 — Model loading and response generation

Generates candidate model responses.

Expected work:

- stage/load GGUF model files
- generate responses for large models
- generate responses for small models
- generate quantization-ablation responses
- run artifact scans and quality checks
- cache generated responses

### Phase 4 — Trial construction

Builds pairwise comparison trials.

Expected work:

- construct A/B candidate pairs
- create stable trial IDs
- create AB/BA ordering
- verify candidate-family coverage
- save trial master

### Phase 5 — LLM judgment collection

Collects judge decisions.

Expected work:

- rubric prompt judgments
- neutral prompt judgments
- small-judge judgments
- logprob caching if available
- judge-output parsing
- judge-output artifact diagnostics

### Phase 6 — Effective winners and preference matrices

Builds clean Primary-4 and Full-5 analysis artifacts.

Expected work:

- reconcile AB/BA judgments
- create effective winners
- create Primary-4 preference matrix
- create Full-5 appendix/sensitivity matrix
- alias headline files to Primary-4 explicitly

### Phase 7 — Core inference

Computes the headline statistics.

Expected work:

- TPS summary
- cluster bootstrap
- prompt-level permutation
- per-family BH correction
- Bradley-Terry residual/quality controls
- technical verification of saved outputs

### Phase 8 — Mechanism decomposition

Runs mechanism and robustness regressions.

Expected work:

- style similarity matrix
- regression feature construction
- nested decomposition models
- quasi-binomial/fractional-logit/GEE-style robustness
- separation diagnostics
- optional mixed model

### Phase 9 — Robustness suite

Runs sensitivity analyses.

Expected work:

- multiverse with candidate-subset axis
- neutral-vs-rubric prompt comparison
- contamination/freshness split
- confirmatory holdout
- judge-scale comparison
- quantization ablation
- Falcon diagnostic/sensitivity analysis
- ranking/leaderboard impact simulation
- final technical audit

### Phase 10 — Human calibration

For a fresh annotation run:

1. Run Cell 10.1 to build the 400-row blind annotation sample.
2. Run Cell 10.2 to export annotator-specific files.
3. Send files to human annotators.

For the completed annotation run:

1. Place the completed workbook at:

   `{analysis_human}/annotator_sheets_completed_primary4_400.xlsx`

2. Do not rerun 10.1 or 10.2 unless regenerating the sample.
3. Run:

   - 10.3 — import completed workbook
   - 10.4 — inter-human agreement
   - 10.5 — human majority consensus
   - 10.6 — human vs LLM panel comparison
   - 10.7 — final human calibration summary

### Phase 11 — Final report bundle

Builds the final machine-readable summary:

- headline design
- Primary-4 scope
- robustness outputs
- human calibration outputs
- interpretation guardrails
- paper framing

### Phase 12 — Publication artifacts

Builds:

- figures
- LaTeX tables
- Hugging Face style dataset artifacts
- `MODEL_CARD.md`
- `DATA_STATEMENT.md`
- `REPRODUCE.md`
- final integrity audit

## Current recommended run order after human annotation is complete

If generation and core statistics already exist, run:

1. Cell 10.3
2. Cell 10.4
3. Cell 10.5
4. Cell 10.6
5. Cell 10.7
6. Cell 11
7. Cell 12.1
8. Cell 12.2
9. Cell 12.4
10. Cell 12.5
11. Cell 12.6

Do not rerun expensive generation or judgment cells unless their outputs are missing or intentionally invalidated.

## Outputs of record

Important release artifacts:

- `{_125_file_text("model_card")}`
- `{_125_file_text("data_statement")}`
- `{_125_file_text("reproduce_readme")}`
- `{_125_file_text("final_report")}`

Important human-calibration artifacts:

- `{_125_file_text("human_import_validation")}`
- `{_125_file_text("human_agreement_metrics")}`
- `{_125_file_text("human_consensus_summary")}`
- `{_125_file_text("human_vs_llm")}`
- `{_125_file_text("human_calibration_summary")}`

Important tables:

- `{_125_path_text("tables")}/table1_main_tps.tex`
- `{_125_path_text("tables")}/table2_decomposition.tex`
- `{_125_path_text("tables")}/table3_robustness.tex`
- `{_125_path_text("tables")}/table4_human_calibration.tex`

## Common pitfalls

### 1. GPU not available

If no GPU is available, do not run the model-generation or LLM-judgment phases. Cached downstream analysis/publication-artifact cells may still be inspected, but the release notebook is GPU-first.

### 2. llama-cpp-python CUDA wheel not installed

Run the GPU setup cell to install the CUDA-enabled `llama-cpp-python` wheel, then restart runtime before loading models. Do not use a source-build fallback for the paper-facing run.

### 3. Missing completed human workbook

If Cell 10.3 fails, confirm that the workbook is named exactly:

`annotator_sheets_completed_primary4_400.xlsx`

and placed at:

`{analysis_human}/annotator_sheets_completed_primary4_400.xlsx`

### 4. Human labels use A1/B1 instead of A/B

This is expected. Cell 10.3 normalizes A1/B1/Tie to A/B/Tie.

### 5. Old hard-coded Mixtral references

Older publication-artifact cells may contain hard-coded `mixtral` rows. Replace those cells with the dynamic Primary-4 versions so the notebook reads families from `MODEL_REGISTRY`.

### 6. Old bootstrap key names

Older table cells may expect:

- `tps_point`
- `tps_ci_lo`
- `tps_ci_hi`

Current outputs may instead use:

- `observed_tps`
- `ci_95_low`
- `ci_95_high`

Use the updated Cell 12.2, which supports both schemas.

### 7. Final report key missing

If `FILES["final_report"]` is missing, use the updated Cell 11. It defines:

`final_report_primary4_human_calibrated.json`

### 8. Rerunning cached cells

Most cells use `run_step` and write output files. If outputs exist and `force=False`, they can be skipped. If `force=True`, the cell will recompute. For expensive phases, set `force=False` after successful completion if you want to avoid reruns.

## Integrity expectations

A successful final run should confirm:

- Primary-4 headline files exist.
- Falcon is not included in the headline analysis.
- 400 shared human annotation items were imported.
- 800 final human judgments from two retained annotators were validated.
- Human agreement metrics were written.
- Human consensus/no-consensus labels were created.
- Human-vs-LLM comparison was written.
- LaTeX tables include the human calibration table.
- MODEL_CARD, DATA_STATEMENT, and REPRODUCE files were written.
- Final audit passes or clearly lists only non-fatal warnings.

## Repository and release

Code:

- this notebook

Data release:

- Hugging Face dataset folder or equivalent release folder, depending on final venue requirements

Preprint:

- arXiv recommended after the manuscript is coherent and all final audit files pass

Pre-registration:

- OSF or repository timestamp can be added if desired, but do not claim formal pre-registration unless it was actually timestamped before analysis.
"""

    atomic_write_text(FILES["reproduce_readme"], text)

    print("REPRODUCE.md written:")
    print(FILES["reproduce_readme"])


run_step(
    [FILES["reproduce_readme"]],
    _step_reproduce_125,
    "reproduce_readme_primary4_human_calibrated",
    force=True,
    validators={
        FILES["reproduce_readme"]: lambda p: file_nonempty(p, 1000)
    },
)

print("\n✓ Cell 12.5 REPRODUCE.md complete")

In [ ]:
# ============================================================================
# Cell 12.6 — Final integrity audit  (Primary-4 + human calibration)
# ============================================================================
"""
Final release integrity audit for the tribal preference study.

Project scope
─────────────
  Headline analysis  : Primary-4  (llama, qwen, gemma, yi)
  Falcon             : diagnostic / sensitivity only — excluded from headline
  Human calibration  : 400 shared items × 2 retained annotators = 800 judgments

Release-clean audit behavior:
  - Expected human-calibration values are two retained annotators and 800 final human judgments.
  - Cohen's κ is checked as the primary two-rater statistic.
  - DATA_STATEMENT checks use the final retained-annotator language.
  - Paper guardrails require honest reporting of moderate agreement rather than overclaiming human validation.

Required outputs validated here
────────────────────────────────
  Phase 10  human_import_validation, human_agreement_metrics,
            human_consensus_summary, human_vs_llm,
            human_calibration_summary

  Phase 12  table1_main_tps.tex, table2_decomposition.tex,
            table3_robustness.tex, table4_human_calibration.tex,
            MODEL_CARD.md, DATA_STATEMENT.md, REPRODUCE.md

Severity policy
───────────────
  Core paper / release artifacts → hard errors  (block complete=True)
  Older / optional outputs        → warnings     (non-blocking)
"""

from __future__ import annotations

import json
import math
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# ── Guard ─────────────────────────────────────────────────────────────────────

_REQUIRED_GLOBALS = ["FILES", "PATHS", "run_step", "atomic_write_json", "file_nonempty"]
_missing = [g for g in _REQUIRED_GLOBALS if g not in globals()]
if _missing:
    raise RuntimeError(f"Missing globals — run earlier cells first: {_missing}")


# ── Path defaults ──────────────────────────────────────────────────────────────

def _default_path(key: str, value: Path) -> Path:
    if key not in PATHS:
        PATHS[key] = value
    return Path(PATHS[key])


_root     = _default_path("root",     Path(globals().get("ROOT_DIR", "/content/drive/MyDrive/tribal_pref_v11")))
_analysis = _default_path("analysis", _root / "analysis")
_human    = _default_path("analysis_human", _analysis / "human_calibration")
_release  = _default_path("release",  _root / "release")
_figures  = _default_path("figures",  _root / "figures")
_tables   = _default_path("tables",   _root / "tables")
_reports  = _default_path("reports",  _analysis / "reports")

_release.mkdir(parents=True, exist_ok=True)


# ── FILES defaults ─────────────────────────────────────────────────────────────

def _fset(key: str, path: Path) -> Path:
    FILES.setdefault(key, path)
    return Path(FILES[key])

_fset("final_audit",    _release / "final_audit.json")
_fset("final_report",   _reports / "final_report_primary4_human_calibrated.json")
_fset("model_card",     _release / "MODEL_CARD.md")
_fset("data_statement", _release / "DATA_STATEMENT.md")
_fset("reproduce_readme", _release / "REPRODUCE.md")

_H = _human
_fset("human_completed_workbook",  _H / "annotator_sheets_completed_primary4_400.xlsx")
_fset("human_key",                 _H / "human_annotation_key_hidden_primary4_400.csv")
_fset("human_completed_long",      _H / "human_annotations_completed_long_primary4_400.csv")
_fset("human_completed_wide",      _H / "human_annotations_completed_wide_primary4_400.csv")
_fset("human_with_key",            _H / "human_annotations_with_key_primary4_400.csv")
_fset("human_import_validation",   _H / "human_import_validation_primary4_400.json")
_fset("human_agreement_metrics",   _H / "human_agreement_metrics_primary4_400.json")
_fset("human_consensus",           _H / "human_consensus_primary4_400.csv")
_fset("human_consensus_summary",   _H / "human_consensus_summary_primary4_400.json")
_fset("human_vs_llm",              _H / "human_vs_llm_panel_primary4_400.json")
_fset("human_vs_llm_rows",         _H / "human_vs_llm_panel_rows_primary4_400.csv")
_fset("human_calibration_summary", _H / "human_calibration_summary_primary4_400.json")


# ── Utility helpers ────────────────────────────────────────────────────────────

def _exists(path) -> bool:
    try:
        return path is not None and Path(path).exists()
    except Exception:
        return False

def _size(path) -> int:
    try:
        return int(Path(path).stat().st_size)
    except Exception:
        return 0

def _read_json(path):
    if not _exists(path):
        return None
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception as exc:
        return {"__read_error__": str(exc)}

def _read_text(path, max_chars: int = 20_000) -> str | None:
    if not _exists(path):
        return None
    try:
        return Path(path).read_text(encoding="utf-8")[:max_chars]
    except Exception as exc:
        return f"__READ_ERROR__: {exc}"

def _csv_shape(path) -> dict | None:
    if not _exists(path):
        return None
    try:
        df = pd.read_csv(path)
        return {"rows": int(len(df)), "columns": list(df.columns), "n_columns": int(len(df.columns))}
    except Exception as exc:
        return {"__read_error__": str(exc)}

def _as_float(x, default: float = np.nan) -> float:
    try:
        return float(x) if x is not None else default
    except Exception:
        return default

def _safe_round(x, n: int = 4):
    v = _as_float(x)
    return round(float(v), n) if np.isfinite(v) else None

def _file_record(name: str, path, min_bytes: int = 1) -> dict:
    ok_size = _size(path) >= min_bytes
    return {
        "name": name,
        "path": str(path) if path is not None else None,
        "exists": _exists(path),
        "bytes": _size(path),
        "min_bytes": min_bytes,
        "ok": _exists(path) and ok_size,
    }

def _make_records(named_paths: dict, min_bytes: int = 1) -> dict:
    return {name: _file_record(name, path, min_bytes) for name, path in named_paths.items()}

def _missing_from(records: dict) -> list:
    return [name for name, rec in records.items() if not rec.get("ok", False)]

def _find_key(*candidates) -> str | None:
    for k in candidates:
        if k in FILES and _exists(FILES[k]):
            return k
    return None

def _fpath(*candidates) -> Path | None:
    k = _find_key(*candidates)
    return Path(FILES[k]) if k else None

def _json_status(path) -> str | None:
    obj = _read_json(path)
    return obj.get("status") if isinstance(obj, dict) else None


# ── Expected artefact manifests ────────────────────────────────────────────────

CORE_JSON = {
    "cluster_bootstrap": _fpath(
        "cluster_bootstrap_tps_primary", "cluster_bootstrap_primary",
        "cluster_bootstrap", "bootstrap_tps",
    ),
    "permutation_result": _fpath("permutation_result", "permutation_tps"),
    "per_family_bh":      _fpath("per_family_bh", "per_family_tps_bh"),
    "multiverse_summary": _fpath("multiverse_summary"),
}

CORE_CSV = {
    "effective_winners_primary":  _fpath("effective_winners_primary", "effective_winners"),
    "preference_matrix_primary":  _fpath("preference_matrix_primary", "preference_matrix"),
    "preference_support_primary": _fpath("preference_support_primary", "preference_support"),
}

HUMAN_ARTEFACTS = {k: Path(FILES[k]) for k in [
    "human_completed_workbook", "human_key",
    "human_completed_long",     "human_completed_wide",
    "human_with_key",           "human_import_validation",
    "human_agreement_metrics",  "human_consensus",
    "human_consensus_summary",  "human_vs_llm",
    "human_vs_llm_rows",        "human_calibration_summary",
]}

PUBLICATION_ARTEFACTS = {
    "final_report":     Path(FILES["final_report"]),
    "model_card":       Path(FILES["model_card"]),
    "data_statement":   Path(FILES["data_statement"]),
    "reproduce_readme": Path(FILES["reproduce_readme"]),
}

FIGURES = {
    "fig1_preference_heatmap":  _figures / "fig1_preference_heatmap.pdf",
    "fig2_residual_heatmap":    _figures / "fig2_residual_heatmap.pdf",
    "fig3_per_family_forest":   _figures / "fig3_per_family_forest.pdf",
    "fig4_decomposition_decay": _figures / "fig4_decomposition_decay.pdf",
    "fig5_specification_curve": _figures / "fig5_specification_curve.pdf",
}

TABLES = {
    "table1_main_tps":          _tables / "table1_main_tps.tex",
    "table2_decomposition":     _tables / "table2_decomposition.tex",
    "table3_robustness":        _tables / "table3_robustness.tex",
    "table4_human_calibration": _tables / "table4_human_calibration.tex",
}

OPTIONAL_ARTEFACTS = {
    "hf_dataset_readme":     _release / "hf_dataset" / "README.md",
    "final_technical_audit": _fpath("final_technical_audit"),
    "ranking_simulation":    _fpath("ranking_simulation"),
    "falcon_validation":     _fpath("falcon_validation"),
}


# ── Main audit function ────────────────────────────────────────────────────────

def _run_final_integrity_audit() -> None:
    print("=" * 100)
    print("FINAL INTEGRITY AUDIT — PRIMARY-4 + HUMAN CALIBRATION")
    print("=" * 100)

    errors:   list[dict] = []
    warnings: list[dict] = []

    # ── 1. File-existence checks ───────────────────────────────────────────────

    required_groups = {
        "core_json":   _make_records(CORE_JSON,             min_bytes=20),
        "core_csv":    _make_records(CORE_CSV,              min_bytes=20),
        "human":       _make_records(HUMAN_ARTEFACTS,       min_bytes=20),
        "publication": _make_records(PUBLICATION_ARTEFACTS, min_bytes=100),
        "figures":     _make_records(FIGURES,               min_bytes=100),
        "tables":      _make_records(TABLES,                min_bytes=100),
    }
    optional_records = _make_records(OPTIONAL_ARTEFACTS, min_bytes=20)

    missing_by_group: dict[str, list] = {}
    for group, records in required_groups.items():
        missing = _missing_from(records)
        missing_by_group[group] = missing
        if missing:
            errors.append({"type": "missing_required_artifacts", "group": group, "missing": missing})

    optional_missing = _missing_from(optional_records)
    if optional_missing:
        warnings.append({
            "type": "missing_optional_artifacts",
            "missing": optional_missing,
            "note": "Optional — useful for release completeness but not blocking for paper writing.",
        })

    # ── 2a. Import validation must pass ───────────────────────────────────────

    if _json_status(FILES["human_import_validation"]) != "pass":
        errors.append({
            "type": "human_import_validation_not_pass",
            "status": _json_status(FILES["human_import_validation"]),
            "path": str(FILES["human_import_validation"]),
        })

    # ── 2b. Calibration summary — design counts ────────────────────────────────
    # CHANGED: expected values updated to 2-annotator design

    cal_summary = _read_json(FILES["human_calibration_summary"])
    if isinstance(cal_summary, dict) and "__read_error__" not in cal_summary:
        design = cal_summary.get("design", {})
        for key, expected in [
            ("n_items",                 400),
            ("n_human_annotators",        2),   # Release-clean: expected 2 retained annotators
            ("n_total_human_judgments", 800),   # Release-clean: expected 800 final judgments
        ]:
            observed = int(design.get(key, -1))
            if observed != expected:
                errors.append({
                    "type": f"unexpected_{key}",
                    "expected": expected,
                    "observed": observed,
                })
    else:
        errors.append({
            "type": "human_calibration_summary_unreadable",
            "path": str(FILES["human_calibration_summary"]),
        })

    # ── 2c. Consensus row count ────────────────────────────────────────────────

    con_summary = _read_json(FILES["human_consensus_summary"])
    if isinstance(con_summary, dict) and "__read_error__" not in con_summary:
        n_majority    = int(con_summary.get("main_analysis_n_majority_rows",    -1))
        n_no_majority = int(con_summary.get("main_analysis_n_no_majority_rows", -1))
        if n_majority + n_no_majority != 400:
            errors.append({
                "type": "human_consensus_count_mismatch",
                "majority_rows": n_majority,
                "no_majority_rows": n_no_majority,
                "expected_total": 400,
            })
        if n_no_majority > 0:
            warnings.append({
                "type": "human_no_consensus_rows_present",   # CHANGED key name
                "n_no_consensus_rows": n_no_majority,
                "note": "Not fatal — report explicitly and cover via sensitivity analysis.",
            })

    # ── 2d. Inter-rater agreement thresholds ──────────────────────────────────
    # CHANGED: check cohen_kappa instead of fleiss_kappa (correct for 2 raters)
    # Threshold raised to 0.35 — cohen_kappa ~0.48 is moderate, not modest.

    agreement = _read_json(FILES["human_agreement_metrics"])
    if isinstance(agreement, dict) and "__read_error__" not in agreement:
        for metric_key, label, threshold in [
            # CHANGED: fleiss_kappa removed; cohen_kappa is the primary metric
            ("cohen_kappa",               "Cohen's κ",        0.35),
            ("krippendorff_alpha_nominal", "Krippendorff α",   0.35),
        ]:
            v = _as_float(agreement.get(metric_key))
            # cohen_kappa may be None in the JSON if Cell 10.4 stored None for
            # some fallback reason — skip gracefully
            if v is None:
                continue
            if np.isfinite(v) and v < threshold:
                warnings.append({
                    "type": f"low_{metric_key}",
                    metric_key: _safe_round(v, 3),
                    "note": (
                        f"{label} < {threshold} — describe agreement carefully in the paper. "
                        "Expected ~0.48 for 2-annotator design."
                    ),
                })

    # ── 3. Release-text phrase checks ──────────────────────────────────────────
    # Release-clean check: DATA_STATEMENT.md must report 800 final human judgments

    mc_text  = _read_text(FILES["model_card"])     or ""
    ds_text  = _read_text(FILES["data_statement"]) or ""
    rep_text = _read_text(FILES["reproduce_readme"]) or ""

    phrase_checks = {
        "model_card_primary4":         ("Primary-4" in mc_text and "llama, qwen, gemma, yi" in mc_text),
        "model_card_falcon_diagnostic": ("Falcon" in mc_text and "Diagnostic" in mc_text),
        # Release-clean check: DATA_STATEMENT must report 800 final human judgments
        "data_statement_human_800":    (
            "800 total human judgments" in ds_text
            or "800 judgments" in ds_text
        ),
        "reproduce_primary4": (
            "Primary-4" in rep_text and "falcon" in rep_text.lower()
        ),
    }
    for check, ok in phrase_checks.items():
        if not ok:
            errors.append({"type": "release_text_missing_expected_phrase", "check": check})

    # ── 4. Core result sanity (unchanged) ─────────────────────────────────────

    bootstrap   = _read_json(CORE_JSON.get("cluster_bootstrap"))
    permutation = _read_json(CORE_JSON.get("permutation_result"))

    headline_tps = ci_low = ci_high = p_two_sided = None

    if isinstance(bootstrap, dict):
        headline_tps = bootstrap.get("observed_tps", bootstrap.get("tps_point"))
        ci_low       = bootstrap.get("ci_95_low",    bootstrap.get("tps_ci_lo"))
        ci_high      = bootstrap.get("ci_95_high",   bootstrap.get("tps_ci_hi"))

    if isinstance(permutation, dict):
        p_two_sided = permutation.get("p_two_sided",
                      permutation.get("p_value",
                      permutation.get("p")))

    tps_f  = _as_float(headline_tps)
    ci_l_f = _as_float(ci_low)
    ci_h_f = _as_float(ci_high)
    p_f    = _as_float(p_two_sided)

    if np.isfinite(tps_f):
        if tps_f <= 0:
            errors.append({"type": "headline_tps_non_positive", "observed_tps": tps_f})
    else:
        warnings.append({
            "type": "headline_tps_not_parsed",
            "note": "Could not parse TPS from bootstrap JSON — verify table output manually.",
        })

    if np.isfinite(ci_l_f) and np.isfinite(ci_h_f):
        if ci_l_f <= 0 <= ci_h_f:
            warnings.append({
                "type": "bootstrap_ci_crosses_zero",
                "ci_low": ci_l_f, "ci_high": ci_h_f,
                "note": "CI crosses zero — would weaken the headline claim.",
            })
    else:
        warnings.append({
            "type": "bootstrap_ci_not_parsed",
            "note": "Could not parse CI from bootstrap JSON.",
        })

    # ── 5. Table / figure content checks (unchanged) ──────────────────────────

    table4_text = _read_text(TABLES["table4_human_calibration"]) or ""
    if "human calibration" not in table4_text.lower():
        errors.append({
            "type": "human_calibration_table_missing_caption",
            "path": str(TABLES["table4_human_calibration"]),
        })

    fig_records = required_groups["figures"]
    if not all(rec.get("ok", False) for rec in fig_records.values()):
        errors.append({
            "type": "one_or_more_required_figures_missing_or_empty",
            "missing": _missing_from(fig_records),
        })

    # ── 6. Tally and write audit ───────────────────────────────────────────────

    all_records: dict[str, dict] = {}
    for group, records in required_groups.items():
        for name, rec in records.items():
            all_records[f"{group}.{name}"] = rec

    n_required = len(all_records)
    n_ok       = sum(1 for r in all_records.values() if r.get("ok", False))
    n_missing  = n_required - n_ok
    complete   = len(errors) == 0 and n_missing == 0
    can_write  = len(errors) == 0 and n_missing <= 2

    audit = {
        "analysis_label": "final_integrity_audit_primary4_human_calibrated",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "complete": complete,
        "n_required_artifacts": n_required,
        "n_required_artifacts_ok": n_ok,
        "n_required_artifacts_missing_or_empty": n_missing,
        "errors": errors,
        "warnings": warnings,
        "missing_by_group": missing_by_group,
        "optional_missing": optional_missing,
        "project_scope": {
            "headline_analysis":  "Primary-4",
            "headline_families":  ["llama", "qwen", "gemma", "yi"],
            "falcon_status":      "excluded_from_headline_diagnostic_or_sensitivity_only",
            # Release-clean human-calibration scope
            "human_calibration": {
                "n_items":           400,
                "n_annotators":      2,
                "n_total_judgments": 800,
                "annotator_note":    "2 retained annotators; 1 excluded for low IAA",
            },
        },
        "headline_result_parsed": {
            "observed_tps": _safe_round(tps_f,  6),
            "ci_95_low":    _safe_round(ci_l_f, 6),
            "ci_95_high":   _safe_round(ci_h_f, 6),
            "p_two_sided":  _safe_round(p_f,    6),
        },
        "records": {
            group: records
            for group, records in {**required_groups, "optional": optional_records}.items()
        },
        "writing_readiness": {
            "can_start_writing":      can_write,
            "final_submission_ready": complete,
            "note": (
                "Writing can proceed once core results and human calibration are complete. "
                "Set complete=True before final submission / public release."
            ),
        },
        # CHANGED: guardrail updated — κ ~0.48 is moderate, not modest
        "paper_guardrails": [
            "Frame the paper as an audit of family-conditioned preference, not a universal claim about all LLMs.",
            "Report Primary-4 as the headline analysis.",
            "Discuss Falcon only as diagnostic/sensitivity — not part of the headline.",
            "Describe human agreement as moderate (Cohen κ ≈ 0.48, 2 retained annotators).",
            "State that the third annotator was excluded and give the reason briefly.",
            "Handle no-consensus human rows explicitly in the paper.",
            "Do not treat any external AI judge as human annotation.",
        ],
    }

    atomic_write_json(audit, FILES["final_audit"])

    # ── 7. Print summary ───────────────────────────────────────────────────────

    divider = "-" * 100
    print(f"\nFinal audit summary\n{divider}")
    print(json.dumps({
        "complete":                              audit["complete"],
        "n_required_artifacts":                  audit["n_required_artifacts"],
        "n_required_artifacts_ok":               audit["n_required_artifacts_ok"],
        "n_required_artifacts_missing_or_empty": audit["n_required_artifacts_missing_or_empty"],
        "n_errors":                              len(errors),
        "n_warnings":                            len(warnings),
        "final_submission_ready":                audit["writing_readiness"]["final_submission_ready"],
        "can_start_writing":                     audit["writing_readiness"]["can_start_writing"],
        "final_audit_path":                      str(FILES["final_audit"]),
    }, indent=2))

    if errors:
        print(f"\nERRORS\n{divider}")
        print(json.dumps(errors, indent=2))

    if warnings:
        print(f"\nWARNINGS\n{divider}")
        print(json.dumps(warnings, indent=2))

    if complete:
        print("\n✓ FINAL AUDIT PASSED — complete=True")
        print("✓ The project is ready for paper writing and final release.")
    else:
        print("\n⚠ FINAL AUDIT INCOMPLETE")
        print("  Fix the listed errors and rerun Cell 12.6.")


# ── Entry point ────────────────────────────────────────────────────────────────

run_step(
    [FILES["final_audit"]],
    _run_final_integrity_audit,
    "final_integrity_audit_primary4_human_calibrated",
    force=True,
    validators={FILES["final_audit"]: lambda p: file_nonempty(p, 1000)},
)

print("\n✓ Cell 12.6 complete")

## PHASE 13 — Reviewer-Response Analyses for ARR/EACL Revision

This phase adds the missing analyses needed for the revised ARR/EACL submission:

1. Leave-one-family-out TPS
2. Human-calibration baselines
3. Leaderboard / winner-flip simulation
4. Appendix export tables
5. Revised evidence summary
6. Final submission-readiness audit v2

These cells do not rerun generation or judging. They operate on existing Primary-4 artifacts.

In [ ]:
# ============================================================================
# Cell 13.1 — Leave-One-Family-Out TPS
# ============================================================================
"""
Purpose
-------
Directly addresses the reviewer objection that the Primary-4 aggregate TPS may
be dominated by one family, especially Qwen.

For each family f in Primary-4:
  - remove f from both judge and candidate families
  - recompute the preference matrix
  - recompute TPS
  - optionally bootstrap over prompt_id clusters

Outputs
-------
analysis/reviewer_response/leave_one_family_out_tps.csv
analysis/reviewer_response/leave_one_family_out_tps.json
analysis/tables/table5_leave_one_family_out_tps.tex

Interpretation
--------------
This does NOT claim generalization beyond Primary-4.
It asks whether the detected signal is stable when any one audited family is
excluded from the panel.
"""

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Sequence
import json
import sys
import time
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Required globals
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_json",
    "atomic_write_csv",
    "atomic_write_text",
    "json_has",
    "csv_has",
    "file_nonempty",
    "build_preference_matrix",
    "compute_tps",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Paths and file registrations
# ---------------------------------------------------------------------------

if "analysis" not in PATHS:
    if "root" in PATHS:
        PATHS["analysis"] = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        PATHS["analysis"] = Path(ROOT_DIR) / "analysis"
    else:
        PATHS["analysis"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

PATHS.setdefault("analysis_reviewer_response", Path(PATHS["analysis"]) / "reviewer_response")
PATHS.setdefault("tables", Path(PATHS.get("root", Path("/content/drive/MyDrive/tribal_pref_v11"))) / "tables")

Path(PATHS["analysis_reviewer_response"]).mkdir(parents=True, exist_ok=True)
Path(PATHS["tables"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "leave_one_family_out_tps_csv",
    Path(PATHS["analysis_reviewer_response"]) / "leave_one_family_out_tps.csv",
)

FILES.setdefault(
    "leave_one_family_out_tps_json",
    Path(PATHS["analysis_reviewer_response"]) / "leave_one_family_out_tps.json",
)

FILES.setdefault(
    "table5_leave_one_family_out_tps",
    Path(PATHS["tables"]) / "table5_leave_one_family_out_tps.tex",
)


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

PRIMARY_JUDGES_131 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_131 = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]

PRIMARY_FAMILIES_131 = sorted(set(PRIMARY_JUDGES_131).intersection(PRIMARY_CANDS_131))

N_BOOT_LOFO_131 = int(globals().get("N_BOOT_LOFO", 1000))
SEED_131 = int(globals().get("RANDOM_SEED", 20260502))


# ---------------------------------------------------------------------------
# Progress helpers
# ---------------------------------------------------------------------------

def _131_progress(msg: str, indent: int = 0) -> None:
    """Print a timestamped progress message, flushed immediately."""
    ts = datetime.now().strftime("%H:%M:%S")
    prefix = "  " * indent
    print(f"[{ts}] {prefix}{msg}", flush=True)


def _131_progress_bar(
    current: int,
    total: int,
    label: str = "",
    width: int = 30,
) -> None:
    """
    Overwrite the current line with an ASCII progress bar.
    Uses \\r so consecutive calls stay on one line in a terminal / Colab cell.
    Falls back to a plain print if the stream is not a tty.
    """
    filled = int(width * current / total) if total > 0 else 0
    bar = "█" * filled + "░" * (width - filled)
    pct = 100 * current / total if total > 0 else 0
    line = f"\r  [{bar}] {pct:5.1f}%  {current}/{total}  {label}"

    if hasattr(sys.stdout, "isatty") and sys.stdout.isatty() or "google.colab" in sys.modules:
        print(line, end="", flush=True)
    else:
        milestones = {0, total} | {int(total * k / 10) for k in range(1, 10)}
        if current in milestones:
            print(line.strip(), flush=True)


def _131_progress_bar_done() -> None:
    """End a progress-bar line."""
    print(flush=True)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _131_load_effective_winners() -> pd.DataFrame:
    _131_progress("Loading effective winners …")

    path_candidates = [
        FILES.get("effective_winners_primary"),
        FILES.get("effective_winners"),
    ]

    path = None
    for p in path_candidates:
        if p is not None and Path(p).exists():
            path = Path(p)
            break

    if path is None:
        raise FileNotFoundError(
            "Could not find effective_winners_primary/effective_winners. "
            "Run Cell 6.1 first."
        )

    df = pd.read_csv(path)

    required_cols = {
        "prompt_id",
        "judge_family",
        "family_1",
        "family_2",
        "support_family_1",
        "support_family_2",
    }

    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Effective winners file missing required columns: {sorted(missing)}")

    for col in ["judge_family", "family_1", "family_2"]:
        df[col] = df[col].astype(str).str.strip().str.lower()

    df["prompt_id"] = df["prompt_id"].astype(str)

    df["support_family_1"] = pd.to_numeric(df["support_family_1"], errors="coerce")
    df["support_family_2"] = pd.to_numeric(df["support_family_2"], errors="coerce")

    df = df.dropna(subset=["support_family_1", "support_family_2"]).copy()

    _131_progress(
        f"Loaded {len(df):,} rows | {df['prompt_id'].nunique():,} prompts | "
        f"path: {path.name}",
        indent=1,
    )
    return df


def _131_compute_subset_tps(
    eff: pd.DataFrame,
    judge_families: Sequence[str],
    candidate_families: Sequence[str],
) -> Dict[str, Any]:
    judge_families = [str(x).strip().lower() for x in judge_families]
    candidate_families = [str(x).strip().lower() for x in candidate_families]

    sub = eff[
        eff["judge_family"].isin(judge_families)
        & eff["family_1"].isin(candidate_families)
        & eff["family_2"].isin(candidate_families)
    ].copy()

    if sub.empty:
        return {
            "n_rows": 0,
            "n_prompts": 0,
            "diag_mean": np.nan,
            "offdiag_mean": np.nan,
            "tps": np.nan,
            "per_family_tps": {},
        }

    pref, support = build_preference_matrix(sub, judge_families, candidate_families)
    tps_obj = compute_tps(pref, judge_families, candidate_families)

    return {
        "n_rows": int(len(sub)),
        "n_prompts": int(sub["prompt_id"].nunique()),
        "judge_families": list(judge_families),
        "candidate_families": list(candidate_families),
        "diag_mean": float(tps_obj.get("diag_mean", np.nan)),
        "offdiag_mean": float(tps_obj.get("offdiag_mean", np.nan)),
        "tps": float(tps_obj.get("tps", np.nan)),
        "diag_values": tps_obj.get("diag_values", {}),
        "per_family_tps": tps_obj.get("per_family_tps", tps_obj.get("per_family", {})),
        "preference_matrix": pref.round(6).to_dict(),
        "support_matrix": support.round(6).to_dict(),
    }


def _131_bootstrap_subset(
    eff: pd.DataFrame,
    judge_families: Sequence[str],
    candidate_families: Sequence[str],
    n_boot: int = 1000,
    seed: int = 20260502,
    label: str = "",
) -> Dict[str, Any]:
    judge_families = [str(x).strip().lower() for x in judge_families]
    candidate_families = [str(x).strip().lower() for x in candidate_families]

    sub = eff[
        eff["judge_family"].isin(judge_families)
        & eff["family_1"].isin(candidate_families)
        & eff["family_2"].isin(candidate_families)
    ].copy()

    prompts = sorted(sub["prompt_id"].astype(str).unique())

    if len(prompts) < 2:
        return {
            "n_bootstrap": 0,
            "bootstrap_mean": np.nan,
            "bootstrap_sd": np.nan,
            "ci_95_low": np.nan,
            "ci_95_high": np.nan,
        }

    # -----------------------------------------------------------------------
    # KEY FIX: build the group lookup dict ONCE before the loop.
    # The original code did sub[sub["prompt_id"] == p] inside the loop,
    # which is an O(n) DataFrame scan per prompt per iteration — O(n * P * B)
    # total. Dict lookup is O(1), dropping this to O(n + P * B).
    # -----------------------------------------------------------------------
    grouped = {p: grp.reset_index(drop=True) for p, grp in sub.groupby("prompt_id")}

    rng = np.random.default_rng(seed)
    vals = []
    report_every = max(1, n_boot // 10)

    for i in range(n_boot):
        sampled_prompts = rng.choice(prompts, size=len(prompts), replace=True)

        sampled = pd.concat(
            [grouped[p] for p in sampled_prompts],
            ignore_index=True,
        )

        pref, _support = build_preference_matrix(sampled, judge_families, candidate_families)
        obj = compute_tps(pref, judge_families, candidate_families)
        vals.append(float(obj.get("tps", np.nan)))

        if (i + 1) % report_every == 0 or i == 0 or (i + 1) == n_boot:
            _131_progress_bar(i + 1, n_boot, label=label)

    _131_progress_bar_done()

    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return {
            "n_bootstrap": int(n_boot),
            "bootstrap_mean": np.nan,
            "bootstrap_sd": np.nan,
            "ci_95_low": np.nan,
            "ci_95_high": np.nan,
        }

    return {
        "n_bootstrap": int(n_boot),
        "bootstrap_mean": float(np.mean(vals)),
        "bootstrap_sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
        "ci_95_low": float(np.quantile(vals, 0.025)),
        "ci_95_high": float(np.quantile(vals, 0.975)),
    }


def _131_make_latex_table(df: pd.DataFrame) -> str:
    rows = []
    rows.append(r"\begin{table}[t]")
    rows.append(r"\centering")
    rows.append(r"\small")
    rows.append(r"\begin{tabular}{lrrrr}")
    rows.append(r"\toprule")
    rows.append(r"Condition & TPS & 95\% CI low & 95\% CI high & Prompts \\")
    rows.append(r"\midrule")

    for _, r in df.iterrows():
        condition = str(r["condition"]).replace("_", r"\_")
        tps = r["tps"]
        lo = r["ci_95_low"]
        hi = r["ci_95_high"]
        prompts = int(r["n_prompts"]) if pd.notna(r["n_prompts"]) else 0

        rows.append(
            f"{condition} & {tps:.4f} & {lo:.4f} & {hi:.4f} & {prompts} \\\\"
        )

    rows.append(r"\bottomrule")
    rows.append(r"\end{tabular}")
    rows.append(
        r"\caption{Leave-one-family-out sensitivity for Primary-4 TPS. "
        r"Each row removes one family from both judge and candidate roles.}"
    )
    rows.append(r"\label{tab:leave-one-family-out}")
    rows.append(r"\end{table}")

    return "\n".join(rows)


# ---------------------------------------------------------------------------
# Main step
# ---------------------------------------------------------------------------

def _step_leave_one_family_out_131():
    t0_total = time.time()
    n_conditions = 1 + len(PRIMARY_FAMILIES_131)

    _131_progress(
        f"Cell 13.1 — Leave-One-Family-Out TPS  "
        f"({n_conditions} conditions × {N_BOOT_LOFO_131} bootstrap reps each)"
    )
    _131_progress(f"Primary families: {PRIMARY_FAMILIES_131}", indent=1)
    print()

    eff = _131_load_effective_winners()
    print()

    rows = []
    details = {
        "analysis_label": "leave_one_family_out_tps",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "primary_families": PRIMARY_FAMILIES_131,
        "n_bootstrap": N_BOOT_LOFO_131,
        "notes": [
            "Primary-4 only.",
            "Each leave-one-out condition removes the family from both judge and candidate roles.",
            "Bootstrap is prompt-cluster bootstrap over prompt_id.",
            "This targets reviewer concern that aggregate TPS may be dominated by one family.",
        ],
        "conditions": {},
    }

    # ------------------------------------------------------------------
    # Baseline: all Primary-4
    # ------------------------------------------------------------------
    _131_progress(f"[1/{n_conditions}] Baseline — all Primary-4")

    t0 = time.time()
    _131_progress("Computing TPS …", indent=1)
    base = _131_compute_subset_tps(eff, PRIMARY_JUDGES_131, PRIMARY_CANDS_131)
    _131_progress(
        f"TPS={base['tps']:.4f}  diag={base['diag_mean']:.4f}  "
        f"offdiag={base['offdiag_mean']:.4f}  "
        f"rows={base['n_rows']:,}  prompts={base['n_prompts']:,}",
        indent=1,
    )

    _131_progress(f"Bootstrapping ({N_BOOT_LOFO_131} reps) …", indent=1)
    base_boot = _131_bootstrap_subset(
        eff,
        PRIMARY_JUDGES_131,
        PRIMARY_CANDS_131,
        n_boot=N_BOOT_LOFO_131,
        seed=SEED_131,
        label="baseline",
    )
    _131_progress(
        f"95% CI [{base_boot['ci_95_low']:.4f}, {base_boot['ci_95_high']:.4f}]  "
        f"sd={base_boot['bootstrap_sd']:.4f}  "
        f"({time.time() - t0:.1f}s)",
        indent=1,
    )

    details["conditions"]["all_primary4"] = {**base, **base_boot}

    rows.append({
        "condition": "all_primary4",
        "removed_family": "",
        "n_judge_families": len(PRIMARY_JUDGES_131),
        "n_candidate_families": len(PRIMARY_CANDS_131),
        "n_rows": base["n_rows"],
        "n_prompts": base["n_prompts"],
        "diag_mean": base["diag_mean"],
        "offdiag_mean": base["offdiag_mean"],
        "tps": base["tps"],
        "ci_95_low": base_boot["ci_95_low"],
        "ci_95_high": base_boot["ci_95_high"],
        "bootstrap_sd": base_boot["bootstrap_sd"],
    })
    print()

    # ------------------------------------------------------------------
    # Leave-one-family-out
    # ------------------------------------------------------------------
    for idx, fam in enumerate(PRIMARY_FAMILIES_131, start=2):
        judges = [x for x in PRIMARY_JUDGES_131 if x != fam]
        cands  = [x for x in PRIMARY_CANDS_131  if x != fam]
        key    = f"without_{fam}"

        _131_progress(f"[{idx}/{n_conditions}] Leave-one-out — removing '{fam}'")
        _131_progress(
            f"Remaining families: judges={judges}  candidates={cands}",
            indent=1,
        )

        t0 = time.time()
        _131_progress("Computing TPS …", indent=1)
        obj = _131_compute_subset_tps(eff, judges, cands)
        _131_progress(
            f"TPS={obj['tps']:.4f}  diag={obj['diag_mean']:.4f}  "
            f"offdiag={obj['offdiag_mean']:.4f}  "
            f"rows={obj['n_rows']:,}  prompts={obj['n_prompts']:,}",
            indent=1,
        )

        _131_progress(f"Bootstrapping ({N_BOOT_LOFO_131} reps) …", indent=1)
        boot = _131_bootstrap_subset(
            eff,
            judges,
            cands,
            n_boot=N_BOOT_LOFO_131,
            seed=SEED_131 + hash(fam) % 10000,
            label=f"without_{fam}",
        )
        _131_progress(
            f"95% CI [{boot['ci_95_low']:.4f}, {boot['ci_95_high']:.4f}]  "
            f"sd={boot['bootstrap_sd']:.4f}  "
            f"({time.time() - t0:.1f}s)",
            indent=1,
        )

        details["conditions"][key] = {**obj, **boot}

        rows.append({
            "condition": key,
            "removed_family": fam,
            "n_judge_families": len(judges),
            "n_candidate_families": len(cands),
            "n_rows": obj["n_rows"],
            "n_prompts": obj["n_prompts"],
            "diag_mean": obj["diag_mean"],
            "offdiag_mean": obj["offdiag_mean"],
            "tps": obj["tps"],
            "ci_95_low": boot["ci_95_low"],
            "ci_95_high": boot["ci_95_high"],
            "bootstrap_sd": boot["bootstrap_sd"],
        })
        print()

    # ------------------------------------------------------------------
    # Save outputs
    # ------------------------------------------------------------------
    out_df = pd.DataFrame(rows)

    _131_progress("Saving outputs …")
    atomic_write_csv(out_df, FILES["leave_one_family_out_tps_csv"])
    _131_progress(f"CSV  → {FILES['leave_one_family_out_tps_csv']}", indent=1)

    atomic_write_json(details, FILES["leave_one_family_out_tps_json"])
    _131_progress(f"JSON → {FILES['leave_one_family_out_tps_json']}", indent=1)

    latex = _131_make_latex_table(out_df)
    atomic_write_text(latex, FILES["table5_leave_one_family_out_tps"])
    _131_progress(f"TEX  → {FILES['table5_leave_one_family_out_tps']}", indent=1)

    # ------------------------------------------------------------------
    # Summary table
    # ------------------------------------------------------------------
    print()
    print("Leave-one-family-out TPS — summary")
    print("-" * 100)
    print(out_df[[
        "condition",
        "n_rows",
        "n_prompts",
        "diag_mean",
        "offdiag_mean",
        "tps",
        "ci_95_low",
        "ci_95_high",
    ]].round(4).to_string(index=False))

    elapsed = time.time() - t0_total
    print()
    _131_progress(f"Cell 13.1 done in {elapsed:.1f}s  ({elapsed/60:.1f} min)")


run_step(
    [
        FILES["leave_one_family_out_tps_csv"],
        FILES["leave_one_family_out_tps_json"],
        FILES["table5_leave_one_family_out_tps"],
    ],
    _step_leave_one_family_out_131,
    "leave_one_family_out_tps",
    force=False,
    validators={
        FILES["leave_one_family_out_tps_csv"]: lambda p: csv_has(
            p,
            ["condition", "tps", "ci_95_low", "ci_95_high"],
        ),
        FILES["leave_one_family_out_tps_json"]: lambda p: json_has(
            p,
            ["analysis_label", "conditions"],
        ),
        FILES["table5_leave_one_family_out_tps"]: file_nonempty,
    },
)

print("\n✓ Cell 13.1 complete")

In [ ]:
# V15 CLEANUP NOTE - DISABLED BY DEFAULT
# Reason: Secondary human-baseline diagnostic produces alternate valid-row counts and should not be confused with Table 6 canonical human calibration.
# To run this exploratory cell, set RUN_EXPLORATORY_HUMAN_BASELINE_132 = True and remove the guard.
RUN_EXPLORATORY_HUMAN_BASELINE_132 = False

if not RUN_EXPLORATORY_HUMAN_BASELINE_132:
    print("V15 cleanup: skipped exploratory/not-for-paper cell. See tribal_pref_v15_FINAL_NUMBERS.md and PAPER_PATCHES.md.")
else:
    # ============================================================================
    # Cell 13.2 — Human Calibration Baselines (2-annotator version)
    # ============================================================================
    """
    CHANGED from prior version:
    - Per-annotator loop checks annotator_1 and annotator_2 only (annotator_3 column
      no longer exists after the Phase 10 rewrite).
    - Pairwise IAA loop checks only the (1,2) pair — (1,3) and (2,3) no longer exist.
    - The "no_majority" filter in _no_majority row counting now also catches
      "no_consensus" (the new label written by Cell 10.5 2-annotator consensus).
    - n_no_majority_rows key in the JSON is renamed to n_no_consensus_rows with
      n_no_majority_rows kept as an alias so Cell 13.6 / Cell 13.5 backward
      compatibility is maintained.
    - Paper interpretation threshold for "reasonable" stays at +0.05 over
      majority-class — the 2-annotator agree-only subset will have fewer rows
      so the exact_match number will shift; the logic handles it automatically.
    - Everything else is identical to the working version already in the notebook.
    """

    import json
    import numpy as np
    import pandas as pd
    from datetime import datetime, timezone
    from pathlib import Path
    from typing import Any, Dict

    required_globals = [
        "FILES", "PATHS", "run_step",
        "atomic_write_json", "atomic_write_csv", "atomic_write_text",
        "json_has", "csv_has", "file_nonempty",
    ]
    missing_globals = [x for x in required_globals if x not in globals()]
    if missing_globals:
        raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")

    # ---------------------------------------------------------------------------
    # Paths — identical to existing cell
    # ---------------------------------------------------------------------------

    if "analysis" not in PATHS:
        if "root" in PATHS:
            PATHS["analysis"] = Path(PATHS["root"]) / "analysis"
        else:
            PATHS["analysis"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

    PATHS.setdefault("analysis_reviewer_response", Path(PATHS["analysis"]) / "reviewer_response")
    PATHS.setdefault("tables", Path(PATHS.get("root", Path("/content/drive/MyDrive/tribal_pref_v11"))) / "tables")

    Path(PATHS["analysis_reviewer_response"]).mkdir(parents=True, exist_ok=True)
    Path(PATHS["tables"]).mkdir(parents=True, exist_ok=True)

    FILES.setdefault(
        "human_calibration_baselines_json",
        Path(PATHS["analysis_reviewer_response"]) / "human_calibration_baselines.json"
    )
    FILES.setdefault(
        "human_calibration_baselines_csv",
        Path(PATHS["analysis_reviewer_response"]) / "human_calibration_baselines.csv"
    )

    # Always reassemble — never drift
    FILES["table6_human_calibration_baselines"] = (
        Path(PATHS["tables"]) / "table6_human_calibration_baselines.tex"
    )

    # Pre-flight path sanity check
    for _key in [
        "human_calibration_baselines_json",
        "human_calibration_baselines_csv",
        "table6_human_calibration_baselines",
    ]:
        _val = FILES[_key]
        assert isinstance(_val, Path), (
            f"FILES['{_key}'] must be a Path object, got {type(_val)}: {str(_val)[:80]}"
        )
        assert len(str(_val)) < 500, (
            f"FILES['{_key}'] suspiciously long — probably a content string: {str(_val)[:80]}"
        )

    print("✓ Path objects validated")


    # ---------------------------------------------------------------------------
    # Safe atomic text writer — identical to existing cell
    # ---------------------------------------------------------------------------

    def _safe_atomic_write_text(dest_path: Path, text_content: str) -> None:
        dest_path = Path(dest_path)
        assert len(str(dest_path)) < 500, (
            f"dest_path looks like content, not a path (length={len(str(dest_path))})."
        )
        dest_path.parent.mkdir(parents=True, exist_ok=True)
        tmp = dest_path.with_name(dest_path.name + ".tmp")
        try:
            with open(tmp, "w", encoding="utf-8") as f:
                f.write(text_content)
            tmp.replace(dest_path)
        except Exception:
            tmp.unlink(missing_ok=True)
            raise


    # ---------------------------------------------------------------------------
    # Helpers — identical to existing cell except where noted
    # ---------------------------------------------------------------------------

    def _132_load_human_vs_llm_rows() -> pd.DataFrame:
        path_candidates = [
            FILES.get("human_vs_llm_rows"),
            FILES.get("human_consensus"),
            FILES.get("human_filled"),
        ]

        df = None
        used_path = None

        for p in path_candidates:
            if p is not None and Path(p).exists():
                try:
                    df = pd.read_csv(p)
                    used_path = p
                    print(f"✓ Loaded human comparison file: {used_path}")
                    print(f"   Rows: {len(df)}, Columns: {len(df.columns)}")
                    print(f"   Column names: {list(df.columns)}")
                    break
                except Exception as e:
                    print(f"⚠ Failed to load {p}: {e}")

        if df is None:
            raise FileNotFoundError(
                "Could not find any human comparison file "
                "(human_vs_llm_rows, human_consensus, or human_filled). "
                "Run Cells 10.5 / 10.6 first."
            )

        df = df.copy()

        # Human label column — priority-ordered, same as before
        human_col = None
        for col in [
            "human_majority_label", "majority_label", "human_consensus_label",
            "human_consensus",   # written by Cell 10.5 (both 2- and 3-annotator versions)
            "human_label", "label_human", "winner_human",
        ]:
            if col in df.columns:
                human_col = col
                break

        # LLM label column — priority-ordered, same as before
        llm_col = None
        for col in [
            "llm_panel_winner_visible_side",
            "llm_panel_label", "llm_choice", "llm_winner",
            "panel_winner", "winner_llm", "llm_label",
        ]:
            if col in df.columns:
                llm_col = col
                break

        if human_col is None or llm_col is None:
            raise ValueError(
                f"Could not find required label columns.\n"
                f"Available: {list(df.columns)}"
            )

        print(f"✓ Human label column : '{human_col}'")
        print(f"✓ LLM label column   : '{llm_col}'")

        df["_human_label"] = df[human_col].astype(str).str.strip()
        df["_llm_label"]   = df[llm_col].astype(str).str.strip()

        return df


    def _132_exact_match(a, b) -> bool:
        if pd.isna(a) or pd.isna(b):
            return False
        return str(a).strip() == str(b).strip()


    def _132_distribution(series: pd.Series, labels=("A", "B", "Tie")) -> Dict[str, float]:
        vc = series.value_counts(dropna=False).to_dict()
        n = float(len(series))
        return {lab: float(vc.get(lab, 0) / n) if n else 0.0 for lab in labels}


    def _132_expected_independent_match(p: Dict[str, float], q: Dict[str, float]) -> float:
        labels = sorted(set(p.keys()).union(q.keys()))
        return float(sum(float(p.get(x, 0.0)) * float(q.get(x, 0.0)) for x in labels))


    def _132_make_latex_table(df: pd.DataFrame) -> str:
        lines = [
            r"\begin{table}[t]",
            r"\centering",
            r"\small",
            r"\begin{tabular}{lr}",
            r"\toprule",
            r"Quantity & Value \\",
            r"\midrule",
        ]
        for _, r in df.iterrows():
            name = str(r["metric"]).replace("_", r"\_")
            val  = r["value"]
            if isinstance(val, (int, np.integer)):
                val_s = str(int(val))
            else:
                try:
                    val_s = f"{float(val):.3f}"
                except Exception:
                    val_s = str(val)
            lines.append(f"{name} & {val_s} \\\\")
        lines.extend([
            r"\bottomrule",
            r"\end{tabular}",
            (r"\caption{Human-calibration baselines. Exact LLM-human agreement "
             r"should be interpreted relative to simple baselines, because the task "
             r"is subjective and human agreement is modest.}"),
            r"\label{tab:human-baselines}",
            r"\end{table}",
        ])
        return "\n".join(lines)


    # ---------------------------------------------------------------------------
    # Main step
    # ---------------------------------------------------------------------------

    def _step_human_calibration_baselines_132():

        df = _132_load_human_vs_llm_rows()

        labels = ["A", "B", "Tie"]

        # Valid rows: human label is A, B, or Tie (i.e., annotators agreed)
        main = df[
            df["_human_label"].isin(labels) & df["_llm_label"].isin(labels)
        ].copy()

        # CHANGED: also catch "no_consensus" (2-annotator label) in addition to
        # "no_majority" (3-annotator label) so the count is correct regardless of
        # which Phase 10 version produced the file.
        no_consensus = df[
            df["_human_label"].str.lower().isin([
                "no_majority", "no majority",
                "no_consensus", "no consensus",
                "nan", "",
            ])
        ].copy()

        if main.empty:
            raise ValueError(
                "No rows with valid A/B/Tie labels. Check your human annotation file."
            )

        main["exact_match"] = main.apply(
            lambda r: _132_exact_match(r["_human_label"], r["_llm_label"]), axis=1
        )

        n                       = int(len(main))
        exact_match             = float(main["exact_match"].mean())
        human_counts            = main["_human_label"].value_counts().to_dict()
        llm_counts              = main["_llm_label"].value_counts().to_dict()
        majority_label          = max(human_counts.items(), key=lambda kv: kv[1])[0]
        majority_class_baseline = float(human_counts[majority_label] / n)
        uniform_random_baseline = 1.0 / 3.0
        human_dist              = _132_distribution(main["_human_label"], labels=labels)
        llm_dist                = _132_distribution(main["_llm_label"],   labels=labels)
        human_dist_random       = _132_expected_independent_match(human_dist, human_dist)
        llm_dist_random         = _132_expected_independent_match(human_dist, llm_dist)

        # -----------------------------------------------------------------------
        # Per-annotator agreement with LLM panel
        # CHANGED: only check annotator_1 and annotator_2 (annotator_3 column
        # no longer exists after the 2-annotator Phase 10 rewrite).
        # The loop gracefully skips any column that is absent, so this also
        # works if the old 3-annotator file is somehow still present on disk.
        # -----------------------------------------------------------------------
        annotator_llm_agreements = {}
        for i in [1, 2]:                          # CHANGED: was [1, 2, 3]
            col = f"annotator_{i}_choice"
            if col in main.columns:
                valid = main[main[col].astype(str).str.strip().isin(labels)].copy()
                if len(valid) > 0:
                    agree = (
                        valid[col].astype(str).str.strip()
                        == valid["_llm_label"].astype(str).str.strip()
                    ).mean()
                    annotator_llm_agreements[f"annotator_{i}_llm_agreement"] = float(agree)
                    annotator_llm_agreements[f"annotator_{i}_n_valid"]       = int(len(valid))

        # -----------------------------------------------------------------------
        # Pairwise inter-annotator agreement
        # CHANGED: only check pair (1, 2) — pairs (1,3) and (2,3) no longer exist.
        # -----------------------------------------------------------------------
        pairwise_iaa = {}
        for i, j in [(1, 2)]:                     # CHANGED: was [(1,2),(1,3),(2,3)]
            col_i = f"annotator_{i}_choice"
            col_j = f"annotator_{j}_choice"
            if col_i in main.columns and col_j in main.columns:
                sub = main[
                    main[col_i].astype(str).str.strip().isin(labels) &
                    main[col_j].astype(str).str.strip().isin(labels)
                ]
                if len(sub) > 0:
                    agree = (
                        sub[col_i].astype(str).str.strip()
                        == sub[col_j].astype(str).str.strip()
                    ).mean()
                    pairwise_iaa[f"annotator_{i}_{j}_pairwise_agreement"] = float(agree)
                    pairwise_iaa[f"annotator_{i}_{j}_n"]                  = int(len(sub))

        # -----------------------------------------------------------------------
        # Assemble output JSON
        # -----------------------------------------------------------------------
        n_no_consensus = int(len(no_consensus))

        out = {
            "analysis_label":   "human_calibration_baselines_primary4_400",
            "generated_at":     datetime.now(timezone.utc).isoformat(),
            "n_valid_consensus_rows":   n,
            # Keep old key name so Cell 13.6 backward-compat read works
            "n_valid_majority_rows":    n,
            "n_no_consensus_rows":      n_no_consensus,
            # Keep old key name so Cell 13.5 / 13.6 read it without change
            "n_no_majority_rows":       n_no_consensus,
            "labels": labels,
            "human_label_counts":           {str(k): int(v) for k, v in human_counts.items()},
            "llm_panel_label_counts":       {str(k): int(v) for k, v in llm_counts.items()},
            "human_label_distribution":     human_dist,
            "llm_panel_label_distribution": llm_dist,
            "metrics": {
                "llm_human_exact_match":              exact_match,
                "uniform_random_3way_baseline":       uniform_random_baseline,
                "human_majority_class_baseline":      majority_class_baseline,
                "human_majority_class_label":         majority_label,
                "human_distribution_random_baseline": human_dist_random,
                "llm_distribution_random_baseline":   llm_dist_random,
                "absolute_gain_over_majority_class":  float(exact_match - majority_class_baseline),
                "absolute_gain_over_uniform_random":  float(exact_match - uniform_random_baseline),
            },
            "annotator_llm_agreements": annotator_llm_agreements,
            "pairwise_iaa":             pairwise_iaa,
            "paper_safe_interpretation": (
                "Human calibration should be reported as an external reference point and "
                "task-difficulty diagnostic. If exact agreement is close to a simple "
                "majority-class baseline, the paper should not claim strong human validation."
            ),
        }

        # -----------------------------------------------------------------------
        # Assemble output CSV
        # -----------------------------------------------------------------------
        rows = [
            {"metric": "n_valid_consensus_rows",            "value": n},
            {"metric": "n_no_consensus_rows",               "value": n_no_consensus},
            {"metric": "llm_human_exact_match",             "value": exact_match},
            {"metric": "uniform_random_3way_baseline",      "value": uniform_random_baseline},
            {"metric": "human_majority_class_baseline",     "value": majority_class_baseline},
            {"metric": "absolute_gain_over_majority_class", "value": float(exact_match - majority_class_baseline)},
            {"metric": "absolute_gain_over_uniform_random", "value": float(exact_match - uniform_random_baseline)},
            {"metric": "human_distribution_random_baseline","value": human_dist_random},
            {"metric": "llm_distribution_random_baseline",  "value": llm_dist_random},
        ]
        for k, v in annotator_llm_agreements.items():
            if isinstance(v, float):
                rows.append({"metric": k, "value": v})
        for k, v in pairwise_iaa.items():
            if isinstance(v, float):
                rows.append({"metric": k, "value": v})

        out_df = pd.DataFrame(rows)

        # LaTeX table: first 9 rows (main metrics only)
        latex_content = _132_make_latex_table(out_df.head(9))

        # -----------------------------------------------------------------------
        # Write outputs — path FIRST, content SECOND
        # -----------------------------------------------------------------------
        atomic_write_json(out, FILES["human_calibration_baselines_json"])
        atomic_write_csv(out_df, FILES["human_calibration_baselines_csv"])
        _safe_atomic_write_text(
            FILES["table6_human_calibration_baselines"],
            latex_content,
        )

        # -----------------------------------------------------------------------
        # Console summary
        # -----------------------------------------------------------------------
        print("\nHuman calibration baselines")
        print("-" * 100)
        print(out_df.to_string(index=False))

        print(f"\nHuman label counts : {human_counts}")
        print(f"LLM panel counts   : {llm_counts}")

        if annotator_llm_agreements:
            print("\nPer-annotator LLM agreement:")
            for k, v in annotator_llm_agreements.items():
                if isinstance(v, float):
                    print(f"  {k}: {v:.3f}")

        if pairwise_iaa:
            print("\nPairwise inter-annotator agreement:")
            for k, v in pairwise_iaa.items():
                if isinstance(v, float):
                    print(f"  {k}: {v:.3f}")

        print("\nSaved:")
        print("  JSON:", FILES["human_calibration_baselines_json"])
        print("  CSV :", FILES["human_calibration_baselines_csv"])
        print("  TEX :", FILES["table6_human_calibration_baselines"])

        # Paper interpretation guidance
        print("\n--- PAPER INTERPRETATION ---")
        print(f"  LLM-human exact match    : {exact_match:.3f}")
        print(f"  Uniform random baseline  : {uniform_random_baseline:.3f}  "
              f"(+{exact_match - uniform_random_baseline:.3f} gain)")
        print(f"  Majority class baseline  : {majority_class_baseline:.3f}  "
              f"(+{exact_match - majority_class_baseline:.3f} gain)")
        print(f"  n valid (agree rows)     : {n}")
        print(f"  n no-consensus rows      : {n_no_consensus}")

        gain = exact_match - majority_class_baseline
        if gain < 0.05:
            print("  ⚠  Gain over majority-class is <0.05 — do NOT claim strong human validation.")
            print(f"     Report: 'LLM panel agreement with human consensus label is "
                  f"{exact_match * 100:.1f}%, slightly above the "
                  f"{majority_class_baseline * 100:.1f}% majority-class baseline.'")
        else:
            print(f"  ✓  Gain over majority-class: {gain:.3f} — reasonable for paper reporting.")


    # ---------------------------------------------------------------------------
    # Run
    # ---------------------------------------------------------------------------

    run_step(
        [
            FILES["human_calibration_baselines_json"],
            FILES["human_calibration_baselines_csv"],
            FILES["table6_human_calibration_baselines"],
        ],
        _step_human_calibration_baselines_132,
        "human_calibration_baselines",
        force=True,
        validators={
            FILES["human_calibration_baselines_json"]: lambda p: json_has(
                p, ["analysis_label", "metrics"]
            ),
            FILES["human_calibration_baselines_csv"]: lambda p: csv_has(
                p, ["metric", "value"]
            ),
            FILES["table6_human_calibration_baselines"]: file_nonempty,
        },
    )

    print("\n✓ Cell 13.2 complete")

In [ ]:
# ============================================================================
# Cell 13.3 — Leaderboard / Winner-Flip Simulation (Fixed + Robust)
# ============================================================================
"""
Purpose
-------
Turn the abstract TPS effect into a practical sensitivity analysis.

Question
--------
How often does the winner of a pairwise comparison change when the judge panel
composition changes?

Regimes
-------
1. balanced_panel:
   Mean support across all Primary-4 judge families (baseline reference).

2. single_family_judge:
   Winner under each individual judge family.

3. in_pair_same_family_panel:
   Mean support using only judge families that match one of the two candidates.

4. family_1_own_judge / family_2_own_judge:
   Winner under only the judge family matching candidate family_1 or family_2.

5. leave_one_family_out_balanced:
   Balanced panel after removing each judge family.

Outputs
-------
analysis/reviewer_response/leaderboard_flip_rows.csv
analysis/reviewer_response/leaderboard_flip_summary.csv
analysis/reviewer_response/leaderboard_flip_simulation.json
tables/table7_leaderboard_flip_simulation.tex

Interpretation
--------------
This is NOT a real public leaderboard. It is an internal sensitivity simulation
over the study's pairwise comparison graph.
"""

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
import json
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Required globals check
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_json",
    "atomic_write_csv",
    "atomic_write_text",
    "json_has",
    "csv_has",
    "file_nonempty",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

if "analysis" not in PATHS:
    if "root" in PATHS:
        PATHS["analysis"] = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        PATHS["analysis"] = Path(ROOT_DIR) / "analysis"
    else:
        PATHS["analysis"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

PATHS.setdefault(
    "analysis_reviewer_response",
    Path(PATHS["analysis"]) / "reviewer_response",
)
PATHS.setdefault(
    "tables",
    Path(PATHS.get("root", Path("/content/drive/MyDrive/tribal_pref_v11"))) / "tables",
)

Path(PATHS["analysis_reviewer_response"]).mkdir(parents=True, exist_ok=True)
Path(PATHS["tables"]).mkdir(parents=True, exist_ok=True)

# Always reassemble from PATHS to prevent any string-drift from prior runs
FILES["leaderboard_flip_rows"] = (
    Path(PATHS["analysis_reviewer_response"]) / "leaderboard_flip_rows.csv"
)
FILES["leaderboard_flip_summary"] = (
    Path(PATHS["analysis_reviewer_response"]) / "leaderboard_flip_summary.csv"
)
FILES["leaderboard_flip_simulation"] = (
    Path(PATHS["analysis_reviewer_response"]) / "leaderboard_flip_simulation.json"
)
FILES["table7_leaderboard_flip_simulation"] = (
    Path(PATHS["tables"]) / "table7_leaderboard_flip_simulation.tex"
)

# Pre-flight sanity check — catch path corruption before any computation
for _key in [
    "leaderboard_flip_rows",
    "leaderboard_flip_summary",
    "leaderboard_flip_simulation",
    "table7_leaderboard_flip_simulation",
]:
    _val = FILES[_key]
    assert isinstance(_val, Path), (
        f"FILES['{_key}'] must be a Path, got {type(_val)}: {str(_val)[:80]}"
    )
    assert len(str(_val)) < 500, (
        f"FILES['{_key}'] suspiciously long — likely a content string, not a path: "
        f"{str(_val)[:80]}"
    )

print("✓ Path objects validated")


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

PRIMARY_JUDGES_133 = [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES]
PRIMARY_CANDS_133  = [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES]


# ---------------------------------------------------------------------------
# Safe atomic text writer (local copy — guards against argument-swap bug)
# ---------------------------------------------------------------------------

def _safe_atomic_write_text_133(dest_path: Path, text_content: str) -> None:
    """
    Write text_content to dest_path atomically via a .tmp file.
    Argument order: path FIRST, content SECOND — matches atomic_write_json/csv.
    Hard assertion prevents content-as-path confusion (Errno 36).
    """
    dest_path = Path(dest_path)
    assert len(str(dest_path)) < 500, (
        f"dest_path looks like content, not a path "
        f"(len={len(str(dest_path))}). Arguments may be swapped."
    )
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest_path.with_name(dest_path.name + ".tmp")
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            f.write(text_content)
        tmp.replace(dest_path)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


# ---------------------------------------------------------------------------
# Data loader
# ---------------------------------------------------------------------------

def _133_load_effective_winners() -> pd.DataFrame:
    path_candidates = [
        FILES.get("effective_winners_primary"),
        FILES.get("effective_winners"),
    ]

    path = None
    for p in path_candidates:
        if p is not None and Path(p).exists():
            path = Path(p)
            break

    if path is None:
        raise FileNotFoundError(
            "Could not find effective_winners_primary / effective_winners. "
            "Run Cell 6.1 first."
        )

    print(f"✓ Loading effective winners: {path}")
    df = pd.read_csv(path)
    print(f"   Rows: {len(df)}, Columns: {list(df.columns)}")

    required_cols = {
        "prompt_id", "judge_family",
        "family_1", "family_2",
        "support_family_1", "support_family_2",
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"Effective winners file missing required columns: {sorted(missing)}"
        )

    for col in ["judge_family", "family_1", "family_2"]:
        df[col] = df[col].astype(str).str.strip().str.lower()

    df["prompt_id"]        = df["prompt_id"].astype(str)
    df["support_family_1"] = pd.to_numeric(df["support_family_1"], errors="coerce")
    df["support_family_2"] = pd.to_numeric(df["support_family_2"], errors="coerce")
    df = df.dropna(subset=["support_family_1", "support_family_2"]).copy()

    # Restrict to Primary-4
    df = df[
        df["judge_family"].isin(PRIMARY_JUDGES_133)
        & df["family_1"].isin(PRIMARY_CANDS_133)
        & df["family_2"].isin(PRIMARY_CANDS_133)
    ].copy()

    # Falcon leak guard
    for col in ["judge_family", "family_1", "family_2"]:
        if "falcon" in df[col].unique():
            raise RuntimeError(f"Falcon leaked into Cell 13.3 via column '{col}'.")

    print(f"   After Primary-4 filter: {len(df)} rows")
    return df


# ---------------------------------------------------------------------------
# Core helpers
# ---------------------------------------------------------------------------

def _133_winner_from_support(support_family_1: float, tie_band: float = 1e-12) -> str:
    if pd.isna(support_family_1):
        return "missing"
    if support_family_1 > 0.5 + tie_band:
        return "family_1"
    if support_family_1 < 0.5 - tie_band:
        return "family_2"
    return "tie"


def _133_flip(a: str, b: str) -> bool:
    """True if two non-missing winner labels differ (tie vs. win counts as a flip)."""
    if a == "missing" or b == "missing":
        return False
    return a != b


def _133_wilson_ci(k: int, n: int, z: float = 1.96):
    """Wilson score confidence interval for a proportion."""
    if n == 0:
        return (np.nan, np.nan)
    p_hat = k / n
    denom = 1 + z ** 2 / n
    centre = (p_hat + z ** 2 / (2 * n)) / denom
    half   = (z * np.sqrt(p_hat * (1 - p_hat) / n + z ** 2 / (4 * n ** 2))) / denom
    return (max(0.0, float(centre - half)), min(1.0, float(centre + half)))


# ---------------------------------------------------------------------------
# Flip summary builder
# ---------------------------------------------------------------------------

def _133_summarize_flips(rows: pd.DataFrame) -> pd.DataFrame:
    summaries = []

    regime_cols = [
        c for c in rows.columns
        if c.endswith("_winner") and c != "balanced_panel_winner"
    ]

    for regime_col in regime_cols:
        regime = regime_col.replace("_winner", "")

        valid = rows[
            rows["balanced_panel_winner"].isin(["family_1", "family_2", "tie"])
            & rows[regime_col].isin(["family_1", "family_2", "tie"])
        ].copy()

        n = len(valid)

        if n == 0:
            summaries.append({
                "regime": regime,
                "n_items": 0,
                "flip_rate_vs_balanced": np.nan,
                "flip_ci_lo": np.nan,
                "flip_ci_hi": np.nan,
                "non_tie_flip_rate_vs_balanced": np.nan,
                "non_tie_flip_ci_lo": np.nan,
                "non_tie_flip_ci_hi": np.nan,
                "tie_involved_rate": np.nan,
            })
            continue

        valid["flip_vs_balanced"] = valid.apply(
            lambda r: _133_flip(r["balanced_panel_winner"], r[regime_col]), axis=1
        )

        n_flips  = int(valid["flip_vs_balanced"].sum())
        flip_rate = float(n_flips / n)
        ci_lo, ci_hi = _133_wilson_ci(n_flips, n)

        non_tie = valid[
            valid["balanced_panel_winner"].isin(["family_1", "family_2"])
            & valid[regime_col].isin(["family_1", "family_2"])
        ].copy()

        n_nt = len(non_tie)
        if n_nt > 0:
            n_nt_flips       = int((non_tie["balanced_panel_winner"] != non_tie[regime_col]).sum())
            non_tie_flip     = float(n_nt_flips / n_nt)
            nt_ci_lo, nt_ci_hi = _133_wilson_ci(n_nt_flips, n_nt)
        else:
            non_tie_flip = np.nan
            nt_ci_lo = nt_ci_hi = np.nan

        tie_involved = float(
            (
                (valid["balanced_panel_winner"] == "tie")
                | (valid[regime_col] == "tie")
            ).mean()
        )

        summaries.append({
            "regime": regime,
            "n_items": n,
            "flip_rate_vs_balanced": flip_rate,
            "flip_ci_lo": ci_lo,
            "flip_ci_hi": ci_hi,
            "non_tie_flip_rate_vs_balanced": non_tie_flip,
            "non_tie_flip_ci_lo": nt_ci_lo,
            "non_tie_flip_ci_hi": nt_ci_hi,
            "tie_involved_rate": tie_involved,
        })

    return pd.DataFrame(summaries)


# ---------------------------------------------------------------------------
# Same-family vs. cross-family pair breakdown
# ---------------------------------------------------------------------------

def _133_summarize_flips_by_pair_type(rows: pd.DataFrame) -> pd.DataFrame:
    """
    For each regime, break flip rates down by whether the pair is
    same-family (f1 == f2, impossible in Primary-4 but guarded) or
    cross-family, and further by whether one candidate is the judge's
    own family (relevant for the in_pair_same_family_panel regime).

    For the single_<judge>_winner regimes, annotate whether the judge
    family matches family_1, family_2, or neither.
    """
    rows = rows.copy()

    # Pair type: same or cross
    rows["pair_type"] = rows.apply(
        lambda r: "same_family" if r["family_1"] == r["family_2"] else "cross_family",
        axis=1,
    )

    regime_cols = [
        c for c in rows.columns
        if c.endswith("_winner") and c != "balanced_panel_winner"
    ]

    breakdown_rows = []

    for regime_col in regime_cols:
        regime = regime_col.replace("_winner", "")

        for pair_type in ["cross_family", "same_family"]:
            sub = rows[
                (rows["pair_type"] == pair_type)
                & rows["balanced_panel_winner"].isin(["family_1", "family_2", "tie"])
                & rows[regime_col].isin(["family_1", "family_2", "tie"])
            ].copy()

            n = len(sub)
            if n == 0:
                breakdown_rows.append({
                    "regime": regime,
                    "pair_type": pair_type,
                    "n_items": 0,
                    "flip_rate": np.nan,
                    "flip_ci_lo": np.nan,
                    "flip_ci_hi": np.nan,
                })
                continue

            sub["flip"] = sub.apply(
                lambda r: _133_flip(r["balanced_panel_winner"], r[regime_col]), axis=1
            )
            n_flips = int(sub["flip"].sum())
            flip_rate = float(n_flips / n)
            ci_lo, ci_hi = _133_wilson_ci(n_flips, n)

            breakdown_rows.append({
                "regime": regime,
                "pair_type": pair_type,
                "n_items": n,
                "flip_rate": flip_rate,
                "flip_ci_lo": ci_lo,
                "flip_ci_hi": ci_hi,
            })

    return pd.DataFrame(breakdown_rows)


# ---------------------------------------------------------------------------
# LaTeX table builder
# ---------------------------------------------------------------------------

def _133_make_latex_table(df: pd.DataFrame) -> str:
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\begin{tabular}{lrrrr}",
        r"\toprule",
        r"Regime & $n$ & Flip rate & 95\% CI & Non-tie flip rate \\",
        r"\midrule",
    ]

    for _, r in df.iterrows():
        regime   = str(r["regime"]).replace("_", r"\_")
        n        = int(r["n_items"])
        flip     = r.get("flip_rate_vs_balanced", np.nan)
        ci_lo    = r.get("flip_ci_lo", np.nan)
        ci_hi    = r.get("flip_ci_hi", np.nan)
        non_tie  = r.get("non_tie_flip_rate_vs_balanced", np.nan)

        flip_s    = "---" if pd.isna(flip)    else f"{100 * float(flip):.1f}\\%"
        non_tie_s = "---" if pd.isna(non_tie) else f"{100 * float(non_tie):.1f}\\%"

        if pd.isna(ci_lo) or pd.isna(ci_hi):
            ci_s = "---"
        else:
            ci_s = f"[{100 * float(ci_lo):.1f}, {100 * float(ci_hi):.1f}]"

        lines.append(f"{regime} & {n} & {flip_s} & {ci_s} & {non_tie_s} \\\\")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        (
            r"\caption{Winner-flip sensitivity relative to the balanced Primary-4 judge panel. "
            r"Flip rate is the fraction of pairwise items whose winner changes when judge "
            r"composition changes. Non-tie flip rate excludes items where either panel "
            r"declares a tie. CIs are Wilson score 95\%.}"
        ),
        r"\label{tab:leaderboard-flip}",
        r"\end{table}",
    ])

    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Main step
# ---------------------------------------------------------------------------

def _step_leaderboard_flip_simulation_133():
    print("=" * 100)
    print("LEADERBOARD / WINNER-FLIP SIMULATION — PRIMARY-4")
    print("=" * 100)

    eff = _133_load_effective_winners()

    group_cols = ["prompt_id", "family_1", "family_2"]
    item_rows: List[Dict[str, Any]] = []

    for (prompt_id, f1, f2), g in eff.groupby(group_cols, dropna=False):

        row: Dict[str, Any] = {
            "prompt_id":      prompt_id,
            "family_1":       f1,
            "family_2":       f2,
            "n_judge_rows":   int(len(g)),
            "is_same_family": (f1 == f2),
        }

        # ---- Regime 1: Balanced panel (reference) ----
        balanced_support = float(g["support_family_1"].mean())
        row["balanced_panel_support_family_1"] = balanced_support
        row["balanced_panel_winner"]           = _133_winner_from_support(balanced_support)

        # ---- Regime 2: Single-family judges ----
        for judge in PRIMARY_JUDGES_133:
            sub     = g[g["judge_family"] == judge]
            support = float(sub["support_family_1"].mean()) if not sub.empty else np.nan
            row[f"single_{judge}_support_family_1"] = support
            row[f"single_{judge}_winner"]           = _133_winner_from_support(support)

        # ---- Regime 3: In-pair same-family panel ----
        # Only judge families matching either candidate
        in_pair     = g[g["judge_family"].isin([f1, f2])]
        in_pair_sup = float(in_pair["support_family_1"].mean()) if not in_pair.empty else np.nan
        row["in_pair_same_family_panel_support_family_1"] = in_pair_sup
        row["in_pair_same_family_panel_winner"]           = _133_winner_from_support(in_pair_sup)

        # ---- Regime 4: Candidate-own judge ----
        f1_judge = g[g["judge_family"] == f1]
        f2_judge = g[g["judge_family"] == f2]

        f1_own_sup = float(f1_judge["support_family_1"].mean()) if not f1_judge.empty else np.nan
        f2_own_sup = float(f2_judge["support_family_1"].mean()) if not f2_judge.empty else np.nan

        row["family_1_own_judge_support_family_1"] = f1_own_sup
        row["family_1_own_judge_winner"]           = _133_winner_from_support(f1_own_sup)
        row["family_2_own_judge_support_family_1"] = f2_own_sup
        row["family_2_own_judge_winner"]           = _133_winner_from_support(f2_own_sup)

        # ---- Regime 5: Leave-one-family-out balanced panels ----
        for removed in PRIMARY_JUDGES_133:
            sub     = g[g["judge_family"] != removed]
            support = float(sub["support_family_1"].mean()) if not sub.empty else np.nan
            row[f"balanced_without_{removed}_support_family_1"] = support
            row[f"balanced_without_{removed}_winner"]           = _133_winner_from_support(support)

        item_rows.append(row)

    rows_df = pd.DataFrame(item_rows)

    print(f"\n✓ Built item-level flip table: {len(rows_df)} items")
    print(f"  Same-family pairs : {int(rows_df['is_same_family'].sum())}")
    print(f"  Cross-family pairs: {int((~rows_df['is_same_family']).sum())}")

    # ---- Summaries ----
    summary_df   = _133_summarize_flips(rows_df)
    breakdown_df = _133_summarize_flips_by_pair_type(rows_df)

    # ---- Console output ----
    print("\nFlip summary (all pairs)")
    print("-" * 100)
    _display_cols = [
        "regime", "n_items",
        "flip_rate_vs_balanced", "flip_ci_lo", "flip_ci_hi",
        "non_tie_flip_rate_vs_balanced",
    ]
    _display_cols = [c for c in _display_cols if c in summary_df.columns]
    print(summary_df[_display_cols].round(4).to_string(index=False))

    print("\nFlip breakdown by pair type (cross-family only is the key number)")
    print("-" * 100)
    _bd_cols = ["regime", "pair_type", "n_items", "flip_rate", "flip_ci_lo", "flip_ci_hi"]
    _bd_cols = [c for c in _bd_cols if c in breakdown_df.columns]
    print(breakdown_df[_bd_cols].round(4).to_string(index=False))

    # ---- Paper interpretation guidance ----
    print("\n--- PAPER INTERPRETATION ---")
    for _, sr in summary_df.iterrows():
        if pd.isna(sr.get("flip_rate_vs_balanced")):
            continue
        flip_pct = 100 * float(sr["flip_rate_vs_balanced"])
        regime   = str(sr["regime"])
        print(
            f"  {regime:<50s}: {flip_pct:.1f}% flip rate "
            f"[{100*sr.get('flip_ci_lo', 0):.1f}–{100*sr.get('flip_ci_hi', 0):.1f}%]"
        )

    # ---- Assemble JSON output ----
    out = {
        "analysis_label":           "leaderboard_flip_simulation_primary4",
        "generated_at":             datetime.now(timezone.utc).isoformat(),
        "n_items":                  int(len(rows_df)),
        "primary_judge_families":   PRIMARY_JUDGES_133,
        "primary_candidate_families": PRIMARY_CANDS_133,
        "regimes": {
            "balanced_panel":
                "Mean support across all Primary-4 judge families (reference).",
            "single_family_judge":
                "Winner under each individual judge family.",
            "in_pair_same_family_panel":
                "Mean support using only judges matching one of the two candidate families.",
            "family_1_own_judge":
                "Winner under only the judge family matching candidate family_1.",
            "family_2_own_judge":
                "Winner under only the judge family matching candidate family_2.",
            "leave_one_family_out_balanced":
                "Balanced panel after removing one judge family.",
        },
        "summary":              summary_df.to_dict(orient="records"),
        "breakdown_by_pair_type": breakdown_df.to_dict(orient="records"),
        "paper_safe_interpretation": (
            "This simulation quantifies practical sensitivity of pairwise comparison "
            "outcomes to judge panel composition. It should be described as an internal "
            "winner-flip analysis, not as a claim about any specific public leaderboard "
            "unless external leaderboard data are explicitly added."
        ),
    }

    # ---- Write outputs (path FIRST, content SECOND) ----
    atomic_write_csv(rows_df,    FILES["leaderboard_flip_rows"])
    atomic_write_csv(summary_df, FILES["leaderboard_flip_summary"])
    atomic_write_json(out,       FILES["leaderboard_flip_simulation"])

    # Use local safe wrapper — correct argument order guaranteed
    _safe_atomic_write_text_133(
        FILES["table7_leaderboard_flip_simulation"],    # path FIRST
        _133_make_latex_table(summary_df),              # content SECOND
    )

    print("\nSaved:")
    print("  Rows JSON :", FILES["leaderboard_flip_rows"])
    print("  Summary   :", FILES["leaderboard_flip_summary"])
    print("  Full JSON :", FILES["leaderboard_flip_simulation"])
    print("  TEX       :", FILES["table7_leaderboard_flip_simulation"])


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

run_step(
    [
        FILES["leaderboard_flip_rows"],
        FILES["leaderboard_flip_summary"],
        FILES["leaderboard_flip_simulation"],
        FILES["table7_leaderboard_flip_simulation"],
    ],
    _step_leaderboard_flip_simulation_133,
    "leaderboard_flip_simulation_primary4",
    force=False,
    validators={
        FILES["leaderboard_flip_rows"]: lambda p: csv_has(
            p,
            ["prompt_id", "family_1", "family_2", "balanced_panel_winner"],
        ),
        FILES["leaderboard_flip_summary"]: lambda p: csv_has(
            p,
            ["regime", "flip_rate_vs_balanced"],
        ),
        FILES["leaderboard_flip_simulation"]: lambda p: json_has(
            p,
            ["analysis_label", "summary"],
        ),
        FILES["table7_leaderboard_flip_simulation"]: file_nonempty,
    },
)

print("\n✓ Cell 13.3 complete")

In [ ]:
# ============================================================================
# Cell 13.4 — Appendix Tables Export (Fixed + Robust)
# ============================================================================
"""
Purpose
-------
Remove all placeholder appendix language by exporting real appendix artifacts.

Creates a compact appendix manifest and supporting files for:
  - leave-one-family-out TPS results
  - human-calibration baselines
  - leaderboard / winner-flip simulation
  - multiverse / specification curve summary
  - Falcon diagnostic summary
  - GEE / mechanism model availability summary

Outputs
-------
analysis/reviewer_response/appendix_artifact_manifest.json
analysis/reviewer_response/appendix_artifact_manifest.csv
release/APPENDIX_ARTIFACTS.md
"""

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
import json
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Required globals check
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_json",
    "atomic_write_csv",
    "atomic_write_text",
    "json_has",
    "csv_has",
    "file_nonempty",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(
        f"Missing required globals. Run earlier cells first: {missing_globals}"
    )


# ---------------------------------------------------------------------------
# Paths — always reassemble, never setdefault (prevents string-drift)
# ---------------------------------------------------------------------------

if "analysis" not in PATHS:
    if "root" in PATHS:
        PATHS["analysis"] = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        PATHS["analysis"] = Path(ROOT_DIR) / "analysis"
    else:
        PATHS["analysis"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

PATHS["analysis_reviewer_response"] = Path(PATHS["analysis"]) / "reviewer_response"
PATHS["release"] = Path(
    PATHS.get("root", Path("/content/drive/MyDrive/tribal_pref_v11"))
) / "release"

Path(PATHS["analysis_reviewer_response"]).mkdir(parents=True, exist_ok=True)
Path(PATHS["release"]).mkdir(parents=True, exist_ok=True)

FILES["appendix_artifact_manifest_json"] = (
    Path(PATHS["analysis_reviewer_response"]) / "appendix_artifact_manifest.json"
)
FILES["appendix_artifact_manifest_csv"] = (
    Path(PATHS["analysis_reviewer_response"]) / "appendix_artifact_manifest.csv"
)
FILES["appendix_artifacts_md"] = (
    Path(PATHS["release"]) / "APPENDIX_ARTIFACTS.md"
)

# Pre-flight sanity check — catch path corruption before any computation
for _key in [
    "appendix_artifact_manifest_json",
    "appendix_artifact_manifest_csv",
    "appendix_artifacts_md",
]:
    _val = FILES[_key]
    assert isinstance(_val, Path), (
        f"FILES['{_key}'] must be a Path, got {type(_val)}: {str(_val)[:80]}"
    )
    assert len(str(_val)) < 500, (
        f"FILES['{_key}'] suspiciously long — likely a content string, not a path: "
        f"{str(_val)[:80]}"
    )

print("✓ Path objects validated")


# ---------------------------------------------------------------------------
# Safe atomic text writer (local copy — guards against argument-swap bug)
# ---------------------------------------------------------------------------

def _safe_atomic_write_text_134(dest_path: Path, text_content: str) -> None:
    """
    Write text_content to dest_path atomically via a .tmp file.
    Argument order: path FIRST, content SECOND.
    Hard assertion prevents content-as-path confusion (Errno 36).
    """
    dest_path = Path(dest_path)
    assert len(str(dest_path)) < 500, (
        f"dest_path looks like content, not a path "
        f"(len={len(str(dest_path))}). Arguments may be swapped."
    )
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest_path.with_name(dest_path.name + ".tmp")
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            f.write(text_content)
        tmp.replace(dest_path)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _134_path_for_key(key: str) -> Optional[Path]:
    if key not in FILES:
        return None
    val = FILES[key]
    # Reject anything that looks like content rather than a path
    if not isinstance(val, (str, Path)):
        return None
    try:
        p = Path(val)
        if len(str(p)) > 500:
            return None
        return p
    except Exception:
        return None


def _134_human_size(n_bytes: int) -> str:
    if n_bytes < 1024:
        return f"{n_bytes} B"
    if n_bytes < 1024 ** 2:
        return f"{n_bytes / 1024:.1f} KB"
    return f"{n_bytes / 1024 ** 2:.1f} MB"


def _134_file_status(
    key: str,
    description: str,
    required_for_submission: bool = True,
) -> Dict[str, Any]:
    path   = _134_path_for_key(key)
    exists = bool(path is not None and path.exists())
    size_bytes = int(path.stat().st_size) if exists else 0
    nonempty   = exists and size_bytes > 0

    return {
        "key":                    key,
        "description":            description,
        "path":                   str(path) if path is not None else "",
        "exists":                 exists,
        "size_bytes":             size_bytes,
        "size_human":             _134_human_size(size_bytes) if exists else "—",
        "required_for_submission": bool(required_for_submission),
        "status":                 "ok" if nonempty else (
                                      "empty" if exists else "missing"
                                  ),
    }


def _134_try_read_json(key: str) -> Dict[str, Any]:
    path = _134_path_for_key(key)
    if path is None or not path.exists():
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        return {"_read_error": str(e)}


def _134_try_read_csv_shape(key: str) -> Dict[str, Any]:
    path = _134_path_for_key(key)
    if path is None or not path.exists():
        return {"rows": None, "columns": None}
    try:
        df = pd.read_csv(path)
        return {"rows": int(len(df)), "columns": list(df.columns)}
    except Exception as e:
        return {"rows": None, "columns": None, "_read_error": str(e)}


def _134_extract_headline_numbers(summaries: Dict[str, Any]) -> Dict[str, Any]:
    """
    Pull the most paper-critical numbers out of the loaded JSON summaries
    so they appear at the top of the manifest and can be sanity-checked
    at a glance without opening individual files.
    """
    headline: Dict[str, Any] = {}

    # Human calibration baselines
    hcb = summaries.get("human_calibration_baselines_json", {})
    if hcb and "metrics" in hcb:
        m = hcb["metrics"]
        headline["human_llm_exact_match"]           = m.get("llm_human_exact_match")
        headline["human_majority_class_baseline"]   = m.get("human_majority_class_baseline")
        headline["human_uniform_random_baseline"]   = m.get("uniform_random_3way_baseline")
        headline["human_gain_over_majority_class"]  = m.get("absolute_gain_over_majority_class")

    # Leave-one-family-out
    lofo = summaries.get("leave_one_family_out_tps_json", {})
    if lofo and "conditions" in lofo:
        conds = lofo["conditions"]
        headline["leave_one_family_out_conditions"] = {
            k: v.get("tps") if isinstance(v, dict) else v
            for k, v in conds.items()
        }

    # Leaderboard flip
    lfs = summaries.get("leaderboard_flip_simulation", {})
    if lfs and "summary" in lfs:
        flip_summary = lfs["summary"]
        # Pull flip rates for in_pair and single-family regimes
        headline["winner_flip_rates"] = {
            row.get("regime", "unknown"): round(float(row["flip_rate_vs_balanced"]), 4)
            for row in flip_summary
            if isinstance(row, dict)
            and row.get("flip_rate_vs_balanced") is not None
            and not (
                isinstance(row.get("flip_rate_vs_balanced"), float)
                and np.isnan(row["flip_rate_vs_balanced"])
            )
        }

    # Per-family BH
    pfbh = summaries.get("per_family_bh", {})
    if pfbh and "families" in pfbh:
        headline["per_family_bh_pvalues"] = {
            fam: pfbh.get("p_two_sided_bh", {}).get(fam)
            for fam in pfbh["families"]
        }

    return headline


def _134_md_manifest(
    rows: List[Dict[str, Any]],
    summaries: Dict[str, Any],
    headline: Dict[str, Any],
    missing_required: List[str],
) -> str:
    lines: List[str] = []

    lines += [
        "# Appendix Artifact Manifest",
        "",
        f"Generated: {datetime.now(timezone.utc).isoformat()}",
        "",
        (
            "This file lists the concrete appendix artifacts for the ARR/EACL revision. "
            "There should be no appendix placeholders or TODO claims in the submitted paper."
        ),
        "",
    ]

    # Submission gate
    if missing_required:
        lines += [
            "## ⚠ BLOCKING: Missing required artifacts",
            "",
            "The following required artifacts are missing or empty. "
            "Fix these before submission:",
            "",
        ]
        for key in missing_required:
            lines.append(f"- `{key}`")
        lines.append("")
    else:
        lines += [
            "## ✓ All required artifacts present",
            "",
        ]

    # Headline numbers
    if headline:
        lines += [
            "## Headline numbers (sanity check)",
            "",
            "```json",
            json.dumps(headline, indent=2, ensure_ascii=False),
            "```",
            "",
        ]

    # Artifact table
    lines += [
        "## Core reviewer-response artifacts",
        "",
        "| Artifact | Status | Size | Required | Path |",
        "|---|:---:|---:|:---:|---|",
    ]
    for r in rows:
        status_icon = "✓" if r["status"] == "ok" else ("⚠" if r["status"] == "empty" else "❌")
        req_icon    = "✓" if r["required_for_submission"] else "—"
        lines.append(
            f"| {r['description']} "
            f"| {status_icon} {r['status']} "
            f"| {r['size_human']} "
            f"| {req_icon} "
            f"| `{Path(r['path']).name if r['path'] else '—'}` |"
        )
    lines.append("")

    # Per-artifact JSON summaries (compact, no arbitrary char truncation)
    json_keys = [
        "leave_one_family_out_tps_json",
        "human_calibration_baselines_json",
        "leaderboard_flip_simulation",
        "per_family_bh",
        "candidate_subset_audit",
        "human_agreement_metrics",
        "human_consensus_summary",
    ]

    lines += ["## Summary values", ""]

    for key in json_keys:
        obj = summaries.get(key, {})
        if not obj:
            lines += [f"### {key}", "", "_not available_", ""]
            continue

        # Trim very large objects by dropping known-large list fields
        trimmed = {
            k: v for k, v in obj.items()
            if k not in {
                "null_distributions", "bootstrap_samples", "permutation_samples",
                "raw_rows", "trial_rows", "judgment_rows",
            }
        }
        lines += [
            f"### {key}",
            "",
            "```json",
            json.dumps(trimmed, indent=2, ensure_ascii=False, default=str),
            "```",
            "",
        ]

    # CSV shapes
    csv_keys = [
        "leave_one_family_out_tps_csv",
        "human_calibration_baselines_csv",
        "leaderboard_flip_summary",
    ]
    lines += ["## CSV file shapes", ""]
    for key in csv_keys:
        shape = summaries.get(f"{key}_shape", {})
        if shape.get("rows") is not None:
            lines.append(
                f"- `{key}`: {shape['rows']} rows × {len(shape.get('columns', []))} cols"
                f" — {shape.get('columns', [])}"
            )
        else:
            lines.append(f"- `{key}`: _not available_")
    lines.append("")

    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Artifact specification list
# ---------------------------------------------------------------------------

ARTIFACT_SPECS: List[tuple] = [
    # (FILES key, human description, required_for_submission)

    # Leave-one-family-out
    ("leave_one_family_out_tps_csv",           "Leave-one-family-out TPS table (CSV)",        True),
    ("leave_one_family_out_tps_json",          "Leave-one-family-out TPS results (JSON)",     True),
    ("table5_leave_one_family_out_tps",        "LaTeX Table 5: leave-one-family-out TPS",     True),

    # Human calibration
    ("human_calibration_baselines_csv",        "Human-calibration baselines (CSV)",           True),
    ("human_calibration_baselines_json",       "Human-calibration baselines (JSON)",          True),
    ("table6_human_calibration_baselines",     "LaTeX Table 6: human-calibration baselines",  True),

    # Leaderboard / winner-flip
    ("leaderboard_flip_summary",               "Winner-flip summary table (CSV)",             True),
    ("leaderboard_flip_simulation",            "Winner-flip simulation (JSON)",               True),
    ("table7_leaderboard_flip_simulation",     "LaTeX Table 7: winner-flip simulation",       True),

    # Multiverse / specification curve
    ("multiverse_results",                     "Multiverse / specification curve artifact",   True),

    # Per-family BH
    ("per_family_bh",                          "Per-family TPS with BH correction (JSON)",    True),

    # Mechanism decomposition
    ("decomp_table",                           "Mechanism / decomposition table",             True),
    ("gee_result",                             "GEE / quasi-binomial model result",           True),

    # Falcon diagnostics
    ("falcon_diagnostics",                     "Falcon diagnostic artifact",                  True),
    ("candidate_subset_audit",                 "Candidate-subset / Falcon exclusion audit",   True),

    # A/B reversal reconciliation
    ("effective_winners_reconciliation_audit", "A/B reversal reconciliation audit",           True),

    # Human agreement
    ("human_agreement_metrics",                "Human agreement metrics (JSON)",              True),
    ("human_consensus_summary",               "Human consensus summary (JSON)",               True),
    ("human_vs_llm",                           "Human vs LLM panel comparison (CSV/JSON)",    True),

    # Optional enrichments
    ("leaderboard_flip_rows",                  "Winner-flip item-level rows (CSV)",           False),
    ("bt_residuals_primary",                   "Bradley-Terry residuals (CSV)",               False),
    ("separation_diagnostics",                 "GEE separation diagnostics (JSON)",           False),
]


# ---------------------------------------------------------------------------
# Main step
# ---------------------------------------------------------------------------

def _step_appendix_tables_export_134():
    print("=" * 100)
    print("APPENDIX TABLES EXPORT — PRIMARY-4")
    print("=" * 100)

    # ---- Build per-artifact status rows ----
    rows = [
        _134_file_status(key, desc, req)
        for key, desc, req in ARTIFACT_SPECS
    ]

    # ---- Identify blocking missing artifacts ----
    missing_required = [
        r["key"]
        for r in rows
        if r["required_for_submission"] and r["status"] != "ok"
    ]

    # ---- Collect JSON summaries ----
    summaries: Dict[str, Any] = {}

    for key in [
        "leave_one_family_out_tps_json",
        "human_calibration_baselines_json",
        "leaderboard_flip_simulation",
        "per_family_bh",
        "candidate_subset_audit",
        "human_agreement_metrics",
        "human_consensus_summary",
        "human_vs_llm",
    ]:
        obj = _134_try_read_json(key)
        if obj:
            summaries[key] = obj

    for key in [
        "leave_one_family_out_tps_csv",
        "human_calibration_baselines_csv",
        "leaderboard_flip_summary",
    ]:
        summaries[f"{key}_shape"] = _134_try_read_csv_shape(key)

    # ---- Extract headline numbers for sanity check ----
    headline = _134_extract_headline_numbers(summaries)

    # ---- Assemble manifest JSON ----
    manifest = {
        "analysis_label":        "appendix_artifact_manifest",
        "generated_at":          datetime.now(timezone.utc).isoformat(),
        "n_artifacts":           int(len(rows)),
        "n_ok":                  int(sum(r["status"] == "ok" for r in rows)),
        "n_empty":               int(sum(r["status"] == "empty" for r in rows)),
        "n_missing":             int(sum(r["status"] == "missing" for r in rows)),
        "n_required_ok":         int(sum(
                                     r["status"] == "ok"
                                     for r in rows
                                     if r["required_for_submission"]
                                 )),
        "n_required_total":      int(sum(r["required_for_submission"] for r in rows)),
        "missing_required":      missing_required,
        "submission_ready":      len(missing_required) == 0,
        "headline_numbers":      headline,
        "artifacts":             rows,
        "paper_policy": {
            "no_placeholder_appendix_text":                        True,
            "appendix_should_contain_real_tables":                 True,
            "main_paper_should_not_depend_on_appendix_for_core_claim": True,
            "human_calibration_is_external_reference_not_validation": True,
            "falcon_excluded_from_headline":                       True,
        },
    }

    df = pd.DataFrame(rows)

    # ---- Write outputs (path FIRST, content SECOND — correct order) ----
    atomic_write_json(manifest, FILES["appendix_artifact_manifest_json"])
    atomic_write_csv(df,        FILES["appendix_artifact_manifest_csv"])

    # Local safe wrapper — correct argument order guaranteed
    _safe_atomic_write_text_134(
        FILES["appendix_artifacts_md"],                                   # path FIRST
        _134_md_manifest(rows, summaries, headline, missing_required),    # content SECOND
    )

    # ---- Console summary ----
    print("\nAppendix artifact manifest")
    print("-" * 100)

    _display_cols = ["key", "status", "size_human", "required_for_submission"]
    _display_cols = [c for c in _display_cols if c in df.columns]
    print(df[_display_cols].to_string(index=False))

    print(
        f"\nRequired: {manifest['n_required_ok']} / {manifest['n_required_total']} OK"
    )
    print(f"Optional: {manifest['n_ok'] - manifest['n_required_ok']} OK")
    print(f"Missing:  {manifest['n_missing']} | Empty: {manifest['n_empty']}")

    if missing_required:
        print("\n⚠  BLOCKING — Missing required artifacts:")
        for key in missing_required:
            print(f"   ❌  {key}")
        print(
            "\n   Fix these before running Cell 13.5 / 13.6 (submission readiness audit)."
        )
    else:
        print("\n✓  All required artifacts present — appendix is placeholder-free.")

    if headline:
        print("\nHeadline numbers (sanity check):")
        for k, v in headline.items():
            if isinstance(v, dict):
                print(f"  {k}:")
                for kk, vv in v.items():
                    print(f"    {kk}: {vv}")
            elif isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")

    print("\nSaved:")
    print("  JSON:", FILES["appendix_artifact_manifest_json"])
    print("  CSV :", FILES["appendix_artifact_manifest_csv"])
    print("  MD  :", FILES["appendix_artifacts_md"])


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

run_step(
    [
        FILES["appendix_artifact_manifest_json"],
        FILES["appendix_artifact_manifest_csv"],
        FILES["appendix_artifacts_md"],
    ],
    _step_appendix_tables_export_134,
    "appendix_tables_export",
    force=False,
    validators={
        FILES["appendix_artifact_manifest_json"]: lambda p: json_has(
            p,
            ["analysis_label", "artifacts", "paper_policy", "submission_ready"],
        ),
        FILES["appendix_artifact_manifest_csv"]: lambda p: csv_has(
            p,
            ["key", "status", "path"],
        ),
        FILES["appendix_artifacts_md"]: file_nonempty,
    },
)

print("\n✓ Cell 13.4 complete")

In [ ]:
# ============================================================================
# Cell 13.5 — Revised Paper Evidence Summary (Fixed + Robust)
# ============================================================================
"""
Purpose
-------
Create a clean evidence summary for rewriting the ARR/EACL paper.

Converts all reviewer-response analyses into paper-safe claims, and
explicitly flags any submission blockers (missing CI, p-value, etc.).

Outputs
-------
analysis/reviewer_response/revised_paper_evidence_summary.json
release/REVISED_PAPER_EVIDENCE_SUMMARY.md
"""

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple
import json
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Required globals check
# ---------------------------------------------------------------------------

required_globals = [
    "FILES",
    "PATHS",
    "PRIMARY_JUDGE_FAMILIES",
    "PRIMARY_CANDIDATE_FAMILIES",
    "run_step",
    "atomic_write_json",
    "atomic_write_text",
    "json_has",
    "file_nonempty",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(
        f"Missing required globals. Run earlier cells first: {missing_globals}"
    )


# ---------------------------------------------------------------------------
# Paths — always reassemble, never setdefault (prevents string-drift)
# ---------------------------------------------------------------------------

if "analysis" not in PATHS:
    if "root" in PATHS:
        PATHS["analysis"] = Path(PATHS["root"]) / "analysis"
    elif "ROOT_DIR" in globals():
        PATHS["analysis"] = Path(ROOT_DIR) / "analysis"
    else:
        PATHS["analysis"] = Path("/content/drive/MyDrive/tribal_pref_v11/analysis")

PATHS["analysis_reviewer_response"] = Path(PATHS["analysis"]) / "reviewer_response"
PATHS["release"] = Path(
    PATHS.get("root", Path("/content/drive/MyDrive/tribal_pref_v11"))
) / "release"

Path(PATHS["analysis_reviewer_response"]).mkdir(parents=True, exist_ok=True)
Path(PATHS["release"]).mkdir(parents=True, exist_ok=True)

FILES["revised_paper_evidence_summary_json"] = (
    Path(PATHS["analysis_reviewer_response"]) / "revised_paper_evidence_summary.json"
)
FILES["revised_paper_evidence_summary_md"] = (
    Path(PATHS["release"]) / "REVISED_PAPER_EVIDENCE_SUMMARY.md"
)

# Pre-flight sanity check
for _key in [
    "revised_paper_evidence_summary_json",
    "revised_paper_evidence_summary_md",
]:
    _val = FILES[_key]
    assert isinstance(_val, Path), (
        f"FILES['{_key}'] must be a Path, got {type(_val)}: {str(_val)[:80]}"
    )
    assert len(str(_val)) < 500, (
        f"FILES['{_key}'] suspiciously long — likely a content string, not a path: "
        f"{str(_val)[:80]}"
    )

print("✓ Path objects validated")


# ---------------------------------------------------------------------------
# Safe atomic text writer
# ---------------------------------------------------------------------------

def _safe_atomic_write_text_135(dest_path: Path, text_content: str) -> None:
    """Path FIRST, content SECOND. Hard assertion prevents Errno 36."""
    dest_path = Path(dest_path)
    assert len(str(dest_path)) < 500, (
        f"dest_path looks like content, not a path "
        f"(len={len(str(dest_path))}). Arguments may be swapped."
    )
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest_path.with_name(dest_path.name + ".tmp")
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            f.write(text_content)
        tmp.replace(dest_path)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _135_safe_path(key: str) -> Optional[Path]:
    """Return a Path for FILES[key] only if it looks like an actual path."""
    if key not in FILES:
        return None
    val = FILES[key]
    try:
        p = Path(val)
        if len(str(p)) > 500:
            return None
        return p
    except Exception:
        return None


def _135_read_json_key(key: str) -> Dict[str, Any]:
    path = _135_safe_path(key)
    if path is None or not path.exists():
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠  Could not read {key}: {e}")
        return {}


def _135_read_csv_key(key: str) -> pd.DataFrame:
    path = _135_safe_path(key)
    if path is None or not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"⚠  Could not read CSV {key}: {e}")
        return pd.DataFrame()


def _135_fmt(x, digits: int = 4) -> str:
    """Format a scalar for display. Returns the string 'NA' only for genuinely missing values."""
    if x is None:
        return "NA"
    try:
        f = float(x)
        if np.isnan(f):
            return "NA"
        return f"{f:.{digits}f}"
    except (TypeError, ValueError):
        return str(x)


def _135_extract_ci(bootstrap: Dict[str, Any]) -> Tuple[Optional[float], Optional[float]]:
    """Robustly extract 95% CI bounds from the bootstrap result JSON."""
    ci_low  = bootstrap.get("ci_95_low")
    ci_high = bootstrap.get("ci_95_high")

    if ci_low is None or ci_high is None:
        ci = bootstrap.get("ci_95")
        if isinstance(ci, (list, tuple)) and len(ci) == 2:
            ci_low, ci_high = ci[0], ci[1]

    # Try nested structure some versions use
    if ci_low is None:
        ci_low  = bootstrap.get("ci", {}).get("low")
        ci_high = bootstrap.get("ci", {}).get("high")

    try:
        ci_low  = float(ci_low)  if ci_low  is not None else None
        ci_high = float(ci_high) if ci_high is not None else None
    except (TypeError, ValueError):
        ci_low = ci_high = None

    return ci_low, ci_high


def _135_extract_p(perm: Dict[str, Any]) -> Optional[float]:
    """Robustly extract the primary p-value from the permutation result JSON."""
    for key in [
        "p_two_sided_bh_global",
        "p_two",
        "p_two_sided",
        "p_value",
        "p",
        "p_one_sided",
    ]:
        val = perm.get(key)
        if val is not None:
            try:
                return float(val)
            except (TypeError, ValueError):
                continue
    return None


def _135_per_family_table(per_family: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Extract per-family TPS + BH p-values into a flat list of records.
    Handles the multiple JSON shapes that different pipeline versions produce.
    """
    rows: List[Dict[str, Any]] = []

    # Try to find per-family TPS values
    tps_map: Dict[str, Any] = {}
    for key in ["per_family_tps", "per_family", "results", "family_tps"]:
        candidate = per_family.get(key)
        if isinstance(candidate, dict):
            tps_map = candidate
            break

    # Try to find BH-corrected p-values
    p_two_bh: Dict[str, Any] = per_family.get("p_two_sided_bh", {}) or {}
    p_one_bh: Dict[str, Any] = per_family.get("p_one_sided_bh", {}) or {}

    # Try to find bootstrap CIs per family
    bs_ci: Dict[str, Any] = per_family.get("bootstrap_ci_per_family", {}) or {}

    families = sorted(
        set(tps_map.keys())
        | set(p_two_bh.keys())
        | set(p_one_bh.keys())
    )

    for fam in families:
        tps_val = tps_map.get(fam)
        p2_bh   = p_two_bh.get(fam)
        p1_bh   = p_one_bh.get(fam)
        fam_ci  = bs_ci.get(fam, {})
        ci_lo   = fam_ci.get("ci_95_low")  or fam_ci.get("low")
        ci_hi   = fam_ci.get("ci_95_high") or fam_ci.get("high")

        rows.append({
            "family":          fam,
            "tps":             _135_fmt(tps_val),
            "ci_lo":           _135_fmt(ci_lo) if ci_lo is not None else "NA",
            "ci_hi":           _135_fmt(ci_hi) if ci_hi is not None else "NA",
            "p_two_sided_bh":  _135_fmt(p2_bh, digits=4) if p2_bh is not None else "NA",
            "p_one_sided_bh":  _135_fmt(p1_bh, digits=4) if p1_bh is not None else "NA",
            "significant_bh":  (
                bool(float(p2_bh) < 0.05)
                if p2_bh is not None
                else None
            ),
        })

    return rows


def _135_derive_submission_blockers(
    headline_tps: Optional[float],
    ci_low: Optional[float],
    ci_high: Optional[float],
    p_perm: Optional[float],
    per_family_rows: List[Dict[str, Any]],
    appendix_manifest: Dict[str, Any],
    human_metrics: Dict[str, Any],
) -> List[str]:
    """
    Return a list of blocking issues that must be resolved before submission.
    Empty list = submission-ready on these criteria.
    """
    blockers: List[str] = []

    if headline_tps is None or np.isnan(headline_tps):
        blockers.append("PRIMARY TPS is missing or NaN — run Cell 6.2.")

    if ci_low is None or ci_high is None:
        blockers.append(
            "Bootstrap CI is missing — run Cell 7.1 and confirm "
            "cluster_bootstrap_tps_primary.json contains ci_95_low / ci_95_high."
        )
    elif ci_low >= 0.0 and headline_tps is not None:
        # CI excludes zero from above — this is the good case, no blocker
        pass
    elif ci_high <= 0.0:
        blockers.append(
            f"Bootstrap 95% CI [{_135_fmt(ci_low)}, {_135_fmt(ci_high)}] is entirely "
            f"below zero — TPS is negative, core claim fails."
        )
    elif ci_low < 0.0 and ci_high > 0.0:
        blockers.append(
            f"Bootstrap 95% CI [{_135_fmt(ci_low)}, {_135_fmt(ci_high)}] straddles "
            f"zero — CI does not exclude zero, cannot claim significant positive TPS."
        )

    if p_perm is None or np.isnan(p_perm):
        blockers.append(
            "Permutation p-value is missing — run Cell 7.2 / 7.3 and confirm "
            "permutation_result.json contains a p_two_sided field."
        )
    elif float(p_perm) >= 0.05:
        blockers.append(
            f"Permutation p = {_135_fmt(p_perm)} ≥ 0.05 — global TPS not significant."
        )

    # Multiverse ≥ 85% positive
    mv_pct = appendix_manifest.get("headline_numbers", {}).get("multiverse_pct_positive")
    if mv_pct is None:
        blockers.append(
            "Multiverse % positive is not in appendix manifest — "
            "confirm Cell 9.1 ran and multiverse_results.json was saved."
        )
    elif float(mv_pct) < 0.85:
        blockers.append(
            f"Multiverse % positive = {float(mv_pct):.1%} < 85% threshold — "
            "specification curve does not support the core claim robustly."
        )

    # Appendix completeness
    n_missing = appendix_manifest.get("n_missing", 0)
    if n_missing and int(n_missing) > 0:
        blockers.append(
            f"Appendix manifest shows {n_missing} missing required artifact(s) — "
            "run Cell 13.4 and fix all ❌ items."
        )

    # Human calibration: warn (not hard block) if gain < 0.05
    if human_metrics:
        gain = human_metrics.get("absolute_gain_over_majority_class")
        if gain is not None and float(gain) < 0.05:
            blockers.append(
                f"Human calibration gain over majority-class baseline is only "
                f"{float(gain):.3f} — do NOT claim strong human validation. "
                "Use hedged language: 'consistent with' / 'external reference point'."
            )

    return blockers


def _135_make_md(summary: Dict[str, Any]) -> str:
    lines: List[str] = []

    lines += [
        "# Revised Paper Evidence Summary",
        "",
        f"Generated: {summary['generated_at']}",
        f"Target: {summary.get('target', 'ARR August 2026 → EACL 2027')}",
        "",
    ]

    # Submission gate — most important section, put it first
    blockers = summary.get("submission_blockers", [])
    if blockers:
        lines += [
            "## ⚠ SUBMISSION BLOCKERS",
            "",
            "Fix ALL of the following before submitting:",
            "",
        ]
        for b in blockers:
            lines.append(f"- ❌  {b}")
        lines.append("")
    else:
        lines += [
            "## ✓ No submission blockers found",
            "",
            "All gating criteria pass. Proceed with paper revision.",
            "",
        ]

    # Headline evidence
    lines += ["## Headline evidence", ""]
    he = summary.get("headline_evidence", {})
    for k, v in he.items():
        lines.append(f"- **{k}**: {v}")
    lines.append("")

    # Revised thesis
    lines += ["## Revised thesis", "", summary.get("revised_thesis", ""), ""]

    # Per-family table
    pf = summary.get("per_family_table", [])
    if pf:
        lines += [
            "## Per-family TPS and BH-corrected p-values",
            "",
            "| Family | TPS | CI low | CI high | p (two-sided BH) | Significant? |",
            "|---|---:|---:|---:|---:|:---:|",
        ]
        for r in pf:
            sig = "✓" if r.get("significant_bh") else ("—" if r.get("significant_bh") is None else "✗")
            lines.append(
                f"| {r['family']} "
                f"| {r['tps']} "
                f"| {r['ci_lo']} "
                f"| {r['ci_hi']} "
                f"| {r['p_two_sided_bh']} "
                f"| {sig} |"
            )
        lines.append("")

    # Paper-safe claims
    lines += ["## Paper-safe contribution claims", ""]
    for c in summary.get("paper_safe_contribution_claims", []):
        lines.append(f"- {c}")
    lines.append("")

    # Claims to avoid
    lines += ["## Claims to avoid", ""]
    for c in summary.get("claims_to_avoid", []):
        lines.append(f"- ⚠  {c}")
    lines.append("")

    # Leave-one-family-out
    loo = summary.get("reviewer_response_analyses", {}).get("leave_one_family_out_tps", [])
    if loo:
        lines += [
            "## Leave-one-family-out TPS",
            "",
            "| Condition | TPS | CI low | CI high |",
            "|---|---:|---:|---:|",
        ]
        for r in loo:
            cond  = r.get("condition") or r.get("removed_family", "unknown")
            tps   = _135_fmt(r.get("tps"))
            c_lo  = _135_fmt(r.get("ci_95_low"))
            c_hi  = _135_fmt(r.get("ci_95_high"))
            lines.append(f"| {cond} | {tps} | {c_lo} | {c_hi} |")
        lines.append("")

    # Human calibration
    hm = summary.get("reviewer_response_analyses", {}).get("human_calibration_baselines", {})
    if hm:
        lines += ["## Human calibration baselines", ""]
        for k, v in hm.items():
            lines.append(f"- **{k}**: {_135_fmt(v) if isinstance(v, (int, float)) else v}")
        lines.append("")

    # Winner-flip simulation
    flip = summary.get("reviewer_response_analyses", {}).get("leaderboard_flip_simulation", [])
    if flip:
        lines += [
            "## Winner-flip simulation",
            "",
            "| Regime | n | Flip rate | Non-tie flip rate |",
            "|---|---:|---:|---:|",
        ]
        for r in flip:
            if isinstance(r, dict):
                regime   = str(r.get("regime", "unknown")).replace("_", " ")
                n        = r.get("n_items", "NA")
                flip_r   = _135_fmt(r.get("flip_rate_vs_balanced"))
                nt_flip  = _135_fmt(r.get("non_tie_flip_rate_vs_balanced"))
                lines.append(f"| {regime} | {n} | {flip_r} | {nt_flip} |")
        lines.append("")

    # Recommended sentences
    lines += [
        "## Recommended abstract sentence",
        "",
        summary.get("recommended_abstract_sentence", ""),
        "",
        "## Recommended conclusion sentence",
        "",
        summary.get("recommended_conclusion_sentence", ""),
        "",
    ]

    # Full JSON dump for machine readers (no char truncation)
    lines += [
        "## Full JSON (machine-readable)",
        "",
        "```json",
        json.dumps(
            {
                k: v
                for k, v in summary.items()
                if k not in {"reviewer_response_analyses"}
            },
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        "```",
        "",
    ]

    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Main step
# ---------------------------------------------------------------------------

def _step_revised_paper_evidence_summary_135():
    print("=" * 100)
    print("REVISED PAPER EVIDENCE SUMMARY — PRIMARY-4")
    print("=" * 100)

    # ---- Load all upstream artifacts ----
    tps_summary     = _135_read_json_key("tps_summary_primary") or _135_read_json_key("tps_summary")
    bootstrap       = _135_read_json_key("cluster_bootstrap_tps_primary") or _135_read_json_key("bootstrap_result")
    perm            = _135_read_json_key("permutation_result") or _135_read_json_key("permutation_tps")
    per_family_raw  = _135_read_json_key("per_family_bh")
    loo_df          = _135_read_csv_key("leave_one_family_out_tps_csv")
    human_base      = _135_read_json_key("human_calibration_baselines_json")
    flip_df         = _135_read_csv_key("leaderboard_flip_summary")
    appendix_mfst   = _135_read_json_key("appendix_artifact_manifest_json")

    # ---- Extract headline numbers ----
    headline_tps = (
        bootstrap.get("observed_tps")
        or bootstrap.get("tps")
        or tps_summary.get("tps")
    )
    try:
        headline_tps = float(headline_tps) if headline_tps is not None else None
    except (TypeError, ValueError):
        headline_tps = None

    ci_low, ci_high = _135_extract_ci(bootstrap)
    p_perm          = _135_extract_p(perm)
    human_metrics   = human_base.get("metrics", {}) if isinstance(human_base, dict) else {}

    # ---- Per-family table ----
    per_family_table = _135_per_family_table(per_family_raw)

    # ---- Leave-one-family-out records ----
    loo_records: List[Dict[str, Any]] = []
    if not loo_df.empty:
        keep_cols = [
            c for c in [
                "condition", "removed_family", "tps",
                "ci_95_low", "ci_95_high", "n_prompts",
            ]
            if c in loo_df.columns
        ]
        loo_records = loo_df[keep_cols].to_dict(orient="records")

    # ---- Winner-flip records ----
    flip_records: List[Dict[str, Any]] = (
        flip_df.to_dict(orient="records") if not flip_df.empty else []
    )

    # ---- CI gate ----
    ci_excludes_zero = (
        ci_low is not None
        and ci_high is not None
        and ci_low > 0.0
    )

    # ---- Submission blockers ----
    blockers = _135_derive_submission_blockers(
        headline_tps    = headline_tps,
        ci_low          = ci_low,
        ci_high         = ci_high,
        p_perm          = p_perm,
        per_family_rows = per_family_table,
        appendix_manifest = appendix_mfst,
        human_metrics   = human_metrics,
    )

    # ---- Assemble summary dict ----
    summary: Dict[str, Any] = {
        "analysis_label":   "revised_paper_evidence_summary",
        "generated_at":     datetime.now(timezone.utc).isoformat(),
        "target":           "ARR August 2026 → EACL 2027",
        "primary_panel": {
            "judge_families":     [str(x).strip().lower() for x in PRIMARY_JUDGE_FAMILIES],
            "candidate_families": [str(x).strip().lower() for x in PRIMARY_CANDIDATE_FAMILIES],
        },

        "submission_blockers":     blockers,
        "submission_ready":        len(blockers) == 0,

        "headline_evidence": {
            "primary_tps":                          _135_fmt(headline_tps),
            "ci_95":                                f"[{_135_fmt(ci_low)}, {_135_fmt(ci_high)}]",
            "ci_excludes_zero":                     ci_excludes_zero,
            "permutation_p":                        _135_fmt(p_perm),
            "permutation_p_significant":            (
                                                        bool(float(p_perm) < 0.05)
                                                        if p_perm is not None else None
                                                    ),
            "per_family_table_rows":                len(per_family_table),
            "leave_one_family_out_available":       not loo_df.empty,
            "human_llm_exact_match":                _135_fmt(
                                                        human_metrics.get("llm_human_exact_match")
                                                    ),
            "human_majority_class_baseline":        _135_fmt(
                                                        human_metrics.get("human_majority_class_baseline")
                                                    ),
            "human_gain_over_majority_class":       _135_fmt(
                                                        human_metrics.get("absolute_gain_over_majority_class")
                                                    ),
            "leaderboard_flip_simulation_available": not flip_df.empty,
            "appendix_submission_ready":            bool(
                                                        appendix_mfst.get("submission_ready", False)
                                                    ),
        },

        "per_family_table": per_family_table,

        "revised_thesis": (
            "We propose a transparent audit framework for evaluator provenance effects "
            "in LLM-as-judge panels. Applied to a controlled Primary-4 open-weight panel, "
            "the audit detects a statistically robust but heterogeneous family-conditioned "
            "signal. The point is not that all model families behave alike; the point is "
            "that family-conditioned effects can be measured and should be audited rather "
            "than assumed absent."
        ),

        "paper_safe_contribution_claims": [
            "The paper introduces a transparent signed-matrix audit for family-conditioned "
            "evaluator effects in LLM-as-judge panels.",
            "The Primary-4 case study shows a positive average same-family signal "
            f"(TPS = {_135_fmt(headline_tps)}, "
            f"95% CI [{_135_fmt(ci_low)}, {_135_fmt(ci_high)}]).",
            "The signal is heterogeneous across families — per-family auditing is "
            "necessary, not just a global average.",
            "Human calibration is used as an external reference point and "
            "task-difficulty diagnostic, not as a perfect gold standard.",
            "Winner-flip simulation quantifies practical sensitivity of pairwise "
            "comparison outcomes to judge-panel composition.",
            "Leave-one-family-out analysis confirms the global signal is not "
            "driven by a single judge family.",
        ],

        "claims_to_avoid": [
            "Do not claim all LLM judges generally prefer their own family — "
            "the signal is heterogeneous.",
            "Do not claim TPS proves causal recognition of family identity — "
            "it is a preference signal, not a mechanism proof.",
            "Do not claim human calibration strongly validates LLM judging if "
            "the gain over the majority-class baseline is < 0.05.",
            "Do not include Falcon in the headline analysis or abstract.",
            "Do not present appendix content as future work or placeholder TODOs.",
            "Do not use the word 'sycophancy' for the same-family preference signal "
            "without adding a definitional footnote distinguishing it from "
            "user-flattery sycophancy.",
        ],

        "reviewer_response_analyses": {
            "leave_one_family_out_tps":      loo_records,
            "human_calibration_baselines":   human_metrics,
            "leaderboard_flip_simulation":   flip_records,
            "per_family_results":            {r["family"]: r for r in per_family_table},
        },

        "recommended_abstract_sentence": (
            "In a Primary-4 open-weight case study, the audit detects a positive "
            "same-family signal that is statistically robust but heterogeneous "
            f"(TPS = {_135_fmt(headline_tps)}, "
            f"95\\% CI [{_135_fmt(ci_low)}, {_135_fmt(ci_high)}], "
            f"permutation $p = {_135_fmt(p_perm)}$), "
            "with the strongest effect concentrated in one family and another near "
            "zero, motivating per-family auditing rather than assumptions of either "
            "neutrality or universal bias."
        ),

        "recommended_conclusion_sentence": (
            "Evaluator provenance effects should therefore be treated as auditable, "
            "panel-specific properties of LLM-as-judge systems, not as universal "
            "constants or negligible implementation details."
        ),
    }

    # ---- Write outputs (path FIRST, content SECOND) ----
    atomic_write_json(summary, FILES["revised_paper_evidence_summary_json"])

    _safe_atomic_write_text_135(
        FILES["revised_paper_evidence_summary_md"],     # path FIRST
        _135_make_md(summary),                          # content SECOND
    )

    # ---- Console summary ----
    print("\nHeadline evidence")
    print("-" * 100)
    print(json.dumps(summary["headline_evidence"], indent=2))

    print("\nPer-family TPS")
    print("-" * 100)
    if per_family_table:
        pf_df = pd.DataFrame(per_family_table)
        print(pf_df.to_string(index=False))
    else:
        print("  (not available — check per_family_bh JSON)")

    if blockers:
        print("\n⚠  SUBMISSION BLOCKERS")
        print("-" * 100)
        for b in blockers:
            print(f"  ❌  {b}")
    else:
        print("\n✓  No submission blockers — all gating criteria pass.")

    print("\nSaved:")
    print("  JSON:", FILES["revised_paper_evidence_summary_json"])
    print("  MD  :", FILES["revised_paper_evidence_summary_md"])


# ---------------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------------

run_step(
    [
        FILES["revised_paper_evidence_summary_json"],
        FILES["revised_paper_evidence_summary_md"],
    ],
    _step_revised_paper_evidence_summary_135,
    "revised_paper_evidence_summary",
    force=False,
    validators={
        FILES["revised_paper_evidence_summary_json"]: lambda p: json_has(
            p,
            [
                "analysis_label",
                "revised_thesis",
                "headline_evidence",
                "paper_safe_contribution_claims",
                "submission_blockers",
                "submission_ready",
            ],
        ),
        FILES["revised_paper_evidence_summary_md"]: file_nonempty,
    },
)

print("\n✓ Cell 13.5 complete")

In [ ]:
# ============================================================================
# Cell 13.6 — Final Submission Readiness Audit v2
# ============================================================================
"""
Purpose
-------
A stricter final audit for the revised ARR/EACL submission.

This audit checks whether the new reviewer-response analyses are present:
- leave-one-family-out TPS
- human baselines
- leaderboard/winner-flip simulation
- appendix artifact manifest
- revised evidence summary

Outputs
-------
release/final_submission_readiness_audit_v2.json

This does not replace the earlier final audit. It extends it.
"""

from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List
import json
import pandas as pd


required_globals = [
    "FILES",
    "PATHS",
    "run_step",
    "atomic_write_json",
    "json_has",
    "file_nonempty",
]

missing_globals = [x for x in required_globals if x not in globals()]
if missing_globals:
    raise RuntimeError(f"Missing required globals. Run earlier cells first: {missing_globals}")


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------

PATHS.setdefault(
    "release",
    Path(PATHS.get("root", Path("/content/drive/MyDrive/tribal_pref_v11"))) / "release",
)

Path(PATHS["release"]).mkdir(parents=True, exist_ok=True)

FILES.setdefault(
    "final_submission_readiness_audit_v2",
    Path(PATHS["release"]) / "final_submission_readiness_audit_v2.json",
)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _136_status(key: str, required: bool = True) -> Dict[str, Any]:
    path = FILES.get(key)
    if path is None:
        return {
            "key": key,
            "path": "",
            "required": required,
            "exists": False,
            "size_bytes": 0,
            "status": "missing_key",
        }

    path = Path(path)
    exists = path.exists()
    size = int(path.stat().st_size) if exists else 0

    if exists and size > 0:
        status = "ok"
    else:
        status = "missing_or_empty"

    return {
        "key": key,
        "path": str(path),
        "required": required,
        "exists": bool(exists),
        "size_bytes": size,
        "status": status,
    }


def _136_read_json(key: str) -> Dict[str, Any]:
    path = FILES.get(key)
    if path is None:
        return {}
    path = Path(path)
    if not path.exists():
        return {}
    try:
        return json.load(open(path, "r", encoding="utf-8"))
    except Exception:
        return {}


def _136_hard_error(condition: bool, message: str, errors: List[str]):
    if condition:
        errors.append(message)


def _136_warning(condition: bool, message: str, warnings: List[str]):
    if condition:
        warnings.append(message)


# ---------------------------------------------------------------------------
# Main step
# ---------------------------------------------------------------------------

def _step_final_submission_readiness_audit_v2_136():
    required_keys = [
        # Original core evidence
        "tps_summary_primary",
        "cluster_bootstrap_tps_primary",
        "per_family_bh",
        "human_agreement_metrics",
        "human_consensus_summary",
        "human_vs_llm",
        "final_report",

        # Phase 13 evidence
        "leave_one_family_out_tps_csv",
        "leave_one_family_out_tps_json",
        "human_calibration_baselines_json",
        "human_calibration_baselines_csv",
        "leaderboard_flip_summary",
        "leaderboard_flip_simulation",
        "appendix_artifact_manifest_json",
        "appendix_artifacts_md",
        "revised_paper_evidence_summary_json",
        "revised_paper_evidence_summary_md",

        # LaTeX tables
        "table5_leave_one_family_out_tps",
        "table6_human_calibration_baselines",
        "table7_leaderboard_flip_simulation",
    ]

    optional_but_desirable_keys = [
        "decomp_table",
        "gee_result",
        "multiverse_results",
        "falcon_diagnostics",
        "candidate_subset_audit",
        "effective_winners_reconciliation_audit",
        "model_card",
        "data_statement",
        "reproduce_readme",
        "appendix_artifact_manifest_csv",
    ]

    required_status = [_136_status(k, required=True) for k in required_keys]
    optional_status = [_136_status(k, required=False) for k in optional_but_desirable_keys]

    errors = []
    warnings = []

    missing_required = [r for r in required_status if r["status"] != "ok"]
    missing_optional = [r for r in optional_status if r["status"] != "ok"]

    _136_hard_error(
        len(missing_required) > 0,
        f"Missing required revised-submission artifacts: {[r['key'] for r in missing_required]}",
        errors,
    )

    _136_warning(
        len(missing_optional) > 0,
        f"Missing optional/desirable artifacts: {[r['key'] for r in missing_optional]}",
        warnings,
    )

    # Check appendix manifest.
    appendix_manifest = _136_read_json("appendix_artifact_manifest_json")
    if appendix_manifest:
        _136_warning(
            int(appendix_manifest.get("n_missing_or_empty", 0)) > 0,
            "Appendix manifest still reports missing_or_empty artifacts. Remove claims or generate files.",
            warnings,
        )
    else:
        _136_hard_error(
            True,
            "Appendix artifact manifest could not be read.",
            errors,
        )

    # Check revised evidence summary.
    evidence = _136_read_json("revised_paper_evidence_summary_json")
    if evidence:
        claims_to_avoid = evidence.get("claims_to_avoid", [])
        _136_warning(
            len(claims_to_avoid) == 0,
            "Revised evidence summary does not list claims to avoid.",
            warnings,
        )
    else:
        _136_hard_error(
            True,
            "Revised paper evidence summary could not be read.",
            errors,
        )

    # Check human baselines.
    human_base = _136_read_json("human_calibration_baselines_json")
    if human_base:
        metrics = human_base.get("metrics", {})
        exact = metrics.get("llm_human_exact_match")
        majority = metrics.get("human_majority_class_baseline")

        if exact is not None and majority is not None:
            _136_warning(
                float(exact) <= float(majority) + 0.02,
                "LLM-human exact match is close to majority-class baseline. Do not overclaim human validation.",
                warnings,
            )
    else:
        _136_hard_error(
            True,
            "Human calibration baseline JSON could not be read.",
            errors,
        )

    # Check leave-one-family-out.
    loo = _136_read_json("leave_one_family_out_tps_json")
    if loo:
        conditions = loo.get("conditions", {})
        _136_hard_error(
            "without_qwen" not in conditions,
            "Leave-one-family-out results do not include without_qwen.",
            errors,
        )
    else:
        _136_hard_error(
            True,
            "Leave-one-family-out JSON could not be read.",
            errors,
        )

    complete = len(errors) == 0

    audit = {
        "analysis_label": "final_submission_readiness_audit_v2",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "complete": bool(complete),
        "final_submission_ready": bool(complete),
        "n_required_artifacts": int(len(required_status)),
        "n_required_ok": int(sum(r["status"] == "ok" for r in required_status)),
        "n_required_missing_or_empty": int(sum(r["status"] != "ok" for r in required_status)),
        "n_optional_artifacts": int(len(optional_status)),
        "n_optional_ok": int(sum(r["status"] == "ok" for r in optional_status)),
        "n_optional_missing_or_empty": int(sum(r["status"] != "ok" for r in optional_status)),
        "errors": errors,
        "warnings": warnings,
        "required_status": required_status,
        "optional_status": optional_status,
        "paper_policy": {
            "main_claim": (
                "Audit framework detects robust but heterogeneous family-conditioned "
                "signal in Primary-4 case study."
            ),
            "do_not_claim_universal_family_bias": True,
            "do_not_claim_causal_mechanism_without_mechanism_test": True,
            "do_not_overclaim_human_calibration": True,
            "falcon_headline_excluded": True,
            "appendix_placeholders_allowed": False,
        },
    }

    atomic_write_json(audit, FILES["final_submission_readiness_audit_v2"])

    print("\nFinal submission readiness audit v2")
    print("-" * 100)
    print(json.dumps({
        "complete": audit["complete"],
        "final_submission_ready": audit["final_submission_ready"],
        "n_required_ok": audit["n_required_ok"],
        "n_required_missing_or_empty": audit["n_required_missing_or_empty"],
        "n_optional_ok": audit["n_optional_ok"],
        "n_optional_missing_or_empty": audit["n_optional_missing_or_empty"],
        "n_errors": len(errors),
        "n_warnings": len(warnings),
    }, indent=2))

    if errors:
        print("\nERRORS")
        print("-" * 100)
        for e in errors:
            print("❌", e)

    if warnings:
        print("\nWARNINGS")
        print("-" * 100)
        for w in warnings:
            print("⚠", w)

    print("\nSaved:", FILES["final_submission_readiness_audit_v2"])


run_step(
    [FILES["final_submission_readiness_audit_v2"]],
    _step_final_submission_readiness_audit_v2_136,
    "final_submission_readiness_audit_v2",
    force=False,
    validators={
        FILES["final_submission_readiness_audit_v2"]: lambda p: json_has(
            p,
            ["analysis_label", "complete", "final_submission_ready", "errors", "warnings"],
        )
    },
)

print("\n✓ Cell 13.6 complete")

## PHASE 14 — Cache-aware reviewer-response diagnostics

This release cell validates existing Phase 14 outputs before doing expensive recomputation. Set `PHASE14_FORCE_RECOMPUTE = True` only when you intentionally want to regenerate the diagnostics.


In [ ]:
# ============================================================
# PHASE 14 — Cache-aware reviewer-response diagnostics
# ============================================================
# Purpose:
#   Runs the three reviewer-response checks only when their outputs are missing
#   or invalid. If outputs already exist, this cell loads and summarizes them
#   instead of rerunning the expensive 5,000-cluster bootstrap.
#
# Set PHASE14_FORCE_RECOMPUTE = True only when you intentionally want to
# regenerate the outputs from master_judgments.csv.
# ============================================================

from pathlib import Path
import json
import subprocess

import numpy as np
import pandas as pd

try:
    DRIVE = Path(PATHS["root"])
except Exception:
    DRIVE = Path("/content/drive/MyDrive/tribal_pref_v11")

OUT = DRIVE / "analysis" / "phase14"
OUT.mkdir(parents=True, exist_ok=True)

PRIMARY4 = ["llama", "qwen", "gemma", "yi"]
N_BOOT = 5000
SEED = 20260502
PHASE14_FORCE_RECOMPUTE = False

# New canonical name plus backwards-compatible legacy name.
CI_TPS_PATH = OUT / "per_family_tps_bootstrap_ci.csv"
CI_LEGACY_PATH = OUT / "per_family_fps_bootstrap_ci.csv"  # legacy filename from earlier scratch cell
INCON_CSV_PATH = OUT / "per_family_inconsistency_rates.csv"
INCON_JSON_PATH = OUT / "inconsistency_test.json"
EMB_JSON_PATH = OUT / "embedding_model_check.json"


def _json_load(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _file_nonempty(path: Path) -> bool:
    try:
        return Path(path).exists() and Path(path).stat().st_size > 0
    except Exception:
        return False


def _valid_ci(path: Path) -> bool:
    if not _file_nonempty(path):
        return False
    try:
        df = pd.read_csv(path)
        cols = set(df.columns)
        metric_ok = ("tps" in cols) or ("fps" in cols)  # accepts legacy column
        needed = {"scope", "ci_lo", "ci_hi"}
        scopes = set(df["scope"].astype(str).str.lower()) if "scope" in df.columns else set()
        return metric_ok and needed.issubset(cols) and {"global", *PRIMARY4}.issubset(scopes)
    except Exception:
        return False


def _valid_incon_csv(path: Path) -> bool:
    if not _file_nonempty(path):
        return False
    try:
        df = pd.read_csv(path)
        needed = {"judge_family", "n_pairs", "n_inconsistent", "inconsistency_rate"}
        fams = set(df["judge_family"].astype(str).str.lower()) if "judge_family" in df.columns else set()
        return needed.issubset(df.columns) and set(PRIMARY4).issubset(fams)
    except Exception:
        return False


def _valid_incon_json(path: Path) -> bool:
    if not _file_nonempty(path):
        return False
    try:
        obj = _json_load(path)
        return all(k in obj for k in ["chi2", "pval", "dof", "verdict", "paper_sentence", "per_family"])
    except Exception:
        return False


def _valid_emb_json(path: Path) -> bool:
    if not _file_nonempty(path):
        return False
    try:
        obj = _json_load(path)
        return "embedding_model" in obj
    except Exception:
        return False


def _ci_cache_path() -> Path | None:
    if _valid_ci(CI_TPS_PATH):
        return CI_TPS_PATH
    if _valid_ci(CI_LEGACY_PATH):
        return CI_LEGACY_PATH
    return None


def _write_tps_alias_if_needed(path: Path) -> None:
    """If only the legacy file exists, create a canonical TPS-named alias without rerunning."""
    if path == CI_LEGACY_PATH and not CI_TPS_PATH.exists():
        df = pd.read_csv(path)
        if "fps" in df.columns and "tps" not in df.columns:
            df = df.rename(columns={"fps": "tps"})
        df.to_csv(CI_TPS_PATH, index=False)
        print(f"  Created canonical alias: {CI_TPS_PATH}")


def _phase14_all_cache_valid() -> bool:
    return (
        _ci_cache_path() is not None
        and _valid_incon_csv(INCON_CSV_PATH)
        and _valid_incon_json(INCON_JSON_PATH)
        and _valid_emb_json(EMB_JSON_PATH)
    )


def _print_cached_summary() -> None:
    print("PHASE 14 CACHE HIT — validated outputs already present. No bootstrap rerun.")
    ci_path = _ci_cache_path()
    if ci_path is not None:
        _write_tps_alias_if_needed(ci_path)
        ci = pd.read_csv(CI_TPS_PATH if CI_TPS_PATH.exists() else ci_path)
        metric_col = "tps" if "tps" in ci.columns else "fps"
        print("\nCached per-family TPS bootstrap CIs:")
        print(ci[["scope", metric_col, "ci_lo", "ci_hi"]].to_string(index=False))
    if _valid_incon_json(INCON_JSON_PATH):
        inc = _json_load(INCON_JSON_PATH)
        print("\nCached inconsistency test:")
        print(f"  chi2={inc.get('chi2')}  p={inc.get('pval')}  dof={inc.get('dof')}")
        print(f"  verdict={inc.get('verdict')}")
    if _valid_emb_json(EMB_JSON_PATH):
        emb = _json_load(EMB_JSON_PATH)
        print("\nCached embedding-model check:")
        print(f"  embedding_model={emb.get('embedding_model')}")


# ---------------------------------------------------------------------------
# Manual chi-square helper: avoids adding scipy as a hard dependency.
# ---------------------------------------------------------------------------
def chi2_contingency_manual(observed):
    from math import erfc, sqrt
    obs = np.array(observed, dtype=float)
    row_sums = obs.sum(axis=1, keepdims=True)
    col_sums = obs.sum(axis=0, keepdims=True)
    total = obs.sum()
    expected = row_sums * col_sums / total
    chi2_stat = ((obs - expected) ** 2 / expected).sum()
    dof = (obs.shape[0] - 1) * (obs.shape[1] - 1)
    z = ((chi2_stat / dof) ** (1/3) - (1 - 2/(9*dof))) / sqrt(2/(9*dof))
    p_val = 0.5 * erfc(z / sqrt(2))
    return float(chi2_stat), float(p_val), int(dof)


def _load_and_reconcile_primary4() -> pd.DataFrame:
    print("Loading master_judgments.csv ...")
    jdg_path = DRIVE / "judgments" / "master" / "master_judgments.csv"
    jdg = pd.read_csv(jdg_path)
    print(f"  shape: {jdg.shape}")

    jdg4 = jdg[
        jdg["judge_family"].isin(PRIMARY4)
        & jdg["family_1"].isin(PRIMARY4)
        & jdg["family_2"].isin(PRIMARY4)
    ].copy()
    print(f"  Primary-4 rows: {len(jdg4)}")
    print(f"  Order values:   {sorted(jdg4['order'].unique())}")
    print(f"  Winner values:  {sorted(jdg4['winner'].unique())}")

    print("\nBuilding reconciled support scores ...")
    jdg4["pair_key"] = jdg4.apply(
        lambda r: "_".join(sorted([r["family_1"], r["family_2"]])), axis=1
    )

    order_vals = list(jdg4["order"].unique())
    ab_lab, ba_lab = None, None
    for v in order_vals:
        sv = str(v).strip().upper()
        if sv in ("AB", "A", "0", "FIRST", "1"):
            ab_lab = v
        elif sv in ("BA", "B", "2", "SECOND"):
            ba_lab = v

    if ab_lab is None or ba_lab is None:
        s = sorted(order_vals, key=str)
        ab_lab, ba_lab = s[0], s[1]
    print(f"  AB label={ab_lab!r}  BA label={ba_lab!r}")

    def support_for_family1(row):
        w = str(row["winner"]).strip().upper()
        if row["order"] == ab_lab:
            if w == "A":
                return 1.0
            if w == "B":
                return 0.0
            return 0.5
        if w == "B":
            return 1.0
        if w == "A":
            return 0.0
        return 0.5

    jdg4["support_f1"] = jdg4.apply(support_for_family1, axis=1)

    merge_key = ["judge_family", "prompt_id", "pair_key", "family_1", "family_2"]
    ab_rows = jdg4[jdg4["order"] == ab_lab][merge_key + ["support_f1"]].copy()
    ba_rows = jdg4[jdg4["order"] == ba_lab][merge_key + ["support_f1"]].copy()
    merged = ab_rows.merge(ba_rows, on=merge_key, suffixes=("_ab", "_ba"))
    print(f"  Matched AB/BA pairs: {len(merged)}")

    merged["support"] = ((merged["support_f1_ab"] + merged["support_f1_ba"]) / 2).round(2)
    merged["inconsistent"] = (
        ((merged["support_f1_ab"] == 1.0) & (merged["support_f1_ba"] == 0.0))
        | ((merged["support_f1_ab"] == 0.0) & (merged["support_f1_ba"] == 1.0))
    )

    print("\n  Support value distribution:")
    print(merged["support"].value_counts().sort_index().to_string())
    print(f"\n  Global inconsistency rate: {merged['inconsistent'].mean():.4f}")
    return merged


def _compute_tps(df: pd.DataFrame):
    rows = []
    for _, r in df.iterrows():
        rows.append({"judge": r["judge_family"], "cand": r["family_1"],
                     "support": r["support"], "prompt": r["prompt_id"]})
        rows.append({"judge": r["judge_family"], "cand": r["family_2"],
                     "support": 1.0 - r["support"], "prompt": r["prompt_id"]})
    long = pd.DataFrame(rows)
    pm = long.groupby(["judge", "cand"])["support"].mean().unstack()

    fams = [f for f in PRIMARY4 if f in pm.index and f in pm.columns]
    diag = [pm.loc[f, f] for f in fams]
    offdiag = [pm.loc[f, g] for f in fams for g in fams if f != g]
    global_tps = np.mean(diag) - np.mean(offdiag)

    per_family = {}
    for f in fams:
        d = pm.loc[f, f]
        off = [pm.loc[f, g] for g in fams if g != f]
        per_family[f] = d - np.mean(off)
    return global_tps, per_family


def _run_bootstrap_if_needed(merged: pd.DataFrame) -> None:
    cached = _ci_cache_path()
    if cached is not None and not PHASE14_FORCE_RECOMPUTE:
        print("\nSTEP 1 CACHE HIT — per-family TPS bootstrap CIs already present.")
        _write_tps_alias_if_needed(cached)
        ci = pd.read_csv(CI_TPS_PATH if CI_TPS_PATH.exists() else cached)
        metric_col = "tps" if "tps" in ci.columns else "fps"
        print(ci[["scope", metric_col, "ci_lo", "ci_hi"]].to_string(index=False))
        return

    print("\n" + "=" * 60)
    print(f"STEP 1: Per-family TPS bootstrap CIs (n_boot={N_BOOT})")
    print("=" * 60)

    obs_global, obs_family = _compute_tps(merged)
    print(f"\n  Observed global TPS: {obs_global:.5f}")
    print("  Observed per-family TPS:")
    for f, v in obs_family.items():
        print(f"    {f:8s}: {v:.5f}")

    prompts = merged["prompt_id"].unique()
    n_prompts = len(prompts)
    print(f"\n  Bootstrapping {n_prompts} prompt clusters × {N_BOOT} draws ...")

    rng = np.random.default_rng(SEED)
    by_prompt = {p: g for p, g in merged.groupby("prompt_id")}
    boot_global = []
    boot_family = {f: [] for f in PRIMARY4}

    for _ in range(N_BOOT):
        samp = rng.choice(prompts, size=n_prompts, replace=True)
        boot_df = pd.concat([by_prompt[p] for p in samp], ignore_index=True)
        try:
            g, pf = _compute_tps(boot_df)
            boot_global.append(g)
            for f in PRIMARY4:
                if f in pf:
                    boot_family[f].append(pf[f])
        except Exception:
            pass

    boot_global = np.array(boot_global)
    print(f"  Valid draws: {len(boot_global)}")

    results = []
    lo_g, hi_g = np.percentile(boot_global, [2.5, 97.5])
    results.append({"scope": "Global", "tps": round(obs_global, 5),
                    "ci_lo": round(lo_g, 5), "ci_hi": round(hi_g, 5)})
    print(f"  Global   TPS={obs_global:.5f}  95%CI=[{lo_g:.5f}, {hi_g:.5f}]")

    for f in PRIMARY4:
        arr = np.array(boot_family[f])
        lo, hi = np.percentile(arr, [2.5, 97.5])
        results.append({"scope": f, "tps": round(obs_family.get(f, np.nan), 5),
                        "ci_lo": round(lo, 5), "ci_hi": round(hi, 5)})
        print(f"  {f:8s} TPS={obs_family.get(f, np.nan):.5f}  95%CI=[{lo:.5f}, {hi:.5f}]")

    df_ci = pd.DataFrame(results)
    df_ci.to_csv(CI_TPS_PATH, index=False)
    print(f"\n✓ Saved: {CI_TPS_PATH}")


def _run_inconsistency_if_needed(merged: pd.DataFrame) -> None:
    if _valid_incon_csv(INCON_CSV_PATH) and _valid_incon_json(INCON_JSON_PATH) and not PHASE14_FORCE_RECOMPUTE:
        print("\nSTEP 2 CACHE HIT — per-family inconsistency rates already present.")
        inc = _json_load(INCON_JSON_PATH)
        print(f"  chi2={inc.get('chi2')}  p={inc.get('pval')}  dof={inc.get('dof')}")
        print(f"  verdict={inc.get('verdict')}")
        return

    print("\n" + "=" * 60)
    print("STEP 2: Per-family AB/BA inconsistency rates")
    print("=" * 60)

    incon = merged.groupby("judge_family").agg(
        n_pairs=("inconsistent", "count"),
        n_inconsistent=("inconsistent", "sum"),
    ).reset_index()
    incon["n_consistent"] = incon["n_pairs"] - incon["n_inconsistent"]
    incon["inconsistency_rate"] = (incon["n_inconsistent"] / incon["n_pairs"]).round(4)

    print("\n" + incon[["judge_family", "n_pairs", "n_inconsistent", "inconsistency_rate"]].to_string(index=False))

    ct = incon[["n_consistent", "n_inconsistent"]].values
    chi2_val, pval, dof = chi2_contingency_manual(ct)
    print(f"\n  chi2({dof}) = {chi2_val:.3f}, p = {pval:.4f}")

    if pval > 0.05:
        verdict = "UNIFORM — reviewer concern dismissed"
        paper_sent = (
            f"Per-judge-family AB/BA inconsistency rates are statistically uniform "
            f"(chi2({dof})={chi2_val:.2f}, p={pval:.3f}), ruling out "
            f"differential 0.5 imputation as a driver of per-family TPS differences."
        )
    else:
        verdict = "HETEROGENEOUS — report rates in paper"
        paper_sent = (
            f"Per-judge-family AB/BA inconsistency rates differ across families "
            f"(chi2({dof})={chi2_val:.2f}, p={pval:.3f}); per-family rates are reported in the appendix."
        )

    print(f"\n  Verdict: {verdict}")
    print(f"\n  Paper sentence:\n  {paper_sent}")

    incon.to_csv(INCON_CSV_PATH, index=False)
    with open(INCON_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "chi2": round(chi2_val, 4),
            "pval": round(pval, 6),
            "dof": dof,
            "verdict": verdict,
            "paper_sentence": paper_sent,
            "per_family": incon.to_dict(orient="records"),
        }, f, indent=2)
    print(f"\n✓ Saved: {INCON_CSV_PATH}")
    print(f"✓ Saved: {INCON_JSON_PATH}")


def _run_embedding_check_if_needed() -> None:
    if _valid_emb_json(EMB_JSON_PATH) and not PHASE14_FORCE_RECOMPUTE:
        print("\nSTEP 3 CACHE HIT — embedding-model check already present.")
        emb = _json_load(EMB_JSON_PATH)
        print(f"  embedding_model={emb.get('embedding_model')}")
        return

    print("\n" + "=" * 60)
    print("STEP 3: Identify embedding model for style-similarity analysis")
    print("=" * 60)

    emb_name = "not found"
    cfg_path = DRIVE / "config" / "config_latest.json"
    try:
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        for key in ["embedding_model", "style_sim_model", "embedding", "embedder", "encoder", "sentence_model"]:
            if key in cfg:
                emb_name = cfg[key]
                print(f"  Found in config['{key}']: {emb_name}")
                break
        if emb_name == "not found":
            print(f"  Not in config. Top-level keys: {list(cfg.keys())[:15]}")
    except Exception as e:
        print(f"  Config lookup skipped/error: {e}")

    patterns = "SentenceTransformer|all-mpnet|all-MiniLM|embed_model|embedding_model|paraphrase-multilingual"
    grep_hits = []
    try:
        r1 = subprocess.run(
            ["grep", "-r", "--include=*.ipynb", "--include=*.py", "-l", patterns, str(DRIVE)],
            capture_output=True, text=True, timeout=15,
        )
        files = [line for line in r1.stdout.strip().split("\n") if line][:3]
        for fp in files:
            print(f"\n  References in: {Path(fp).name}")
            r2 = subprocess.run(["grep", "-n", patterns, fp], capture_output=True, text=True, timeout=10)
            lines = []
            for line in r2.stdout.split("\n")[:6]:
                if line.strip():
                    print(f"    {line}")
                    lines.append(line)
            grep_hits.append({"file": fp, "lines": lines})
    except Exception as e:
        print(f"  Grep skipped/error: {e}")

    with open(EMB_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump({
            "embedding_model": emb_name,
            "grep_hits": grep_hits,
            "note": "Add the embedding model name to the method section if it is not already reported. Confirm it is not from the audited model families.",
        }, f, indent=2)
    print(f"\n✓ Saved: {EMB_JSON_PATH}")


if _phase14_all_cache_valid() and not PHASE14_FORCE_RECOMPUTE:
    _print_cached_summary()
else:
    print("PHASE 14 CACHE MISS/PARTIAL CACHE — only missing/invalid outputs will be recomputed.")
    merged_primary4 = _load_and_reconcile_primary4()
    _run_bootstrap_if_needed(merged_primary4)
    _run_inconsistency_if_needed(merged_primary4)
    _run_embedding_check_if_needed()
    print("\nPHASE 14 COMPLETE — cache-aware outputs are ready.")


## POINT 5 / PHASE 15 — Cache-aware off-diagonal BT robustness check

This release cell validates `bt_offdiag_result.json` before recomputing the off-diagonal Bradley–Terry robustness check. Set `PHASE15_FORCE_RECOMPUTE = True` only when you intentionally want to regenerate the result.


In [ ]:
# ============================================================
# POINT 5 / PHASE 15 — Cache-aware off-diagonal BT robustness check
# ============================================================
# Purpose:
#   Recomputes BT quality scores using only off-diagonal judgments only when
#   the result JSON is missing or invalid. Otherwise, loads the cached result.
#
# Set PHASE15_FORCE_RECOMPUTE = True only when you intentionally want to rerun.
# ============================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    DRIVE = Path(PATHS["root"])
except Exception:
    DRIVE = Path("/content/drive/MyDrive/tribal_pref_v11")

OUT = DRIVE / "analysis" / "phase15_bt_robustness"
OUT.mkdir(parents=True, exist_ok=True)

PRIMARY4 = ["llama", "qwen", "gemma", "yi"]
PHASE15_FORCE_RECOMPUTE = False
BT_RESULT_PATH = OUT / "bt_offdiag_result.json"


def _load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _valid_bt_result(path: Path) -> bool:
    try:
        obj = _load_json(path)
        if not all(k in obj for k in ["results", "verdict", "paper_sentence"]):
            return False
        results = obj.get("results", {})
        return "standard BT (all)" in results and "off-diagonal BT" in results
    except Exception:
        return False


if _valid_bt_result(BT_RESULT_PATH) and not PHASE15_FORCE_RECOMPUTE:
    print("POINT 5 CACHE HIT — off-diagonal BT result already present. No rerun.")
    obj = _load_json(BT_RESULT_PATH)
    print(f"  Verdict: {obj.get('verdict')}")
    for label, vals in obj.get("results", {}).items():
        print(f"  {label}: M1={vals.get('M1')}  M2={vals.get('M2')}  shift_pct={vals.get('shift_pct')}")
    print(f"  Cached file: {BT_RESULT_PATH}")

else:
    print("POINT 5 CACHE MISS — recomputing off-diagonal BT robustness check.")

    print("Loading master_judgments.csv ...")
    jdg = pd.read_csv(DRIVE / "judgments" / "master" / "master_judgments.csv")
    jdg4 = jdg[
        jdg["judge_family"].isin(PRIMARY4)
        & jdg["family_1"].isin(PRIMARY4)
        & jdg["family_2"].isin(PRIMARY4)
    ].copy()
    print(f"  Total rows: {len(jdg4)}")

    order_vals = list(jdg4["order"].unique())
    ab_candidates = [v for v in order_vals if str(v).upper() in ("AB", "A", "0", "1", "FIRST")]
    ba_candidates = [v for v in order_vals if str(v).upper() in ("BA", "B", "2", "SECOND")]
    if not ab_candidates or not ba_candidates:
        ordered = sorted(order_vals, key=str)
        ab_lab, ba_lab = ordered[0], ordered[1]
    else:
        ab_lab, ba_lab = ab_candidates[0], ba_candidates[0]
    print(f"  AB label={ab_lab!r}  BA label={ba_lab!r}")

    def support_for_family1(row):
        w = str(row["winner"]).strip().upper()
        if row["order"] == ab_lab:
            return 1.0 if w == "A" else (0.0 if w == "B" else 0.5)
        return 1.0 if w == "B" else (0.0 if w == "A" else 0.5)

    jdg4["support_f1"] = jdg4.apply(support_for_family1, axis=1)

    # Use pair_id when available; otherwise fall back to sorted family pair key.
    if "pair_id" not in jdg4.columns:
        jdg4["pair_id"] = jdg4.apply(lambda r: "_".join(sorted([r["family_1"], r["family_2"]])), axis=1)

    merge_key = ["judge_family", "prompt_id", "pair_id", "family_1", "family_2"]
    ab_rows = jdg4[jdg4["order"] == ab_lab][merge_key + ["support_f1"]].copy()
    ba_rows = jdg4[jdg4["order"] == ba_lab][merge_key + ["support_f1"]].copy()
    merged = ab_rows.merge(ba_rows, on=merge_key, suffixes=("_ab", "_ba"))
    merged["support"] = ((merged["support_f1_ab"] + merged["support_f1_ba"]) / 2).round(2)
    print(f"  Matched AB/BA pairs: {len(merged)}")

    rows = []
    for _, r in merged.iterrows():
        rows.append({"judge": r["judge_family"], "cand": r["family_1"],
                     "support": r["support"], "prompt": r["prompt_id"]})
        rows.append({"judge": r["judge_family"], "cand": r["family_2"],
                     "support": 1.0 - r["support"], "prompt": r["prompt_id"]})
    long = pd.DataFrame(rows)
    long["same_family"] = (long["judge"] == long["cand"]).astype(int)
    print(f"Long-format rows: {len(long)}")

    def compute_bt_scores(df, judge_col="judge", cand_col="cand", supp_col="support"):
        fams = PRIMARY4
        n = len(fams)
        idx = {f: i for i, f in enumerate(fams)}
        W = np.zeros((n, n))
        for _, r in df.iterrows():
            j, c = r[judge_col], r[cand_col]
            if j not in idx or c not in idx:
                continue
            s = r[supp_col]
            W[idx[c], idx[j]] += s
            W[idx[j], idx[c]] += (1 - s)

        pi = np.ones(n)
        for _ in range(200):
            pi_new = np.zeros(n)
            for i in range(n):
                denom = sum(
                    (W[i, j] + W[j, i]) / (pi[i] + pi[j])
                    for j in range(n)
                    if i != j and (W[i, j] + W[j, i]) > 0
                )
                numer = sum(W[i, j] for j in range(n) if i != j)
                pi_new[i] = numer / denom if denom > 0 else pi[i]
            pi_new /= pi_new.sum()
            if np.max(np.abs(pi_new - pi)) < 1e-8:
                break
            pi = pi_new
        return {fams[i]: float(pi[i]) for i in range(n)}

    print("\nComputing BT scores from all judgments ...")
    bt_all = compute_bt_scores(long)
    print("  BT scores (all):", {k: round(v, 4) for k, v in bt_all.items()})

    print("\nComputing BT scores from off-diagonal judgments only ...")
    long_offdiag = long[long["same_family"] == 0].copy()
    print(f"  Off-diagonal rows: {len(long_offdiag)} (removed {len(long) - len(long_offdiag)} same-family rows)")
    bt_offdiag = compute_bt_scores(long_offdiag)
    print("  BT scores (off-diagonal):", {k: round(v, 4) for k, v in bt_offdiag.items()})

    try:
        from statsmodels.genmod.generalized_estimating_equations import GEE
        from statsmodels.genmod.families import Binomial

        long["bt_adv_all"] = long.apply(lambda r: bt_all.get(r["cand"], 0) - bt_all.get(r["judge"], 0), axis=1)
        long["bt_adv_offdiag"] = long.apply(lambda r: bt_offdiag.get(r["cand"], 0) - bt_offdiag.get(r["judge"], 0), axis=1)

        for col in ["bt_adv_all", "bt_adv_offdiag"]:
            mu, sd = long[col].mean(), long[col].std()
            long[col + "_z"] = (long[col] - mu) / sd if sd > 0 else 0.0

        results = {}
        for bt_label, bt_col in [("standard BT (all)", "bt_adv_all_z"), ("off-diagonal BT", "bt_adv_offdiag_z")]:
            print(f"\n  Fitting GEE with {bt_label} ...")
            gee_m1 = GEE.from_formula(
                "support ~ same_family", groups="prompt", data=long,
                family=Binomial(), cov_struct=None,
            ).fit(cov_type="robust")
            gee_m2 = GEE.from_formula(
                f"support ~ same_family + {bt_col}", groups="prompt", data=long,
                family=Binomial(), cov_struct=None,
            ).fit(cov_type="robust")

            b1 = float(gee_m1.params["same_family"])
            b2 = float(gee_m2.params["same_family"])
            shift = b2 - b1
            shift_pct = shift / b1 * 100 if b1 != 0 else np.nan
            print(f"  M1 beta = {b1:.4f}")
            print(f"  M2 beta = {b2:.4f}  (shift = {shift:+.4f}, {shift_pct:+.1f}%)")
            results[bt_label] = {
                "M1": round(b1, 4),
                "M2": round(b2, 4),
                "shift": round(shift, 4),
                "shift_pct": round(shift_pct, 1),
            }

        r_std = results["standard BT (all)"]
        r_off = results["off-diagonal BT"]
        stable = abs(r_off["M2"] - r_std["M2"]) < 0.01

        if stable:
            verdict = "ENDOGENEITY CONCERN EMPIRICALLY RESOLVED"
            paper_sent = (
                f"As a further check on potential endogeneity of the BT quality covariate, "
                f"we re-estimated BT scores using only off-diagonal cross-family judgments. "
                f"The M2 same-family coefficient is {r_off['M2']:.3f} "
                f"versus {r_std['M2']:.3f} with standard BT, a difference below 0.01."
            )
        else:
            verdict = f"COEFFICIENTS DIFFER BY {abs(r_off['M2'] - r_std['M2']):.3f} — REPORT HONESTLY"
            paper_sent = f"Standard M2 beta={r_std['M2']:.3f}, off-diagonal M2 beta={r_off['M2']:.3f}."

        print("\n" + "=" * 60)
        print("POINT 5 RESULT SUMMARY")
        print("=" * 60)
        print(f"  Standard BT:     M1={r_std['M1']:.4f}  M2={r_std['M2']:.4f}  shift={r_std['shift_pct']:+.1f}%")
        print(f"  Off-diagonal BT: M1={r_off['M1']:.4f}  M2={r_off['M2']:.4f}  shift={r_off['shift_pct']:+.1f}%")
        print(f"\n  Verdict: {verdict}")
        print(f"\n  Paper sentence:\n  {paper_sent}")

        with open(BT_RESULT_PATH, "w", encoding="utf-8") as f:
            json.dump({"results": results, "verdict": verdict, "paper_sentence": paper_sent}, f, indent=2)
        print(f"\n✓ Saved: {BT_RESULT_PATH}")

    except ImportError:
        raise RuntimeError("statsmodels is required for this check. Install it before running this cell.")


## Primary-4 update summary

Run order after this update:

1. Run cells through Phase 5 only if responses or judgments have changed.
2. From **Cell 6.1 onward**, rerun only when upstream responses/judgments or configuration have changed; otherwise use validated cached artifacts.
3. Report the Primary-4 outputs in the main paper.
4. Put the full 5-family Falcon-retained outputs in an appendix/sensitivity table.

Main outputs to cite in the paper:

- `analysis/preference/tps_summary_primary.json`
- `analysis/preference/preference_matrix_primary.csv`
- `analysis/stats/cluster_bootstrap_tps_primary.json`
- `analysis/stats/permutation_result_primary.json`
- `analysis/bradley_terry/bt_results_primary.json`
- `analysis/mechanism_decomp/nested_logit_decomposition_primary.csv`
- `analysis/robustness/multiverse_grid_with_candidate_subset.csv`
- `analysis/diagnostics/falcon_excluded_report.json`


## Pipeline complete

**What's in `release/`:**
- `MODEL_CARD.md` and `DATA_STATEMENT.md` (Mitchell et al. + Bender & Friedman style)
- `hf_dataset/` (parquet + README dataset card — ready for `huggingface-cli upload`)
- `REPRODUCE.md` (one-document reproduction guide)
- `final_audit.json` (integrity check + headline numbers)

**What's in `figures/`:**
- `fig1_preference_heatmap.{pdf,png}` — observed F×F preference matrix
- `fig2_residual_heatmap.{pdf,png}` — BT-residual matrix
- `fig3_per_family_forest.{pdf,png}` — per-family TPS with bootstrap CIs
- `fig4_decomposition_decay.{pdf,png}` — same_family β across M1→M5
- `fig5_specification_curve.{pdf,png}` — multiverse TPS sorted

**What's in `tables/`:**
- `table1_main_tps.tex`, `table2_decomposition.tex`, `table3_robustness.tex`

**Submission checklist:**
- [ ] `final_audit.json["summary"]["complete"] == True`
- [ ] `final_audit.json["headline"]["multiverse_pct_positive"] >= 0.85`
- [ ] Confirmatory TPS CI excludes zero
- [ ] M5 same_family coefficient and its decay pattern make a clear story
- [ ] Human calibration reports two retained annotators, 800 final judgments, κ/α values, and no-consensus rows honestly; do not claim strong human validation
- [ ] Pre-registration link in OSF, referenced from REPRODUCE.md